# D1: диагностика train-reference

CE: обе GPU C-long2x, непересекающиеся строки. Greedy: GPU0 C-main / GPU1 C-long2x L3 l2-62280. Только inference; полный CE long → stratified top≤4% → v6 greedy + CPU execution. Smoke:131072 CE-строк, все фазы,≤12мин. Production: hard130мин/backend140мин; Secrets/live optional. Согласие моделей — подозрение, не новая разметка. GPU-smoke/push выполняет ревьюер.

In [ ]:
import time,json,os,sys,tempfile
from pathlib import Path
SESSION_START=time.monotonic()
CONFIG={'owner': 'thisarivson', 'session_kind': 'd1', 'gpu_smoke': True, 'working_root': '/kaggle/working', 'input_root': '/kaggle/input', 'live_dataset': 'friend1/nlp-hw02-d1-live', 'weights_source': 'thisarivson/nlp-hw02-l3-live', 'require_secrets': False, 'enable_live_sync': False, 'cache_source': 'thisarivson/nlp-hw02-compact-cache-verified/2', 'cache_fallback_source': 'friend1/nlp-hw02-compact-cache-verified', 'seed_sha256': '6fd712d4b805e8cd49b709e01260aef15ae28875dfc31e659f0a4d81c9c52d36', 'setup_cap_seconds': 240, 'loss_stop_seconds': 420, 'generation_stop_seconds': 580, 'execution_stop_seconds': 610, 'worker_deadline_seconds': 630, 'hard_shutdown_seconds': 660, 'budget_seconds': 630, 'session_ceiling_seconds': 720, 'terminate_grace_seconds': 20, 'loss_batch': 512, 'loss_token_slots': 32768, 'loss_block': 8192, 'fraction': 0.04, 'pilot_per_bin': 256, 'timing_safety': 1.6, 'pilot_tail_reserve': 20.0, 'wave_rows': 4096, 'generation_batch': 512, 'cap': 512, 'wave_timeout_seconds': 120, 'smoke_rows': 131072, 'port_sha256': 'f5838218837c04d54e89dfead59065b43176dbdec4fe7398fe106ab0bc52a2e2'}
WORKING=Path(CONFIG['working_root']);WORKING.mkdir(parents=True,exist_ok=True)
CONFIG['session_start_monotonic']=SESSION_START
CONFIG['deadline_monotonic']=SESSION_START+CONFIG['worker_deadline_seconds']
CONFIG['hard_deadline_monotonic']=SESSION_START+CONFIG['hard_shutdown_seconds']
CONFIG['temp_root']=tempfile.mkdtemp(prefix='nlp-d1-',dir='/kaggle/temp' if Path('/kaggle/temp').is_dir() else None)
MODULE_DIR=WORKING/'src';MODULE_DIR.mkdir(exist_ok=True)
SOURCES={'d1_data.py': '"""D1 CPU contracts: immutable artifacts, train identities, ranks and L2 orders."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport numpy as np\nfrom kaggle_io import sha, write\n\nCACHE_SHA = \'46ef881bba60dba2563a25a7490becf05104a0537b70c3510629bef8c93902be\'\nROWS = 9725737\nMODELS = (\'C-main\', \'C-long2x\')\nEDGES = (53, 79, 105)  # Compact CONTENT, mode included, EOS excluded.\nSEED = 2026101404\nWEIGHTS = {\n    \'C-main\': \'8c11b1038855422b6912e243049a554317978f848edb412ed48aebbe4cb6b886\',\n    \'C-long2x\': \'d10bf0018d0b14fd6cc3c8b9c7121f127b5cec97057544fd63e5824253a55057\'}\n\n\ndef record(path):\n    p = Path(path)\n    return dict(file=p.name, bytes=p.stat().st_size, sha256=sha(p))\n\n\ndef save_npz(path, **arrays):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name+\'.tmp\')\n    with tmp.open(\'wb\') as stream:\n        np.savez_compressed(stream, **arrays)\n    tmp.replace(path)\n\n\ndef publish(path, files, **metadata):\n    """Receipt is the commit point; its blobs must never be modified afterwards."""\n    path = Path(path)\n    write(path, dict(format=\'nlp-hw02-d1-artifact-v1\', files=[record(p) for p in files], **metadata))\n\n\ndef verify_receipt(path):\n    path = Path(path); meta = json.loads(path.read_text())\n    assert meta[\'format\'] == \'nlp-hw02-d1-artifact-v1\'\n    for f in meta[\'files\']:\n        assert Path(f[\'file\']).name == f[\'file\']\n        p = path.parent/f[\'file\']\n        assert p.stat().st_size == f[\'bytes\'] and sha(p) == f[\'sha256\'], str(p)\n    return meta\n\n\ndef length_bins(lengths):\n    return np.searchsorted(EDGES, lengths, side=\'right\')\n\n\ndef ranked_candidates(main, long, lengths, fraction=.08):\n    """Top q separately per length bin. Float32 mean of persisted FP16 scores.\n\n    Deterministic row-id ties. Rounded bin quotas may differ from q*N by <=2.\n    """\n    a, b = np.asarray(main), np.asarray(long)\n    assert a.shape == b.shape == np.asarray(lengths).shape\n    assert np.isfinite(a).all() and np.isfinite(b).all() and 0 < fraction <= .08\n    score = (a.astype(np.float32)+b.astype(np.float32))*.5\n    bins = length_bins(lengths); ranked = []\n    for bin_id in range(4):\n        ids = np.flatnonzero(bins == bin_id)\n        k = min(len(ids), int(math.floor(len(ids)*fraction+.5)))\n        # Partition narrows the expensive sort; explicitly resolve the boundary tie.\n        if k:\n            threshold = np.partition(score[ids], len(ids)-k)[len(ids)-k]\n            greater = ids[score[ids] > threshold]\n            tied = ids[score[ids] == threshold][:k-len(greater)]\n            ids = np.r_[greater, tied]\n            ids = ids[np.lexsort((ids, -score[ids]))]\n        else:\n            ids = ids[:0]\n        ranked.append(ids.astype(np.uint32))\n    return ranked, score\n\n\ndef choose_budget(ranked, pilots, remaining, safety=1.6, tail=240.):\n    """One proportional reduction across bins; both GPUs get identical row IDs.\n\n    Pilot wall includes generation AND overlapped execution. Use the slower model\n    in EACH bin, so a fast short bin cannot hide the long tail. No upward resizing.\n    """\n    rates = []\n    for b in range(4):\n        if not len(ranked[b]):\n            rates.append(0.); continue\n        rates.append(max(p[\'bins\'][b][\'seconds\']/max(1, p[\'bins\'][b][\'rows\']) for p in pilots))\n    cost = safety*sum(len(ids)*rate for ids, rate in zip(ranked, rates))\n    scale = min(1., max(0., remaining-tail)/max(cost, 1e-9))\n    quotas = [int(len(ids)*scale) for ids in ranked]\n    pilot_counts = [max(p[\'bins\'][b][\'rows\'] for p in pilots) for b in range(4)]\n    if any(q < count for q, count in zip(quotas, pilot_counts)):\n        raise TimeoutError(\'Insufficient budget even for pilot-sized common subset\')\n    selected = np.concatenate([ids[:q] for ids, q in zip(ranked, quotas)])\n    return selected, dict(maximum_rows=sum(map(len, ranked)), selected_rows=len(selected),\n                         quotas=quotas, seconds_per_row_by_bin=rates, safety=safety,\n                         estimated_seconds=safety*sum(q*r for q, r in zip(quotas, rates)),\n                         available_seconds=remaining, execution_tail_reserve=tail)\n\n\ndef close(a, b):\n    # Symmetric math.isclose, unlike asymmetric np.isclose.\n    return np.abs(a-b) <= np.maximum(1e-6, 1e-6*np.maximum(np.abs(a), np.abs(b)))\n\n\ndef cluster(values):\n    reps, labels, counts = [], [], []\n    for x in values:\n        for j, rep in enumerate(reps):\n            if math.isclose(float(x), rep, rel_tol=1e-6, abs_tol=1e-6):\n                labels.append(j); counts[j] += 1; break\n        else:\n            labels.append(len(reps)); reps.append(float(x)); counts.append(1)\n    return np.asarray(labels), np.asarray(counts), np.asarray(reps)\n\n\ndef filtered_order(keep, sl, tl, destination, cache_sha, rule, evidence_sha):\n    """Same shuffle/bucketing as clean-cooldown; unmodified L2 loader accepts it."""\n    from l2_sampler import load_order, fixed_split\n    from exp26_common import json_hash\n    root = Path(destination); root.mkdir(parents=True, exist_ok=True)\n    order = np.flatnonzero(keep).astype(np.uint32)\n    assert len(order), \'Refuse an empty train order\'\n    rng = np.random.default_rng(SEED); rng.shuffle(order)\n    for lo in range(0, len(order), 4096):\n        part = order[lo:lo+4096]\n        ns, nt = np.asarray(sl[part], dtype=np.int32), np.asarray(tl[part], dtype=np.int32)\n        part = part[np.lexsort((nt, ns, ns+nt+1))]; full = len(part)//128\n        if full:\n            part[:full*128] = part[:full*128].reshape(full, 128)[rng.permutation(full)].reshape(-1)\n        order[lo:lo+len(part)] = part\n    assert np.array_equal(np.sort(order), np.flatnonzero(keep))\n    np.save(root/\'order.npy\', order, allow_pickle=False)\n    meta = dict(seed=SEED, rows=len(keep), slots=len(order), batch=128, chunk=4096,\n                selected_bijection=True, cache_manifest_sha256=cache_sha, file=record(root/\'order.npy\'),\n                rule=rule, evidence_sha256=evidence_sha,\n                selected_cache_indices_sha256=hashlib.sha256(np.flatnonzero(keep).astype(\'<u4\').tobytes()).hexdigest(),\n                answer_policy=\'retain original pairs only; model programs never become targets\')\n    write(root/\'order.json\', meta)\n    write(root/\'split.json\', {k: dict(ids=v, sha256=json_hash(v)) for k, v in fixed_split().items()})\n    class Identity:\n        def __len__(self): return len(keep)\n    loaded, receipt = load_order(root, Identity(), meta[\'file\'][\'sha256\'])\n    assert np.array_equal(loaded, order) and receipt == meta\n    return meta\n', 'd1_generate.py': '"""Bounded waves of v6 greedy + overlapped CPU execution; no generated targets."""\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nimport final_core as C\nimport final_fast as Fast\nimport final_speed as Speed\nfrom final_representation import restore_compact\nfrom final_executor_v6 import Executor\nfrom d1_data import save_npz, publish, verify_receipt\n\nOPTIONS = Speed.Options(share_cross=True, async_eos=True, eos_every=32,\n                        compact_done=True, race_sampler=True, lean_greedy=True)\n\n\ndef decode_code(tokens, table):\n    active = C.active_tokens(tokens)\n    represented = C.decode_bytes(table, active).decode(\'utf8\', errors=\'replace\')\n    try:\n        return restore_compact(represented), (\'eos\' if C.EOS in active else \'token_cap\'), len(active)\n    except (ValueError, IndexError):\n        return \'\', \'restore_error\', len(active)\n\n\ndef reference_for(cache, row, table, evidence):\n    pos = int(np.searchsorted(evidence[\'row_id\'], row))\n    if pos < len(evidence[\'row_id\']) and int(evidence[\'row_id\'][pos]) == row:\n        return None, {\'status\': \'numeric\', \'value\': float(evidence[\'reference\'][pos])}, \'receipt\'\n    text = C.decode_bytes(table, cache[row][1]).decode(\'utf8\')\n    try:\n        return restore_compact(text), None, \'executed\'\n    except (ValueError, IndexError):\n        return None, {\'status\': \'reference_restore_error\'}, \'restore_error\'\n\n\n@torch.inference_mode()\ndef generation_wave(model, cache, ids, table, device, root, check, execution_deadline,\n                    evidence=None, batch=256, cap=512, executor_factory=Executor, generation_deadline=None):\n    """At most one wave of codes/results resident. Submit after every GPU batch.\n\n    Two CPU workers per GPU = four total, including main\'s missing references.\n    No per-program timeout can grow the queue beyond the current wave.\n    """\n    root = Path(root); root.parent.mkdir(parents=True, exist_ok=True)\n    receipt = root.with_suffix(\'.receipt.json\')\n    ids = np.asarray(ids, dtype=np.uint32)\n    if receipt.exists():\n        meta = verify_receipt(receipt)\n        with np.load(root, allow_pickle=False) as old: assert np.array_equal(old[\'row_id\'], ids)\n        return meta\n    started = time.monotonic(); model.eval()\n    order = sorted(map(int, ids), key=lambda i: (int(cache.source_lengths[i]), i))\n    candidates = {}; references = {}; sources = {}; gen_status = {}; gen_length = {}; offset = 0; oom = 0\n    executor = executor_factory(2); interrupted = None; gpu_seconds = 0.\n    try:\n        while offset < len(order):\n            try: check()\n            except TimeoutError: interrupted = \'phase deadline\'; break\n            take = order[offset:offset+batch]\n            src = ctx = sequences = means = None\n            try:\n                src = nn.utils.rnn.pad_sequence([torch.as_tensor(np.asarray(cache[i][0]).copy(), dtype=torch.long)\n                    for i in take], batch_first=True, padding_value=C.PAD).to(device)\n                ctx = Fast.Context(model, src, cap)\n                gpu_started = time.monotonic()\n                sequences, means, _ = Speed.decode(ctx, deadline=generation_deadline or execution_deadline,\n                    compute_logprob=False, options=OPTIONS)\n                if device.type == \'cuda\': torch.cuda.synchronize(device)\n                gpu_seconds += time.monotonic()-gpu_started\n                jobs = []\n                for i, tokens in zip(take, sequences[:, 0].tolist()):\n                    code, status, length = decode_code(tokens, table)\n                    gen_status[i], gen_length[i] = status, length\n                    candidates[i] = {\'code\': code}\n                    if status != \'restore_error\': jobs.append(candidates[i])\n                    else: candidates[i][\'execution\'] = {\'status\': \'restore_error\'}\n                    if evidence is not None:\n                        refcode, result, origin = reference_for(cache, i, table, evidence)\n                        sources[i] = origin; references[i] = {\'code\': refcode}\n                        if result is None: jobs.append(references[i])\n                        else: references[i][\'execution\'] = result\n                executor.submit(jobs)\n                offset += len(take)\n            except TimeoutError:\n                interrupted = \'phase deadline\'; break\n            except torch.cuda.OutOfMemoryError:\n                oom += 1; batch //= 2\n                if batch < 1: raise\n            except Exception as error:\n                interrupted = type(error).__name__; break\n            finally:\n                del src, ctx, sequences, means\n            if oom and device.type == \'cuda\':\n                import gc\n                gc.collect(); torch.cuda.empty_cache()\n        # Publish every completed GPU batch even if a later batch hit its guard.\n        ids = np.asarray([i for i in ids if int(i) in gen_status], dtype=np.uint32)\n        executor.drain(execution_deadline)\n        for c in [*candidates.values(), *references.values()]:\n            if \'execution\' not in c:\n                try: executor.attach([c])\n                except KeyError: c[\'execution\'] = {\'status\': \'budget_not_evaluated\'}\n        def values(items):\n            out = np.asarray([items[int(i)][\'execution\'].get(\'value\', np.nan) for i in ids], dtype=np.float64)\n            status = np.asarray([items[int(i)][\'execution\'][\'status\'] for i in ids], dtype=\'U48\')\n            # A service crash/nonfinite answer cannot masquerade as a numeric disagreement.\n            status[(status == \'numeric\') & ~np.isfinite(out)] = \'nonfinite\'\n            return out, status\n        answer, status = values(candidates)\n        arrays = dict(row_id=ids, answer=answer, status=status,\n                      generation_status=np.asarray([gen_status[int(i)] for i in ids], dtype=\'U16\'),\n                      generation_length=np.asarray([gen_length[int(i)] for i in ids], dtype=np.uint16))\n        if evidence is not None:\n            arrays[\'reference\'], arrays[\'reference_status\'] = values(references)\n            arrays[\'reference_source\'] = np.asarray([sources[int(i)] for i in ids], dtype=\'U16\')\n        execution = executor.summary()\n        save_npz(root, **arrays)\n        meta = dict(phase=\'generation\', rows=len(ids), seconds=time.monotonic()-started,\n                    execution=execution, OOM_retries=oom, cap=cap, effective_batch=batch,\n                    complete=interrupted is None and len(ids) == len(order) and\n                        bool(np.all(status != \'budget_not_evaluated\')) and\n                        (evidence is None or bool(np.all(arrays[\'reference_status\'] != \'budget_not_evaluated\'))),\n                    interruption=interrupted, target_rows=len(order), GPU_seconds=gpu_seconds,\n                    GPU_rows_per_second=len(ids)/max(gpu_seconds, 1e-9),\n                    pipeline_rows_per_second=len(ids)/max(time.monotonic()-started, 1e-9),\n                    reference_reused=sum(s == \'receipt\' for s in sources.values()),\n                    reference_executed=sum(s == \'executed\' for s in sources.values()))\n        publish(receipt, [root], **meta)\n        return meta\n    finally:\n        executor.close()\n', 'd1_live.py': '"""D1 uses L3/P1 bounded private live-sync; stage immutable data receipts only."""\nimport argparse\nimport json\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\nfrom kaggle_io import sha, write, safe_name\nfrom l2_live import L2LiveSync\nfrom s2_live_sync import audit_tree, privacy_probe\nfrom d1_data import verify_receipt\n\nLIMIT = 1500*1024**2\n\n\ndef stage_snapshot(working, stage, ref, reason):\n    working, stage = Path(working), Path(stage)\n    assert ref == \'friend1/nlp-hw02-d1-live\'\n    stage.mkdir(parents=True, exist_ok=True)\n    audit_tree(working, ref)\n    for p in stage.iterdir():\n        if p.is_dir(): shutil.rmtree(p)\n        else: p.unlink()\n    selected = set(working.glob(\'*.json\'))\n    selected.discard(working/\'session-config.json\')\n    receipts = sorted((working/\'d1\').rglob(\'*.receipt.json\'))\n    complete_folders = {p.parent for p in receipts if p.name == \'complete.receipt.json\'}\n    selected.update(p for p in (working/\'d1\').rglob(\'*.json\') if not p.name.endswith(\'.receipt.json\'))\n    for receipt in receipts:\n        # A complete consolidation replaces loss blocks in CURRENT live version.\n        if receipt.name.startswith(\'block-\') and receipt.parent in complete_folders: continue\n        rec = verify_receipt(receipt)\n        selected.add(receipt)\n        selected.update(receipt.parent/f[\'file\'] for f in rec[\'files\'])\n    total = 0; files = []\n    for path in sorted(selected):\n        if not path.is_file() or \'.tmp\' in path.name: continue\n        assert not path.is_symlink()\n        relative = path.relative_to(working).as_posix()\n        flat = relative.replace(\'/\', \'__\'); target = stage/flat\n        total += path.stat().st_size\n        if total > LIMIT: raise RuntimeError(\'D1 staging size limit\')\n        shutil.copyfile(path, target)\n        files.append(dict(file=flat, original_path=relative, bytes=target.stat().st_size,\n                          sha256=sha(target), gzip=False))\n    write(stage/\'recover-manifest.json\', dict(format=\'nlp-hw02-d1-live-v1\', files=files,\n          payload_bytes=total, reason=reason, unix_time=time.time(), credentials_in_files=False))\n    write(stage/\'dataset-metadata.json\', dict(id=ref, title=\'NLP HW02 D1 live\',\n          licenses=[dict(name=\'other\')], description=\'Private row-loss and label-noise diagnostics; original-pair filtering only.\'))\n    audit_tree(stage, ref)\n\n\ndef restore_snapshot(source, destination):\n    from l3_resume import live_bytes\n    source, destination = Path(source), Path(destination)\n    manifest = json.loads((source/\'recover-manifest.json\').read_text())\n    assert manifest[\'format\'] == \'nlp-hw02-d1-live-v1\'\n    for rec in manifest[\'files\']:\n        p = destination/safe_name(rec[\'original_path\']); p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_bytes(live_bytes(source, rec))\n    return destination\n\n\nclass D1LiveSync(L2LiveSync):\n    def bounded_call(self, args, timeout, final=False):\n        # Retain reviewed auth/privacy/ready/retry/timeout logic, swap only stager.\n        if len(args) > 2 and args[2] == \'stage\' and Path(args[1]).name == \'l2_live.py\':\n            args = [*args]; args[1] = str(Path(__file__).resolve())\n        return super().bounded_call(args, timeout, final=final)\n\n    def signature(self):\n        # Batch new artifacts into milestones; uploading every GPU batch would\n        # waste bandwidth and may keep the Dataset in processing for hours.\n        root = self.working/\'d1\'; milestones = []\n        for model in (\'C-main\', \'C-long2x\'):\n            folder = root/model\n            milestones.append((model, len(list((folder/\'loss\').glob(\'block-*.receipt.json\')))//8,\n                len(list(folder.glob(\'generation-*.receipt.json\')))//16))\n        milestones.extend((str(p.relative_to(root)), p.stat().st_mtime_ns)\n            for p in root.rglob(\'*.receipt.json\') if p.name in (\'complete.receipt.json\', \'selection.receipt.json\'))\n        return sorted(milestones)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'mode\', choices=[\'stage\', \'restore\', \'privacy\'])\n    for name in (\'working\', \'stage\', \'ref\', \'reason\', \'destination\', \'receipt\'): parser.add_argument(\'--\'+name)\n    args = parser.parse_args()\n    try:\n        if args.mode == \'stage\': stage_snapshot(args.working, args.stage, args.ref, args.reason)\n        elif args.mode == \'privacy\': privacy_probe(args.ref, args.receipt)\n        else: restore_snapshot(args.stage, args.destination)\n    except Exception: sys.exit(1)\n', 'd1_loss.py': '"""Teacher-forced row NLL, exact training labels, FP32 CE on FP16 final weights."""\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom exp26_order import collate\nfrom kaggle_io import write\nfrom d1_data import publish, save_npz, verify_receipt\n\n\n@torch.inference_mode()\ndef row_nll(model, cache, ids, device, ce_chunk=4096):\n    src, tgt, labels, _ = collate(cache, ids, device)\n    logits = model(src, tgt)\n    # A full float() of B*T*8195 is unnecessary; limit the FP32 CE workspace.\n    flat = logits.reshape(-1, model.vocab); y = labels.reshape(-1)\n    losses = torch.empty(len(y), dtype=torch.float32, device=device)\n    for lo in range(0, len(y), ce_chunk):\n        losses[lo:lo+ce_chunk] = F.cross_entropy(flat[lo:lo+ce_chunk].float(), y[lo:lo+ce_chunk],\n                                               ignore_index=0, reduction=\'none\')\n    nll = losses.reshape_as(labels).sum(1)\n    lengths = labels.ne(0).sum(1)\n    if not bool(torch.isfinite(nll).all()): raise FloatingPointError(\'Nonfinite D1 NLL\')\n    return nll.cpu().numpy(), lengths.cpu().numpy().astype(np.uint16)\n\n\ndef bucket_order(cache, ids):\n    ids = np.asarray(ids, dtype=np.uint32)\n    sl = cache.source_lengths[ids].astype(np.int32)\n    tl = cache.target_lengths[ids].astype(np.int32)+1\n    return ids[np.lexsort((ids, sl, tl, sl+tl))]\n\n\n@torch.inference_mode()\ndef parity_check(model, cache, ids, device):\n    """Reviewer GPU smoke also compares against the ordinary flattened CE path."""\n    nn, ll = row_nll(model, cache, ids, device)\n    src, tgt, labels, counts = collate(cache, ids, device)\n    logits = model(src, tgt)\n    ordinary = F.cross_entropy(logits.float().reshape(-1, model.vocab), labels.reshape(-1), ignore_index=0)\n    delta = abs(float(nn.sum(dtype=np.float64)/ll.sum())-float(ordinary))\n    assert delta < 1e-5 and int(ll.sum()) == counts[\'target_tokens\']\n    assert all(p.grad is None for p in model.parameters())\n    return dict(passed=True, rows=len(ids), tokens=int(ll.sum()), token_CE=float(ordinary), absolute_delta=delta)\n\n\ndef score_all(model, cache, ids, root, device, check, batch=256, token_slots=32768, block=8192,\n              compact=False):\n    """Immutable row-id blocks permit live recovery; consolidated arrays index full cache.\n\n    Memory stays bounded on long examples; OOM retries the same rows with half batch.\n    A receipt is published only after the corresponding complete block is durable.\n    """\n    root = Path(root); root.mkdir(parents=True, exist_ok=True); started = time.monotonic()\n    ids = np.asarray(ids, dtype=np.uint32); model.eval()\n    n = len(cache); total = 0.; tokens = 0; done = 0; oom = 0; interrupted = False\n    for lo in range(0, len(ids), block):\n        try: check()\n        except TimeoutError: interrupted = True; break\n        output = root/f\'block-{lo:08d}.npz\'; receipt = output.with_suffix(\'.receipt.json\')\n        current = bucket_order(cache, ids[lo:lo+block])\n        if receipt.exists():\n            verify_receipt(receipt)\n            with np.load(output, allow_pickle=False) as saved:\n                assert np.array_equal(saved[\'row_id\'], current)\n                meta = verify_receipt(receipt)\n                total += meta[\'sum_NLL\']; tokens += meta[\'tokens\']\n        else:\n            nlls = np.empty(len(current), np.float32); lens = np.empty(len(current), np.uint16)\n            offset = 0\n            while offset < len(current):\n                try: check()\n                except TimeoutError: interrupted = True; break\n                count = min(batch, len(current)-offset)\n                while count > 1:\n                    take = current[offset:offset+count]\n                    max_t = int(cache.target_lengths[take].max())+1\n                    max_s = int(cache.source_lengths[take].max())\n                    if count*max_t <= token_slots and count*(max_t+max_s) <= 2*token_slots: break\n                    count //= 2\n                try:\n                    a, b = row_nll(model, cache, current[offset:offset+count], device)\n                    nlls[offset:offset+count], lens[offset:offset+count] = a, b\n                    offset += count\n                except torch.cuda.OutOfMemoryError:\n                    oom += 1; batch = count//2\n                    if not batch: raise\n                    import gc\n                    gc.collect()\n                    if device.type == \'cuda\': torch.cuda.empty_cache()\n            current, nlls, lens = current[:offset], nlls[:offset], lens[:offset]\n            if not offset: break\n            block_total = float(nlls.sum(dtype=np.float64)); block_tokens = int(lens.sum())\n            total += block_total; tokens += block_tokens\n            arrays = dict(row_id=current, length=lens, ce=(nlls/lens).astype(np.float16))\n            if not compact: arrays[\'nll\'] = nlls\n            save_npz(output, **arrays)\n            publish(receipt, [output], phase=\'loss-block\', rows=len(current),\n                    sum_NLL=block_total, tokens=block_tokens, complete=not interrupted)\n        done += len(current)\n        write(root/\'progress.json\', dict(rows=done, target_rows=len(ids), seconds=time.monotonic()-started,\n                                       sum_NLL=total, tokens=tokens, batch=batch, OOM_retries=oom))\n        print(\'D1 CE\', done, \'/\', len(ids), \'seconds\', round(time.monotonic()-started, 1), flush=True)\n        if interrupted: break\n    paths = [root/\'ce.npy\', root/\'length.npy\'] if compact else [root/\'ce.npy\', root/\'nll.npy\', root/\'length.npy\']\n    dtypes = [np.float16, np.uint16] if compact else [np.float16, np.float32, np.uint16]\n    arrays = [np.lib.format.open_memmap(str(p)+\'.tmp\', mode=\'w+\', dtype=dt, shape=(n,))\n              for p, dt in zip(paths, dtypes)]\n    arrays[0][:] = np.nan; arrays[-1][:] = 0\n    if not compact: arrays[1][:] = np.nan\n    for receipt in sorted(root.glob(\'block-*.receipt.json\')):\n        verify_receipt(receipt)\n        with np.load(receipt.with_name(receipt.name.replace(\'.receipt.json\', \'.npz\')), allow_pickle=False) as saved:\n            ii, ll = saved[\'row_id\'], saved[\'length\']\n            arrays[0][ii] = saved[\'ce\']; arrays[-1][ii] = ll\n            if not compact: arrays[1][ii] = saved[\'nll\']\n    for a in arrays: a.flush()\n    del arrays\n    for path in paths: Path(str(path)+\'.tmp\').replace(path)\n    summary = dict(phase=\'loss\', complete=done == len(ids), rows=done, cache_rows=n,\n                   sum_NLL=total, tokens=tokens, token_CE=total/tokens if tokens else None, OOM_retries=oom,\n                   seconds=time.monotonic()-started, normalized_dtype=\'float16\', nll_dtype=\'float32\',\n                   scoring=\'target content + EOS; PAD/BOS excluded; FP32 CE; no smoothing\',\n                   compact=compact, target_rows=len(ids), rows_per_second=done/max(time.monotonic()-started, 1e-9))\n    publish(root/(\'complete.receipt.json\' if summary[\'complete\'] else \'partial.receipt.json\'), paths, **summary)\n    return summary\n\n\ndef merge_shards(common, ids, cache):\n    """Publish one C-long2x CE array, retaining NaN for unscored global row IDs."""\n    common = Path(common); ids = np.asarray(ids, dtype=np.uint32)\n    root = common/\'C-long2x/loss\'; root.mkdir(parents=True, exist_ok=True)\n    output = root/\'ce.npy\'\n    ce = np.lib.format.open_memmap(str(output)+\'.tmp\', mode=\'w+\', dtype=np.float16, shape=(len(cache),))\n    ce[:] = np.nan; seen = np.zeros(len(cache), bool); total = 0.; tokens = 0; reports = []\n    for gpu in (0, 1):\n        shard = common/f\'loss-shards/gpu{gpu}\'\n        meta = verify_receipt(shard/(\'complete.receipt.json\' if (shard/\'complete.receipt.json\').exists()\n                                     else \'partial.receipt.json\'))\n        a = np.load(shard/\'ce.npy\', mmap_mode=\'r\', allow_pickle=False)\n        length = np.load(shard/\'length.npy\', mmap_mode=\'r\', allow_pickle=False)\n        current = np.flatnonzero(np.isfinite(a))\n        assert not seen[current].any() and np.isin(current, ids[gpu::2]).all()\n        assert np.array_equal(length[current], cache.target_lengths[current]+1)\n        assert len(current) == meta[\'rows\'] and int(length[current].sum()) == meta[\'tokens\']\n        ce[current] = a[current]; seen[current] = True\n        total += meta[\'sum_NLL\']; tokens += meta[\'tokens\']; reports.append(meta)\n    ce.flush(); del ce; Path(str(output)+\'.tmp\').replace(output)\n    summary = dict(phase=\'loss\', model=\'C-long2x\', weight_sha256=__import__(\'d1_data\').WEIGHTS[\'C-long2x\'],\n                   complete=bool(seen[ids].all()), rows=int(seen.sum()), target_rows=len(ids),\n                   cache_rows=len(cache), sum_NLL=total, tokens=tokens, token_CE=total/tokens if tokens else None,\n                   normalized_dtype=\'float16\', shards=reports, score_policy=\'C-long2x only\')\n    publish(root/(\'complete.receipt.json\' if summary[\'complete\'] else \'partial.receipt.json\'), [output], **summary)\n    return summary\n', 'd1_session.py': '"""D1 setup and independent workers under the reviewed L3/P1 watchdog."""\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom l1_session import Watchdog\nfrom l2_session import disk_guard\nfrom session_runner import stop_workers, tail\nfrom kaggle_io import write, sha\n\n\ndef watchdog_probe(cfg):\n    """Torch-free child forces the actual watchdog exit on the same output mount."""\n    target = Path(cfg[\'working_root\'])/\'watchdog-probe\'; target.mkdir(parents=True, exist_ok=True)\n    script = Path(cfg[\'temp_root\'])/\'d1-watchdog-probe.py\'\n    script.write_text(\'\'\'import sys,time,json\nfrom pathlib import Path\nsys.path.insert(0,sys.argv[1])\nfrom d1_session import Watchdog\nfrom kaggle_io import write\nroot=Path(sys.argv[2]);start=time.monotonic()\nwrite(root/\'durable-batch.json\',{\'rows\':17,\'artifact_durable_before_forced_exit\':True})\ncfg=dict(working_root=str(root),session_kind=\'d1\',session_start_monotonic=start,\n    deadline_monotonic=start+.2,hard_deadline_monotonic=start+.6,budget_seconds=.2,terminate_grace_seconds=.1)\nWatchdog(cfg).start()\nwhile True:time.sleep(.05)\n\'\'\')\n    p = subprocess.run([sys.executable, str(script), str(Path(__file__).parent), str(target)],\n                       capture_output=True, timeout=5, start_new_session=True)\n    meta = json.loads((target/\'session-summary.json\').read_text())\n    assert p.returncode == 0 and meta[\'watchdog_forced_exit\']\n    assert json.loads((target/\'durable-batch.json\').read_text())[\'rows\'] == 17\n    write(target/\'probe.json\', dict(passed=True, actual_forced_exit=True, child_exit_code=p.returncode,\n                                  durable_artifact_preserved=True, same_working_mount=True))\n\n\ndef setup(cfg):\n    import numpy as np\n    from kaggle_io import materialize_dataset\n    from kaggle_runtime import prepare_starter\n    from l1_cache import CompactCache\n    from l3_resume import resolve_cache_mount\n    from d1_data import ROWS, CACHE_SHA, MODELS, WEIGHTS, publish, length_bins\n    scratch, inp = Path(cfg[\'temp_root\']), Path(cfg[\'input_root\'])\n    work = Path(cfg[\'working_root\']); data = work/\'d1\'; data.mkdir(exist_ok=True)\n    if cfg[\'gpu_smoke\']: watchdog_probe(cfg)\n    starter = prepare_starter(resolve_cache_mount(inp, \'thisarivson/nlp-hw02-26b-inputs\', \'datasets\'),\n                              scratch/\'starter\', \'training-inputs\', read_only=True)\n    try:\n        cache_mount = resolve_cache_mount(inp, cfg[\'cache_source\'], \'notebooks\')\n        source = cfg[\'cache_source\']\n    except ValueError:\n        cache_mount = resolve_cache_mount(inp, cfg[\'cache_fallback_source\'], \'datasets\')\n        source = cfg[\'cache_fallback_source\']\n    cache_root = materialize_dataset(cache_mount, scratch/\'compact\', \'compact-manifest.json\')\n    assert sha(cache_root/\'compact-manifest.json\') == CACHE_SHA\n    cache = CompactCache(cache_root, verify=True); assert len(cache) == ROWS\n    summary = json.loads((cache_root/\'cache-summary.json\').read_text())\n    assert summary[\'rows\'] == ROWS and summary[\'roundtrip_rows\'] >= 50000 and summary[\'answer_mismatches\'] == 0\n    assert sha(cache_root/\'tokenizer.json\') == cache.manifest[\'tokenizer\'][\'sha256\']\n    assert sha(starter/\'tokenizer.json\') == cache.manifest[\'original_tokenizer_sha256\']\n    live = resolve_cache_mount(inp, cfg[\'weights_source\'], \'datasets\')\n    models = {}\n    for name in MODELS:\n        hits = list(live.rglob(f\'model-{name}__weights__l2-62280-fp16.pt\'))\n        assert len(hits) == 1 and sha(hits[0]) == WEIGHTS[name]\n        models[name] = str(hits[0])\n    if not cfg[\'gpu_smoke\']:\n        seed_root = resolve_cache_mount(inp, cfg[\'live_dataset\'], \'datasets\')\n        seed = [p for p in seed_root.rglob(\'*\') if p.name in (\'seed-evidence.npz\', \'d1__seed-evidence.npz\')]\n        assert len(seed) == 1 and sha(seed[0]) == cfg[\'seed_sha256\']\n        provenance = [p for p in seed_root.rglob(\'*\') if p.name in (\'seed.receipt.json\', \'d1__seed-provenance.json\')]\n        assert len(provenance) == 1\n        provenance_meta = json.loads(provenance[0].read_text())\n        assert provenance_meta[\'cache_manifest_sha256\'] == CACHE_SHA\n        assert provenance_meta[\'files\'][0][\'sha256\'] == cfg[\'seed_sha256\']\n        shutil.copyfile(provenance[0], data/\'seed-provenance.json\')\n        shutil.copyfile(seed[0], data/\'seed-evidence.npz\')\n    else:\n        from d1_data import save_npz\n        save_npz(data/\'seed-evidence.npz\', row_id=np.empty(0, np.uint32), reference=np.empty(0),\n                 target_length=cache.target_lengths, source_length=cache.source_lengths,\n                 group=np.empty(0, np.uint32), group_size=np.empty(0, np.uint16), conflicting=np.empty(0, bool),\n                 minority=np.empty(0, bool), majority=np.empty(0), protected=np.asarray([8545870], np.uint32))\n    publish(data/\'seed.receipt.json\', [data/\'seed-evidence.npz\'], phase=\'seed\', cache_manifest_sha256=CACHE_SHA)\n    ids = np.arange(ROWS, dtype=np.uint32)\n    if cfg[\'gpu_smoke\']:\n        # Uniform sample estimates full-train CE mix; top4% still ranked within length bins.\n        rng = np.random.default_rng(2026101404)\n        ids = np.sort(rng.choice(ROWS, cfg[\'smoke_rows\'], replace=False)).astype(np.uint32)\n    np.save(data/\'scored-row-ids.npy\', ids, allow_pickle=False)\n    publish(data/\'scored-row-ids.receipt.json\', [data/\'scored-row-ids.npy\'], phase=\'scope\',\n            rows=len(ids), all_train=not cfg[\'gpu_smoke\'], cache_manifest_sha256=CACHE_SHA)\n    # v6 adds the three representation tokens itself; pass the ORIGINAL 8189 BPE.\n    cfg.update(compact_cache=str(cache_root), tokenizer=str(starter/\'tokenizer.json\'),\n               starter_dataset=str(starter), models=models, cache_manifest_sha256=CACHE_SHA)\n    write(work/\'input-preflight.json\', dict(passed=True, cache_source=source, rows=ROWS,\n          all_shard_SHA_verified=True, tokenizer_SHA_verified=True, models=WEIGHTS,\n          seed_sha256=cfg[\'seed_sha256\'], scoring_rows=len(ids), training=False))\n    assert time.monotonic()-cfg[\'session_start_monotonic\'] < cfg[\'setup_cap_seconds\']\n\n\ndef launch(config, worker, watchdog, live=None, cpu_test=False, heartbeat=30.):\n    working = Path(config[\'working_root\']); path = working/\'session-config.json\'\n    write(path, config); handles = []; processes = []; reason = \'workers completed\'; failed = set()\n    try:\n        for gpu in (0, 1):\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'\' if cpu_test else str(gpu), OMP_NUM_THREADS=\'1\',\n                       MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n                       KAGGLE_CONFIG_DIR=str(Path(config[\'temp_root\'])/\'worker-empty-config\'))\n            for secret in (\'KAGGLE_USERNAME\', \'KAGGLE_KEY\', \'KAGGLE_API_TOKEN\',\n                           \'KAGGLE_USER_SECRETS_TOKEN\', \'KAGGLE_IAP_TOKEN\'): env.pop(secret, None)\n            h = (working/f\'gpu{gpu}-process.log\').open(\'w\'); handles.append(h)\n            p = subprocess.Popen([sys.executable, str(worker), \'--gpu\', str(gpu), \'--config\', str(path)],\n                                 env=env, stdout=h, stderr=subprocess.STDOUT, start_new_session=True)\n            processes.append(p); watchdog.register(p)\n        while any(p.poll() is None for p in processes):\n            if time.monotonic() >= config[\'deadline_monotonic\']: reason = \'session deadline\'; break\n            if live is not None and (live.uninsured or time.monotonic()-live.last_success >= 7200.):\n                reason = \'live insurance stale >=2h\'\n                write(working/\'stop-request.json\', dict(reason=reason))\n            for gpu, p in enumerate(processes):\n                if p.poll() not in (None, 0) and gpu not in failed:\n                    failed.add(gpu)\n                    write(working/f\'gpu{gpu}-failure.json\', dict(exit_code=p.poll(), healthy_worker_stopped=False))\n                print(\'D1 GPU\', gpu, \'status\', p.poll(), \' | \'.join(tail(working/f\'gpu{gpu}-process.log\'))[:400], flush=True)\n            disk_guard(working, limit=2*1024**3)\n            time.sleep(min(heartbeat, max(.01, config[\'deadline_monotonic\']-time.monotonic())))\n    except Exception as error:\n        reason = \'supervisor error: \'+type(error).__name__\n    finally:\n        survivors = stop_workers(processes, watchdog)\n        for h in handles: h.close()\n        states = {}\n        for name in (\'C-main\', \'C-long2x\'):\n            summary = working/\'d1\'/name/\'summary.json\'\n            try: states[name] = json.loads(summary.read_text()).get(\'status\') if summary.exists() else \'missing\'\n            except (ValueError, OSError): states[name] = \'unreadable\'\n        report = dict(status=\'complete\' if len(processes) == 2 and all(p.returncode == 0 for p in processes)\n                      and all(s == \'complete\' for s in states.values()) else \'partial\',\n                      worker_status=states, reason=reason, exit_codes=[p.poll() for p in processes],\n                      failed_workers=sorted(failed), unreaped_worker_pids=survivors,\n                      wall_seconds=time.monotonic()-config[\'session_start_monotonic\'],\n                      hard_shutdown_seconds=config[\'hard_shutdown_seconds\'], GPU=not cpu_test, training=False)\n        write(working/\'session-summary.json\', report)\n        if live is not None:\n            report[\'final_live_uploaded\'] = live.final_sync(\'normal-exit\', timeout=max(.1,\n                min(540., watchdog.hard_deadline-time.monotonic()-60.)))\n        write(working/\'session-summary.json\', report)\n        if not survivors: watchdog.finish()\n    return report\n', 'd1_worker.py': '"""D1 worker: one visible GPU, one pinned L3 model, no authentication environment."""\nimport argparse\nimport json\nimport os\nimport signal\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--gpu\', required=True, type=int)\n    parser.add_argument(\'--config\', required=True)\n    args = parser.parse_args(); cfg = json.loads(Path(args.config).read_text())\n    root = Path(cfg[\'working_root\']); sys.path.insert(0, str(root/\'src\'))\n    import numpy as np\n    import torch\n    import final_core as C\n    from final_representation import tokenizer\n    from l1_cache import CompactCache\n    from kaggle_io import sha, write\n    from d1_data import MODELS, WEIGHTS, ranked_candidates, choose_budget, save_npz, publish, verify_receipt\n    from d1_loss import score_all, parity_check, merge_shards\n    from d1_generate import generation_wave\n    name = MODELS[args.gpu]; out = root/\'d1\'/name; out.mkdir(parents=True, exist_ok=True)\n    start = cfg[\'session_start_monotonic\']\n    torch.set_num_threads(1); torch.set_num_interop_threads(1)\n    device = torch.device(\'cuda:0\')\n    assert torch.cuda.device_count() == 1 and \'T4\' in torch.cuda.get_device_name(0)\n    assert os.environ.get(\'CUDA_VISIBLE_DEVICES\') == str(args.gpu)\n    assert str(torch.__version__) == \'2.11.0+cu128\' and torch.version.cuda == \'12.8\'\n    assert not any(os.environ.get(s) for s in (\'KAGGLE_KEY\', \'KAGGLE_API_TOKEN\', \'KAGGLE_USER_SECRETS_TOKEN\'))\n    torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False\n    torch.use_deterministic_algorithms(True); torch.utils.deterministic.fill_uninitialized_memory = False\n    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True\n    cache = CompactCache(cfg[\'compact_cache\'], verify=False)\n    assert cache.manifest_sha == cfg[\'cache_manifest_sha256\']\n    def load_model(which):\n        path = Path(cfg[\'models\'][which]); assert sha(path) == WEIGHTS[which]\n        p = torch.load(path, map_location=\'cpu\', weights_only=True)\n        assert p[\'format\'] == \'nlp-hw02-model-only-v1\' and p[\'optimizer_included\'] is False\n        assert p[\'point\'] == \'l2-62280\' and p[\'model_spec\'][\'vocab\'] == 8195\n        result = C.SimpleTransformer(**p[\'model_spec\'], dropout=0.).eval().half()\n        result.load_state_dict(p[\'model\'], strict=True)\n        return result.to(device)\n    # Both GPUs score C-long2x; C-main is loaded only when generation begins.\n    model = load_model(\'C-long2x\')\n    def terminated(*_): raise TimeoutError(\'D1 graceful stop\')\n    signal.signal(signal.SIGTERM, terminated)\n    write(out/\'identity.json\', dict(model=name, weight_sha256=WEIGHTS[name],\n          cache_manifest_sha256=cache.manifest_sha, tokenizer_sha256=sha(cfg[\'tokenizer\']),\n          point=\'l2-62280\', training=False))\n    tok, table = tokenizer(cfg[\'tokenizer\'], 30)\n    del tok\n\n    def check(until, dependency=False):\n        if time.monotonic() >= until or (root/\'stop-request.json\').exists():\n            raise TimeoutError(\'D1 phase deadline or stop request\')\n        if dependency and any(root.glob(\'gpu*-failure.json\')):\n            raise RuntimeError(\'D1 dependency worker failed\')\n\n    def wait(path, until):\n        while not path.exists():\n            check(until, dependency=True); time.sleep(.25)\n        return json.loads(path.read_text())\n\n    score_ids = np.load(root/\'d1/scored-row-ids.npy\', allow_pickle=False)\n    if cfg[\'gpu_smoke\']:\n        write(out/\'loss-parity.json\', parity_check(model, cache, score_ids[:16], device))\n    loss_until = start+cfg[\'loss_stop_seconds\']\n    common = root/\'d1\'; shard_root = common/f\'loss-shards/gpu{args.gpu}\'\n    loss = score_all(model, cache, score_ids[args.gpu::2], shard_root, device, lambda: check(loss_until),\n                     batch=cfg[\'loss_batch\'], token_slots=cfg[\'loss_token_slots\'], block=cfg[\'loss_block\'], compact=True)\n    publish(shard_root/\'done.receipt.json\', [], phase=\'loss-shard-done\', **{\'summary\': loss})\n    write(out/\'summary.json\', dict(status=\'loss-complete\', loss=loss))\n    candidate_path = common/\'candidates.npz\'\n    for gpu in (0, 1):\n        wait(common/f\'loss-shards/gpu{gpu}/done.receipt.json\', loss_until+120)\n    if args.gpu == 0:\n        merged = merge_shards(common, score_ids, cache)\n        ce = np.load(common/\'C-long2x/loss/ce.npy\', mmap_mode=\'r\', allow_pickle=False)\n        scored = score_ids[np.isfinite(ce[score_ids])]\n        if not len(scored): raise RuntimeError(\'No scored rows\')\n        ranked, _ = ranked_candidates(ce[scored], ce[scored], cache.target_lengths[scored], cfg[\'fraction\'])\n        save_npz(candidate_path, **{f\'bin{b}\': scored[ii] for b, ii in enumerate(ranked)})\n        publish(common/\'candidates.receipt.json\', [candidate_path], phase=\'candidate-ranking\',\n                fraction=cfg[\'fraction\'], compact_content_edges=[53, 79, 105], rows=sum(map(len, ranked)),\n                loss_complete=merged[\'complete\'], scored_rows=len(scored), score_model=\'C-long2x\')\n    wait(common/\'candidates.receipt.json\', loss_until+120)\n    verify_receipt(common/\'candidates.receipt.json\')\n    with np.load(candidate_path, allow_pickle=False) as saved:\n        ranked = [saved[f\'bin{b}\'] for b in range(4)]\n    if args.gpu == 0:\n        del model\n        import gc\n        gc.collect(); torch.cuda.empty_cache()\n        model = load_model(name)\n    evidence = None\n    if args.gpu == 0:\n        with np.load(common/\'seed-evidence.npz\', allow_pickle=False) as seed:\n            evidence = {k: seed[k] for k in (\'row_id\', \'reference\')}\n        if cfg[\'gpu_smoke\']:\n            # Independently execute two original references/bin before the wave, testing reuse\n            # of genuine numeric answers without attaching a fourth smoke input dataset.\n            from d1_generate import reference_for, Executor\n            jobs = []; rows = []\n            for ii in ranked:\n                for row in ii[:2]:\n                    code, value, _ = reference_for(cache, int(row), table, evidence)\n                    if code is not None: jobs.append({\'code\':code}); rows.append(int(row))\n            executor = Executor(2)\n            try:\n                executor.submit(jobs); executor.drain(min(start+cfg[\'execution_stop_seconds\'], time.monotonic()+30))\n                executor.attach(jobs)\n                reused = sorted((r,j[\'execution\'][\'value\']) for r,j in zip(rows,jobs)\n                                if j[\'execution\'][\'status\'] == \'numeric\')\n                evidence = dict(row_id=np.asarray([r for r,v in reused], np.uint32),\n                                reference=np.asarray([v for r,v in reused], np.float64))\n                write(out/\'smoke-reference-cache.json\', dict(real_original_references=True, reused=len(reused)))\n            finally: executor.close()\n    pilot = []; pilot_ids = []\n    for b, ids in enumerate(ranked):\n        take = ids[:cfg[\'pilot_per_bin\']]; pilot_ids.extend(take.tolist())\n        if len(take):\n            metrics = generation_wave(model, cache, take, table, device, out/f\'pilot-bin-{b}.npz\',\n                lambda: check(start+cfg[\'generation_stop_seconds\']),\n                min(start+cfg[\'execution_stop_seconds\'], time.monotonic()+cfg[\'wave_timeout_seconds\']),\n                evidence=evidence, batch=cfg[\'generation_batch\'], cap=cfg[\'cap\'],\n                generation_deadline=start+cfg[\'generation_stop_seconds\'])\n            pilot.append(dict(rows=len(take), seconds=metrics[\'seconds\']))\n        else: pilot.append(dict(rows=0, seconds=0.))\n    write(out/\'pilot.json\', dict(bins=pilot, model=name))\n    selection_receipt = common/\'selection.receipt.json\'\n    if args.gpu == 0:\n        pilots = [wait(common/m/\'pilot.json\', start+cfg[\'generation_stop_seconds\']) for m in MODELS]\n        selected, budget = choose_budget(ranked, pilots, start+cfg[\'generation_stop_seconds\']-time.monotonic(),\n                                         safety=cfg[\'timing_safety\'], tail=cfg[\'pilot_tail_reserve\'])\n        save_npz(common/\'selection.npz\', row_id=selected)\n        publish(selection_receipt, [common/\'selection.npz\'], phase=\'selection\', budget=budget,\n                cache_manifest_sha256=cache.manifest_sha, score_precision=\'C-long2x persisted FP16 CE\')\n    selection = wait(selection_receipt, start+cfg[\'generation_stop_seconds\'])\n    verify_receipt(selection_receipt)\n    with np.load(common/\'selection.npz\', allow_pickle=False) as saved: selected = saved[\'row_id\']\n    assert len(np.unique(selected)) == len(selected)\n    remaining = selected[~np.isin(selected, np.asarray(pilot_ids, dtype=np.uint32))]\n    metrics = []\n    for lo in range(0, len(remaining), cfg[\'wave_rows\']):\n        try: check(start+cfg[\'generation_stop_seconds\'])\n        except TimeoutError: break\n        metrics.append(generation_wave(model, cache, remaining[lo:lo+cfg[\'wave_rows\']], table, device,\n            out/f\'generation-{lo:08d}.npz\', lambda: check(start+cfg[\'generation_stop_seconds\']),\n            min(start+cfg[\'execution_stop_seconds\'], time.monotonic()+cfg[\'wave_timeout_seconds\']),\n            evidence=evidence, batch=cfg[\'generation_batch\'], cap=cfg[\'cap\'],\n            generation_deadline=start+cfg[\'generation_stop_seconds\']))\n        print(\'D1 GREEDY\', name, min(lo+cfg[\'wave_rows\'], len(remaining)), \'/\', len(remaining), flush=True)\n    pilot_complete = all(verify_receipt(out/f\'pilot-bin-{b}.receipt.json\')[\'complete\']\n                         for b in range(4) if len(ranked[b]))\n    generated_rows = len(pilot_ids)+sum(m[\'rows\'] for m in metrics)\n    full_loss = (common/\'C-long2x/loss/complete.receipt.json\').exists()\n    write(out/\'summary.json\', dict(status=\'complete\' if full_loss and generated_rows == len(selected) and\n          pilot_complete and all(m[\'complete\'] for m in metrics) else \'partial\',\n          loss=loss, selected_rows=len(selected), pilot_rows=len(pilot_ids), generated_rows=generated_rows,\n          budget=selection[\'budget\'], generation_waves=len(metrics), weight_sha256=WEIGHTS[name],\n          wall_seconds=time.monotonic()-start))\n\n\nif __name__ == \'__main__\':\n    try:\n        main()\n    except BaseException as error:\n        # Never stringify exceptions: API credential values/URLs must not reach logs.\n        print(\'D1 WORKER FAILED\', type(error).__name__, flush=True)\n        sys.exit(1)\n', 'exp26_cache.py': '"""CPU-only shard cache: exact exclusions, finite references, int16 BPE.\nOnly one shard of text is in RAM. Completed shard receipts are immutable.\n"""\nimport hashlib,json,os,socket,time,urllib.request\nfrom array import array\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nimport pyarrow.parquet as pq\nfrom tokenizers import Tokenizer\nimport exp26_protocol as P\nfrom exp26_common import sha_file,json_hash,read_json,write_json,file_record,verify_record\nfrom exp26_input import load_protected\nfrom exp26_fast_exec import run_chunks\n\nFILES=[\'source.npy\',\'source-offsets.npy\',\'target.npy\',\'target-offsets.npy\',\'provenance-rows.npy\',\'question-sha256.npy\']\n\ndef is_validation(q):return int.from_bytes(hashlib.sha256(f\'{P.SEED}:{q}\'.encode()).digest()[:8],\'big\')%10==0\n\ndef classify(row,protected):\n if not isinstance(row.get(\'question\'),str)or not isinstance(row.get(\'code\'),str):return \'invalid_text\'\n if len(row[\'question\'].encode(\'utf-8\'))>512 or len(row[\'code\'].encode(\'utf-8\'))>=512:return \'length\'\n if row[\'question\']in protected:return \'protected_question\'\n if is_validation(row[\'question\']):return \'hash_heldout\'\n return None\n\ndef download_raw(spec,root):\n root=Path(root);root.mkdir(parents=True,exist_ok=True);p=root/Path(spec[\'path\']).name\n if p.exists():\n  assert p.stat().st_size==spec[\'bytes\']and sha_file(p)==spec[\'sha256\'],p.name\n  return p\n url=f\'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{P.REVISION}/{spec["path"]}\'\n for attempt in range(3):\n  tmp=p.with_suffix(\'.download.tmp\');h=hashlib.sha256()\n  try:\n   with urllib.request.urlopen(url,timeout=30)as response,tmp.open(\'wb\')as out:\n    for block in iter(lambda:response.read(4*1024**2),b\'\'):out.write(block);h.update(block)\n   assert tmp.stat().st_size==spec[\'bytes\']and h.hexdigest()==spec[\'sha256\']\n   tmp.replace(p);assert sha_file(p)==spec[\'sha256\'];return p\n  except (OSError,AssertionError):\n   if attempt==2:raise\n   time.sleep(1+attempt)\n\ndef collect_candidates(path,protected,limit=None):\n rows=[];row_ids=[];counts=Counter();offset=0\n for batch in pq.ParquetFile(path).iter_batches(batch_size=8192,columns=[\'question\',\'code\']):\n  for j,row in enumerate(batch.to_pylist()):\n   counts[\'raw_rows\']+=1;reason=classify(row,protected)\n   if reason:counts[reason]+=1;continue\n   rows.append(row);row_ids.append(offset+j)\n   if limit is not None and len(rows)==limit:return rows,row_ids,dict(counts)\n  offset+=len(batch)\n return rows,row_ids,dict(counts)\n\ndef save_npy(path,value):\n p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+\'.tmp\')\n with tmp.open(\'wb\')as f:np.save(f,value,allow_pickle=False);f.flush();os.fsync(f.fileno())\n tmp.replace(p)\n\ndef token_block(rows,row_ids,tokenizer,token_reuse=None,shard_id=None):\n source=array(\'h\');target=array(\'h\');so=array(\'q\',[0]);to=array(\'q\',[0]);qh=bytearray();reused=0\n for lo in range(0,len(rows),2048):\n  part=rows[lo:lo+2048];locs=row_ids[lo:lo+2048]\n  missing=[i for i,loc in enumerate(locs)if token_reuse is None or (shard_id,loc)not in token_reuse[\'indices\']]\n  qs=tokenizer.encode_batch([part[i][\'question\']for i in missing],add_special_tokens=False)if missing else []\n  cs=tokenizer.encode_batch([part[i][\'code\']for i in missing],add_special_tokens=False)if missing else []\n  encoded={i:([1]+[j+3 for j in q.ids]+[2],[j+3 for j in c.ids])for i,q,c in zip(missing,qs,cs)}\n  for i,r in enumerate(part):\n   if i in encoded:q,c=encoded[i]\n   else:\n    k=token_reuse[\'indices\'][(shard_id,locs[i])];a=token_reuse\n    q=a[\'source\'][a[\'so\'][k]:a[\'so\'][k+1]];c=a[\'target\'][a[\'to\'][k]:a[\'to\'][k+1]];reused+=1\n   source.extend(q);so.append(len(source));target.extend(c);to.append(len(target));qh.extend(hashlib.sha256(r[\'question\'].encode()).digest())\n return [np.frombuffer(source,dtype=np.int16),np.frombuffer(so,dtype=np.int64),np.frombuffer(target,dtype=np.int16),\n         np.frombuffer(to,dtype=np.int64),np.asarray(row_ids,dtype=np.uint32),np.frombuffer(qh,dtype=np.uint8).reshape(-1,32)],reused\n\ndef build_tokens(rows,row_ids,tokenizer,out,token_reuse=None,shard_id=None,identity=None,stop_after_blocks=None,block_rows=16384):\n out=Path(out);out.mkdir(parents=True,exist_ok=True);work=out/\'token-work\';work.mkdir(exist_ok=True)\n identity=identity or json_hash({\'rows\':row_ids,\'questions\':[r[\'question\']for r in rows]})\n rp=work/\'progress.json\';dtypes=[np.int16,np.int64,np.int16,np.int64,np.uint32,np.uint8]\n if rp.exists():\n  progress=read_json(rp);assert progress[\'identity\']==identity\n  for record in progress[\'blocks\']:\n   p=work/record[\'file\']\n   with p.open(\'rb\')as f:f.seek(record[\'offset\']);assert hashlib.sha256(f.read(record[\'bytes\'])).hexdigest()==record[\'sha256\']\n  for name,size in progress[\'file_bytes\'].items():\n   p=work/name;assert p.stat().st_size>=size\n   with p.open(\'r+b\')as f:f.truncate(size)  # Only surplus from an uncommitted block.\n else:\n  progress={\'identity\':identity,\'committed_rows\':0,\'blocks\':[],\'file_bytes\':{},\'pilot_tokens_reused\':0}\n  for name,dtype in zip(FILES,dtypes):\n   p=work/(name+\'.bin\')\n   with p.open(\'wb\')as f:\n    if \'offsets\'in name:f.write(np.asarray([0],dtype=np.int64).tobytes())\n   progress[\'file_bytes\'][p.name]=p.stat().st_size\n  write_json(rp,progress)\n block_number=0\n for lo in range(progress[\'committed_rows\'],len(rows),block_rows):\n  values,reused=token_block(rows[lo:lo+block_rows],row_ids[lo:lo+block_rows],tokenizer,token_reuse,shard_id)\n  for i,(name,value)in enumerate(zip(FILES,values)):\n   p=work/(name+\'.bin\')\n   if i in (1,3):\n    token_name=FILES[i-1]+\'.bin\';start=progress[\'file_bytes\'][token_name]//2-len(values[i-1])\n    value=value[1:]+start\n   raw=value.tobytes();offset=p.stat().st_size\n   with p.open(\'ab\')as f:f.write(raw);f.flush();os.fsync(f.fileno())\n   progress[\'blocks\'].append({\'file\':p.name,\'offset\':offset,\'bytes\':len(raw),\'sha256\':hashlib.sha256(raw).hexdigest()})\n   progress[\'file_bytes\'][p.name]=offset+len(raw)\n  progress[\'committed_rows\']=min(len(rows),lo+block_rows);progress[\'pilot_tokens_reused\']+=reused;write_json(rp,progress)\n  block_number+=1\n  if stop_after_blocks and block_number>=stop_after_blocks:raise KeyboardInterrupt(\'Token-cache interruption fixture\')\n n=len(rows)\n for name,dtype in zip(FILES,dtypes):\n  p=work/(name+\'.bin\');elements=p.stat().st_size//np.dtype(dtype).itemsize\n  shape=(n,32)if name==\'question-sha256.npy\'else(elements,)\n  src=np.memmap(p,dtype=dtype,mode=\'r\',shape=shape)if elements else np.empty(shape,dtype=dtype)\n  tmp=out/(name+\'.tmp\');dst=np.lib.format.open_memmap(tmp,mode=\'w+\',dtype=dtype,shape=shape)\n  for lo in range(0,len(src),1024**2):dst[lo:lo+1024**2]=src[lo:lo+1024**2]\n  dst.flush();del dst,src;tmp.replace(out/name)\n return {\'rows\':n,\'source_tokens\':progress[\'file_bytes\'][\'source.npy.bin\']//2,\n         \'target_content_tokens\':progress[\'file_bytes\'][\'target.npy.bin\']//2,\n         \'target_tokens_with_EOS\':progress[\'file_bytes\'][\'target.npy.bin\']//2+n,\'pilot_tokenizations_reused\':progress[\'pilot_tokens_reused\']}\n\ndef validate_shard(folder,receipt,protected=None):\n for entry in receipt[\'files\']:verify_record(folder,entry)\n src,so,tgt,to,ids,qh=[np.load(Path(folder)/n,mmap_mode=\'r\',allow_pickle=False)for n in FILES]\n n=receipt[\'tokens\'][\'rows\'];assert len(ids)==len(qh)==n and so.shape==to.shape==(n+1,)\n assert src.dtype==tgt.dtype==np.int16 and so.dtype==to.dtype==np.int64 and ids.dtype==np.uint32 and qh.dtype==np.uint8\n assert qh.shape==(n,32)and so[0]==to[0]==0 and so[-1]==len(src)and to[-1]==len(tgt)\n assert (np.diff(so)>=2).all()and (np.diff(to)>=0).all()and (np.diff(ids.astype(np.int64))>0).all()\n assert not n or ids.max()<receipt[\'raw_num_rows\']\n if n:\n  assert src.min()>=1 and src.max()<8192 and (src[so[:-1]]==1).all()and (src[so[1:]-1]==2).all()\n  assert not len(tgt)or (tgt.min()>=3 and tgt.max()<8192)\n if protected is not None:\n  protected_hashes=np.asarray([hashlib.sha256(q.encode()).digest()for q in protected],dtype=\'S32\')\n  for lo in range(0,n,65536):\n   hashes=np.asarray(qh[lo:lo+65536]).copy().view(\'S32\').reshape(-1)\n   assert not np.isin(hashes,protected_hashes).any(),\'Protected question in token cache\'\n return n\n\nSTATUS_NAMES=[\'numeric\',\'non_number\',\'non_finite\',\'missing_solve\',\'execution_error\',\'syntax_error\',\'blocked_size\',\'blocked_name\',\'blocked_attribute\',\'blocked_construct\',\'blocked_import\',\'timeout\',\'timeout_or_worker_exit\',\'worker_error\',\'budget_not_evaluated\']\n\ndef execute_journal(codes,out,identity,workers,progress=None):\n import struct\n out=Path(out);out.mkdir(parents=True,exist_ok=True);p=out/\'reference-journal.bin\';header=out/\'reference-journal.json\'\n fingerprint=json_hash({\'identity\':identity,\'codes\':codes,\'chunk_rows\':2000})\n if header.exists():assert read_json(header)[\'fingerprint\']==fingerprint\n else:write_json(header,{\'fingerprint\':fingerprint,\'status_names\':STATUS_NAMES})\n saved={};good=0\n if p.exists():\n  with p.open(\'rb\')as f:\n   while True:\n    h=f.read(40)\n    if not h:break\n    if len(h)!=40:break\n    i,n,digest=struct.unpack(\'<II32s\',h);assert n<=2000 and i*2000<len(codes)\n    payload=f.read(n)\n    if len(payload)!=n:break\n    assert hashlib.sha256(payload).digest()==digest\n    assert n==min(2000,len(codes)-i*2000)and all(v<len(STATUS_NAMES)for v in payload)\n    saved[i]=[{\'status\':STATUS_NAMES[v]}for v in payload];good=f.tell()\n  with p.open(\'r+b\')as f:f.truncate(good)\n def commit(i,rows):\n  payload=bytes(STATUS_NAMES.index(r[\'status\'])for r in rows)\n  with p.open(\'ab\')as f:f.write(struct.pack(\'<II32s\',i,len(payload),hashlib.sha256(payload).digest())+payload);f.flush();os.fsync(f.fileno())\n result=run_chunks(codes,workers=workers,chunk_rows=2000,saved_chunks=saved,on_chunk=commit,progress=progress)\n result[\'summary\'][\'programs_reused_from_journal\']=sum(len(v)for v in saved.values())\n return result\n\ndef pilot(root,paths,specs,protected,tokenizer,code_sha,workers,limit=50000):\n root=Path(root);receipt_path=root/\'pilot.json\'\n if receipt_path.exists():\n  info=read_json(receipt_path);assert info[\'code_sha256\']==code_sha\n  for e in info[\'files\']:verify_record(root,e)\n  return info,np.load(root/\'pilot.npz\',allow_pickle=False)\n started=time.monotonic();rows=[];locations=[];raw_scanned=0\n for spec,path in zip(specs,paths):\n  part,ids,counts=collect_candidates(path,protected,limit=limit-len(rows));rows.extend(part)\n  locations.extend((spec[\'shard\'],i)for i in ids);raw_scanned+=counts[\'raw_rows\']\n  if len(rows)>=limit:break\n if len(rows)<limit:raise ValueError(\'Insufficient pilot eligible rows\')\n selected_seconds=time.monotonic()-started\n last_report=[0]\n def progress(done,total,seconds):\n  if done-last_report[0]>=10000 or done==total:\n   rate=done/max(seconds,1e-9);print(\'PILOT EXEC\',done,\'/\',total,\'rows;\',round(rate,1),\'rows/s; remaining\',round((total-done)/max(rate,1e-9),1),\'s\',flush=True);last_report[0]=done\n execution=execute_journal([r[\'code\']for r in rows],root/\'pilot-reference\',json_hash({\'code\':code_sha,\'purpose\':\'pilot\'}),workers,progress=progress)\n encode_start=time.monotonic();keep=[i for i,r in enumerate(execution[\'rows\'])if r[\'status\']==\'numeric\']\n tokens=build_tokens([rows[i]for i in keep],[locations[i][1]for i in keep],tokenizer,root/\'pilot-tokens\')\n save_npy(root/\'pilot-shards.npy\',np.asarray([locations[i][0]for i in keep],dtype=np.uint8))\n tokenization_seconds=time.monotonic()-encode_start\n names=sorted({r[\'status\']for r in execution[\'rows\']});lookup={n:i for i,n in enumerate(names)}\n p=root/\'pilot.npz\';tmp=p.with_suffix(\'.npz.tmp\')\n with tmp.open(\'wb\')as f:np.savez(f,shard=np.asarray([a for a,b in locations],dtype=np.uint8),row=np.asarray([b for a,b in locations],dtype=np.uint32),status=np.asarray([lookup[r[\'status\']]for r in execution[\'rows\']],dtype=np.uint8))\n tmp.replace(p)\n elapsed=time.monotonic()-started\n info={\'code_sha256\':code_sha,\'eligible_rows\':len(rows),\'raw_scanned\':raw_scanned,\'status_names\':names,\n       \'selection_seconds\':selected_seconds,\'execution\':execution[\'summary\'],\'tokenization_seconds\':tokenization_seconds,\n       \'total_seconds\':elapsed,\'eligible_rows_per_second\':len(rows)/elapsed,\'tokens\':tokens,\n       \'files\':[file_record(p),file_record(root/\'pilot-shards.npy\')]+[file_record(root/\'pilot-tokens\'/n,\'pilot-tokens/\'+n)for n in FILES]}\n write_json(receipt_path,info)\n return info,np.load(p,allow_pickle=False)\n\ndef prepare_cache(root,inputs,code_sha,workers=2,smoke=False,specs=None,download_fn=download_raw,\n                  pilot_rows=None,min_rows=None,eta_limit_minutes=60,stop_after_shards=None):\n root,inputs=Path(root),Path(inputs);root.mkdir(parents=True,exist_ok=True)\n specs=specs or P.SHARDS\n if not smoke:assert specs==P.SHARDS and pilot_rows in (None,50000)and min_rows in (None,7000000)\n pilot_rows=pilot_rows or (4 if smoke else 50000);min_rows=min_rows or (4 if smoke else 7000000)\n protected,_=load_protected(inputs/\'protected-questions.json\',strict=not smoke)\n tokenizer=Tokenizer.from_file(str(inputs/\'tokenizer.json\'))\n if not smoke:assert sha_file(inputs/\'tokenizer.json\')==P.TOKENIZER_SHA and tokenizer.get_vocab_size()==8189\n identity={\'protocol\':P.CACHE_PROTOCOL,\'code_sha256\':code_sha,\'tokenizer_sha256\':sha_file(inputs/\'tokenizer.json\'),\n           \'protected_sha256\':sha_file(inputs/\'protected-questions.json\'),\'specs\':specs,\'smoke\':smoke}\n identity_sha=json_hash(identity);master=root/\'cache-manifest.json\';old=read_json(master)if master.exists()else None\n if old:\n  assert old[\'identity_sha256\']==identity_sha,\'Cache input/protocol changed; use another directory\'\n  if old.get(\'sealed\'):\n   for e in old[\'shards\']:validate_shard(root/e[\'folder\'],e,protected)\n   print(\'CACHE ALREADY SEALED; no download/selection/execution/tokenization repeated\',old[\'rows\']);return old\n # All60 files are downloaded even when the time rule chooses a processing subset.\n download_start=time.monotonic();paths=[];raw_counts=[]\n for j,spec in enumerate(specs):\n  p=download_fn(spec,root/\'raw\');assert p.stat().st_size==spec[\'bytes\']and sha_file(p)==spec[\'sha256\']\n  paths.append(p);raw_counts.append(pq.ParquetFile(p).metadata.num_rows)\n  if (j+1)%5==0 or j+1==len(specs):print(\'DOWNLOAD\',j+1,\'/\',len(specs),flush=True)\n download_seconds=(old or {}).get(\'download_seconds\',time.monotonic()-download_start)\n info,pdata=pilot(root,paths,specs,protected,tokenizer,code_sha,workers,pilot_rows)\n estimated=download_seconds+info[\'total_seconds\']*sum(raw_counts)/info[\'raw_scanned\']\n mode=(old or {}).get(\'processing_policy\')or (\'minimum7M_if_ETA_over60min\'if estimated>eta_limit_minutes*60 else \'all60\')\n print(\'PILOT\',info[\'eligible_rows\'],\'eligible;\',round(info[\'eligible_rows_per_second\'],1),\'rows/s; full ETA\',round(estimated/60,2),\'min; POLICY\',mode,flush=True)\n manifest={\'format\':\'nlp-hw02-token-cache-v1\',\'experiment\':26,\'identity\':identity,\'identity_sha256\':identity_sha,\n  \'downloaded_shards\':len(specs),\'downloaded_raw_rows\':sum(raw_counts),\'download_seconds\':download_seconds,\n  \'pilot\':info,\'estimated_full_minutes\':estimated/60,\'processing_policy\':mode,\'minimum_rows\':min_rows,\n  \'rows\':0,\'shards\':[],\'sealed\':False,\'fresh_evaluations\':0,\'train_protected_question_overlaps\':0}\n shard_ids=pdata[\'shard\'];row_ids=pdata[\'row\'];status=pdata[\'status\'];names=info[\'status_names\']\n elapsed_processing=info[\'total_seconds\']\n pa={name:np.load(root/\'pilot-tokens\'/name,mmap_mode=\'r\',allow_pickle=False)for name in FILES}\n ps=np.load(root/\'pilot-shards.npy\',allow_pickle=False)\n reuse={\'indices\':{(int(s),int(r)):i for i,(s,r)in enumerate(zip(ps,pa[\'provenance-rows.npy\']))},\n        \'source\':pa[\'source.npy\'],\'target\':pa[\'target.npy\'],\'so\':pa[\'source-offsets.npy\'],\'to\':pa[\'target-offsets.npy\']}\n for spec,path,raw_n in zip(specs,paths,raw_counts):\n  folder=f\'shards/part-{spec["shard"]:05d}\';out=root/folder;rp=out/\'manifest.json\'\n  if rp.exists():\n   receipt=read_json(rp);assert receipt[\'identity_sha256\']==identity_sha and receipt[\'raw_sha256\']==spec[\'sha256\']\n   validate_shard(out,receipt,protected);print(\'REUSE SHARD\',spec[\'shard\'],receipt[\'tokens\'][\'rows\'],flush=True)\n  else:\n   started=time.monotonic();rows,ids,counts=collect_candidates(path,protected)\n   assert counts[\'raw_rows\']==raw_n\n   cached={int(r):names[int(s)]for r,s in zip(row_ids[shard_ids==spec[\'shard\']],status[shard_ids==spec[\'shard\']])}\n   missing=[i for i,loc in enumerate(ids)if loc not in cached]\n   execution=execute_journal([rows[i][\'code\']for i in missing],out,identity_sha,workers)\n   statuses=[None]*len(rows)\n   for i,loc in enumerate(ids):\n    if loc in cached:statuses[i]=cached[loc]\n   for i,e in zip(missing,execution[\'rows\']):statuses[i]=e[\'status\']\n   keep=[i for i,s in enumerate(statuses)if s==\'numeric\']\n   selected=[rows[i]for i in keep];selected_ids=[ids[i]for i in keep]\n   assert all(classify(r,protected)is None for r in selected)\n   encoding_start=time.monotonic();tok=build_tokens(selected,selected_ids,tokenizer,out,reuse,spec[\'shard\'],identity=json_hash({\'cache\':identity_sha,\'raw_sha\':spec[\'sha256\'],\'rows\':selected_ids}))\n   receipt={\'shard\':spec[\'shard\'],\'folder\':folder,\'identity_sha256\':identity_sha,\'raw_sha256\':spec[\'sha256\'],\n    \'raw_num_rows\':raw_n,\'selection_counts\':counts,\'pre_execution_rows\':len(rows),\'reference_statuses\':dict(Counter(statuses)),\n    \'reference_non_numeric_dropped\':len(rows)-len(keep),\'reference_drop_fraction\':1-len(keep)/len(rows)if rows else 0.,\n    \'pilot_reference_results_reused\':len(cached),\'execution\':execution[\'summary\'],\'tokens\':tok,\n    \'encoding_seconds\':time.monotonic()-encoding_start,\'wall_seconds\':time.monotonic()-started,\n    \'protected_question_overlaps\':0,\'hash_heldout_overlaps\':0,\'files\':[file_record(out/n)for n in FILES]}\n   validate_shard(out,receipt,protected);write_json(rp,receipt)\n   import shutil\n   if (out/\'token-work\').exists():shutil.rmtree(out/\'token-work\')  # Only committed temporary binaries.\n  manifest[\'shards\'].append(receipt);manifest[\'rows\']+=receipt[\'tokens\'][\'rows\'];elapsed_processing+=receipt[\'wall_seconds\']\n  manifest[\'processed_shards\']=len(manifest[\'shards\']);manifest[\'processed_raw_rows\']=sum(r[\'raw_num_rows\']for r in manifest[\'shards\'])\n  manifest[\'reference_drop_fraction\']=sum(r[\'reference_non_numeric_dropped\']for r in manifest[\'shards\'])/sum(r[\'pre_execution_rows\']for r in manifest[\'shards\'])\n  revised=download_seconds+elapsed_processing*sum(raw_counts)/manifest[\'processed_raw_rows\']\n  if revised>eta_limit_minutes*60:manifest[\'processing_policy\']=\'minimum7M_if_ETA_over60min\'\n  manifest[\'revised_full_ETA_minutes\']=revised/60;write_json(master,manifest)\n  print(\'CACHE SHARD\',spec[\'shard\'],\'rows\',receipt[\'tokens\'][\'rows\'],\'TOTAL\',manifest[\'rows\'],\n    \'drop\',round(100*manifest[\'reference_drop_fraction\'],2),\'%; revised ETA\',round(revised/60,2),\'min\',flush=True)\n  if stop_after_shards and len(manifest[\'shards\'])>=stop_after_shards:return manifest\n  if manifest[\'processing_policy\']==\'minimum7M_if_ETA_over60min\'and manifest[\'rows\']>=min_rows:break\n if manifest[\'rows\']<min_rows:raise ValueError(\'Not enough filtered rows; GPU training must not start\')\n manifest[\'sealed\']=True;manifest[\'status\']=\'ready_all60\'if len(manifest[\'shards\'])==len(specs)else \'ready_subset_minimum7M\'\n manifest[\'source_tokens\']=sum(r[\'tokens\'][\'source_tokens\']for r in manifest[\'shards\'])\n manifest[\'target_tokens_with_EOS\']=sum(r[\'tokens\'][\'target_tokens_with_EOS\']for r in manifest[\'shards\'])\n manifest[\'unique_row_identity_policy\']=\'Each original(shard,row)once; duplicate question texts not deduplicated\'\n write_json(master,manifest);print(\'CACHE READY\',manifest[\'status\'],manifest[\'rows\'],\'rows;\',len(manifest[\'shards\']),\'processed /\',len(specs),\'downloaded\')\n return manifest\n\nclass TokenCache:\n def __init__(self,root,protected=None,require_minimum=True):\n  self.root=Path(root);self.manifest=read_json(self.root/\'cache-manifest.json\')\n  assert self.manifest[\'sealed\']and self.manifest[\'fresh_evaluations\']==0\n  if require_minimum:assert self.manifest[\'rows\']>=7000000 and self.manifest[\'downloaded_shards\']==60 and not self.manifest[\'identity\'][\'smoke\']\n  self.parts=[];starts=[0];sl=[];tl=[]\n  for receipt in self.manifest[\'shards\']:\n   folder=self.root/receipt[\'folder\'];n=validate_shard(folder,receipt,protected)\n   arrays={name:np.load(folder/name,mmap_mode=\'r\',allow_pickle=False)for name in FILES}\n   self.parts.append(arrays);starts.append(starts[-1]+n)\n   sl.append(np.diff(arrays[\'source-offsets.npy\']).astype(np.uint16));tl.append(np.diff(arrays[\'target-offsets.npy\']).astype(np.uint16))\n  self.starts=np.asarray(starts,dtype=np.int64);self.source_lengths=np.concatenate(sl);self.target_lengths=np.concatenate(tl)\n  assert len(self)==self.manifest[\'rows\'];self.manifest_sha=sha_file(self.root/\'cache-manifest.json\')\n def __len__(self):return int(self.starts[-1])\n def __getitem__(self,index):\n  index=int(index)\n  if not 0<=index<len(self):raise IndexError(index)\n  shard=int(np.searchsorted(self.starts,index,side=\'right\')-1);j=index-int(self.starts[shard]);a=self.parts[shard]\n  so,to=a[\'source-offsets.npy\'],a[\'target-offsets.npy\']\n  return a[\'source.npy\'][so[j]:so[j+1]],a[\'target.npy\'][to[j]:to[j+1]]\n def provenance(self,index):\n  shard=int(np.searchsorted(self.starts,int(index),side=\'right\')-1);j=int(index)-int(self.starts[shard])\n  return self.manifest[\'shards\'][shard][\'shard\'],int(self.parts[shard][\'provenance-rows.npy\'][j])\n\n\ndef copy_cache(drive_root,local_root,code_sha,protected):\n from exp26_common import copy_verified\n drive_root,local_root=Path(drive_root),Path(local_root);m=read_json(drive_root/\'cache-manifest.json\')\n assert m[\'sealed\']and m[\'rows\']>=7000000 and m[\'identity\'][\'code_sha256\']==code_sha and m[\'identity\'][\'tokenizer_sha256\']==P.TOKENIZER_SHA and m[\'identity\'][\'protected_sha256\']==P.PROTECTED_SHA\n for j,r in enumerate(m[\'shards\']):\n  for e in r[\'files\']:copy_verified(drive_root/r[\'folder\']/e[\'file\'],local_root/r[\'folder\']/e[\'file\'],e[\'sha256\'])\n  copy_verified(drive_root/r[\'folder\']/\'manifest.json\',local_root/r[\'folder\']/\'manifest.json\')\n  print(\'CACHE COPY\',j+1,\'/\',len(m[\'shards\']),flush=True)\n copy_verified(drive_root/\'cache-manifest.json\',local_root/\'cache-manifest.json\',overwrite=True)\n return TokenCache(local_root,protected)\n', 'exp26_common.py': '"""Atomic local/Drive files; immutable SHA receipts."""\nimport hashlib,json,os,shutil,uuid\nfrom pathlib import Path\n\ndef sha_file(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\')as f:\n  for block in iter(lambda:f.read(4*1024**2),b\'\'):h.update(block)\n return h.hexdigest()\n\ndef json_hash(obj):return hashlib.sha256(json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\ndef read_json(path):return json.loads(Path(path).read_text())\ndef write_json(path,obj):\n p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+\'.tmp-\'+uuid.uuid4().hex[:8])\n with tmp.open(\'w\')as f:json.dump(obj,f,ensure_ascii=False,indent=2);f.write(\'\\n\');f.flush();os.fsync(f.fileno())\n tmp.replace(p)\n\ndef copy_verified(source,target,expected=None,overwrite=False):\n source,target=Path(source),Path(target);expected=expected or sha_file(source)\n if sha_file(source)!=expected:raise ValueError(f\'Source SHA mismatch: {source.name}\')\n if target.exists():\n  if sha_file(target)==expected:return expected\n  if not overwrite:raise ValueError(f\'Preserve conflicting file: {target.name}\')\n target.parent.mkdir(parents=True,exist_ok=True);tmp=target.with_name(target.name+\'.tmp-\'+uuid.uuid4().hex[:8])\n shutil.copyfile(source,tmp)\n if sha_file(tmp)!=expected:raise ValueError(\'Copy SHA mismatch\')\n tmp.replace(target)\n if sha_file(target)!=expected:raise ValueError(\'Destination SHA mismatch\')\n return expected\n\ndef file_record(path,relative=None):\n p=Path(path);return {\'file\':relative or p.name,\'bytes\':p.stat().st_size,\'sha256\':sha_file(p)}\n\ndef verify_record(root,row):\n from pathlib import PurePosixPath\n name=row[\'file\'];p=PurePosixPath(name)\n if p.is_absolute()or \'..\'in p.parts or \'\\\\\'in name:raise ValueError(\'Unsafe manifest path\')\n target=Path(root)/name\n if not target.is_file()or target.stat().st_size!=row[\'bytes\']or sha_file(target)!=row[\'sha256\']:raise ValueError(f\'File SHA/size mismatch: {name}\')\n return target\n', 'exp26_core.py': 'import ast,contextlib,gc,hashlib,json,math,random,time\n\nfrom array import array\n\nfrom pathlib import Path\n\nfrom collections import Counter\n\nimport torch\n\nfrom torch import nn\n\nfrom torch.nn import functional as F\n\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\n\nfrom torch.nn.utils.rnn import pad_sequence\n\nfrom tokenizers import Tokenizer\n\nVOCAB=8192\n\nMAX_POSITIONS=2048\n\nGENERATION_CAP=512\n\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError(\'Position overflow\')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\ndef sync(device):\n if device.type==\'cuda\':torch.cuda.synchronize(device)\n\ndef init_tokenizer(path):\n tokenizer=Tokenizer.from_file(str(path))\n if tokenizer.get_vocab_size()!=VOCAB-TOKEN_OFFSET:raise ValueError(\'Unexpected vocabulary\')\n visible=list(range(33,127))+list(range(161,173))+list(range(174,256));bchars={b:chr(b)for b in visible}\n for i,b in enumerate(b for b in range(256)if b not in bchars):bchars[b]=chr(256+i)\n charbytes={c:b for b,c in bchars.items()}\n tokenbytes=[b\'\',b\'\',b\'\']+[bytes(charbytes[c]for c in tokenizer.id_to_token(i))for i in range(VOCAB-3)]\n return tokenizer,tokenbytes\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b\'\'.join(chunks)\n\ndef source_batch(tokenizer,questions,device):\n return pad_sequence([torch.tensor([BOS]+encode(tokenizer,q)+[EOS])for q in questions],batch_first=True,padding_value=PAD).to(device)\n\ndef collate_cache(cache,indices,device):\n src=pad_sequence([torch.tensor(cache[i][0],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n tgt=pad_sequence([torch.tensor([BOS]+list(cache[i][1]),dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n labels=pad_sequence([torch.tensor(list(cache[i][1])+[EOS],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n counts={\'source_tokens\':sum(len(cache[i][0])for i in indices),\'target_tokens\':sum(len(cache[i][1])+1 for i in indices),\n  \'source_slots\':src.numel(),\'target_slots\':labels.numel(),\'pairs\':len(indices)}\n return src.to(device),tgt.to(device),labels.to(device),counts\n\ndef bucket_plan(cache,batch,steps,seed=230042,chunk_pairs=4096):\n if chunk_pairs%batch:raise ValueError(\'Chunk must be divisible by batch\')\n rng=random.Random(seed);plans=[]\n while len(plans)<steps:\n  chunk=rng.sample(range(len(cache)),min(chunk_pairs,len(cache)))\n  chunk.sort(key=lambda i:(len(cache[i][0])+len(cache[i][1])+1,len(cache[i][0]),len(cache[i][1])))\n  batches=[chunk[i:i+batch]for i in range(0,len(chunk)-batch+1,batch)]\n  if not batches:raise ValueError(\'Too few data for this batch size\')\n  rng.shuffle(batches);plans.extend(batches)\n return plans[:steps]\n\ndef sdpa_flags():\n return {k:bool(getattr(torch.backends.cuda,k+\'_sdp_enabled\')())for k in [\'flash\',\'mem_efficient\',\'math\',\'cudnn\']if hasattr(torch.backends.cuda,k+\'_sdp_enabled\')}\n\ndef profile_sdpa(call,device):\n if device.type!=\'cuda\':return {\'status\':\'CPU_smoke\',\'operators\':[]}\n with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA])as prof:\n  call();sync(device)\n ops=sorted({e.key for e in prof.key_averages()if \'scaled_dot_product\'in e.key})\n return {\'status\':\'observed\',\'operators\':ops,\'scope\':\'one inference preflight outside timers; other shapes/backward may use a different backend\'}\n\ndef fused_adamw_available(device):\n if device.type!=\'cuda\':return False\n try:\n  p=nn.Parameter(torch.ones(1,device=device));opt=torch.optim.AdamW([p],lr=1e-4,fused=True);p.grad=torch.ones_like(p);opt.step();sync(device)\n  del opt,p;return True\n except (TypeError,RuntimeError):return False\n\nclass CachedDecoder:\n    @torch.inference_mode()\n    def __init__(self, model, src, capacity):\n        model.eval()\n        if not 0 < capacity <= model.positions.num_embeddings:\n            raise ValueError(\'Invalid cache capacity\')\n        self.model, self.capacity, self.position = model, capacity, 0\n        self.batch_size = len(src)\n        self.width = model.tokens.embedding_dim\n        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads\n        self.head_dim = self.width // self.heads\n        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))\n        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).\n        self.source_allowed = src.ne(PAD)[:, None, None, :]\n        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)\n        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))\n        self.layers = []\n        for layer in model.transformer.decoder.layers:\n            if not layer.norm_first:\n                raise ValueError(\'This experiment requires Pre-LN decoder layers\')\n            for attention in (layer.self_attn, layer.multihead_attn):\n                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads\n                        or attention.bias_k is not None or attention.bias_v is not None\n                        or attention.add_zero_attn):\n                    raise ValueError(\'Unsupported attention configuration\')\n            attention = layer.multihead_attn\n            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],\n                                    attention.in_proj_bias[self.width:])\n            cross_k, cross_v = projected_kv.chunk(2, dim=-1)\n            shape = (len(src), self.heads, capacity, self.head_dim)\n            self.layers.append({\n                \'self_k\': memory.new_empty(shape), \'self_v\': memory.new_empty(shape),\n                \'cross_k\': self.split_heads(cross_k), \'cross_v\': self.split_heads(cross_v),\n            })\n\n    def split_heads(self, values):\n        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)\n\n    def attend(self, attention, q, k, v, allowed):\n        # One query at the latest position; all supplied keys are from its past/current.\n        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.\n        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,\n                                                dropout_p=0.0, is_causal=False)\n        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)\n        return attention.out_proj(output)\n\n    @torch.inference_mode()\n    def step(self, token_ids):\n        if self.position >= self.capacity:\n            raise ValueError(\'Cache capacity exceeded\')\n        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:\n            raise ValueError(\'One token per batch row is required\')\n        t = self.position\n        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)\n        x = x + self.position_vectors[t][None, None, :]\n        self.valid[:, t] = token_ids.ne(PAD)\n        self_allowed = self.valid[:, None, None, :t + 1]\n        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):\n            attention = layer.self_attn\n            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)\n            cache[\'self_k\'][:, :, t:t + 1] = self.split_heads(k)\n            cache[\'self_v\'][:, :, t:t + 1] = self.split_heads(v)\n            self_output = self.attend(attention, self.split_heads(q),\n                cache[\'self_k\'][:, :, :t + 1], cache[\'self_v\'][:, :, :t + 1], self_allowed)\n            x = x + layer.dropout1(self_output)\n\n            attention = layer.multihead_attn\n            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])\n            cross_output = self.attend(attention, self.split_heads(q),\n                                      cache[\'cross_k\'], cache[\'cross_v\'], self.source_allowed)\n            x = x + layer.dropout2(cross_output)\n            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))\n            x = x + layer.dropout3(feed_forward)\n\n        if self.model.transformer.decoder.norm is not None:\n            x = self.model.transformer.decoder.norm(x)\n        self.position += 1\n        return self.model.head(x[:, 0])\n\ndef code_properties(code):\n    def scoped_nodes(node):\n        stack = [node]\n        while stack:\n            current = stack.pop()\n            yield current\n            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):\n                stack.extend(ast.iter_child_nodes(current))\n    try:\n        tree = ast.parse(code)\n        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'solve\']\n        has_return = any(isinstance(n, ast.Return) and n.value is not None\n            for f in functions for stmt in f.body for n in scoped_nodes(stmt))\n        return {\'parses_as_python\': True, \'defines_solve\': bool(functions), \'solve_returns_value\': has_return,\n                \'syntax_error_type\': None, \'syntax_error_message\': None, \'indentation_error\': False}\n    except (SyntaxError, ValueError, RecursionError) as error:\n        return {\'parses_as_python\': False, \'defines_solve\': False, \'solve_returns_value\': False,\n                \'syntax_error_type\': type(error).__name__, \'syntax_error_message\': str(error),\n                \'indentation_error\': isinstance(error, IndentationError)}\n\nclass BudgetExpired(Exception):pass\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired(\'Session budget reached; completed run boundaries are preserved\')\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n@torch.inference_mode()\ndef generate_preallocated(model,src,cap=GENERATION_CAP,deadline=None,check_every=16):\n """Only output storage and termination checks change; same encoder/KV/argmax."""\n cache=CachedDecoder(model,src,cap)\n output=torch.full((len(src),cap),PAD,device=src.device,dtype=torch.long)\n token=torch.full((len(src),),BOS,device=src.device,dtype=torch.long)\n done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n for step in range(cap):\n  if step%check_every==0:check_deadline(deadline)\n  logits=cache.step(token);logits[:,[PAD,BOS]]=-torch.inf\n  token=logits.argmax(-1).masked_fill(done,PAD);output[:,step]=token;done|=token.eq(EOS)\n  # Only one device->CPU synchronization per 16 steps, plus final output transfer.\n  # Rows already at EOS remain PAD; active_tokens removes these extra slots.\n  if (step+1)%check_every==0 and bool(done.all()):break\n return output[:,:cache.position],cache.position\n\ndef restore_order(ordered_values,original_indices,size):\n if len(ordered_values)!=size or sorted(original_indices)!=list(range(size)):\n  raise ValueError(\'Not a complete permutation\')\n restored=[None]*size\n for value,i in zip(ordered_values,original_indices):restored[i]=value\n return restored\n\ndef inspect_output(row,ids,tokenizer,tokenbytes):\n active=active_tokens(ids);raw=decode_bytes(tokenbytes,active)\n try:code=raw.decode(\'utf-8\');valid=True\n except UnicodeDecodeError:code=raw.decode(\'utf-8\',errors=\'replace\');valid=False\n example={\'question\':row[\'question\'],\'generated_code\':code,\'generated_token_ids\':active,\n  \'generated_tokens_including_eos\':len(active),\'generated_utf8_bytes\':len(raw),\'valid_utf8\':valid,\n  \'eos_emitted\':EOS in active,\'reached_token_cap\':len(active)==GENERATION_CAP,\n  \'token_cap_without_eos\':len(active)==GENERATION_CAP and EOS not in active,\n  \'byte_budget_stop\':False,\'stop_reason\':\'eos\'if EOS in active else\'token_cap\',**code_properties(code)}\n if \'code\'in row:\n  gold=encode(tokenizer,row[\'code\'])+[EOS]\n  example.update(reference_code=row[\'code\'],reference_utf8_bytes=len(row[\'code\'].encode()),\n   reference_bpe_tokens_including_eos=len(gold),exact_code_match=valid and code==row[\'code\'],exact_tokens_with_eos=active==gold)\n return example\n\ndef sha_file(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\')as f:\n  for chunk in iter(lambda:f.read(1024**2),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef write_json(path,value):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+\'.tmp\')\n tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\n\nMODEL_SPEC=dict(d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680)\n', 'exp26_eval.py': '"""Identical25 model/greedy/CE/exec math; new point names and paired comparisons."""\nimport gc,hashlib,json,math,time\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport exp26_core as core\nimport exp26_exec as numeric\nimport exp26_protocol as P\nfrom exp26_common import json_hash,write_json,read_json\n\ndef state_digest(state):\n    h=hashlib.sha256()\n    for k,v in sorted(state.items()):\n        a=v.detach().cpu().contiguous().numpy();h.update(k.encode());h.update(str(a.shape).encode());h.update(a.tobytes())\n    return h.hexdigest()\n\ndef load_model(path, device, dropout=.1, scaled_tokens=False, spec=None):\n    model=core.SimpleTransformer(**(spec or core.MODEL_SPEC),dropout=dropout)\n    payload=torch.load(path,map_location=\'cpu\',weights_only=True)\n    state=payload[\'model\'] if payload.get(\'format\')in (\'nlp-hw02-training-v1\',\'nlp-hw02-wsd-v1\')else payload\n    model.load_state_dict(state,strict=True)\n    del payload\n    if scaled_tokens:\n        with torch.no_grad():model.tokens.weight.div_(math.sqrt(model.width))\n    del state\n    return model.to(device)\n\n@torch.inference_mode()\ndef ce_one(model_path,rows,tokenizer,device,spec=None,batch=32):\n    started=time.perf_counter();model=load_model(model_path,device,spec=spec).eval();total=torch.zeros((),device=device);denominator=0\n    cached=[([core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS],core.encode(tokenizer,r[\'code\'])) for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:len(cached[i][0])+len(cached[i][1]))\n    try:\n        for lo in range(0,len(order),batch):\n            src,tgt,labels,counts=core.collate_cache(cached,order[lo:lo+batch],device)\n            logits=model(src,tgt)\n            total+=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD,reduction=\'sum\')\n            denominator+=counts[\'target_tokens\']\n            del src,tgt,labels,logits\n        value=float(total.cpu())/denominator\n        if not math.isfinite(value):raise FloatingPointError(\'CE nonfinite\')\n        return {\'CE\':value,\'useful_target_tokens\':denominator,\'rows\':len(rows),\'precision\':\'FP32\',\'batch\':batch,\n                \'seconds\':time.perf_counter()-started}\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\n@torch.inference_mode()\ndef generate_one(model_path,rows,tokenizer_path,device,outdir,name,spec=None,batch=512,cap=512,half=True):\n    """Full timer begins before loading model tensors; ends at answers-file close."""\n    gc.collect()\n    if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n    start=time.perf_counter();model=load_model(model_path,device,spec=spec).eval()\n    tokenizer,tokenbytes=core.init_tokenizer(tokenizer_path)\n    if half:model.half()\n    try:\n        core.generate_preallocated(model,core.source_batch(tokenizer,[rows[0][\'question\']],device),cap=min(8,cap))\n        core.sync(device);generation_start=time.perf_counter()\n        encoded=[[core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS] for r in rows]\n        order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));ordered=[];steps_sum=row_steps_sum=0\n        for lo in range(0,len(order),batch):\n            indices=order[lo:lo+batch]\n            src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=core.PAD).to(device)\n            tokens,steps=core.generate_preallocated(model,src,cap=cap)\n            steps_sum+=steps;row_steps_sum+=steps*len(indices)\n            for i,ids in zip(indices,tokens.cpu().tolist()):\n                active=core.active_tokens(ids)\n                ordered.append({\'id\':i,\'question\':rows[i][\'question\'],\'generated_token_ids\':active,\n                    \'generated_code\':core.decode_bytes(tokenbytes,active).decode(\'utf-8\',errors=\'replace\'),\n                    \'eos_emitted\':core.EOS in active})\n        examples=core.restore_order(ordered,order,len(rows));core.sync(device)\n        generation_seconds=time.perf_counter()-generation_start\n        outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n        answers=outdir/f\'{name}-answers.jsonl\'\n        with answers.open(\'w\',encoding=\'utf-8\') as f:\n            for e in examples:f.write(json.dumps({\'id\':e[\'id\'],\'question\':e[\'question\'],\'code\':e[\'generated_code\']},ensure_ascii=False)+\'\\n\')\n        full_seconds=time.perf_counter()-start\n        core.write_json(outdir/f\'{name}-generations.json\',examples)\n        timing={\'full_seconds\':full_seconds,\'generation_seconds\':generation_seconds,\'rows\':len(rows),\n                \'decode_steps_sum\':steps_sum,\'decode_row_steps_sum\':row_steps_sum,\n                \'answers_sha256\':core.sha_file(answers),\n                \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None,\n                \'timer\':\'model state load,restore,tokenizer,FP16/H2D,warmup,sorted greedy/order restore/answers write; download/cache/exec/CE/JSON dump outside\'}\n        timing[\'model_sha256\']=core.sha_file(model_path)\n        timing[\'tokenizer_sha256\']=core.sha_file(tokenizer_path)\n        return examples,timing\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef paired_mcnemar(score,control):\n    assert score[\'eligible_indices\']==control[\'eligible_indices\']\n    s,c=set(score[\'match_indices\']),set(control[\'match_indices\']);g,l=len(s-c),len(c-s);n=g+l\n    p=min(1.,(2*sum(math.comb(n,i) for i in range(min(g,l)+1)))/(1<<n)) if n else 1.\n    return {\'gains\':g,\'losses\':l,\'discordant\':n,\'exact_two_sided_p\':p,\n            \'delta_percentage_points\':100*(g-l)/score[\'denominator\']}\n\ndef length_accuracy(rows,tokenizer,score):\n    lengths=[len(core.encode(tokenizer,r[\'code\'])) for r in rows];eligible=set(score[\'eligible_indices\']);matched=set(score[\'match_indices\'])\n    groups={k:[] for k in [\'<60\',\'60–90\',\'90–120\',\'≥120\']}\n    for i,n in enumerate(lengths):\n        if i in eligible:groups[\'<60\' if n<60 else \'60–90\' if n<90 else \'90–120\' if n<120 else \'≥120\'].append(i)\n    return {k:{\'denominator\':len(ids),\'matches\':sum(i in matched for i in ids),\n               \'accuracy\':sum(i in matched for i in ids)/len(ids) if ids else None} for k,ids in groups.items()}\n\ndef evaluate_point(model_path,sets,refs,tokenizer_path,device,out,point,D0=None,previous=None,spec=None,smoke=False):\n out=Path(out);out.mkdir(parents=True,exist_ok=True);digest=core.sha_file(model_path)\n identity={\'model_sha256\':digest,\'protocol_sha256\':json_hash(P.TRAIN_PROTOCOL),\'tokenizer_sha256\':core.sha_file(tokenizer_path),\'sets_sha256\':json_hash(sets),\'torch\':str(torch.__version__),\'CUDA\':torch.version.cuda,\'smoke\':smoke}\n receipt=out/\'metrics.json\'\n if receipt.exists():\n  m=read_json(receipt);assert m[\'identity\']==identity;return m\n tokenizer,_=core.init_tokenizer(tokenizer_path);m={\'point\':point,\'identity\':identity,\'splits\':{}}\n for split,rows in sets.items():\n  gen_path=out/(split+\'-generations.json\');time_path=out/(split+\'-generation-time.json\')\n  if gen_path.exists()and time_path.exists():\n   gens=read_json(gen_path);timing=read_json(time_path)\n   assert len(gens)==len(rows)and [g[\'question\']for g in gens]==[r[\'question\']for r in rows]\n   assert timing[\'model_sha256\']==digest and timing[\'tokenizer_sha256\']==identity[\'tokenizer_sha256\']\n   assert timing[\'answers_sha256\']==core.sha_file(out/(split+\'-answers.jsonl\'))\n  else:\n   gens,timing=generate_one(model_path,rows,tokenizer_path,device,out,split,spec=spec,batch=2 if smoke else 512,cap=8 if smoke else 512,half=True)\n   write_json(time_path,timing)\n  execution=numeric.evaluate_programs([g[\'generated_code\']for g in gens]);assert execution[\'summary\'][\'complete\']\n  write_json(out/(split+\'-execution.json\'),execution);score=numeric.compare_exec(refs[split],execution)\n  if not smoke:assert score[\'denominator\']==(4096 if split==\'heldout4k\'else 991)\n  ce=ce_one(model_path,rows,tokenizer,device,spec=spec,batch=2 if smoke else 32)\n  row={\'exec\':score,\'CE\':ce,\'length_accuracy\':length_accuracy(rows,tokenizer,score),\'generation_time\':timing}\n  if D0 is not None:row[\'McNemar_against_D0\']=paired_mcnemar(score,D0[\'splits\'][split][\'exec\'])\n  if previous is not None:row[\'McNemar_against_previous\']=paired_mcnemar(score,previous[\'splits\'][split][\'exec\'])\n  m[\'splits\'][split]=row;write_json(out/\'evaluation-progress.json\',m)\n write_json(receipt,m);return m\n\ndef verify_D0(m):\n expected={\'heldout4k\':(1677,.4764168630129111),\'old1024\':(422,.4863326656808216)}\n for split,(matches,ce)in expected.items():\n  s=m[\'splits\'][split];assert abs(s[\'exec\'][\'matches\']-matches)<=3 and abs(s[\'CE\'][\'CE\']-ce)<=1e-4,\'D0 reproduction failed; stop before training\'\n return {\'status\':\'passed\',\'expected4k\':1677,\'expected991\':422,\'match_tolerance\':3,\'CE_abs_tolerance\':1e-4}\n', 'exp26_exec.py': '"""Bounded numeric-program execution for this public dataset; no arbitrary imports/I/O."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef execute_task(payload):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    tree,status=filter_program(code)\n    if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:signal.setitimer(signal.ITIMER_REAL,0)\n\ndef create_pool(workers,memory_bytes):\n    return mp.get_context(\'spawn\').Pool(workers,initializer=worker_init,\n        initargs=(memory_bytes,),maxtasksperchild=256)\n\ndef evaluate_programs(codes,workers=2,wall_limit=1.0,cpu_limit=2.0,\n                      parent_timeout=3.0,memory_bytes=768*1024**2,deadline=None,progress=None):\n    """Bounded inflight tasks; every async result has a parent deadline.\n    A killed worker loses its task even if Pool replaces it: terminate/recreate\n    the WHOLE pool, retain completed results and retry collateral pending tasks.\n    Pool.map is deliberately never used. Output order and duplicates preserved.\n    """\n    if parent_timeout<=wall_limit:raise ValueError(\'Parent deadline must exceed program deadline\')\n    unique=list(dict.fromkeys(codes));todo=deque(range(len(unique)));answers={};pending={}\n    pool=None;restarts=0;start=time.monotonic();cancelled=False;last_progress=-1\n    try:\n        while todo or pending:\n            if deadline is not None and time.monotonic()>=deadline:\n                cancelled=True;break\n            if pool is None:pool=create_pool(workers,memory_bytes)\n            while todo and len(pending)<workers:\n                i=todo.popleft();tree,status=filter_program(unique[i])\n                if status:answers[i]={\'status\':status};continue\n                pending[i]=(pool.apply_async(execute_task,((unique[i],wall_limit,cpu_limit),)),time.monotonic())\n            for i,(future,submitted)in list(pending.items()):\n                if future.ready():\n                    try:answers[i]=future.get(timeout=0)\n                    except Exception as e:answers[i]={\'status\':\'worker_error\',\'error_type\':type(e).__name__}\n                    del pending[i]\n            timed=[i for i,(_,submitted)in pending.items()if time.monotonic()-submitted>=parent_timeout]\n            if timed:\n                for i in timed:answers[i]={\'status\':\'timeout_or_worker_exit\'}\n                retry=[i for i in pending if i not in timed]\n                pool.terminate();pool.join();pool=None;restarts+=1;pending.clear()\n                todo.extendleft(reversed(retry))\n            if progress and len(answers)!=last_progress and (len(answers)%1000==0 or len(answers)==len(unique)):\n                progress(len(answers),len(unique));last_progress=len(answers)\n            if pending:time.sleep(.001)\n        if cancelled:\n            for i in todo:answers[i]={\'status\':\'budget_not_evaluated\'}\n            for i in pending:answers[i]={\'status\':\'budget_not_evaluated\'}\n    finally:\n        if pool is not None:pool.terminate();pool.join()\n    by_code={c:answers.get(i,{\'status\':\'budget_not_evaluated\'})for i,c in enumerate(unique)}\n    rows=[dict(by_code[c])for c in codes]\n    return {\'rows\':rows,\'summary\':{\'programs\':len(codes),\'unique_programs\':len(unique),\n        \'statuses\':dict(Counter(r[\'status\']for r in rows)),\'pool_restarts\':restarts,\n        \'seconds\':time.monotonic()-start,\'complete\':not cancelled}}\n\ndef compare_exec(reference_results,generation_results,rel_tol=1e-6,abs_tol=1e-6):\n    refs=reference_results[\'rows\']if isinstance(reference_results,dict)else reference_results\n    gens=generation_results[\'rows\']if isinstance(generation_results,dict)else generation_results\n    if len(refs)!=len(gens):raise ValueError(\'Order/length mismatch\')\n    eligible=[i for i,r in enumerate(refs)if r[\'status\']==\'numeric\']\n    matches=[i for i in eligible if gens[i][\'status\']==\'numeric\'and math.isclose(\n        gens[i][\'value\'],refs[i][\'value\'],rel_tol=rel_tol,abs_tol=abs_tol)]\n    return {\'denominator\':len(eligible),\'matches\':len(matches),\n        \'accuracy\':len(matches)/len(eligible)if eligible else None,\n        \'eligible_indices\':eligible,\'match_indices\':matches,\n        \'reference_statuses\':dict(Counter(r[\'status\']for r in refs)),\n        \'generation_statuses\':dict(Counter(r[\'status\']for r in gens)),\n        \'rel_tol\':rel_tol,\'abs_tol\':abs_tol}\n', 'exp26_fast_exec.py': '"""Reference filter: ~2000 programs per worker task; per-program SIGALRM.\nNo Torch/NumPy import in spawned workers. A dead chunk is retried per-program.\n"""\nimport multiprocessing as mp,time\nfrom collections import Counter,deque\nimport exp26_exec as numeric\n\nMEMORY=768*1024**2\n\ndef execute_chunk(codes,wall_limit,cpu_limit):\n answers={}\n for c in dict.fromkeys(codes):answers[c]=numeric.execute_task((c,wall_limit,cpu_limit))\n return [answers[c]for c in codes]\n\ndef run_chunks(codes,workers=2,chunk_rows=2000,wall_limit=1.,cpu_limit=2.,chunk_timeout=None,\n               kill_first_worker_fixture=False,saved_chunks=None,on_chunk=None,progress=None):\n if not codes:return {\'rows\':[],\'summary\':{\'programs\':0,\'seconds\':0.,\'chunk_retries\':0,\'pool_restarts\':0}}\n chunks=[codes[i:i+chunk_rows]for i in range(0,len(codes),chunk_rows)];done=dict(saved_chunks or {});todo=deque(i for i in range(len(chunks))if i not in done);pending={}\n pool=None;old_pids=set();start=time.monotonic();retries=restarts=0;killed=False\n try:\n  while todo or pending:\n   if pool is None:\n    pool=mp.get_context(\'spawn\').Pool(workers,initializer=numeric.worker_init,initargs=(MEMORY,))\n    old_pids={p.pid for p in pool._pool}\n   while todo and len(pending)<workers:\n    i=todo.popleft();pending[i]=(pool.apply_async(execute_chunk,(chunks[i],wall_limit,cpu_limit)),time.monotonic())\n   if kill_first_worker_fixture and not killed and pending:\n    pool._pool[0].terminate();killed=True\n   for i,(future,_)in list(pending.items()):\n    if future.ready():\n     try:\n      row=future.get(timeout=0)\n     except Exception:continue  # A failed result follows the per-program fallback below.\n     if len(row)!=len(chunks[i]):continue\n     if on_chunk:on_chunk(i,row)  # Drive/journal errors must propagate.\n     done[i]=row;del pending[i]\n     if progress:progress(sum(len(v)for v in done.values()),len(codes),time.monotonic()-start)\n   pids={p.pid for p in pool._pool}\n   death=pids!=old_pids or any(p.exitcode is not None for p in pool._pool)\n   overdue=[i for i,(_,t)in pending.items()if time.monotonic()-t>(chunk_timeout or max(30.,len(chunks[i])*.02+10.))]\n   if pending and (death or overdue or any(f.ready()for f,_ in pending.values())):\n    retry=list(pending);pool.terminate();pool.join();pool=None;pending.clear();restarts+=1\n    # Pool.map never waits for a task lost with its worker. Collateral chunks also\n    # get a fresh, bounded per-task pool and preserve their input order.\n    for i in retry:\n     evaluated=numeric.evaluate_programs(chunks[i],workers=workers,wall_limit=wall_limit,\n       cpu_limit=cpu_limit,parent_timeout=max(3.,wall_limit+2.),memory_bytes=MEMORY)\n     assert evaluated[\'summary\'][\'complete\'];done[i]=evaluated[\'rows\'];retries+=1\n     if on_chunk:on_chunk(i,done[i])\n     if progress:progress(sum(len(v)for v in done.values()),len(codes),time.monotonic()-start)\n   elif death and not pending:\n    pool.terminate();pool.join();pool=None;restarts+=1\n   if pending:time.sleep(.005)\n finally:\n  if pool is not None:pool.terminate();pool.join()\n rows=[r for i in range(len(chunks))for r in done[i]]\n return {\'rows\':rows,\'summary\':{\'programs\':len(codes),\'statuses\':dict(Counter(r[\'status\']for r in rows)),\n   \'seconds\':time.monotonic()-start,\'chunk_retries\':retries,\'pool_restarts\':restarts,\'chunk_rows\':chunk_rows,\n   \'workers\':workers,\'wall_limit\':wall_limit,\'cpu_limit\':cpu_limit}}\n', 'exp26_input.py': '"""Only prepared public inputs; fresh references are absent from these archives."""\nimport hashlib,shutil,zipfile\nfrom pathlib import Path,PurePosixPath\nimport exp26_protocol as P\nfrom exp26_common import sha_file,read_json,copy_verified\n\nA_NAMES={\'tokenizer.json\',\'protected-questions.json\'}\nB_NAMES=set(P.INPUT_FILES)\n\ndef import_starter(archive,out,kind):\n archive,out=Path(archive),Path(out);expected=P.STARTER_A_SHA if kind==\'cache-inputs\'else P.STARTER_B_SHA\n if sha_file(archive)!=expected:raise ValueError(\'Need exact prepared26 starter; not the entire25 results archive\')\n with zipfile.ZipFile(archive)as z:\n  if z.testzip()is not None:raise ValueError(\'Input ZIP CRC mismatch\')\n  import json\n  m=json.loads(z.read(\'manifest.json\'));names=A_NAMES if kind==\'cache-inputs\'else B_NAMES\n  assert m[\'experiment\']==26 and m[\'kind\']==kind and {e[\'file\']for e in m[\'files\']}==names\n  assert set(z.namelist())==names|{\'manifest.json\'}and len(z.namelist())==len(names)+1\n  for row in m[\'files\']:\n   n=row[\'file\'];assert n in names and row[\'sha256\']==P.INPUT_FILES[n]\n   content=z.read(n);assert len(content)==row[\'bytes\']and hashlib.sha256(content).hexdigest()==row[\'sha256\']\n   target=out/n\n   if target.exists()and sha_file(target)!=row[\'sha256\']:raise ValueError(\'Keep conflicting input\')\n  out.mkdir(parents=True,exist_ok=True)\n  for row in m[\'files\']:\n   n=row[\'file\'];target=out/n\n   if not target.exists():\n    tmp=target.with_suffix(target.suffix+\'.tmp\')\n    with z.open(n)as f,tmp.open(\'wb\')as dst:shutil.copyfileobj(f,dst)\n    assert sha_file(tmp)==row[\'sha256\'];tmp.replace(target)\n return {\'kind\':kind,\'archive_sha256\':expected,\'files\':len(names)}\n\ndef load_protected(path,strict=True):\n obj=read_json(path)\n if strict:assert sha_file(path)==P.PROTECTED_SHA\n groups=obj[\'groups\']\n if strict:assert {k:len(v)for k,v in groups.items()}=={\'test\':1919,\'old1024\':1024,\'heldout4k\':4096,\'fresh1024\':1024}\n assert all(isinstance(q,str)for values in groups.values()for q in values)\n return set(q for values in groups.values()for q in values),groups\n\ndef validate_training_inputs(root):\n for name,digest in P.INPUT_FILES.items():assert sha_file(Path(root)/name)==digest,name\n old=read_json(Path(root)/\'old1024.json\');held=read_json(Path(root)/\'heldout4k.json\');test=read_json(Path(root)/\'test.json\')\n protected,groups=load_protected(Path(root)/\'protected-questions.json\')\n assert [r[\'question\']for r in old]==groups[\'old1024\']and [r[\'question\']for r in held]==groups[\'heldout4k\']\n assert [r[\'question\']for r in test]==groups[\'test\']\n assert len(set(groups[\'heldout4k\']))==4096 and not set(groups[\'heldout4k\'])&(set(groups[\'old1024\'])|set(groups[\'test\'])|set(groups[\'fresh1024\']))\n return {\'heldout4k\':held,\'old1024\':old},test,protected\n', 'exp26_order.py': '"""One global seeded permutation; local bucketing remains a bijection."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nimport exp26_protocol as P\nfrom exp26_common import read_json,write_json,json_hash,sha_file,file_record,copy_verified\n\ndef make_order(cache,seed=P.ORDER_SEED,batch=128,chunk=4096):\n assert chunk%batch==0\n rng=np.random.default_rng(seed);order=rng.permutation(len(cache)).astype(np.uint32)\n for lo in range(0,len(order),chunk):\n  part=order[lo:lo+chunk];ns=cache.source_lengths[part].astype(np.int32);nt=cache.target_lengths[part].astype(np.int32)\n  part=part[np.lexsort((nt,ns,ns+nt+1))];full=len(part)//batch\n  if full:part[:full*batch]=part[:full*batch].reshape(full,batch)[rng.permutation(full)].reshape(-1)\n  order[lo:lo+len(part)]=part\n seen=np.zeros(len(cache),dtype=np.bool_);seen[order]=True\n assert len(order)==len(cache)and int(order.max())<len(cache)and seen.all()\n return order\n\ndef prepare_order(cache,drive_root,local_root,smoke=False):\n drive_root,local_root=Path(drive_root),Path(local_root);drive_root.mkdir(parents=True,exist_ok=True);local_root.mkdir(parents=True,exist_ok=True)\n identity={\'cache_manifest_sha256\':cache.manifest_sha,\'rows\':len(cache),\'seed\':P.ORDER_SEED,\'batch\':128,\'chunk\':4096,\'smoke\':smoke}\n rec=drive_root/\'order.json\'\n if rec.exists():\n  meta=read_json(rec);assert meta[\'identity\']==identity\n  assert sha_file(drive_root/\'order.npy\')==meta[\'file\'][\'sha256\']\n else:\n  order=make_order(cache);p=local_root/\'order.npy\';tmp=p.with_suffix(\'.npy.tmp\')\n  with tmp.open(\'wb\')as f:np.save(f,order,allow_pickle=False)\n  tmp.replace(p);copy_verified(p,drive_root/\'order.npy\')\n  meta={\'identity\':identity,\'file\':file_record(drive_root/\'order.npy\'),\'unique_rows\':len(cache),\'duplicates_before_exhaustion\':0,\n        \'semantics\':\'global permutation of all row identities; sort in4096 chunks then shuffle128-row batches; partial tail preserved\'}\n  write_json(rec,meta)\n copy_verified(drive_root/\'order.npy\',local_root/\'order.npy\',meta[\'file\'][\'sha256\'])\n order=np.load(local_root/\'order.npy\',mmap_mode=\'r\',allow_pickle=False)\n assert order.dtype==np.uint32 and len(order)==len(cache)\n return order,meta\n\ndef collate(cache,indices,device):\n import torch\n pairs=[cache[int(i)]for i in indices];ns=[len(s)for s,t in pairs];nt=[len(t)for s,t in pairs]\n src=np.zeros((len(pairs),max(ns)),dtype=np.int64);tgt=np.zeros((len(pairs),max(nt)+1),dtype=np.int64);labels=np.zeros_like(tgt)\n for j,(s,t)in enumerate(pairs):\n  src[j,:len(s)]=s;tgt[j,0]=1;tgt[j,1:len(t)+1]=t;labels[j,:len(t)]=t;labels[j,len(t)]=2\n counts={\'pairs\':len(pairs),\'source_tokens\':sum(ns),\'target_tokens\':sum(n+1 for n in nt),\'source_slots\':src.size,\'target_slots\':tgt.size}\n return torch.from_numpy(src).to(device),torch.from_numpy(tgt).to(device),torch.from_numpy(labels).to(device),counts\n\ndef prefix_identity(order,cursor,rows):\n assert 0<=cursor<=len(order)\n return {\'presentations\':int(cursor),\'unique_rows_seen\':int(cursor),\'unique_fraction_of_cache\':cursor/rows,\n         \'seen_rows_repeated\':0,\'row_indices_prefix_sha256\':hashlib.sha256(np.asarray(order[:cursor],dtype=np.uint32).tobytes()).hexdigest()}\n', 'exp26_protocol.py': '"""Frozen experiment26 protocol and SHA provenance."""\nREVISION = \'b2724689ff079e9c22efd548fe134bfb29036dbb\'\nSEED = 42\nORDER_SEED = 230042\nTOKENIZER_SHA = \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\'\nPROTECTED_SHA = \'7c034c9767334e4c0f1a12a14bf18353bd1d0a9bf175da251fe74473561cabab\'\nINITIAL_SHA = \'daedd90fe51fef95dc151bc28e2f1ed0c66de80e5aa50a9ad8723ec50204c647\'\nINITIAL_TENSORS = \'7e588b942e0aab8e39873e63a6acf7d6446623db17c5be50f297a6bb4b472489\'\nNORMALIZED_INITIAL_TENSORS = \'974f9b6efca5e95e8b4e9269858984d6c70bdf62e354f490931d1b984005e380\'\nD0_MODEL_SHA = \'1ac82d0c62686456024280aa9821a9c71067387b9b6db2911b72def9652f7109\'\nD0_TENSORS = \'994f4a36e2100aeee5d267cbeb8551d5026340138c1615f72236acd100a7c9b8\'\nD0_SOURCE_CHECKPOINT_SHA = \'597451d05c0130b4aef09eb0d504180b8056ca9f1368a0961454310e99ffb4c6\'\nSTARTER_A_SHA = \'67356bfa699c6b701163cb9e0bce0749c2c539b949cfc26d490484e9454fae7f\'\nSTARTER_B_SHA = \'8376b6632faed4c90c83790f7775bb679c1a8bb7419bba85caa49364d0324e35\'\nSHARDS = [{\'shard\': 0, \'path\': \'data/train/part-00000.parquet\', \'bytes\': 26169796, \'sha256\': \'91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec\'}, {\'shard\': 1, \'path\': \'data/train/part-00001.parquet\', \'bytes\': 26166175, \'sha256\': \'5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068\'}, {\'shard\': 2, \'path\': \'data/train/part-00002.parquet\', \'bytes\': 26162244, \'sha256\': \'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de\'}, {\'shard\': 3, \'path\': \'data/train/part-00003.parquet\', \'bytes\': 26167481, \'sha256\': \'b2b63db46b005e229aa5e91f1010adc763ca9f4b806561b328432fbc9e5c11e9\'}, {\'shard\': 4, \'path\': \'data/train/part-00004.parquet\', \'bytes\': 26185069, \'sha256\': \'272ac87a686134a3e3ba6488985879bd8ea567dcaba1e1145aae0cb724a0afe5\'}, {\'shard\': 5, \'path\': \'data/train/part-00005.parquet\', \'bytes\': 26168730, \'sha256\': \'89687321a1df7338e543a7d8c46d7bc0bd6596d855f9d3e43995e2a9cff858b7\'}, {\'shard\': 6, \'path\': \'data/train/part-00006.parquet\', \'bytes\': 26157073, \'sha256\': \'4ecc34f83241a21dd3d9b28dc9477da9bd297dffee0936c490ca9b75a32612aa\'}, {\'shard\': 7, \'path\': \'data/train/part-00007.parquet\', \'bytes\': 26162520, \'sha256\': \'0e8c5310122ba05a7c92d48ca59af0d5a467a58fe57e3e074f8a1768dd37f9cb\'}, {\'shard\': 8, \'path\': \'data/train/part-00008.parquet\', \'bytes\': 26199557, \'sha256\': \'44f6cbd7a6b821c80108ef6dcb5b4b8e136ad6deaaeccca894717ea4e0b2e1d4\'}, {\'shard\': 9, \'path\': \'data/train/part-00009.parquet\', \'bytes\': 26157219, \'sha256\': \'1ef7d1642dd5e5b8e90c7e239214bf0fd301315d5ef45830e663ffa72e7bd028\'}, {\'shard\': 10, \'path\': \'data/train/part-00010.parquet\', \'bytes\': 26164309, \'sha256\': \'41caae2f7236c8839ef30ad915320b69844aad4d01c2d9c8d65a38befa029408\'}, {\'shard\': 11, \'path\': \'data/train/part-00011.parquet\', \'bytes\': 26190841, \'sha256\': \'4a22ce4fcedbd4cf09b53cfe5d0491530f39411d54b407d2965d2e210f1af264\'}, {\'shard\': 12, \'path\': \'data/train/part-00012.parquet\', \'bytes\': 26151272, \'sha256\': \'53f510526897ccc5103722f3a5724340abbf01c26a67d139dff9e9a430b15116\'}, {\'shard\': 13, \'path\': \'data/train/part-00013.parquet\', \'bytes\': 26129342, \'sha256\': \'f4e6d924bbd7a8e711fa9bdf4173e5d0e2aa9db0204861444b8c86ccf61f6916\'}, {\'shard\': 14, \'path\': \'data/train/part-00014.parquet\', \'bytes\': 24278231, \'sha256\': \'87ba111400e3742e80efd106ee5ea236fea2e6d92bba752033108a1d728c14d5\'}, {\'shard\': 15, \'path\': \'data/train/part-00015.parquet\', \'bytes\': 23578713, \'sha256\': \'f9f71f028f59d44418139920615cad652ad91d3671aaec5b4dc5dad29515c315\'}, {\'shard\': 16, \'path\': \'data/train/part-00016.parquet\', \'bytes\': 23580532, \'sha256\': \'204fa2af3be3f09282118cfc29af9417ce05229fe100502d529bf4a16bc8542f\'}, {\'shard\': 17, \'path\': \'data/train/part-00017.parquet\', \'bytes\': 23559509, \'sha256\': \'07ffc050aca01ae824d747c1d690ae50b3cb46126195ebcc2d847bc7b338d274\'}, {\'shard\': 18, \'path\': \'data/train/part-00018.parquet\', \'bytes\': 23575652, \'sha256\': \'48226fa0da896a28ecba6d68978ac21e444c80ee5c8dbf7cdc4ae6f7d7a07ba0\'}, {\'shard\': 19, \'path\': \'data/train/part-00019.parquet\', \'bytes\': 23588643, \'sha256\': \'47078937f949663ab7a31756dd4fd2445a3e2ed4b4f34a2e2ad28914ed5ba40b\'}, {\'shard\': 20, \'path\': \'data/train/part-00020.parquet\', \'bytes\': 23581860, \'sha256\': \'1b17ed45f8bea8065150371c2da75b2117be55217a9863c74a2b5a46013d0583\'}, {\'shard\': 21, \'path\': \'data/train/part-00021.parquet\', \'bytes\': 23572499, \'sha256\': \'62f8a983f0e7d198e495df52942ce295af377648aedf7a3c3c303e7183cac492\'}, {\'shard\': 22, \'path\': \'data/train/part-00022.parquet\', \'bytes\': 23578899, \'sha256\': \'f42aed0c6814b1491b658fd5a61bdd3cd65ff0c9696e972d198183b6fd33f2c0\'}, {\'shard\': 23, \'path\': \'data/train/part-00023.parquet\', \'bytes\': 23566965, \'sha256\': \'b7802cff1236dc757f114c17a1ecc0065a6f057db5ad658aeb98cf23d3d45fa4\'}, {\'shard\': 24, \'path\': \'data/train/part-00024.parquet\', \'bytes\': 23594948, \'sha256\': \'f4b297dd098807319c8ce6ff6b4318ae70ee9167409a13c20a590b21853a2eb2\'}, {\'shard\': 25, \'path\': \'data/train/part-00025.parquet\', \'bytes\': 23592190, \'sha256\': \'1966e2ad5b2fc25c9a91c1ece78e6cc3c42b2cd6e76f0e0d300af1ac0031c4fa\'}, {\'shard\': 26, \'path\': \'data/train/part-00026.parquet\', \'bytes\': 23586227, \'sha256\': \'b6d6b7edd181e5c4919669ae799cb0c5002185a3dc8f6f0d41f0dce664f45dfb\'}, {\'shard\': 27, \'path\': \'data/train/part-00027.parquet\', \'bytes\': 23587897, \'sha256\': \'69907e05bf7f1b088fb36a388c6f0be46477a586c657f5c0dc076decec203135\'}, {\'shard\': 28, \'path\': \'data/train/part-00028.parquet\', \'bytes\': 23551367, \'sha256\': \'e9b572b4b11d4127b9c3031695ef569067da6d9fa493f24c4ead65e8e9df95a5\'}, {\'shard\': 29, \'path\': \'data/train/part-00029.parquet\', \'bytes\': 27914406, \'sha256\': \'a6606ce87f15bfafb51553d3be682dfe88f42657e6dc8e249beea4c6dab9b192\'}, {\'shard\': 30, \'path\': \'data/train/part-00030.parquet\', \'bytes\': 30255832, \'sha256\': \'92ef433b1be78433fd238dfbcbc6b68cbdcfaf4060d7074e8002b1c4241fc9c7\'}, {\'shard\': 31, \'path\': \'data/train/part-00031.parquet\', \'bytes\': 30272658, \'sha256\': \'f35b0c6f4e36bf472220c2adf16bf9ea401564b6bce37e75e6511235911f29e4\'}, {\'shard\': 32, \'path\': \'data/train/part-00032.parquet\', \'bytes\': 30292629, \'sha256\': \'5ac54d930269ffb55bb7520c3f3f5d0de84c890b704c876feb162717a9b54207\'}, {\'shard\': 33, \'path\': \'data/train/part-00033.parquet\', \'bytes\': 30281085, \'sha256\': \'51d6bd884e8203175b75c4e7f5d531ea09a82fe487f2320353d1c4c54ea328e3\'}, {\'shard\': 34, \'path\': \'data/train/part-00034.parquet\', \'bytes\': 30301581, \'sha256\': \'d263a24b5b5321f84a68610675813f753e3125b16589d70aad265b9d3938927b\'}, {\'shard\': 35, \'path\': \'data/train/part-00035.parquet\', \'bytes\': 30237616, \'sha256\': \'7708377ca2b0cf2e61113edda14abae22b9e84ebd57362177a750c05115d5103\'}, {\'shard\': 36, \'path\': \'data/train/part-00036.parquet\', \'bytes\': 28578586, \'sha256\': \'5f8d8eb234d3fd0b2b02bc6fa96e13d078855e7e5e94a6289e9c648904f9a791\'}, {\'shard\': 37, \'path\': \'data/train/part-00037.parquet\', \'bytes\': 23504723, \'sha256\': \'7328070d46d4bc68da93e424b899f542bbb52580721d92462bfdf0bb2bb785ca\'}, {\'shard\': 38, \'path\': \'data/train/part-00038.parquet\', \'bytes\': 23501763, \'sha256\': \'3bd57e3b3a19d3a1cfad96d11fa789f56b0f06501eb7c3ee09ec576981de798e\'}, {\'shard\': 39, \'path\': \'data/train/part-00039.parquet\', \'bytes\': 23550818, \'sha256\': \'e81fedee1fa299f0db0c089e93716a385311662aa8954419308f0c21ec9a79ec\'}, {\'shard\': 40, \'path\': \'data/train/part-00040.parquet\', \'bytes\': 23478550, \'sha256\': \'31920d538b7bde332561dbb2afea2afe1dc8aea637c87dff79fcec9c445a406d\'}, {\'shard\': 41, \'path\': \'data/train/part-00041.parquet\', \'bytes\': 23544280, \'sha256\': \'96a68e4aba353771a6e090d5d098db0fc5c5a1c3377a78a17d5a539898609de4\'}, {\'shard\': 42, \'path\': \'data/train/part-00042.parquet\', \'bytes\': 23532881, \'sha256\': \'1855f54d0c3bdf8d1d89ba58e340ea03131c86d9b55d353a6c7685d92a0de703\'}, {\'shard\': 43, \'path\': \'data/train/part-00043.parquet\', \'bytes\': 23487888, \'sha256\': \'0602fa8c93d5d635ec539b27075557ed83cf76d7366a1880bf07e646239b4c99\'}, {\'shard\': 44, \'path\': \'data/train/part-00044.parquet\', \'bytes\': 23528291, \'sha256\': \'6240f1e5f4ff2832d554931d528d34b6bfa6fcdd1c64b44f273f5f9e7ab27a4d\'}, {\'shard\': 45, \'path\': \'data/train/part-00045.parquet\', \'bytes\': 23520482, \'sha256\': \'24e457560cf58387c2d3a34f415518f90a3f06ea411ab20d6ecfb678deb2aeb2\'}, {\'shard\': 46, \'path\': \'data/train/part-00046.parquet\', \'bytes\': 23553675, \'sha256\': \'69d8e27a658513a129687863696ede7b62c334d2eb69bccba7c5bf19752e4578\'}, {\'shard\': 47, \'path\': \'data/train/part-00047.parquet\', \'bytes\': 23534383, \'sha256\': \'fb6cc779bd2ae25cf5329655d4c6c176ae40fb360b110bbf8806969f72856a98\'}, {\'shard\': 48, \'path\': \'data/train/part-00048.parquet\', \'bytes\': 23547600, \'sha256\': \'1245557b20c3419f576602fbebba464d04187f9b0d4cedc660829b47f20dd955\'}, {\'shard\': 49, \'path\': \'data/train/part-00049.parquet\', \'bytes\': 23527054, \'sha256\': \'0ec6d7d2b1a6c0e7481284618a11f9df10fea752e7c41b9d73faedf2090f90a0\'}, {\'shard\': 50, \'path\': \'data/train/part-00050.parquet\', \'bytes\': 23524938, \'sha256\': \'b5a9518358b663512003d62e2f7706add86b63a47969a20e3a1110566869a57e\'}, {\'shard\': 51, \'path\': \'data/train/part-00051.parquet\', \'bytes\': 23524651, \'sha256\': \'a562a888cfcaafa07a16b54dba79633fc936df34fe61d09303ab11117466d20b\'}, {\'shard\': 52, \'path\': \'data/train/part-00052.parquet\', \'bytes\': 39245172, \'sha256\': \'e5c605de3a44056b7d5ebaa6442eb3bf2f9898fae29c7560392a65df170f6837\'}, {\'shard\': 53, \'path\': \'data/train/part-00053.parquet\', \'bytes\': 39347988, \'sha256\': \'e6ccfe88296f320ff4f1d5a881fa8af842ac15d610ad6de8d08fb81ffde0e157\'}, {\'shard\': 54, \'path\': \'data/train/part-00054.parquet\', \'bytes\': 39355070, \'sha256\': \'82c1eaac80deea7b2ef90e306079b6165a80b011491c2aee5a073da7cf3ef8a4\'}, {\'shard\': 55, \'path\': \'data/train/part-00055.parquet\', \'bytes\': 39311389, \'sha256\': \'1a793279bf52a90ebbeb290d9c386067b69d28a87e0c31fb89ab90a9d87be751\'}, {\'shard\': 56, \'path\': \'data/train/part-00056.parquet\', \'bytes\': 39287567, \'sha256\': \'a984ff588e0ab606793ecc0d21b1f29ca39dc03572374aa406c5362a04a7117c\'}, {\'shard\': 57, \'path\': \'data/train/part-00057.parquet\', \'bytes\': 39302585, \'sha256\': \'27c35d99709bb983e7a072d33031777cd154c35ae267b3605b4bdafff56fb9e1\'}, {\'shard\': 58, \'path\': \'data/train/part-00058.parquet\', \'bytes\': 39321706, \'sha256\': \'6d802692d5ee78e300e8fb94e5b556cc24aa274b622b5806e411edf7e0433673\'}, {\'shard\': 59, \'path\': \'data/train/part-00059.parquet\', \'bytes\': 7472207, \'sha256\': \'69e1c814c52a97519fc485e7e14f9500cff74585e06c92d648eb2377da3484dd\'}]\nINPUT_FILES = {\'tokenizer.json\': \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\', \'protected-questions.json\': \'7c034c9767334e4c0f1a12a14bf18353bd1d0a9bf175da251fe74473561cabab\', \'initial.pt\': \'daedd90fe51fef95dc151bc28e2f1ed0c66de80e5aa50a9ad8723ec50204c647\', \'control-D0.pt\': \'1ac82d0c62686456024280aa9821a9c71067387b9b6db2911b72def9652f7109\', \'heldout4k.json\': \'28a3c0c9088c5b4a17d75d216e4abc4262cc9ca62884966c43e7ec09b6f040c2\', \'old1024.json\': \'2ba73d6034ea8c258c3ecda5c8c110143cba7fcd800dbdbdb7ce5cc666d408b2\', \'heldout4k-reference-execution.json\': \'3855680e9a37292aa2e351b8e8e4f7a5cbbe76444096d136470e4a9be8797111\', \'old1024-reference-execution.json\': \'f7cd4cac0ff393cdd622f5d9018b762bcae1f7a397da334dbad5fca3ece2bcf3\', \'test.json\': \'20cacb334e3fc81291433d2173d9085b17a15e1834bd84aea67b350403182503\'}\nCACHE_PROTOCOL = {\'version\': \'26a-v1\', \'revision\': \'b2724689ff079e9c22efd548fe134bfb29036dbb\', \'selection_seed\': 42, \'question_max_bytes\': 512, \'code_max_bytes_exclusive\': 512, \'reference_filter\': \'finite_numeric_solve; bool->int; sameAST/math/sympy limits as25\', \'tokens_dtype\': \'int16\', \'offsets_dtype\': \'int64\', \'source_specials\': \'BOS+question+EOS\', \'target_specials\': \'content only; EOS in train collate\', \'token_offset\': 3, \'pilot_eligible_rows\': 50000, \'execution_chunk_rows\': 2000, \'minimum_rows_if_ETA_over60min\': 7000000, \'all_download_shards\': 60, \'uniqueness\': \'original (shard,row) identities; text duplicates are not removed\'}\nTRAIN_PROTOCOL = {\'version\': \'26b-v1\', \'experiment\': 26, \'model\': {\'d_model\': 256, \'heads\': 4, \'encoder_layers\': 4, \'decoder_layers\': 4, \'swiglu_dim\': 680}, \'batch\': 128, \'main_updates\': 50000, \'peak_lr\': 0.003, \'floor_lr\': 0.0003, \'warmup_updates\': 300, \'main_cooldown_start\': 40000, \'main_cooldown_updates\': 10000, \'branch_lengths\': {6250: 1250, 12500: 2500, 25000: 5000}, \'init_seed\': 42, \'order_seed\': 230042, \'bucket_chunk_rows\': 4096, \'dropout\': 0.0, \'weight_decay\': 0.01, \'clip\': 1.0, \'AMP\': \'FP16+GradScaler init1024 unscale before clip\', \'fused_AdamW\': True, \'normalized_token_init\': True, \'generation\': \'greedy FP16 sorted b512 cap512\', \'CE\': \'FP32 b32 tokenweighted EOS withoutPAD\', \'primary\': \'exec-match finite reference rel=abs1e-6 heldout4096; old991 secondary\', \'nominal_points\': [0.8, 1.6, 3.2, 6.4], \'actual_point_pairs\': [960000, 1920000, 3840000, 6400000], \'main_pairs\': 6400000, \'all_compute_pairs\': 7520000, \'rules\': {\'compatibility_pp\': 1.0, \'mean_slope_data_compute_pp\': 2.0, \'last_segment_capacity_hint_pp\': 1.0}, \'fresh\': \'question exclusion only; never train/encode/evaluate\'}\n\nTRAIN_PROTOCOL[\'update_count_semantics\']=\'fixed presentation steps; native GradScaler skipped AdamW steps are separately reported, no retry/repeated row\'\n', 'exp26_train.py': '"""WSD main trajectory + reversible side cooldowns; every Drive checkpoint is complete."""\nimport copy,gc,hashlib,json,math,random,time,uuid\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nimport exp26_core as core\nimport exp26_eval as evaluation\nimport exp26_protocol as P\nfrom exp26_common import sha_file,json_hash,read_json,write_json,copy_verified\nfrom exp26_order import collate,prefix_identity\n\n\ndef main_lr(update,cfg):\n if not 1<=update<=cfg[\'main_updates\']:raise ValueError(\'Main update out of range\')\n if update<=cfg[\'warmup_updates\']:return cfg[\'peak_lr\']*update/cfg[\'warmup_updates\']\n if update<=cfg[\'main_cooldown_start\']:return cfg[\'peak_lr\']\n fraction=(update-cfg[\'main_cooldown_start\'])/cfg[\'main_cooldown_updates\']\n return cfg[\'peak_lr\']+(cfg[\'floor_lr\']-cfg[\'peak_lr\'])*fraction\n\ndef branch_lr(update,length,cfg):\n assert 1<=update<=length\n return cfg[\'peak_lr\']+(cfg[\'floor_lr\']-cfg[\'peak_lr\'])*update/length\n\ndef cpu(value):\n if isinstance(value,torch.Tensor):return value.detach().cpu().contiguous()\n if isinstance(value,dict):return {k:cpu(v)for k,v in value.items()}\n if isinstance(value,list):return [cpu(v)for v in value]\n if isinstance(value,tuple):return tuple(cpu(v)for v in value)\n return value\n\ndef rng_state():\n n=np.random.get_state()\n return {\'torch_cpu\':torch.get_rng_state(),\'torch_cuda\':torch.cuda.get_rng_state_all()if torch.cuda.is_available()else[],\n         \'python\':random.getstate(),\'numpy\':(n[0],n[1].tolist(),int(n[2]),int(n[3]),float(n[4]))}\n\ndef restore_rng(rng,device):\n torch.set_rng_state(rng[\'torch_cpu\'])\n if device.type==\'cuda\':torch.cuda.set_rng_state_all(rng[\'torch_cuda\'])\n random.setstate(rng[\'python\']);n=rng[\'numpy\'];np.random.set_state((n[0],np.asarray(n[1],dtype=np.uint32),n[2],n[3],n[4]))\n\ndef schedule_state(state,cfg):\n phase=\'branch\'if state[\'phase\'].startswith(\'branch\')else\'main\'\n return {\'format\':\'wsd-main-and-side-linear-v1\',\'mode\':phase,\'main_updates\':state[\'main_updates\'],\n         \'branch_origin\':state[\'branch_origin\'],\'branch_updates\':state[\'branch_updates\'],\n         \'branch_length\':state[\'branch_length\'],\'last_lr\':state[\'last_lr\'],\n         \'warmup_updates\':cfg[\'warmup_updates\'],\'main_cooldown_start\':cfg[\'main_cooldown_start\'],\n         \'main_cooldown_updates\':cfg[\'main_cooldown_updates\'],\'peak_lr\':cfg[\'peak_lr\'],\'floor_lr\':cfg[\'floor_lr\']}\n\nclass Checkpoints:\n def __init__(self,drive,local,identity):\n  self.drive,self.local=Path(drive),Path(local);self.identity=identity\n  self.drive.mkdir(parents=True,exist_ok=True);self.local.mkdir(parents=True,exist_ok=True)\n def save(self,model,opt,scaler,state,meta,cfg,spec,role=\'periodic\',latest=True):\n  core.sync(next(model.parameters()).device);started=time.perf_counter();name=f\'checkpoints/{role}-{meta["compute_updates"]:06d}-{uuid.uuid4().hex[:10]}.pt\'\n  m=cpu(model.state_dict());digest=evaluation.state_digest(m)\n  payload={\'format\':\'nlp-hw02-wsd-v1\',\'experiment\':26,\'identity\':self.identity,\'config\':cfg,\'model_spec\':spec,\n    \'model\':m,\'optimizer\':cpu(opt.state_dict()),\'scaler\':scaler.state_dict(),\'scaler_enabled\':scaler.is_enabled(),\n    \'scheduler\':schedule_state(state,cfg),\'rng\':cpu(rng_state()),\'training\':copy.deepcopy(state),\'experiment_history\':copy.deepcopy(meta),\n    \'model_tensor_sha256\':digest,\'purpose\':\'CPU_smoke\'if self.identity[\'smoke\']else\'Colab_T4_training\'}\n  target=self.local/name;target.parent.mkdir(parents=True,exist_ok=True);tmp=target.with_suffix(\'.pt.tmp\');torch.save(payload,tmp);tmp.replace(target)\n  digest_file=sha_file(target);copy_verified(target,self.drive/name,digest_file)\n  receipt={\'file\':name,\'sha256\':digest_file,\'bytes\':target.stat().st_size,\'model_tensor_sha256\':digest,\n           \'main_updates\':state[\'main_updates\'],\'branch_updates\':state[\'branch_updates\'],\'phase\':state[\'phase\'],\n           \'data_cursor\':state[\'data_cursor\'],\'role\':role,\'identity\':self.identity}\n  write_json(self.drive/(name+\'.json\'),receipt)\n  if latest:write_json(self.drive/\'latest.json\',receipt)\n  meta[\'checkpoint_io_seconds\']+=time.perf_counter()-started\n  return receipt\n def local_file(self,receipt):\n  assert receipt[\'identity\']==self.identity\n  source=self.drive/receipt[\'file\'];assert source.stat().st_size==receipt[\'bytes\']\n  target=self.local/receipt[\'file\'];copy_verified(source,target,receipt[\'sha256\']);return target\n def load(self,receipt,device):\n  file=self.local_file(receipt);p=torch.load(file,map_location=\'cpu\',weights_only=True)\n  assert p[\'format\']==\'nlp-hw02-wsd-v1\'and p[\'identity\']==self.identity\n  cfg=p[\'config\'];model=core.SimpleTransformer(**p[\'model_spec\'],dropout=cfg[\'dropout\']).to(device);model.load_state_dict(p[\'model\'],strict=True)\n  assert evaluation.state_digest(model.state_dict())==p[\'model_tensor_sha256\']==receipt[\'model_tensor_sha256\']\n  fused=not self.identity[\'smoke\'];opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'peak_lr\'],weight_decay=cfg[\'weight_decay\'],**({\'fused\':True}if fused else{}))\n  opt.load_state_dict(p[\'optimizer\']);scaler=torch.amp.GradScaler(device.type,enabled=p[\'scaler_enabled\'],init_scale=1024);scaler.load_state_dict(p[\'scaler\'])\n  state,meta=p[\'training\'],p[\'experiment_history\'];assert p[\'scheduler\']==schedule_state(state,cfg)\n  if state[\'main_updates\']or state[\'branch_updates\']:assert math.isclose(opt.param_groups[0][\'lr\'],state[\'last_lr\'],rel_tol=1e-12)\n  assert state[\'data_cursor\']==cfg[\'batch\']*(state[\'main_updates\']+state[\'branch_updates\'])\n  steps=state[\'main_updates\']+state[\'branch_updates\']-state[\'AMP_skips\']\n  assert all(int(s[\'step\'])==steps for s in opt.state.values())\n  restore_rng(p[\'rng\'],device);model.train();del p;return model,opt,scaler,state,meta\n\n\ndef sync_folder(local,drive):\n local,drive=Path(local),Path(drive)\n for p in sorted(local.rglob(\'*\')):\n  if p.is_file()and not p.name.endswith(\'.tmp\'):copy_verified(p,drive/p.relative_to(local),overwrite=True)\n\n\ndef initialize(initial,device,cfg,spec,smoke=False):\n if not smoke:assert sha_file(initial)==P.INITIAL_SHA\n model=evaluation.load_model(initial,device,dropout=cfg[\'dropout\'],scaled_tokens=True,spec=spec)\n if not smoke:assert evaluation.state_digest(model.state_dict())==P.NORMALIZED_INITIAL_TENSORS\n random.seed(P.SEED);np.random.seed(P.SEED);torch.manual_seed(P.SEED)\n if device.type==\'cuda\':torch.cuda.manual_seed_all(P.SEED)\n model.train();opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'peak_lr\'],weight_decay=cfg[\'weight_decay\'],**({\'fused\':True}if not smoke else{}))\n amp=device.type==\'cuda\';scaler=torch.amp.GradScaler(device.type,enabled=amp,init_scale=1024)\n state={\'phase\':\'main\',\'main_updates\':0,\'branch_origin\':None,\'branch_updates\':0,\'branch_length\':0,\'data_cursor\':0,\n        \'counts\':{},\'last_lr\':0.,\'active_anchor\':None,\'AMP_skips\':0}\n meta={\'compute_updates\':0,\'AMP_skips\':0,\'finite_loss_and_grad\':True,\'all_counts\':{},\'train_seconds\':0.,\'checkpoint_io_seconds\':0.,\n       \'completed_points\':[],\'point_training\':{},\'logs\':[],\'sessions\':[], \'finite_loss\':True,\'finite_successful_grad\':True}\n return model,opt,scaler,state,meta\n\n\ndef train_chunk(model,opt,scaler,state,meta,cfg,cache,order,device,updates,interrupt_fixture=None):\n core.sync(device);started=time.perf_counter();records=[];counts=Counter()\n for _ in range(updates):\n  cursor=state[\'data_cursor\'];end=cursor+cfg[\'batch\']\n  if end>len(order):raise ValueError(\'No repeats before exhaustion; cache too small for the specified trajectory\')\n  src,tgt,labels,c=collate(cache,order[cursor:end],device);opt.zero_grad(set_to_none=True)\n  branch=state[\'phase\']==\'branch\';step=state[\'branch_updates\']+1 if branch else state[\'main_updates\']+1\n  lr=branch_lr(step,state[\'branch_length\'],cfg)if branch else main_lr(step,cfg)\n  for g in opt.param_groups:g[\'lr\']=lr\n  with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=scaler.is_enabled()):\n   logits=model(src,tgt);loss=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD)\n  if interrupt_fixture==meta[\'compute_updates\']+1:raise KeyboardInterrupt(\'Uncommitted-update fixture\')\n  if scaler.is_enabled():\n   scaler.scale(loss).backward();scaler.unscale_(opt);found=scaler._found_inf_per_device(opt)\n   assert len(found)==1;skipped=next(iter(found.values())).detach()\n  else:loss.backward();skipped=loss.new_zeros(())\n  norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'clip\'])\n  records.append(torch.stack([loss.detach().float(),torch.isfinite(loss.detach()).float(),torch.isfinite(norm.detach()).float(),skipped.float()]))\n  if scaler.is_enabled():scaler.step(opt);scaler.update()\n  else:opt.step()\n  state[\'branch_updates\'if branch else\'main_updates\']=step;state[\'data_cursor\']=end;state[\'last_lr\']=lr\n  counts.update(c);meta[\'compute_updates\']+=1\n  del src,tgt,labels,logits,loss,norm\n core.sync(device);elapsed=time.perf_counter()-started;records=torch.stack(records).cpu().tolist()\n skips=sum(int(r[3]>0)for r in records);loss_finite=all(bool(r[1])for r in records)\n successful_grad_finite=all(bool(r[2])or r[3]>0 for r in records)\n meta[\'AMP_skips\']+=skips;state[\'AMP_skips\']+=skips\n meta[\'finite_loss_and_grad\']&=all(bool(r[1])and bool(r[2])for r in records)\n meta[\'finite_loss\']&=loss_finite;meta[\'finite_successful_grad\']&=successful_grad_finite\n if not loss_finite or not successful_grad_finite:raise FloatingPointError(\'Nonfinite loss/unsafe successful gradient; resume last verified checkpoint\')\n for destination in (state[\'counts\'],meta[\'all_counts\']):\n  for k,v in counts.items():destination[k]=destination.get(k,0)+v\n meta[\'train_seconds\']+=elapsed\n return {\'phase\':state[\'phase\'],\'main_update\':state[\'main_updates\'],\'branch_update\':state[\'branch_updates\'],\'lr\':lr,\n         \'CE_last_chunk\':sum(r[0]for r in records)/len(records),\'AMP_skips_chunk\':skips,\'loss_scale\':scaler.get_scale(),\'seconds_chunk\':elapsed,\'pairs_per_second\':counts[\'pairs\']/elapsed,\n         \'source_tokens_per_second\':counts[\'source_tokens\']/elapsed,\'target_tokens_per_second\':counts[\'target_tokens\']/elapsed,\'compute_updates\':meta[\'compute_updates\']}\n\n\ndef run_training(cache,order,order_meta,inputs,drive,local,code_sha,device,evaluate_callback,\n                 stop_main_updates=50000,checkpoint_seconds=600,session_minutes=120,smoke=False,cfg=None,spec=None,\n                 max_updates_this_call=None,interrupt_fixture=None):\n cfg=copy.deepcopy(cfg or P.TRAIN_PROTOCOL);spec=spec or core.MODEL_SPEC\n if not smoke:\n  assert cfg==P.TRAIN_PROTOCOL and spec==core.MODEL_SPEC and device.type==\'cuda\'\n  assert \'T4\'in torch.cuda.get_device_name(device)and core.fused_adamw_available(device)\n  assert len(cache)>=7000000 and cfg[\'batch\']*cfg[\'main_updates\']<len(cache)\n  assert stop_main_updates in (6250,12500,25000,50000)\n identity={\'protocol_sha256\':json_hash(cfg),\'code_sha256\':code_sha,\'cache_manifest_sha256\':cache.manifest_sha,\n           \'order_sha256\':order_meta[\'file\'][\'sha256\'],\'tokenizer_sha256\':sha_file(Path(inputs)/\'tokenizer.json\'),\n           \'initial_sha256\':sha_file(Path(inputs)/\'initial.pt\'),\'smoke\':smoke,\'fused_adamw\':not smoke,\n           \'torch\':str(torch.__version__),\'CUDA\':torch.version.cuda}\n drive,local=Path(drive),Path(local);mgr=Checkpoints(drive,local,identity)\n latest=drive/\'latest.json\';started=time.perf_counter();last_checkpoint=started;updates_this_call=0\n if latest.exists():\n  model,opt,scaler,state,meta=mgr.load(read_json(latest),device);print(\'RESUME\',state[\'phase\'],state[\'main_updates\'],state[\'branch_updates\'],\'cursor\',state[\'data_cursor\'],flush=True)\n else:\n  model,opt,scaler,state,meta=initialize(Path(inputs)/\'initial.pt\',device,cfg,spec,smoke)\n  mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'initial\');last_checkpoint=time.perf_counter()\n initial_train_seconds=meta[\'train_seconds\'];meta[\'sessions\'].append({\'session_index\':len(meta[\'sessions\'])+1,\'start_compute_updates\':meta[\'compute_updates\'],\'requested_main_stop\':stop_main_updates})\n try:\n  while True:\n   # Evaluate a completed branch/final checkpoint before advancing the main cursor.\n   if state[\'phase\']in (\'branch_eval\',\'final_eval\'):\n    origin=state[\'branch_origin\']if state[\'phase\']==\'branch_eval\'else cfg[\'main_updates\'];tag=f\'p{origin:06d}\'\n    receipt_path=drive/\'points\'/tag/\'checkpoint.json\'\n    if receipt_path.exists():point_receipt=read_json(receipt_path)\n    else:\n     point_receipt=mgr.save(model,opt,scaler,state,meta,cfg,spec,role=tag);write_json(receipt_path,point_receipt)\n    assert point_receipt[\'model_tensor_sha256\']==evaluation.state_digest(model.state_dict())\n    prev=meta[\'completed_points\'][-1]if meta[\'completed_points\']else None\n    metrics=evaluate_callback(mgr.local_file(point_receipt),tag,prev)\n    point_info={\'nominal_main_updates\':origin,\'nominal_pairs\':origin*cfg[\'batch\'],\n       \'actual_updates\':state[\'main_updates\']+state[\'branch_updates\'],\'successful_optimizer_steps\':state[\'main_updates\']+state[\'branch_updates\']-state[\'AMP_skips\'],\'AMP_skips_trajectory\':state[\'AMP_skips\'],\'actual_pairs\':state[\'data_cursor\'],\n       \'cooldown_updates\':state[\'branch_length\']if state[\'phase\']==\'branch_eval\'else cfg[\'main_cooldown_updates\'],\n       \'counts\':copy.deepcopy(state[\'counts\']),\'data\':prefix_identity(order,state[\'data_cursor\'],len(cache)),\n       \'compute_updates_total\':meta[\'compute_updates\'],\'train_seconds_total\':meta[\'train_seconds\'],\'metrics\':metrics}\n    meta[\'point_training\'][tag]=point_info\n    if tag not in meta[\'completed_points\']:meta[\'completed_points\'].append(tag)\n    write_json(drive/\'points\'/tag/\'training.json\',point_info)\n    if state[\'phase\']==\'final_eval\':\n     state[\'phase\']=\'complete\';complete_receipt=mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'complete\');write_json(drive/\'final-checkpoint.json\',complete_receipt);break\n    # Preserve experiment costs, but restore every component of the main training trajectory.\n    anchor=state[\'active_anchor\'];saved_meta=meta\n    del model,opt,scaler;gc.collect()\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    model,opt,scaler,state,_=mgr.load(anchor,device);meta=saved_meta\n    assert state[\'phase\']==\'main\'and state[\'data_cursor\']==origin*cfg[\'batch\']\n    mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'main-restored\');last_checkpoint=time.perf_counter();continue\n   if state[\'phase\']==\'complete\':break\n   if state[\'phase\']==\'main\':\n    tag=f\'p{state["main_updates"]:06d}\'\n    if state[\'main_updates\']in cfg[\'branch_lengths\']and tag not in meta[\'completed_points\']:\n     anchor_path=drive/\'anchors\'/f\'{state["main_updates"]:06d}.json\'\n     if anchor_path.exists():\n      anchor=read_json(anchor_path);assert anchor[\'model_tensor_sha256\']==evaluation.state_digest(model.state_dict())\n     else:\n      anchor=mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'anchor\',latest=False);write_json(anchor_path,anchor)\n     state.update(phase=\'branch\',branch_origin=state[\'main_updates\'],branch_updates=0,branch_length=cfg[\'branch_lengths\'][state[\'main_updates\']],active_anchor=anchor)\n     mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'branch-start\');last_checkpoint=time.perf_counter();continue\n    if state[\'main_updates\']==cfg[\'main_updates\']:\n     state[\'phase\']=\'final_eval\';r=mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'final-evaluation-pending\');write_json(drive/\'points\'/f\'p{cfg["main_updates"]:06d}\'/\'checkpoint.json\',r);continue\n    if state[\'main_updates\']>=stop_main_updates:\n     mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'requested-stop\');break\n   if time.perf_counter()-started>=session_minutes*60 or(max_updates_this_call is not None and updates_this_call>=max_updates_this_call):\n    mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'session-stop\');break\n   if state[\'phase\']==\'branch\':remaining=state[\'branch_length\']-state[\'branch_updates\']\n   else:\n    future=[v for v in cfg[\'branch_lengths\']if v>state[\'main_updates\']]+[cfg[\'main_updates\'],stop_main_updates]\n    remaining=min(future)-state[\'main_updates\']\n   n=min(25 if not smoke else 1,remaining)\n   if max_updates_this_call is not None:n=min(n,max_updates_this_call-updates_this_call)\n   log=train_chunk(model,opt,scaler,state,meta,cfg,cache,order,device,n,interrupt_fixture)\n   updates_this_call+=n\n   if state[\'phase\']==\'branch\'and state[\'branch_updates\']==state[\'branch_length\']:\n    state[\'phase\']=\'branch_eval\';r=mgr.save(model,opt,scaler,state,meta,cfg,spec,role=\'branch-evaluation-pending\');write_json(drive/\'points\'/f\'p{state["branch_origin"]:06d}\'/\'checkpoint.json\',r);continue\n   if meta[\'compute_updates\']%(1000 if not smoke else 1)==0:\n    log[\'unique_seen\']=state[\'data_cursor\'];log[\'unique_fraction_of_cache\']=state[\'data_cursor\']/len(cache)\n    meta[\'logs\'].append(log);print(\'TRAIN\',json.dumps(log),flush=True)\n    write_json(drive/\'live-progress.json\',dict(log,session_elapsed_seconds=time.perf_counter()-started,train_seconds_total=meta[\'train_seconds\'],resume_from=read_json(latest)[\'file\']))\n   if time.perf_counter()-last_checkpoint>=checkpoint_seconds:\n    mgr.save(model,opt,scaler,state,meta,cfg,spec);last_checkpoint=time.perf_counter()\n except (KeyboardInterrupt,FloatingPointError,torch.cuda.OutOfMemoryError)as error:\n  # A Python interrupt can land inside a fused update. Never serialize such live state.\n  write_json(drive/\'interruption.json\',{\'error\':type(error).__name__,\'message\':str(error)[:300],\n      \'resume_from\':read_json(latest)[\'file\'],\'uncommitted_work_may_be_repeated\':True})\n  print(\'INTERRUPTED: restart from the last verified Drive checkpoint; uncommitted chunk discarded\',flush=True)\n  if not isinstance(error,KeyboardInterrupt):raise\n finally:\n  session={\'wall_seconds\':time.perf_counter()-started,\'new_train_seconds\':meta[\'train_seconds\']-initial_train_seconds,\n           \'updates_this_call\':updates_this_call,\'checkpoint_file\':read_json(latest)[\'file\']}\n  write_json(drive/f\'session-{uuid.uuid4().hex[:10]}.json\',session)\n  del model,opt,scaler;gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache()\n return {\'state\':state,\'history\':meta,\'latest\':read_json(latest),\'session\':session,\'complete\':state[\'phase\']==\'complete\'}\n\n\ndef decisions(points,D0,cfg=None):\n cfg=cfg or P.TRAIN_PROTOCOL;keys=[f\'p{x:06d}\'for x in sorted(cfg[\'branch_lengths\'])]+[f\'p{cfg["main_updates"]:06d}\']\n if any(k not in points for k in keys):return {\'status\':\'incomplete; no slope/quality decision until all4 points evaluated\'}\n get=lambda k:points[k][\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\']\n acc=[get(k)for k in keys];base=D0[\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\']\n delta0=100*(acc[0]-base);slope=100*(acc[-1]-acc[0])/3.;last=100*(acc[-1]-acc[-2])\n return {\'status\':\'complete\',\'compatibility_0_8M_delta_pp\':delta0,\'compatible_with_D0_primary4k\':abs(delta0)<=1.,\n         \'mean_nominal_slope_pp_per_doubling\':slope,\'data_compute_remain_main_lever_by_rule\':slope>=2.,\n         \'last_nominal_segment_delta_pp\':last,\'capacity_hint_next_model_B_by_rule\':last<1.,\n         \'actual_exposure_mean_slope_pp_per_doubling\':100*(acc[-1]-acc[0])/math.log2(6.4/.96),\n         \'actual_exposure_last_slope_pp_per_doubling\':last/math.log2(6.4/3.84),\n         \'interpretation\':\'Thresholds are registered heuristics, not proof of capacity. One seed; filtered data/global permutation/WSD differ from25; fresh untouched.\'}\n', 'exp26_workflow.py': '"""Colab evaluation orchestration; frozen input sets, Drive receipts, no fresh eval."""\nimport gc,math,time\nfrom pathlib import Path\nimport torch\nimport exp26_protocol as P\nimport exp26_core as core\nimport exp26_eval as E\nimport exp26_exec as numeric\nfrom exp26_common import read_json,write_json,sha_file,json_hash,copy_verified\nfrom exp26_train import sync_folder\n\n\ndef initialize_experiment(drive,code_sha,cache,order_meta,inputs):\n drive=Path(drive);drive.mkdir(parents=True,exist_ok=True)\n identity={\'code_sha256\':code_sha,\'protocol_sha256\':json_hash(P.TRAIN_PROTOCOL),\n    \'cache_manifest_sha256\':cache.manifest_sha,\'order_sha256\':order_meta[\'file\'][\'sha256\'],\n    \'input_files\':P.INPUT_FILES,\'torch\':str(torch.__version__),\'CUDA\':torch.version.cuda}\n path=drive/\'experiment-identity.json\'\n if path.exists():assert read_json(path)==identity,\'Existing experiment differs; preserve it and use another Drive folder\'\n else:write_json(path,identity)\n write_json(drive/\'protocol.json\',P.TRAIN_PROTOCOL)\n write_json(drive/\'input-identity.json\',{\'starter_sha256\':P.STARTER_B_SHA,\'files\':P.INPUT_FILES,\n    \'D0_source_full_checkpoint_sha256\':P.D0_SOURCE_CHECKPOINT_SHA,\'D0_tensor_sha256\':P.D0_TENSORS,\n    \'normalized_initial_tensor_sha256\':P.NORMALIZED_INITIAL_TENSORS,\'fresh\':\'question exclusion only\'})\n copy_verified(cache.root/\'cache-manifest.json\',drive/\'cache-manifest.json\',cache.manifest_sha)\n return identity\n\n\ndef reference_execution(sets,inputs,drive):\n """Recheck reference finite indices and values once; subsequent sessions reuse verified receipts."""\n root=Path(drive)/\'references\';root.mkdir(parents=True,exist_ok=True);refs={}\n for split,rows in sets.items():\n  expected=read_json(Path(inputs)/(split+\'-reference-execution.json\'))\n  identity={\'rows_sha256\':json_hash(rows),\'saved_reference_sha256\':sha_file(Path(inputs)/(split+\'-reference-execution.json\')),\n    \'policy\':\'same25 sandbox finite solve bool->int rel=abs1e-6\'}\n  path=root/(split+\'-reference-execution.json\');rp=root/(split+\'-receipt.json\')\n  if path.exists()and rp.exists():\n   receipt=read_json(rp);assert receipt[\'identity\']==identity and sha_file(path)==receipt[\'sha256\'];execution=read_json(path)\n  else:\n   print(\'REFERENCE CPU\',split,len(rows),\'programs\',flush=True)\n   execution=numeric.evaluate_programs([r[\'code\']for r in rows]);assert execution[\'summary\'][\'complete\']\n   expected_count=4096 if split==\'heldout4k\'else 991\n   assert sum(r[\'status\']==\'numeric\'for r in execution[\'rows\'])==expected_count\n   comparison=numeric.compare_exec(expected,execution)\n   assert comparison[\'matches\']==comparison[\'denominator\']==expected_count,\'Reference values/finite indices changed\'\n   assert [i for i,r in enumerate(execution[\'rows\'])if r[\'status\']==\'numeric\']==comparison[\'eligible_indices\']\n   write_json(path,execution);write_json(rp,{\'identity\':identity,\'sha256\':sha_file(path),\'gate\':comparison})\n  refs[split]=execution\n return refs\n\n\ndef restore_evaluation(drive,local):\n drive,local=Path(drive),Path(local)\n if not drive.exists():return\n local.mkdir(parents=True,exist_ok=True)\n for p in drive.rglob(\'*\'):\n  if p.is_file()and p.suffix in (\'.json\',\'.jsonl\'):copy_verified(p,local/p.relative_to(drive),overwrite=True)\n\n\ndef prepare_D0(sets,refs,inputs,drive,local,device):\n drive,local=Path(drive),Path(local);restore_evaluation(drive/\'D0\',local/\'D0\')\n # The compact file changes serialization only; its model tensors must still be exactly D0.\n state=torch.load(Path(inputs)/\'control-D0.pt\',map_location=\'cpu\',weights_only=True)\n assert E.state_digest(state)==P.D0_TENSORS;del state;gc.collect()\n print(\'D0 CONTROL: expected heldout4k1677/4096, old422/991; match tolerance3, CE tolerance1e-4\',flush=True)\n m=E.evaluate_point(Path(inputs)/\'control-D0.pt\',sets,refs,Path(inputs)/\'tokenizer.json\',device,local/\'D0\',\'D0\')\n gate=E.verify_D0(m);write_json(local/\'D0\'/\'gate.json\',gate);sync_folder(local/\'D0\',drive/\'D0\')\n print(\'D0 gate\',gate,flush=True);return m\n\n\ndef point_callback(sets,refs,inputs,drive,local,device,D0):\n drive,local=Path(drive),Path(local)\n def evaluate(model_path,tag,previous):\n  point_drive=drive/\'points\'/tag/\'evaluation\';point_local=local/\'points\'/tag/\'evaluation\'\n  restore_evaluation(point_drive,point_local)\n  previous_metrics=read_json(drive/\'points\'/previous/\'evaluation\'/\'metrics.json\')if previous is not None else None\n  started=time.perf_counter();m=E.evaluate_point(model_path,sets,refs,Path(inputs)/\'tokenizer.json\',device,point_local,tag,D0,previous_metrics)\n  write_json(point_local/\'evaluation-session.json\',{\'wall_seconds\':time.perf_counter()-started,\'previous\':previous})\n  sync_folder(point_local,point_drive)\n  print(\'POINT\',tag,{s:{\'matches\':r[\'exec\'][\'matches\'],\'denominator\':r[\'exec\'][\'denominator\'],\'CE\':r[\'CE\'][\'CE\']}for s,r in m[\'splits\'].items()},flush=True)\n  return m\n return evaluate\n\n\ndef generate_final_test(test,inputs,drive,local,device,smoke=False):\n drive,local=Path(drive),Path(local);final_path=drive/\'final-checkpoint.json\'\n if not final_path.exists():return {\'status\':\'not_complete; test not generated\'}\n receipt=read_json(final_path);target=local/receipt[\'file\'];copy_verified(drive/receipt[\'file\'],target,receipt[\'sha256\'])\n assert receipt[\'phase\']==\'complete\'\n if not smoke:assert receipt[\'main_updates\']==50000 and receipt[\'data_cursor\']==6400000 and len(test)==1919\n out=local/\'test\';restore_evaluation(drive/\'test\',out);timing_path=out/\'test-generation-time.json\'\n if timing_path.exists()and (out/\'test-answers.jsonl\').exists()and(out/\'test-generations.json\').exists():\n  timing=read_json(timing_path)\n  assert timing[\'model_sha256\']==receipt[\'sha256\']and timing[\'tokenizer_sha256\']==P.TOKENIZER_SHA\n  assert timing[\'answers_sha256\']==sha_file(out/\'test-answers.jsonl\')\n  examples=read_json(out/\'test-generations.json\')\n else:\n  examples,timing=E.generate_one(target,test,Path(inputs)/\'tokenizer.json\',device,out,\'test\',batch=2 if smoke else 512,cap=8 if smoke else 512,half=True)\n  write_json(timing_path,timing)\n assert len(examples)==len(test) and [r[\'question\']for r in examples]==[r[\'question\']for r in test]and [r[\'id\']for r in examples]==list(range(len(test)))\n write_json(out/\'submission-candidate.json\',{\'answers\':\'test-answers.jsonl\',\'rows\':len(test),\'smoke\':smoke,\'answers_sha256\':timing[\'answers_sha256\'],\n    \'checkpoint_sha256\':receipt[\'sha256\'],\'full_seconds\':timing[\'full_seconds\'],\'official_bot_format_verified\':False})\n sync_folder(out,drive/\'test\');return timing\n\n\ndef collect_report(drive,local,cfg=None):\n """Only committed training state/point receipts; live interrupted tensors are never reported as saved."""\n cfg=cfg or P.TRAIN_PROTOCOL\n drive,local=Path(drive),Path(local);metrics={};table=[]\n for origin in sorted(cfg[\'branch_lengths\'])+[cfg[\'main_updates\']]:\n  tag=f\'p{origin:06d}\';base=drive/\'points\'/tag\n  if not(base/\'training.json\').exists()or not(base/\'evaluation\'/\'metrics.json\').exists():continue\n  info=read_json(base/\'training.json\');m=read_json(base/\'evaluation\'/\'metrics.json\');metrics[tag]=m\n  table.append({\'point\':tag,\'nominal_M_pairs\':info[\'nominal_pairs\']/1e6,\'actual_M_pairs\':info[\'actual_pairs\']/1e6,\n    \'exec4k\':m[\'splits\'][\'heldout4k\'][\'exec\'][\'matches\'],\'exec991\':m[\'splits\'][\'old1024\'][\'exec\'][\'matches\'],\n    \'accuracy4k\':m[\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\'],\'CE4k\':m[\'splits\'][\'heldout4k\'][\'CE\'][\'CE\'],\n    \'unique_fraction_of_cache\':info[\'data\'][\'unique_fraction_of_cache\'],\'target_tokens\':info[\'counts\'][\'target_tokens\'],\n    \'compute_updates_total\':info[\'compute_updates_total\'],\'train_seconds_total\':info[\'train_seconds_total\']})\n D0=read_json(drive/\'D0\'/\'metrics.json\');from exp26_train import decisions\n decision=decisions(metrics,D0,cfg);write_json(drive/\'decision.json\',decision)\n latest=read_json(drive/\'latest.json\');path=local/latest[\'file\'];copy_verified(drive/latest[\'file\'],path,latest[\'sha256\'])\n payload=torch.load(path,map_location=\'cpu\',weights_only=True);history=payload[\'experiment_history\'];committed=payload[\'training\'];del payload\n counters=history[\'all_counts\'];seconds=history[\'train_seconds\']\n totals={\'committed_main_updates\':committed[\'main_updates\'],\'committed_branch_updates\':committed[\'branch_updates\'],\n    \'phase\':committed[\'phase\'],\'all_compute_updates\':history[\'compute_updates\'],\'train_seconds\':seconds,\n    \'pairs_per_second\':counters.get(\'pairs\',0)/seconds if seconds else None,\n    \'useful_source_tokens_per_second\':counters.get(\'source_tokens\',0)/seconds if seconds else None,\n    \'useful_target_tokens_per_second\':counters.get(\'target_tokens\',0)/seconds if seconds else None,\n    \'all_compute_presentations\':counters.get(\'pairs\',0),\'checkpoint_io_seconds_committed\':history[\'checkpoint_io_seconds\'],\n    \'AMP_skips\':history[\'AMP_skips\'],\'successful_optimizer_steps_all_compute\':history[\'compute_updates\']-history[\'AMP_skips\'],\n    \'AMP_skips_final_trajectory\':committed[\'AMP_skips\'],\'successful_optimizer_steps_final_trajectory\':committed[\'main_updates\']+committed[\'branch_updates\']-committed[\'AMP_skips\'],\n    \'finite_loss\':history[\'finite_loss\'],\'finite_successful_grad\':history[\'finite_successful_grad\'],\'finite_loss_and_grad\':history[\'finite_loss_and_grad\'],\n    \'unique_main_rows_seen\':committed[\'main_updates\']*cfg[\'batch\'],\'unique_trajectory_rows_seen\':committed[\'data_cursor\'],\n    \'recorded_session_wall_seconds\':sum(read_json(p)[\'wall_seconds\']for p in drive.glob(\'session-*.json\')),\n    \'wall_timer_scope\':\'training controller, evaluations and checkpoint I/O per completed/interrupted Python call; setup/cache copy/D0 outside; kernel killed without finally has only heartbeat/last checkpoint\',\n    \'uncommitted_work_after_last_checkpoint_may_be_repeated\':True}\n report={\'experiment\':26,\'table\':table,\'totals\':totals,\'decision\':decision,\'latest_checkpoint\':latest,\n   \'protocol\':cfg,\'fresh_encoded\':0,\'fresh_executed\':0,\'GPU_budget_minutes_target\':120,\n   \'baseline_D0_saved\':{\'exec4k\':1677,\'exec991\':422,\'CE4k\':.4764168630129111}}\n write_json(drive/\'summary.json\',report);return report\n', 'exp27_decode.py': '"""Sampling with shared encoder/cross KV, numeric voting, complete pipeline timers."""\nimport gc,json,math,time\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport exp26_core as C\nimport exp26_eval as E\nimport exp26_exec as N\nfrom exp26_fast_exec import run_chunks\nfrom exp26_common import read_json,write_json,json_hash,sha_file\n\nTEMPERATURES={0.7:32,0.5:16,1.0:16}\n\ndef configurations():\n    configs=[{\'name\':\'greedy\',\'mode\':\'greedy\',\'k\':0,\'temperature\':None,\'greedy_vote\':False}]\n    for t,n in TEMPERATURES.items():\n        for k in (4,8,16,32):\n            if k>n: continue\n            for g in (False,True):\n                configs.append({\'name\':f\'vote-t{t}-k{k}-g{int(g)}\',\'mode\':\'vote\',\'temperature\':t,\'k\':k,\'greedy_vote\':g})\n            if t==0.7:\n                configs.append({\'name\':f\'fallback-k{k}\',\'mode\':\'fallback\',\'temperature\':t,\'k\':k,\'greedy_vote\':False})\n    return configs\n\ndef choose(row,cfg):\n    greedy=row[\'greedy\']\n    if cfg[\'mode\']==\'greedy\': return greedy\n    samples=row[\'samples\'][str(cfg[\'temperature\'])][:cfg[\'k\']]\n    if cfg[\'mode\']==\'fallback\':\n        return greedy if greedy[\'execution\'][\'status\']==\'numeric\' else next((s for s in samples if s[\'execution\'][\'status\']==\'numeric\'),greedy)\n    votes=([greedy] if cfg[\'greedy_vote\'] else [])+samples\n    good=[v for v in votes if v[\'execution\'][\'status\']==\'numeric\']\n    if not good:return greedy\n    # isclose is not transitive: explicitly use connected components of its pairwise graph.\n    parent=list(range(len(good)))\n    def root(i):\n        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]\n        return i\n    for i in range(len(good)):\n        for j in range(i):\n            if math.isclose(good[i][\'execution\'][\'value\'],good[j][\'execution\'][\'value\'],rel_tol=1e-6,abs_tol=1e-6):\n                parent[root(i)]=root(j)\n    clusters={}\n    for i,v in enumerate(good):clusters.setdefault(root(i),[]).append(v)\n    def key(vs):\n        contains=greedy[\'execution\'][\'status\']==\'numeric\' and any(math.isclose(v[\'execution\'][\'value\'],greedy[\'execution\'][\'value\'],rel_tol=1e-6,abs_tol=1e-6) for v in vs)\n        return len(vs),contains,sum(v[\'mean_logprob\'] for v in vs)/len(vs)\n    winning=max(clusters.values(),key=key)\n    return max(winning,key=lambda v:v[\'mean_logprob\'])\n\nclass EncoderContext:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads\n        self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,None,:]\n        self.positions=model.positions(torch.arange(capacity,device=src.device))\n        self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            att=layer.multihead_attn\n            k,v=F.linear(memory,att.in_proj_weight[self.width:],att.in_proj_bias[self.width:]).chunk(2,-1)\n            shape=(len(src),-1,self.heads,self.dim)\n            self.cross.append((k.reshape(shape).transpose(1,2)[:,None],v.reshape(shape).transpose(1,2)[:,None]))\n        # [question,1,head,source,dim], stride0 broadcast to lanes at SDPA; no repeat/contiguous.\n\nclass SampleDecoder:\n    def __init__(self,context,k):\n        self.c=context;self.k=k;self.t=0;self.b=len(context.src)\n        self.valid=torch.zeros(self.b,k,context.capacity,dtype=torch.bool,device=context.src.device)\n        shape=(self.b,k,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n    def heads(self,x):return x.reshape(self.b,self.k,1,self.c.heads,self.c.dim).transpose(2,3)\n    def attend(self,att,q,k,v,mask):\n        # k=1 keeps the original4-D SDPA route; sampling uses5-D broadcasting.\n        if self.k==1:\n            y=F.scaled_dot_product_attention(q[:,0],k[:,0],v[:,0],attn_mask=mask[:,0],dropout_p=0.)[:,None]\n        else:y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)\n        y=y.transpose(2,3).contiguous().reshape(self.b,self.k,1,self.c.width)\n        return att.out_proj(y)\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t\n        x=c.model.tokens(tokens[:,:,None])*math.sqrt(c.width)+c.positions[t][None,None,None,:]\n        self.valid[:,:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,c.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,:,t:t+1]=self.heads(k);sv[:,:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:,:t+1],sv[:,:,:,:t+1],self.valid[:,:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,c.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,:,0])\n\n@torch.inference_mode()\ndef decode_context(context,k,temperature,deadline=None,forced=None):\n    d=SampleDecoder(context,k);device=context.src.device\n    token=torch.full((d.b,k),C.BOS,device=device,dtype=torch.long);done=torch.zeros_like(token,dtype=torch.bool)\n    out=torch.zeros(d.b,k,context.capacity,device=device,dtype=torch.long)\n    sums=torch.zeros(d.b,k,device=device);lengths=torch.zeros(d.b,k,device=device,dtype=torch.long)\n    for t in range(context.capacity):\n        if t%16==0:C.check_deadline(deadline)\n        logits=d.step(token).float();logits[:,:,[C.PAD,C.BOS]]=-torch.inf\n        if forced is not None:pick=forced[:,:,t]\n        elif temperature is None:pick=logits.argmax(-1)\n        else:pick=torch.multinomial(F.softmax(logits.reshape(-1,logits.shape[-1])/temperature,-1),1).reshape(d.b,k)\n        logprob=F.log_softmax(logits,-1).gather(-1,pick[:,:,None]).squeeze(-1)\n        sums+=torch.where(done,0.,logprob);lengths+=(~done).long()\n        token=pick.masked_fill(done,C.PAD);out[:,:,t]=token;done|=token.eq(C.EOS)\n        if (t+1)%16==0 and bool(done.all()):break\n    return out[:,:,:d.t].cpu(),(sums/lengths.clamp_min(1)).cpu()\n\ndef candidates(ids,logprobs,tokenbytes):\n    return [[{\'code\':C.decode_bytes(tokenbytes,C.active_tokens(seq)).decode(\'utf8\',errors=\'replace\'),\n              \'length\':len(C.active_tokens(seq)), \'mean_logprob\':float(lp)} for seq,lp in zip(group,lps)]\n            for group,lps in zip(ids.tolist(),logprobs.tolist())]\n\ndef generate_grid(model_path,rows,tokenizer_path,device,out,spec=None,grid=None,batch_rows=512,cap=512,deadline=None,controls=None):\n    """One encoder pass/question for greedy + all temperatures; completed question batches resume."""\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);grid=grid or TEMPERATURES\n    identity={\'model\':sha_file(model_path),\'tokenizer\':sha_file(tokenizer_path),\'rows\':json_hash(rows),\n              \'grid\':{str(k):v for k,v in grid.items()},\'batch_rows\':batch_rows,\'cap\':cap,\'spec\':spec,\n              \'controls\':json_hash(controls) if controls is not None else None,\'runtime\':str(torch.__version__)}\n    meta=out/\'candidate-identity.json\'\n    if meta.exists():assert read_json(meta)==identity,\'Sampling input/config changed\'\n    else:write_json(meta,identity)\n    start=time.perf_counter();model=E.load_model(model_path,device,spec=spec).eval()\n    if device.type==\'cuda\':model.half();torch.cuda.reset_peak_memory_stats(device)\n    tokenizer,tokenbytes=C.init_tokenizer(tokenizer_path)\n    encoded=[[C.BOS]+C.encode(tokenizer,r[\'question\'])+[C.EOS] for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));qbatch=max(1,batch_rows//max(grid.values()))\n    allrows=[None]*len(rows);new_batches=0;resumed_batches=0\n    try:\n        for lo in range(0,len(order),qbatch):\n            C.check_deadline(deadline);indices=order[lo:lo+qbatch];file=out/f\'batch-{lo:06d}.json\'\n            if file.exists():\n                part=read_json(file);assert [r[\'id\'] for r in part]==indices\n                receipt=read_json(file.with_suffix(\'.sha.json\'));assert sha_file(file)==receipt[\'sha256\']\n                resumed_batches+=1\n            else:\n                src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=C.PAD).to(device)\n                ctx=EncoderContext(model,src,cap)\n                forced=None\n                if controls is not None:\n                    forced=torch.zeros(len(indices),1,cap,dtype=torch.long,device=device)\n                    for j,i in enumerate(indices):\n                        active=controls[i][\'generated_token_ids\'][:cap];forced[j,0,:len(active)]=torch.tensor(active,device=device)\n                gids,glp=decode_context(ctx,1,None,deadline,forced=forced);greedy=candidates(gids,glp,tokenbytes)\n                part=[{\'id\':i,\'question\':rows[i][\'question\'],\'greedy\':greedy[j][0],\'samples\':{}} for j,i in enumerate(indices)]\n                if controls is not None:\n                    for r in part:assert r[\'greedy\'][\'code\']==controls[r[\'id\']][\'generated_code\'],\'Forced greedy replay differs\'\n                for t,k in grid.items():\n                    torch.manual_seed(270042+lo+int(t*10000))\n                    if device.type==\'cuda\':torch.cuda.manual_seed_all(270042+lo+int(t*10000))\n                    ids,lps=decode_context(ctx,k,t,deadline);generated=candidates(ids,lps,tokenbytes)\n                    for r,g in zip(part,generated):r[\'samples\'][str(t)]=g\n                    del ids,lps\n                del ctx,src,gids,glp\n                write_json(file,part);write_json(file.with_suffix(\'.sha.json\'),{\'sha256\':sha_file(file)})\n                new_batches+=1\n            for r in part:allrows[r[\'id\']]=r\n            if lo%(qbatch*16)==0:print(\'SAMPLING\',len([r for r in allrows if r is not None]),\'/\',len(rows),\'seconds\',round(time.perf_counter()-start,1),flush=True)\n        C.sync(device)\n        return allrows,{\'load_generate_seconds\':time.perf_counter()-start,\'new_batches\':new_batches,\'resumed_batches\':resumed_batches,\n                       \'batch_candidate_rows\':batch_rows,\'question_batch\':qbatch,\'cross_KV\':\'shared5D broadcast; no physical expansion\',\n                       \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None}\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef execute_candidates(rows,out,deadline=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    flat=[c for r in rows for c in [r[\'greedy\']]+[c for group in r[\'samples\'].values() for c in group]]\n    codes=[c[\'code\'] for c in flat];identity=json_hash(codes);path=out/\'identity.json\'\n    if path.exists():assert read_json(path)[\'codes_sha256\']==identity\n    else:write_json(path,{\'codes_sha256\':identity,\'candidate_count\':len(codes)})\n    saved={}\n    for p in out.glob(\'chunk-*.json\'):\n        saved[int(p.stem.split(\'-\')[1])]=read_json(p)\n        assert sha_file(p)==read_json(p.with_suffix(\'.sha\'))[\'sha256\']\n    def commit(i,value):\n        p=out/f\'chunk-{i:06d}.json\';write_json(p,value);write_json(p.with_suffix(\'.sha\'),{\'sha256\':sha_file(p)})\n        C.check_deadline(deadline)\n    results=run_chunks(codes,workers=2,chunk_rows=2000,saved_chunks=saved,on_chunk=commit)\n    for c,e in zip(flat,results[\'rows\']):c[\'execution\']=e\n    extra={\'candidate_count\':len(flat),\'reused_candidates\':sum(len(v) for v in saved.values()),\n           \'committed_distinct_program_calls\':sum(len(set(codes[i:i+2000])) for i in range(0,len(codes),2000)),\n           \'new_committed_distinct_program_calls\':sum(len(set(codes[i:i+2000])) for i in range(0,len(codes),2000) if i//2000 not in saved),\n           \'failed_chunk_extra_calls_unknown\':results[\'summary\'][\'chunk_retries\']>0}\n    results[\'summary\'].update(extra);write_json(out/\'summary.json\',results[\'summary\'])\n    return results[\'summary\']\n\ndef score_configs(rows,reference,original,tokenizer,greedy_score,configs=None):\n    result=[]\n    for cfg in configs or configurations():\n        winners=[choose(r,cfg) for r in rows];score=N.compare_exec(reference,[w[\'execution\'] for w in winners])\n        result.append({\'config\':cfg,\'exec\':score,\'McNemar_against_greedy\':E.paired_mcnemar(score,greedy_score),\n                       \'length_accuracy\':E.length_accuracy(original,tokenizer,score),\n                       \'executed_candidates_used\':len(rows)*(1+cfg[\'k\']),\n                       \'mean_winning_length\':sum(w[\'length\'] for w in winners)/len(winners)})\n    return result\n\ndef time_test(model_path,test,tokenizer_path,device,out,cfg,spec=None,batch_rows=512,cap=512,deadline=None):\n    """No cache reuse. Timer includes input read, checkpoint/tokenizer load, generation, exec, vote, answers."""\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    if list((out/\'candidates\').glob(\'batch-*.json\')):raise ValueError(\'Fresh timing requires empty test candidates\')\n    C.sync(device);start=time.perf_counter()\n    test=read_json(test) if isinstance(test,(Path,str)) else test\n    if cfg[\'mode\']==\'greedy\':\n        gens,timing=E.generate_one(model_path,test,tokenizer_path,device,out,\'test\',spec=spec,batch=512,cap=cap,half=device.type==\'cuda\')\n        programs=[g[\'generated_code\'] for g in gens];count=0\n    else:\n        grid={cfg[\'temperature\']:cfg[\'k\']}\n        generated,gen_timing=generate_grid(model_path,test,tokenizer_path,device,out/\'candidates\',spec,grid,batch_rows,cap,deadline)\n        execution=execute_candidates(generated,out/\'execution\',deadline)\n        programs=[choose(r,cfg)[\'code\'] for r in generated];count=len(test)*(1+cfg[\'k\'])\n        timing=dict(gen_timing,execution=execution)\n    path=out/\'test-answers.jsonl\'\n    with path.open(\'w\') as f:\n        for i,(row,code) in enumerate(zip(test,programs)):\n            f.write(json.dumps({\'id\':i,\'question\':row[\'question\'],\'code\':code},ensure_ascii=False)+\'\\n\')\n    C.sync(device);full=time.perf_counter()-start\n    report={\'config\':cfg,\'full_pipeline_seconds\':full,\'rows\':len(test),\'executed_candidates\':count,\n            \'answers_sha256\':sha_file(path),\'model_sha256\':sha_file(model_path),\'tokenizer_sha256\':sha_file(tokenizer_path),\'components\':timing,\'complete\':True,\n            \'timer\':\'test read, checkpoint/tokenizer load, encoder/decode, candidate execution, voting, final answer write; SHA/report outside\'}\n    write_json(out/\'timing.json\',report);return report\n\ndef select_config(tested):\n    eligible=[r for r in tested[1:] if r[\'timing\'][\'full_pipeline_seconds\']<=120 and r[\'McNemar_against_greedy\'][\'delta_percentage_points\']>=1.5]\n    chosen=max(eligible,key=lambda r:(r[\'exec\'][\'matches\'],-r[\'timing\'][\'full_pipeline_seconds\'])) if eligible else tested[0]\n    return bool(eligible),chosen\n\ndef run_experiment(model_path,sets,refs,inputs,device,out,control_metrics,control_out,spec=None,deadline=None,batch_rows=512):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);tokenizer,_=C.init_tokenizer(Path(inputs)/\'tokenizer.json\');splits={}\n    start=time.perf_counter()\n    for split,original in sets.items():\n        controls=read_json(Path(control_out)/(split+\'-generations.json\'))\n        generated,timing=generate_grid(model_path,original,Path(inputs)/\'tokenizer.json\',device,out/split/\'candidates\',spec,\n                                       batch_rows=batch_rows,deadline=deadline,controls=controls)\n        execution=execute_candidates(generated,out/split/\'execution\',deadline)\n        configs=score_configs(generated,refs[split],original,tokenizer,control_metrics[\'splits\'][split][\'exec\'])\n        assert configs[0][\'exec\'][\'match_indices\']==control_metrics[\'splits\'][split][\'exec\'][\'match_indices\'],\'Greedy execution replay gate\'\n        splits[split]={\'configs\':configs,\'generation\':timing,\'execution\':execution,\'candidate_count\':len(original)*65}\n        write_json(out/\'progress.json\',{\'experiment\':27,\'status\':\'partial\',\'splits\':splits})\n    primary=splits[\'heldout4k\'][\'configs\'];ranked=sorted(primary[1:],key=lambda r:(-r[\'exec\'][\'matches\'],r[\'config\'][\'k\'],r[\'config\'][\'name\']))[:3]\n    tested=[]\n    # Mandatory greedy baseline plus2–3 best alternatives; each gets a fresh full timer.\n    for row in [primary[0]]+ranked:\n        C.check_deadline(deadline);cfg=row[\'config\'];dest=out/\'test\'/cfg[\'name\']\n        p=dest/\'timing.json\'\n        timing=read_json(p) if p.exists() else time_test(model_path,Path(inputs)/\'test.json\',Path(inputs)/\'tokenizer.json\',device,dest,cfg,spec,batch_rows,deadline=deadline)\n        tested.append(dict(row,timing=timing));write_json(out/\'timed-configs.json\',tested)\n    accepted,chosen=select_config(tested)\n    report={\'experiment\':27,\'status\':\'complete\',\'model_sha256\':sha_file(model_path),\'splits\':splits,\'test\':tested,\n            \'decision\':{\'accepted\':accepted,\'config\':chosen[\'config\'],\'delta_pp\':chosen[\'McNemar_against_greedy\'][\'delta_percentage_points\'],\n                        \'full_test_seconds\':chosen[\'timing\'][\'full_pipeline_seconds\'],\'selection\':\'max exec4k among top3 measured alternatives with fulltest<=120s and gain>=1.5pp\'},\n            \'wall_seconds\':time.perf_counter()-start,\'logprob\':\'mean untempered model logprob including EOS, PAD/BOS masked\',\n            \'clustering\':\'pairwise isclose1e-6 connected components; tie count, greedy membership, mean candidate logprob\',\n            \'fresh_encoded\':0,\'fresh_evaluated\':0}\n    write_json(out/\'27-summary.json\',report);return report\n', 'exp27b_fast.py': '"""Physical candidate cross-KV, 4D SDPA and fresh full pipeline timings."""\nimport gc,math,time\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport exp26_core as C\nimport exp26_eval as E\nimport exp26_exec as N\nimport exp27_decode as D\nfrom exp26_common import read_json,write_json,json_hash,sha_file\nfrom exp26_fast_exec import run_chunks\n\ndef config(k=0):\n    return {\'name\':\'greedy\' if not k else f\'vote-t0.7-k{k}-g0\',\'mode\':\'vote\' if k else \'greedy\',\n            \'temperature\':.7 if k else None,\'k\':k,\'greedy_vote\':False}\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            self.cross.append(tuple(t.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous() for t in (k,v)))\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n@torch.inference_mode()\ndef decode(context,k=1,temperature=None,deadline=None,check_every=16):\n    d=Decoder(context,k);device=context.src.device\n    token=torch.full((d.n,),C.BOS,device=device,dtype=torch.long);done=torch.zeros(d.n,dtype=torch.bool,device=device)\n    output=torch.full((d.n,context.capacity),C.PAD,device=device,dtype=torch.long)\n    sums=torch.zeros(d.n,device=device);lengths=torch.zeros(d.n,dtype=torch.long,device=device)\n    for t in range(context.capacity):\n        if t%check_every==0:C.check_deadline(deadline)\n        logits=d.step(token).float();logits[:,[C.PAD,C.BOS]]=-torch.inf\n        pick=logits.argmax(-1) if temperature is None else torch.multinomial(F.softmax(logits/temperature,-1),1).squeeze(-1)\n        logprob=F.log_softmax(logits,-1).gather(-1,pick[:,None]).squeeze(-1)\n        sums.add_(torch.where(done,0.,logprob));lengths.add_((~done).long())\n        token=pick.masked_fill(done,C.PAD);output[:,t]=token;done|=token.eq(C.EOS)\n        if (t+1)%check_every==0 and bool(done.all().cpu()):break\n    return output[:,:d.t].reshape(len(context.src),k,d.t).cpu(),(sums/lengths.clamp_min(1)).reshape(-1,k).cpu(),d.cross_bytes\n\ndef pack(ids,lps,tokenbytes):\n    result=D.candidates(ids,lps,tokenbytes)\n    for group,seqs in zip(result,ids.tolist()):\n        for c,seq in zip(group,seqs):c[\'token_ids\']=C.active_tokens(seq)\n    return result\n\ndef generate(model_path,rows,tokenizer_path,device,out,k=0,batch_rows=2048,cap=512,spec=None,deadline=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    identity={\'model\':sha_file(model_path),\'rows\':json_hash(rows),\'tokenizer\':sha_file(tokenizer_path),\n              \'k\':k,\'batch_rows\':batch_rows,\'cap\':cap,\'spec\':spec,\'decoder\':\'physical4D-v1\',\'torch\':str(torch.__version__)}\n    if (out/\'identity.json\').exists():assert read_json(out/\'identity.json\')==identity\n    else:write_json(out/\'identity.json\',identity)\n    started=time.perf_counter();model=E.load_model(model_path,device,spec=spec).eval()\n    if device.type==\'cuda\':model.half();torch.cuda.reset_peak_memory_stats(device)\n    tokenizer,tokenbytes=C.init_tokenizer(tokenizer_path)\n    encoded=[[C.BOS]+C.encode(tokenizer,r[\'question\'])+[C.EOS] for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));qbatch=max(1,batch_rows//max(1,k)) if k else min(512,batch_rows);allrows=[None]*len(rows)\n    load_seconds=time.perf_counter()-started;new=resumed=0;cross_bytes=0;decode_seconds=0.\n    try:\n        for lo in range(0,len(order),qbatch):\n            C.check_deadline(deadline);indices=order[lo:lo+qbatch];file=out/f\'batch-{lo:06d}.json\'\n            if file.exists():\n                assert sha_file(file)==read_json(file.with_suffix(\'.sha.json\'))[\'sha256\'];part=read_json(file)\n                assert [r[\'id\'] for r in part]==indices;resumed+=1\n            else:\n                src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=C.PAD).to(device)\n                C.sync(device);begin=time.perf_counter();ctx=Context(model,src,cap)\n                ids,lps,_=decode(ctx,deadline=deadline);greedy=pack(ids,lps,tokenbytes)\n                part=[{\'id\':i,\'question\':rows[i][\'question\'],\'greedy\':greedy[j][0],\'samples\':{}} for j,i in enumerate(indices)]\n                if k:\n                    seed=270042+lo+7000;torch.manual_seed(seed)\n                    if device.type==\'cuda\':torch.cuda.manual_seed_all(seed)\n                    ids,lps,memory=decode(ctx,k,.7,deadline);samples=pack(ids,lps,tokenbytes);cross_bytes=max(cross_bytes,memory)\n                    for r,g in zip(part,samples):\n                        for c in g:c.pop(\'token_ids\') # greedy tokens retained for correctness gate\n                        r[\'samples\'][\'0.7\']=g\n                C.sync(device);decode_seconds+=time.perf_counter()-begin\n                del ctx,src,ids,lps\n                write_json(file,part);write_json(file.with_suffix(\'.sha.json\'),{\'sha256\':sha_file(file)});new+=1\n            for r in part:allrows[r[\'id\']]=r\n            if lo%(qbatch*8)==0:print(\'FAST SAMPLE\',lo+len(indices),\'/\',len(rows),\'k\',k,\'batch\',batch_rows,flush=True)\n        C.sync(device)\n        return allrows,{\'load_generate_write_seconds\':time.perf_counter()-started,\'load_seconds\':load_seconds,\'generation_seconds\':decode_seconds,\n            \'batch_candidate_rows\':batch_rows,\'question_batch\':qbatch,\'new_batches\':new,\'resumed_batches\':resumed,\n            \'cross_KV\':\'physical repeat_interleave; all SDPA4D\',\'cross_KV_peak_bytes\':cross_bytes,\'check_every\':16,\n            \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None}\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef execute(rows,out,deadline=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    flat=[c for r in rows for c in [r[\'greedy\']]+r[\'samples\'].get(\'0.7\',[])];codes=[c[\'code\'] for c in flat]\n    identity={\'codes\':json_hash(codes),\'wall\':3.,\'cpu\':2.}\n    if (out/\'identity.json\').exists():assert read_json(out/\'identity.json\')==identity\n    else:write_json(out/\'identity.json\',identity)\n    saved={}\n    for p in out.glob(\'chunk-*.json\'):\n        assert sha_file(p)==read_json(p.with_suffix(\'.sha.json\'))[\'sha256\'];saved[int(p.stem.split(\'-\')[1])]=read_json(p)\n    def commit(i,value):\n        p=out/f\'chunk-{i:06d}.json\';write_json(p,value);write_json(p.with_suffix(\'.sha.json\'),{\'sha256\':sha_file(p)});C.check_deadline(deadline)\n    result=run_chunks(codes,workers=2,chunk_rows=2000,wall_limit=3.,cpu_limit=2.,saved_chunks=saved,on_chunk=commit)\n    for c,e in zip(flat,result[\'rows\']):c[\'execution\']=e\n    result[\'summary\'].update(timeouts=sum(\'timeout\' in r[\'status\'] for r in result[\'rows\']),complete=True)\n    result[\'summary\'][\'timeout_limits\']={kind:sum(r.get(\'timeout_limit\')==kind for r in result[\'rows\']) for kind in (\'wall\',\'cpu\')}\n    write_json(out/\'summary.json\',result[\'summary\']);return result[\'summary\']\n\ndef probe_batch(model_path,rows,tokenizer_path,device,out,k=16,spec=None,cap=512,choices=(2048,1024),deadline=None):\n    out=Path(out);p=out/\'batch-choice.json\'\n    if p.exists():return read_json(p)\n    tok,_=C.init_tokenizer(tokenizer_path)\n    longest=sorted(rows,key=lambda r:len(C.encode(tok,r[\'question\'])),reverse=True);attempts=[]\n    for size in choices:\n        sample=longest[:max(1,size//max(1,k))]\n        try:\n            _,timing=generate(model_path,sample,tokenizer_path,device,out/f\'probe-{size}\',k,size,cap,spec,deadline)\n            attempts.append({\'batch\':size,\'status\':\'passed\',**timing})\n            report={\'batch_rows\':size,\'attempts\':attempts,\'probe\':\'longest inputs, full cap; 2048 then OOM fallback1024\'}\n            write_json(p,report);return report\n        except torch.cuda.OutOfMemoryError:\n            gc.collect();torch.cuda.empty_cache();attempts.append({\'batch\':size,\'status\':\'OOM\'});write_json(out/\'batch-attempts.json\',attempts)\n    raise RuntimeError(\'Sampler OOM at both candidate batches\')\n\ndef time_test(model_path,test,tokenizer_path,device,out,cfg,spec=None,batch_rows=2048,cap=512,deadline=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);p=out/\'timing.json\'\n    if p.exists():\n        report=read_json(p);assert report[\'model_sha256\']==sha_file(model_path) and report[\'config\']==cfg\n        assert report[\'answers_sha256\']==sha_file(out/\'test-answers.jsonl\');return report\n    # Every incomplete attempt is retained. Timing restarts with entirely fresh candidates.\n    attempt=out/f\'attempt-{len(list(out.glob("attempt-*"))):03d}\';attempt.mkdir()\n    C.sync(device);started=time.perf_counter();rows=read_json(test) if isinstance(test,(str,Path)) else test\n    generated,gen=generate(model_path,rows,tokenizer_path,device,attempt/\'candidates\',cfg[\'k\'],batch_rows,cap,spec,deadline)\n    execution=execute(generated,attempt/\'execution\',deadline);begin=time.perf_counter()\n    winners=[D.choose(r,cfg) for r in generated];vote_seconds=time.perf_counter()-begin;begin=time.perf_counter()\n    with (out/\'test-answers.jsonl\').open(\'w\',encoding=\'utf8\') as f:\n        for i,(row,c) in enumerate(zip(rows,winners)):f.write(__import__(\'json\').dumps({\'id\':i,\'question\':row[\'question\'],\'code\':c[\'code\']},ensure_ascii=False)+\'\\n\')\n    C.sync(device);write_seconds=time.perf_counter()-begin;full=time.perf_counter()-started\n    report={\'config\':cfg,\'model_sha256\':sha_file(model_path),\'tokenizer_sha256\':sha_file(tokenizer_path),\'rows\':len(rows),\'complete\':True,\n        \'answers_sha256\':sha_file(out/\'test-answers.jsonl\'),\'full_pipeline_seconds\':full,\'executed_candidates\':len(rows)*(1+cfg[\'k\']),\n        \'components\':{\'generation\':gen,\'execution\':execution,\'voting_seconds\':vote_seconds,\'answer_write_seconds\':write_seconds},\n        \'timer\':\'test read, model/tokenizer load, generation/candidate journal, execution, voting, answers close; hashes/report outside\',\n        \'attempt\':attempt.name,\'cache_reuse\':False}\n    write_json(p,report);return report\n\ndef select_k(timings):\n    eligible=[r for r in timings if r[\'config\'][\'k\'] and r[\'complete\'] and r[\'full_pipeline_seconds\']<=120.]\n    return max(eligible,key=lambda r:r[\'config\'][\'k\'])[\'config\'] if eligible else config()\n\ndef run(model_path,sets,refs,inputs,device,out,control_out,deadline):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);summary=out/\'27b-summary.json\'\n    if summary.exists() and read_json(summary).get(\'status\')==\'complete\':return read_json(summary)\n    report={\'experiment\':\'27b\',\'status\':\'partial\',\'model_sha256\':sha_file(model_path),\'test\':[]};write_json(summary,report)\n    probe=probe_batch(model_path,sets[\'heldout4k\'],Path(inputs)/\'tokenizer.json\',device,out/\'memory\',deadline=deadline)\n    report[\'batch_probe\']=probe;batch=probe[\'batch_rows\']\n    rows,timing=generate(model_path,sets[\'heldout4k\'],Path(inputs)/\'tokenizer.json\',device,out/\'heldout4k/candidates\',16,batch,deadline=deadline)\n    controls=read_json(Path(control_out)/\'heldout4k-generations.json\')\n    mismatch=[i for i,(r,c) in enumerate(zip(rows,controls)) if r[\'greedy\'][\'token_ids\']!=c[\'generated_token_ids\']]\n    report[\'greedy_gate\']={\'token_identical\':not mismatch,\'mismatch_indices\':mismatch,\'rows\':len(rows)}\n    write_json(summary,report)\n    if mismatch:raise AssertionError(\'Fast greedy token identity failed; see27b-summary\')\n    ex=execute(rows,out/\'heldout4k/execution\',deadline);wins=[D.choose(r,config(16)) for r in rows]\n    score=N.compare_exec(refs[\'heldout4k\'],[w[\'execution\'] for w in wins]);matches=score[\'matches\']\n    report[\'heldout4k\']={\'generation\':timing,\'execution\':ex,\'exec\':score,\'quality_gate\':abs(matches-2707)<=45,\'expected_matches\':2707,\'tolerance\':45}\n    write_json(summary,report)\n    if abs(matches-2707)>45:raise AssertionError(\'Fast k16 quality outside registered tolerance\')\n    for k in (0,8,16):\n        C.check_deadline(deadline)\n        timed=time_test(model_path,Path(inputs)/\'test.json\',Path(inputs)/\'tokenizer.json\',device,out/\'test\'/config(k)[\'name\'],config(k),batch_rows=batch,deadline=deadline)\n        report[\'test\'].append(timed);write_json(summary,report)\n    report.update(status=\'complete\',decision={\'config\':select_k(report[\'test\']),\'rule\':\'largest k8/k16 with fresh fulltest<=120s\',\n        \'voting_available\':select_k(report[\'test\'])[\'k\']>0},fresh_encoded=0,fresh_evaluated=0)\n    write_json(summary,report);return report\n', 'final_core.py': "# Generated by build_final_inference.py from exp26_core.py; do not edit.\nimport math,time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError('Position overflow')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b''.join(chunks)\n\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n\nclass BudgetExpired(Exception):pass\n\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired('Session budget reached; completed run boundaries are preserved')\n\n\ndef sync(device):\n if device.type=='cuda':torch.cuda.synchronize(device)\n\n", 'final_decode.py': '"""V3 decode loop; frozen Context/Decoder mathematics live in final_fast."""\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, check_every=16, compute_logprob=True):\n    from final_fast import Decoder\n    d = Decoder(context, k)\n    device = context.src.device\n    token = torch.full((d.n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(d.n, dtype=torch.bool, device=device)\n    # KV, output, running LP/length buffers are allocated once, never concatenated.\n    output = torch.full((d.n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums = torch.zeros(d.n, device=device)\n    lengths = torch.zeros(d.n, dtype=torch.long, device=device)\n    for t in range(context.capacity):\n        if t % check_every == 0:\n            C.check_deadline(deadline)\n        logits = d.step(token)\n        if temperature is not None or compute_logprob:\n            logits = logits.float()\n        logits[:, :2] = -torch.inf  # PAD/BOS: contiguous slice avoids a per-step index tensor.\n        pick = logits.argmax(-1) if temperature is None else torch.multinomial(\n            F.softmax(logits / temperature, -1), 1).squeeze(-1)\n        if compute_logprob:\n            logprob = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n            sums.add_(torch.where(done, 0., logprob))\n        lengths.add_(~done)\n        token = pick.masked_fill(done, C.PAD)\n        output[:, t] = token\n        done |= token.eq(C.EOS)\n        if (t + 1) % check_every == 0 and bool(done.all().cpu()):\n            break\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (sums / lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_execute.py': '"""Persistent Torch-free service: fresh numeric namespaces, pristine SymPy forks.\n\nstdin: JSON lists of [id, code], or null to finish. stdout: [id, execution, metrics].\nThe launcher can kill the complete service process group, including its workers.\n"""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport resource\nimport select\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nimport final_numeric as numeric\n\nMEMORY = 768 * 1024**2\nWALL = 3.\nCPU = 2.\n\n\ndef kill_program(pid):\n    # Each program owns a group; kill descendants even after a successful result.\n    try:\n        os.killpg(pid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        try:\n            os.kill(pid, signal.SIGKILL)  # Child may not yet have called setsid().\n        except ProcessLookupError:\n            pass\n    until = time.monotonic()+.1\n    while time.monotonic() < until:\n        try:\n            if os.waitpid(pid, os.WNOHANG)[0]:\n                return\n        except ChildProcessError:\n            return\n        time.sleep(.001)\n    raise RuntimeError(\'Program could not be reaped; replace pristine worker\')\n\n\ndef isolated_task(code, tree, connection):\n    reader, writer = os.pipe()\n    parent = os.getpid()\n    pid = os.fork()  # This parent has no Torch/CUDA and no reader/BLAS threads.\n    if pid == 0:\n        try:\n            os.close(reader)\n            connection.close()\n            os.setsid()\n            if sys.platform.startswith(\'linux\'):\n                import ctypes\n                signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n                if ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) != 0 or os.getppid() != parent:\n                    os.killpg(os.getpgrp(), signal.SIGKILL)\n            resource.setrlimit(resource.RLIMIT_CPU, (int(CPU)+1, int(CPU)+1))\n            result = numeric.execute_task((code, WALL, CPU), tree=tree, isolated=True)\n            os.write(writer, json.dumps(result, separators=(\',\', \':\')).encode())\n        finally:\n            os._exit(0)\n    os.close(writer)\n    try:\n        # Same outer limit as v2; catches code that repeatedly catches timer exceptions.\n        if select.select([reader], [], [], WALL+2.)[0]:\n            data = os.read(reader, 4096)\n            if data:\n                return json.loads(data)\n        return {\'status\': \'timeout_or_worker_exit\'}\n    finally:\n        os.close(reader)\n        kill_program(pid)\n\n\ndef worker(connection):\n    # Library diagnostics and user output cannot corrupt the JSON protocol.\n    with open(os.devnull, \'w\') as sink:\n        os.dup2(sink.fileno(), 1)\n        os.dup2(sink.fileno(), 2)\n    numeric.worker_init(MEMORY)\n    pristine = None\n    while True:\n        item = connection.recv()\n        if item is None:\n            return\n        index, code = item\n        started = time.monotonic()\n        used = time.process_time()\n        before = resource.getrusage(resource.RUSAGE_CHILDREN)\n        tree, status = numeric.filter_program(code)\n        if status:\n            result = {\'status\': status}\n        elif any((isinstance(node, numeric.ast.Import) and any(a.name == \'sympy\' for a in node.names)) or\n                 (isinstance(node, numeric.ast.ImportFrom) and node.module == \'sympy\')\n                 for node in numeric.ast.walk(tree)):\n            if pristine is None:\n                numeric.prepare_pristine_worker()\n                pristine = dict(numeric._MODULES)\n            numeric._MODULES.clear()\n            numeric._MODULES.update(pristine)\n            result = isolated_task(code, tree, connection)\n        else:\n            # Only builtins and a fresh math namespace are reachable on this path.\n            # No user code can access SymPy objects/caches; env and timers are per task.\n            result = numeric.execute_task((code, WALL, CPU), tree=tree)\n        after = resource.getrusage(resource.RUSAGE_CHILDREN)\n        connection.send((index, result, {\'task_wall_seconds\': time.monotonic()-started,\n            \'task_cpu_seconds\': time.process_time()-used + after.ru_utime+after.ru_stime-before.ru_utime-before.ru_stime}))\n\n\ndef serve(workers):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    # Independent supervisor kills the process group on parent death (Linux).\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]))\n', 'final_execute_v6.py': '"""Torch-free v6 service. Same isolated worker, nice10, live 2 -> up to4 slots."""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nfrom final_execute import worker, WALL\n\n\ndef lower_priority():\n    error = None\n    try:\n        effective = os.nice(10)\n    except PermissionError as exc:\n        effective, error = os.getpriority(os.PRIO_PROCESS, 0), str(exc)\n    return {\'nice_requested_increment\': 10, \'nice\': effective,\n            \'nice_applied\': error is None, \'nice_error\': error}\n\n\ndef serve(workers, priority):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        def state():\n            print(json.dumps({\'workers\': len(slots), **priority}), flush=True)\n        state()\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                elif isinstance(batch, dict):\n                    target = batch[\'workers\']\n                    if not isinstance(target, int) or not len(slots) <= target <= 4:\n                        raise ValueError(\'Only expansion up to four workers is supported\')\n                    slots.extend(start() for _ in range(target-len(slots)))\n                    state()\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]), lower_priority())\n', 'final_executor_v6.py': '"""V6 execution cache with nice10 service and lossless 2 -> up to4 expansion."""\nimport math\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    def __init__(self, workers=None, service_path=None):\n        self.workers = workers if workers is not None else 2\n        if isinstance(self.workers, bool) or not isinstance(self.workers, int) or not 1 <= self.workers <= 4:\n            raise ValueError(\'Require one to four execution workers\')\n        env = dict(os.environ, OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                   PYTHONHASHSEED=\'0\')\n        self.child = subprocess.Popen([sys.executable, str(service_path or Path(__file__).with_name(\'final_execute_v6.py\')),\n                                      str(self.workers)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                      stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env,\n                                      start_new_session=True)\n        self.service_state = {}\n        self.worker_history = [self.workers]\n        self.codes = {}\n        self.results = {}\n        self.requests = 0\n        self.started = time.monotonic()\n        self.error = None\n        self.task_wall_seconds = self.task_cpu_seconds = 0.\n        self.first_submit = self.last_result = None\n        def read():\n            try:\n                for line in self.child.stdout:\n                    message = json.loads(line)\n                    if isinstance(message, dict):\n                        self.service_state = message\n                        continue\n                    index, value, *metadata = message\n                    if metadata and metadata[0]:\n                        self.task_wall_seconds += metadata[0][\'task_wall_seconds\']\n                        self.task_cpu_seconds += metadata[0][\'task_cpu_seconds\']\n                    self.last_result = time.monotonic()\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def resize(self, workers):\n        if not isinstance(workers, int) or not self.workers <= workers <= 4:\n            raise ValueError(\'Only expansion up to four workers is supported\')\n        self.child.stdin.write(json.dumps({\'workers\': workers})+\'\\n\')\n        self.child.stdin.flush()\n        self.workers = workers\n        self.worker_history.append(workers)\n\n    def submit(self, candidates):\n        jobs = []\n        for candidate in candidates:\n            self.requests += 1\n            code = candidate[\'code\']\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            if self.first_submit is None:\n                self.first_submit = time.monotonic()\n            self.child.stdin.write(json.dumps(jobs) + \'\\n\')\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        for candidate in candidates:\n            index = self.codes[candidate[\'code\']]\n            candidate[\'execution\'] = self.results.get(index, {\'status\': \'budget_not_evaluated\'})\n\n    def drain(self, deadline):\n        while (len(self.results) < len(self.codes) or self.service_state.get(\'workers\') != self.workers) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or \'executor_exit_\'+str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(.002)\n        return len(self.results) == len(self.codes) and self.service_state.get(\'workers\') == self.workers\n\n    def summary(self):\n        return {\'workers\': self.workers, \'worker_history\': self.worker_history,\n                \'service_state\': dict(self.service_state), \'requested_candidates\': self.requests,\n                \'unique_programs\': len(self.codes), \'deduplicated\': self.requests-len(self.codes),\n                \'completed_unique\': len(self.results), \'service_wall_seconds\': time.monotonic()-self.started,\n                \'sum_worker_task_wall_seconds\': self.task_wall_seconds,\n                \'sum_worker_task_cpu_seconds\': self.task_cpu_seconds,\n                \'first_submit_to_last_result_seconds\': (self.last_result-self.first_submit\n                    if self.last_result is not None and self.first_submit is not None else None),\n                \'service_wall_includes_GPU_wait\': True,\n                \'wall_limit\': 3., \'cpu_limit\': 2., \'memory_bytes\': 768*1024**2,\n                \'overlap\': True, \'isolation\': \'fresh numeric namespaces / pristine SymPy fork\', \'error\': self.error}\n\n    def close(self):\n        # Never pool.join() unbounded in the CUDA process.\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=.1)\n        self.child.stdout.close()\n\n\ndef verify_winners(winners, deadline, workers=None):\n    """Fresh service: compare the actual chosen CODE against its voted execution."""\n    executor = Executor(workers)\n    eligible = [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] == \'numeric\']\n    items = [{\'code\': winners[i][\'code\']} for i in eligible]\n    try:\n        executor.submit(items)\n        complete = executor.drain(deadline)\n        executor.attach(items)\n        mismatches = [i for i, item in zip(eligible, items) if\n                      item[\'execution\'][\'status\'] != \'numeric\' or not math.isclose(\n                          item[\'execution\'][\'value\'], winners[i][\'execution\'][\'value\'],\n                          rel_tol=1e-6, abs_tol=1e-6)]\n        return {\'checked_rows\': len(eligible), \'complete\': complete,\n                \'mismatch_rows\': mismatches, \'execution\': executor.summary()}\n    finally:\n        executor.close()\n', 'final_fast.py': '# Generated by build_final_inference.py from exp27b_fast.py; do not edit.\nimport math\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            self.cross.append(tuple(t.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous() for t in (k,v)))\n\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n\nfrom final_decode import decode\n', 'final_numeric.py': '"""Final-only numeric policy; historical experiment executors remain unchanged."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport random\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\',\'random\',\'format\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Attribute) and isinstance(n.ctx,(ast.Store,ast.Del)):\n            return None,\'blocked_attribute_mutation\'\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    # All sympify bindings import this parser dynamically, including object methods.\n    # Symbol names use the separately validated symbols() syntax, never parse_expr.\n    from sympy.parsing import sympy_parser\n    def denied_parser(*args,**kwargs):\n        raise ValueError(\'String expression parsing forbidden\')\n    sympy_parser.parse_expr=denied_parser\n    from sympy.core.cache import clear_cache\n    clear_cache()\n    from sympy.core.random import seed\n    seed(0)\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef prepare_pristine_worker():\n    """Run once in a Torch-free, single-threaded parent; never run user code here."""\n    _MODULES.clear()\n    safe_import(\'math\')\n    safe_import(\'sympy\')  # Parser deny patch, cache clear and SymPy seed happen ONCE.\n    random.seed(0)\n\ndef execute_task(payload, tree=None, isolated=False):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    if not isolated:\n        _MODULES.clear()  # Historical/reference path; fork children inherit pristine modules.\n    random.seed(0)\n    if tree is None:\n        tree,status=filter_program(code)\n        if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.signal(signal.SIGPROF,lambda *_:(_ for _ in ()).throw(ProgramTimeout()))\n    signal.setitimer(signal.ITIMER_PROF,cpu_limit)\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:\n        signal.setitimer(signal.ITIMER_REAL,0)\n        signal.setitimer(signal.ITIMER_PROF,0)\n', 'final_representation.py': "# Generated by build_final_inference.py from s3_representation.py; do not edit.\nfrom tokenizers import Tokenizer,AddedToken\nMODES={'header_tail':'<|header_tail|>','header':'<|header|>','raw':'<|raw|>'}\nNUMBERS=[f'<|N{i}|>' for i in range(1,21)]\n\ndef restore_compact(text):\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError('Missing representation mode')\n    fragment = text[len(MODES[mode]):]\n    if mode == 'raw':\n        return fragment\n    if mode == 'header':\n        return 'def solve() -> int:\\n' + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith('    '):\n        raise ValueError('Missing final result expression')\n    lines[-1] = '    result = ' + lines[-1][4:]\n    body = ''.join(lines)\n    return 'def solve() -> int:\\n' + body + ('' if body.endswith('\\n') else '\\n') + '    return result\\n'\n\n\ndef tokenizer(path, variant):\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else NUMBERS\n    assert tok.add_special_tokens([AddedToken(t, normalized=False, special=True) for t in added]) == len(added)\n    # Original byte alphabet is unchanged; added specials decode literally, never get skipped.\n    visible = list(range(33,127)) + list(range(161,173)) + list(range(174,256))\n    bchars = {b:chr(b) for b in visible}\n    for i,b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256+i)\n    charbytes = {c:b for b,c in bchars.items()}\n    tokenbytes = [b'',b'',b''] + [bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)]\n    tokenbytes += [t.encode() for t in added]\n    assert all(tok.encode(t, add_special_tokens=False).ids == [8189+i] for i,t in enumerate(added))\n    return tok, tokenbytes\n\n", 'final_speed.py': '"""Opt-in decoder experiments. Frozen final_fast/final_decode remain the control."""\nfrom dataclasses import dataclass\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\nimport final_fast as Fast\n\n\nGRAPH_REFUSAL = (\n    \'CUDA graphs disabled: prefix length, KV slice addresses and active batch change every step. \'\n    \'One graph per position/batch would require hundreds of captures; full-capacity masked SDPA \'\n    \'changes reduction shapes and cannot promise bit-exact greedy. No untested graph fallback.\')\n\n\n@dataclass(frozen=True)\nclass Options:\n    preallocated_kv: bool = True  # Already present in the frozen baseline; no new speedup.\n    share_cross: bool = False\n    async_eos: bool = False\n    eos_every: int = 16\n    compact_done: bool = False\n    compact_fraction: float = .25\n    race_sampler: bool = False\n    lean_greedy: bool = False\n    cuda_graphs: bool = False\n\n    def __post_init__(self):\n        if not self.preallocated_kv:\n            raise ValueError(\'Baseline already preallocates KV; a cat regression is not an optimization\')\n        if self.cuda_graphs:\n            raise NotImplementedError(GRAPH_REFUSAL)\n        if self.eos_every < 1 or not 0 < self.compact_fraction <= 1:\n            raise ValueError(\'Invalid EOS/compaction interval\')\n\n\nclass Decoder(Fast.Decoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        if not share_cross or k != 1:\n            super().__init__(context, k)\n            return\n        # Immutable cross KV and source mask can be shared for greedy, no replication.\n        self.c, self.k, self.t, self.n = context, k, 0, len(context.src)\n        self.allowed, self.cross = context.allowed, context.cross\n        self.cross_bytes = sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [(context.positions.new_empty(shape), context.positions.new_empty(shape))\n                        for _ in context.cross]\n        self.valid = torch.zeros(self.n, context.capacity, dtype=torch.bool, device=context.src.device)\n\n    @torch.inference_mode()\n    def select(self, index):\n        """Copy only initialized self-KV prefix, then continue at the SAME position."""\n        if index.ndim != 1 or not len(index):\n            raise ValueError(\'Nonempty 1-D active indices required\')\n        n, cap = len(index), self.c.capacity\n        self.allowed = self.allowed.index_select(0, index)\n        self.cross = [tuple(t.index_select(0, index) for t in pair) for pair in self.cross]\n        # Release each old layer as soon as its replacement is ready. Holding both\n        # complete caches would make compaction itself an avoidable T4 OOM peak.\n        for number in range(len(self.self_kv)):\n            pair = self.self_kv[number]\n            new_pair = []\n            for old in pair:\n                new = old.new_empty((n, self.c.heads, cap, self.c.dim))\n                new[:, :, :self.t].copy_(old[:, :, :self.t].index_select(0, index))\n                new_pair.append(new)\n            self.self_kv[number] = tuple(new_pair)\n        self.n = n\n        self.valid = self.valid.index_select(0, index)\n\n\ndef race_pick(probabilities, noise):\n    """Independent Exp(1) race: argmax(p/E) has categorical probabilities p.\n\n    Uses exactly the same FP32 softmax as baseline; RNG consumption can change.\n    Avoids multinomial\'s probability validation/reductions. Noise storage is reused.\n    """\n    noise.exponential_()\n    return torch.div(probabilities, noise, out=noise).argmax(-1)\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, compute_logprob=True,\n           options=None, stats=None):\n    options = options or Options()\n    d = Decoder(context, k, options.share_cross)\n    device, n = context.src.device, d.n\n    token = torch.full((n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(n, dtype=torch.bool, device=device)\n    output = torch.full((n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums, lengths = torch.zeros(n, device=device), torch.zeros(n, dtype=torch.long, device=device)\n    final_sums, final_lengths = torch.zeros_like(sums), torch.zeros_like(lengths)\n    original = torch.arange(n, device=device)\n    compacted = False\n    noise = None\n    host = event = None\n    pending = False\n    async_eos = options.async_eos and device.type == \'cuda\'\n    if async_eos:\n        host = torch.empty(n, dtype=torch.bool, pin_memory=True)\n        event = torch.cuda.Event()\n    history = []\n    checks = 0\n\n    def flush():\n        final_sums.index_copy_(0, original, sums)\n        final_lengths.index_copy_(0, original, lengths)\n\n    def inspect(snapshot):\n        nonlocal token, done, sums, lengths, original, compacted, noise, host\n        if bool(snapshot.all()):\n            return True\n        if not options.compact_done:\n            return False\n        keep = (~snapshot).nonzero().flatten()  # CPU only: never CUDA nonzero/item.\n        if options.compact_done and len(keep) <= d.n*(1-options.compact_fraction):\n            flush()\n            index = keep.to(device)\n            before = d.n\n            d.select(index)\n            token, done = token[index], done[index]\n            sums, lengths, original = sums[index], lengths[index], original[index]\n            compacted, noise = True, None\n            history.append({\'step\': d.t, \'before\': before, \'after\': d.n})\n            if async_eos:\n                host = torch.empty(d.n, dtype=torch.bool, pin_memory=True)\n        return False\n\n    try:\n        for t in range(context.capacity):\n            if t % 16 == 0:\n                C.check_deadline(deadline)\n            if pending and event.query():\n                pending = False\n                if inspect(host):\n                    break\n            logits = d.step(token)\n            if temperature is not None or compute_logprob:\n                logits = logits.float()\n            logits[:, :2] = -torch.inf\n            if temperature is None:\n                pick = logits.argmax(-1)\n            else:\n                probabilities = F.softmax(logits / temperature, -1)\n                if options.race_sampler:\n                    if noise is None:\n                        noise = torch.empty_like(probabilities)\n                    pick = race_pick(probabilities, noise)\n                else:\n                    pick = torch.multinomial(probabilities, 1).squeeze(-1)\n            if compute_logprob:\n                lp = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n                sums.add_(torch.where(done, 0., lp))\n            if compute_logprob or not options.lean_greedy:\n                lengths.add_(~done)\n            token = pick.masked_fill(done, C.PAD)\n            if compacted:\n                output[:, t].index_copy_(0, original, token)\n            else:\n                output[:, t] = token\n            done |= token.eq(C.EOS)\n            if (t+1) % options.eos_every == 0:\n                if async_eos:\n                    if not pending:\n                        host.copy_(done, non_blocking=True)\n                        event.record()\n                        pending = True\n                        checks += 1\n                else:\n                    checks += 1\n                    if inspect(done.cpu()):\n                        break\n    finally:\n        # Also retain pinned storage on deadline/OOM exceptions until copy finishes.\n        if pending:\n            event.synchronize()\n    flush()\n    if stats is not None:\n        stats.update(steps=d.t, eos_checks=checks, compactions=history,\n                     async_eos=async_eos, rng_stream_may_change=options.race_sampler or options.compact_done)\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (final_sums / final_lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'kaggle_io.py': '"""Filesystem utilities shared by local preparation and commit notebooks; no credentials."""\nimport hashlib, json, os, shutil, tarfile, zipfile\nfrom pathlib import Path, PurePosixPath\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024**2),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef write(path, value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\');tmp.replace(path)\n\ndef safe_name(name):\n    p=PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name: raise ValueError(\'Unsafe archive path\')\n    if any(x in (\'kaggle.json\',\'.aws\',\'.ssh\') or x.startswith(\'.env\') for x in p.parts):\n        raise ValueError(\'Credentials are not dataset inputs\')\n    return p\n\ndef place(source, target):\n    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n    if target.exists():\n        if sha(source)!=sha(target): raise ValueError(\'Conflicting split download: \'+target.name)\n        return\n    shutil.copyfile(source,target)\n\ndef extract(archive, target):\n    """CRC/SHA checks happen again at application boundaries. No links/traversal/overwrite."""\n    target=Path(target);target.mkdir(parents=True,exist_ok=True)\n    # TAR may contain a ZIP whose end record makes is_zipfile(TAR) return True.\n    # Select the actual outer container by its explicit filename.\n    if Path(archive).suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(archive) as z:\n            if z.testzip(): raise ValueError(\'ZIP CRC failure\')\n            for info in z.infolist():\n                p=safe_name(info.filename)\n                if \'__MACOSX\' in p.parts or p.name==\'.DS_Store\': continue\n                if (info.external_attr>>16)&0o170000==0o120000: raise ValueError(\'ZIP symlink\')\n                if info.is_dir(): continue\n                dst=target/p;dst.parent.mkdir(parents=True,exist_ok=True)\n                tmp=dst.with_name(dst.name+\'.extract-tmp\')\n                with z.open(info) as src,tmp.open(\'wb\') as out: shutil.copyfileobj(src,out)\n                if dst.exists():\n                    if sha(tmp)!=sha(dst): raise ValueError(\'Conflicting ZIP member \'+str(p))\n                    tmp.unlink()\n                else: tmp.replace(dst)\n    else:\n        with tarfile.open(archive) as t:\n            for info in t:\n                p=safe_name(info.name)\n                if info.isdir(): continue\n                if not info.isfile(): raise ValueError(\'TAR links/devices forbidden\')\n                dst=target/p;dst.parent.mkdir(parents=True,exist_ok=True)\n                tmp=dst.with_name(dst.name+\'.extract-tmp\')\n                with t.extractfile(info) as src,tmp.open(\'wb\') as out: shutil.copyfileobj(src,out)\n                if dst.exists():\n                    if sha(tmp)!=sha(dst): raise ValueError(\'Conflicting TAR member\')\n                    tmp.unlink()\n                else: tmp.replace(dst)\n\ndef find_one(root, marker):\n    hits=list(Path(root).rglob(marker))\n    if len(hits)!=1: raise ValueError(f\'Expected one {marker}, found {len(hits)} under {root}\')\n    return hits[0]\n\ndef resolve_dataset(input_root, configured):\n    input_root=Path(input_root);path=input_root/Path(configured)\n    if path.is_dir():return path\n    # Support both /input/slug and the current /input/datasets/owner/slug layout,\n    # plus notebook-source outputs under /input/notebooks/owner/slug.\n    slug=Path(configured).name\n    hits=[p for p in input_root.rglob(slug) if p.is_dir()]\n    if len(hits)!=1:\n        visible=[str(p.relative_to(input_root)) for p in input_root.rglob(\'*\') if p.is_dir() and len(p.relative_to(input_root).parts)<=4]\n        raise ValueError(f\'Cannot resolve configured dataset {configured}; folders={visible[:60]}\')\n    return hits[0]\n\ndef materialize_dataset(dataset, working, marker):\n    """Kaggle normally expands uploads; also accept a still-packed payload archive."""\n    hits=list(Path(dataset).rglob(marker))\n    if hits: return find_one(dataset,marker).parent\n    out=Path(working)\n    for p in sorted(Path(dataset).rglob(\'*\')):\n        if p.is_file() and (p.suffix in (\'.zip\',\'.tar\') or p.name.endswith(\'.tar.gz\')):\n            extract(p,out)\n    return find_one(out,marker).parent\n\ndef disk_guard(root, limit=18*1024**3):\n    used=sum(p.stat().st_size for p in Path(root).rglob(\'*\') if p.is_file())\n    if used>limit: raise RuntimeError(f\'Working output exceeds reserve: {used} bytes\')\n    if shutil.disk_usage(root).free<512*1024**2: raise RuntimeError(\'Less than 512 MiB free\')\n    return used\n', 'kaggle_runtime.py': '"""Runtime adapters: immutable inputs, portable complete state, bounded local retention."""\nimport copy,json,os,shutil,time,uuid\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom kaggle_io import materialize_dataset,resolve_dataset,sha,write,disk_guard\n\nCACHE_SHA=\'2c691fef64fea8b4186b8acae54561d5620080286196a4390a23108776e80add\'\nORDER_SHA=\'737dac5a146ad15bb0ec70cb06fea59493277e4332dea440b2bdf2a6316c1637\'\nCP_SHA=\'8400c8e59da764c65fbcb574758acc72e8e034aedeb59b10bf979465f110ab5e\'\n\ndef copy_output(source,target):\n    """Only copy prior selected run output; preserve every source file, reject local conflicts."""\n    from exp26_common import copy_verified\n    source,target=Path(source),Path(target)\n    if source.resolve()==target.resolve():return\n    for p in source.rglob(\'*\'):\n        if p.is_file():copy_verified(p,target/p.relative_to(source))\n\ndef reuse_verified_references(source,target,sets,inputs):\n    """Reuse the pinned previous session\'s successful reference gate across GPU workers."""\n    from exp26_common import read_json,json_hash,copy_verified\n    import exp26_exec as N\n    source,target,inputs=Path(source),Path(target),Path(inputs)\n    records={}\n    for split,rows in sets.items():\n        name=split+\'-reference-execution.json\';receipt_name=split+\'-receipt.json\'\n        execution=read_json(source/name);receipt=read_json(source/receipt_name)\n        identity={\'rows_sha256\':json_hash(rows),\'saved_reference_sha256\':sha(inputs/name),\n                  \'policy\':\'same25 sandbox finite solve bool->int rel=abs1e-6\'}\n        assert receipt[\'identity\']==identity and receipt[\'sha256\']==sha(source/name),\'Reference reuse identity/SHA\'\n        expected=read_json(inputs/name);comparison=N.compare_exec(expected,execution)\n        count=4096 if split==\'heldout4k\' else 991\n        assert comparison[\'matches\']==comparison[\'denominator\']==count,\'Reference reuse values\'\n        assert [i for i,r in enumerate(execution[\'rows\']) if r[\'status\']==\'numeric\']==comparison[\'eligible_indices\'],\'Reference reuse finite indices\'\n        assert receipt[\'gate\']==comparison,\'Reference reuse gate receipt\'\n        copy_verified(source/name,target/name,receipt[\'sha256\'])\n        copy_verified(source/receipt_name,target/receipt_name)\n        records[split]={\'sha256\':receipt[\'sha256\'],\'numeric\':count}\n    write(target.parent/\'reference-reuse.json\',{\'source\':\'pinned previous output gpu0-a26/references\',\n          \'policy\':\'reuse once-verified reference receipts; no new numeric filtering or timeout change\',\'splits\':records})\n    return records\n\ndef install_runtime_adapters():\n    import torch\n    import exp26_train as T\n    import exp26_eval as E\n    import exp26_workflow as W\n    import exp26_exec as N\n    from exp26_fast_exec import run_chunks\n    from exp26_common import read_json,write_json\n    original=T.Checkpoints\n    class RetainedCheckpoints(original):\n        def __init__(self,drive,local,identity):\n            # Input/protocol/tensor/scheduler identity remains strict. Runtime fields are provenance.\n            latest=Path(drive)/\'latest.json\'\n            if latest.exists():\n                old=read_json(latest)[\'identity\']\n                for k in set(old)|set(identity):\n                    if k not in (\'torch\',\'CUDA\'):assert old.get(k)==identity.get(k),f\'Resume identity: {k}\'\n                if old!=identity:\n                    write_json(Path(drive)/\'runtime-migration.json\',{\'source_runtime\':{k:old[k] for k in (\'torch\',\'CUDA\')},\n                        \'execution_runtime\':{k:identity[k] for k in (\'torch\',\'CUDA\')},\n                        \'checkpoint_identity_retains_source_provenance\':True,\'bitwise_cross_runtime_equivalence_claimed\':False})\n                identity=old\n            super().__init__(drive,local,identity)\n        def save(self,*args,**kwargs):\n            model,opt,scaler,state,meta,cfg,spec=args\n            role=kwargs.get(\'role\',\'periodic\');latest=kwargs.get(\'latest\',True)\n            T.core.sync(next(model.parameters()).device);started=time.perf_counter()\n            name=f\'checkpoints/{role}-{meta["compute_updates"]:06d}-{uuid.uuid4().hex[:10]}.pt\'\n            weights=T.cpu(model.state_dict());digest=E.state_digest(weights)\n            payload={\'format\':\'nlp-hw02-wsd-v1\',\'experiment\':cfg.get(\'experiment\',26),\'identity\':self.identity,\n                \'config\':cfg,\'model_spec\':spec,\'model\':weights,\'optimizer\':T.cpu(opt.state_dict()),\n                \'scaler\':scaler.state_dict(),\'scaler_enabled\':scaler.is_enabled(),\'scheduler\':T.schedule_state(state,cfg),\n                \'rng\':T.cpu(T.rng_state()),\'training\':copy.deepcopy(state),\'experiment_history\':copy.deepcopy(meta),\n                \'model_tensor_sha256\':digest,\'purpose\':\'CPU_smoke\' if self.identity[\'smoke\'] else \'Kaggle_T4_training\',\n                \'execution_runtime\':{\'torch\':str(torch.__version__),\'CUDA\':torch.version.cuda}}\n            target=self.local/name;target.parent.mkdir(parents=True,exist_ok=True);tmp=target.with_suffix(\'.pt.tmp\')\n            torch.save(payload,tmp);tmp.replace(target);digest_file=sha(target)\n            from exp26_common import copy_verified\n            copy_verified(target,self.drive/name,digest_file)\n            receipt={\'file\':name,\'sha256\':digest_file,\'bytes\':target.stat().st_size,\'model_tensor_sha256\':digest,\n                \'main_updates\':state[\'main_updates\'],\'branch_updates\':state[\'branch_updates\'],\'phase\':state[\'phase\'],\n                \'data_cursor\':state[\'data_cursor\'],\'role\':role,\'identity\':self.identity}\n            write_json(self.drive/(name+\'.json\'),receipt)\n            if latest:write_json(self.drive/\'latest.json\',receipt)\n            meta[\'checkpoint_io_seconds\']+=time.perf_counter()-started\n            keep={receipt[\'file\']}\n            for selector in (\'latest.json\',\'final-checkpoint.json\'):\n                p=self.drive/selector\n                if p.exists():keep.add(read_json(p)[\'file\'])\n            if state.get(\'active_anchor\'):keep.add(state[\'active_anchor\'][\'file\'])\n            for p in (self.drive/\'points\').glob(\'*/checkpoint.json\'):keep.add(read_json(p)[\'file\'])\n            for root in {self.drive,self.local}:\n                for p in (root/\'checkpoints\').glob(\'*.pt\'):\n                    if p.relative_to(root).as_posix() not in keep:\n                        p.unlink();p.with_suffix(\'.pt.json\').unlink(missing_ok=True)\n            disk_guard(self.drive.parents[0] if self.drive.name.startswith(\'gpu\') else self.drive)\n            return receipt\n    T.Checkpoints=RetainedCheckpoints\n    def fast(codes,**kwargs):\n        result=run_chunks(codes,workers=2,chunk_rows=2000)\n        result[\'summary\'][\'complete\']=True;return result\n    proxy=SimpleNamespace(evaluate_programs=fast,compare_exec=N.compare_exec)\n    E.numeric=proxy;W.numeric=proxy # leave N untouched for the dead-chunk per-program fallback\n    return RetainedCheckpoints\n\ndef prepare_starter(dataset,prepared,kind,read_only=False):\n    from exp26_input import import_starter\n    from exp26_common import copy_verified\n    import exp26_protocol as P\n    dataset,prepared=Path(dataset),Path(prepared)\n    name=\'26a-cache-inputs-tokenizer-and-exclusions.zip\' if kind==\'cache-inputs\' else \'26b-training-inputs-d0-initial-and-evaluation.zip\'\n    archives=list(dataset.rglob(name))\n    direct=[p.parent for p in dataset.rglob(\'protected-questions.json\') if (p.parent/\'tokenizer.json\').exists()]\n    if not archives and not direct:\n        root=materialize_dataset(dataset,prepared.parent/(\'unpacked-\'+kind),name)\n        archives=list(root.rglob(name))\n    if archives:\n        assert len(archives)==1\n        import_starter(archives[0],prepared,kind)\n    else:\n        assert len(direct)==1\n        names={\'protected-questions.json\',\'tokenizer.json\'} if kind==\'cache-inputs\' else set(P.INPUT_FILES)\n        if read_only:\n            for n in names:assert sha(direct[0]/n)==P.INPUT_FILES[n]\n            return direct[0]\n        for n in names:copy_verified(direct[0]/n,prepared/n,P.INPUT_FILES[n])\n    return prepared\n\ndef inputs(config,working):\n    from exp26_input import import_starter,validate_training_inputs\n    from exp26_cache import TokenCache\n    from exp26_common import read_json,sha_file\n    import exp26_protocol as P\n    working=Path(working);input_root=Path(config[\'input_root\'])\n    scratch=Path(config[\'temp_root\']) if config.get(\'session_kind\')==\'s2\' else working\n    if config.get(\'session_kind\')==\'s2\':assert not scratch.resolve().is_relative_to(working.resolve())\n    prepared=prepare_starter(resolve_dataset(input_root,config[\'starter_dataset\']),scratch/\'inputs\',\'training-inputs\',\n                             read_only=config.get(\'session_kind\')==\'s2\')\n    sets,test,protected=validate_training_inputs(prepared)\n    cache_root=materialize_dataset(resolve_dataset(input_root,config[\'cache_dataset\']),scratch/\'unpacked-cache\',\'cache-manifest.json\')\n    assert sha_file(cache_root/\'cache-manifest.json\')==CACHE_SHA,\'Pinned cache manifest required\'\n    cache=TokenCache(cache_root,protected)\n    # Initial26b input or explicitly attached previous S1 output. Never choose by mtime.\n    resume_base=resolve_dataset(input_root,config[\'resume_dataset\'])\n    run=materialize_dataset(resume_base,scratch/\'unpacked-resume\',\'latest.json\')\n    assert sha_file(run/\'order.npy\')==ORDER_SHA\n    order_meta=read_json(run/\'order.json\');assert order_meta[\'file\'][\'sha256\']==ORDER_SHA\n    return prepared,sets,test,cache,run,order_meta\n\ndef prepare_a(config,working,run):\n    from exp26_common import read_json,sha_file\n    root=Path(working)/\'gpu0-a26\'\n    resume=resolve_dataset(config[\'input_root\'],config[\'previous_output\']) if config.get(\'previous_output\') else None\n    if config.get(\'previous_output\'):\n        copy_output(resume/\'gpu0-a26\',root)\n    else:\n        latest=read_json(Path(run)/\'latest.json\')\n        assert latest[\'sha256\']==CP_SHA and sha_file(Path(run)/latest[\'file\'])==CP_SHA\n        assert latest[\'main_updates\']==46875 and latest[\'phase\']==\'main\'\n        copy_output(run,root)\n    return root\n\ndef evaluate_a(model_path,tag,previous,sets,refs,inputs,root,device,D0):\n    import exp26_eval as E\n    from exp26_common import read_json,write_json\n    prevpath=root/\'points\'/str(previous)/\'evaluation/metrics.json\'\n    available=prevpath.exists() and all((root/\'points\'/str(previous)/\'evaluation\'/(split+\'-generations.json\')).exists() for split in sets)\n    prev=read_json(prevpath) if available else None\n    out=root/\'points\'/tag/\'evaluation\'\n    metrics=E.evaluate_point(model_path,sets,refs,inputs/\'tokenizer.json\',device,out,tag,D0,prev)\n    metrics[\'paired_comparison_basis\']=previous if available else \'D0; previous generations unavailable in input\'\n    write_json(out/\'metrics.json\',metrics);return metrics\n', 'l1_cache.py': '"""CPU compact cache, exact row identities from sealed 26a + 26c. No Torch."""\nimport argparse, hashlib, json, math, multiprocessing as mp, os, shutil, subprocess, sys, time\nfrom array import array\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom pathlib import Path\nimport numpy as np\nfrom tokenizers import Tokenizer\nimport s3_representation as R\nfrom kaggle_io import sha, write, materialize_dataset, resolve_dataset\n\nBASE_SHA = \'2c691fef64fea8b4186b8acae54561d5620080286196a4390a23108776e80add\'\nEXT_SHA = \'7417b6ffc0a0cfeeafb501610fdee8b43bd1bcea2935ded64cc964cba79800a3\'\nROWS = 9725737\nORDER_SEED = 2026101201\nFILES = [\'source.npy\', \'source-offsets.npy\', \'target.npy\', \'target-offsets.npy\',\n         \'provenance-rows.npy\', \'question-sha256.npy\']\n\ndef record(path):\n    path = Path(path)\n    return {\'file\': path.name, \'bytes\': path.stat().st_size, \'sha256\': sha(path)}\n\ndef decode(table, ids):\n    return b\'\'.join(table[int(i)] for i in ids if i >= 3).decode(\'utf8\')\n\ndef convert_shard(job):\n    original, receipt, destination, tokpath, sample_n = job\n    original, destination = Path(original), Path(destination)\n    src = original / receipt[\'folder\']; out = destination / receipt[\'folder\']\n    out.mkdir(parents=True, exist_ok=True)\n    for rec in receipt[\'files\']:\n        p = src / rec[\'file\']\n        assert p.stat().st_size == rec[\'bytes\'] and sha(p) == rec[\'sha256\']\n    tok, table = R.tokenizer(tokpath, 30)\n    target = np.load(src/\'target.npy\', mmap_mode=\'r\', allow_pickle=False)\n    offsets = np.load(src/\'target-offsets.npy\', mmap_mode=\'r\', allow_pickle=False)\n    n = len(offsets)-1; new_offsets = np.empty(n+1, dtype=np.int64); new_offsets[0] = 0\n    chosen = set(np.random.default_rng(ORDER_SEED+receipt[\'shard\']).choice(n, min(sample_n,n), replace=False).tolist())\n    samples = []; started = time.monotonic(); total = 0; modes = {}\n    raw = out/\'target.bin.tmp\'\n    with raw.open(\'wb\') as stream:\n        for lo in range(0, n, 2048):\n            codes = [decode(table, target[offsets[i]:offsets[i+1]]) for i in range(lo, min(n,lo+2048))]\n            fragments = [R.compact(code) for code in codes]\n            encoded = tok.encode_batch(fragments, add_special_tokens=False)\n            block = array(\'h\')\n            for j, (code, text, enc) in enumerate(zip(codes, fragments, encoded)):\n                i = lo+j; ids = [v+3 for v in enc.ids]\n                block.extend(ids); total += len(ids); new_offsets[i+1] = total\n                mode = text.split(\'|>\',1)[0]+\'|>\'; modes[mode] = modes.get(mode,0)+1\n                if i in chosen:\n                    roundtrip = decode(table, ids)\n                    assert roundtrip == text, \'BPE round-trip mismatch\'\n                    restored = R.restore_compact(roundtrip)\n                    samples.append({\'shard\': receipt[\'shard\'], \'local_row\': i, \'original\': code, \'restored\': restored})\n            stream.write(block.tobytes())\n    mm = np.lib.format.open_memmap(out/\'target.npy\', mode=\'w+\', dtype=np.int16, shape=(total,))\n    binary = np.memmap(raw, mode=\'r\', dtype=np.int16)\n    for lo in range(0,total,4*1024**2): mm[lo:lo+4*1024**2] = binary[lo:lo+4*1024**2]\n    mm.flush(); del mm, binary; raw.unlink()\n    np.save(out/\'target-offsets.npy\',new_offsets,allow_pickle=False)\n    for name in FILES:\n        if name not in (\'target.npy\',\'target-offsets.npy\'): shutil.copyfile(src/name,out/name)\n    result = {\'shard\':receipt[\'shard\'], \'folder\':receipt[\'folder\'], \'rows\':n, \'target_tokens\':total,\n              \'original_shard_manifest_sha256\':sha(src/\'manifest.json\'),\n              \'original_files\':receipt[\'files\'], \'files\':[record(out/name) for name in FILES],\n              \'modes\':modes, \'seconds\':time.monotonic()-started}\n    write(out/\'manifest.json\',result)\n    write(out/\'roundtrip-sample.json\',samples)\n    print(\'COMPACT SHARD\',receipt[\'shard\'],\'rows\',n,\'seconds\',round(result[\'seconds\'],1),flush=True)\n    return result\n\nclass CompactCache:\n    def __init__(self, root, verify=True):\n        self.root=Path(root); self.manifest=json.loads((self.root/\'compact-manifest.json\').read_text())\n        assert self.manifest[\'sealed\'] and self.manifest[\'representation\']==30\n        self.manifest_sha=sha(self.root/\'compact-manifest.json\'); self.parts=[]; starts=[0]; sl=[]; tl=[]\n        for rec in self.manifest[\'shards\']:\n            folder=self.root/rec[\'folder\']\n            if verify:\n                for f in rec[\'files\']:\n                    p=folder/f[\'file\']; assert p.stat().st_size==f[\'bytes\'] and sha(p)==f[\'sha256\']\n            a={n:np.load(folder/n,mmap_mode=\'r\',allow_pickle=False) for n in FILES}\n            n=len(a[\'source-offsets.npy\'])-1\n            assert n==rec[\'rows\']==len(a[\'target-offsets.npy\'])-1\n            for prefix in (\'source\',\'target\'):\n                assert a[prefix+\'.npy\'].dtype==np.int16\n                o=a[prefix+\'-offsets.npy\']; assert o.dtype==np.int64 and o[0]==0 and o[-1]==len(a[prefix+\'.npy\'])\n                lengths=np.diff(o);assert np.all(lengths>0)\n                assert int(lengths.max())+(prefix==\'target\')<=2048,\'Position/length overflow\'\n                if verify:\n                    ids=a[prefix+\'.npy\'];assert int(ids.min())>=1 and int(ids.max())<8195,\'Token ID overflow\'\n            self.parts.append(a); starts.append(starts[-1]+n)\n            sl.append(np.diff(a[\'source-offsets.npy\']).astype(np.uint16)); tl.append(np.diff(a[\'target-offsets.npy\']).astype(np.uint16))\n        self.starts=np.asarray(starts); self.source_lengths=np.concatenate(sl); self.target_lengths=np.concatenate(tl)\n        assert len(self)==self.manifest[\'rows\']\n    def __len__(self): return int(self.starts[-1])\n    def __getitem__(self,index):\n        index=int(index)\n        if not 0<=index<len(self): raise IndexError(index)\n        k=int(np.searchsorted(self.starts,index,side=\'right\')-1); j=index-int(self.starts[k]); a=self.parts[k]\n        so,to=a[\'source-offsets.npy\'],a[\'target-offsets.npy\']\n        return a[\'source.npy\'][so[j]:so[j+1]],a[\'target.npy\'][to[j]:to[j+1]]\n\ndef make_order(cache, seed=ORDER_SEED, batch=128, chunk=4096):\n    rng=np.random.default_rng(seed); order=rng.permutation(len(cache)).astype(np.uint32)\n    for lo in range(0,len(order),chunk):\n        part=order[lo:lo+chunk]; ns=cache.source_lengths[part].astype(np.int32); nt=cache.target_lengths[part].astype(np.int32)\n        part=part[np.lexsort((nt,ns,ns+nt+1))]; full=len(part)//batch\n        if full: part[:full*batch]=part[:full*batch].reshape(full,batch)[rng.permutation(full)].reshape(-1)\n        order[lo:lo+len(part)]=part\n    assert np.array_equal(np.sort(order),np.arange(len(cache),dtype=np.uint32))\n    return order\n\ndef build(base, extension, tokpath, output, workers=4, sample_rows=50000, smoke=False):\n    output=Path(output); output.mkdir(parents=True,exist_ok=True); started=time.monotonic()\n    roots=[Path(base),Path(extension)]; originals=[json.loads((p/\'cache-manifest.json\').read_text()) for p in roots]\n    hashes=[sha(p/\'cache-manifest.json\') for p in roots]\n    if not smoke:\n        assert hashes==[BASE_SHA,EXT_SHA]\n        assert [r[\'shard\'] for m in originals for r in m[\'shards\']]==list(range(60))\n        assert sum(m[\'rows\'] for m in originals)==ROWS\n    assert all(m[\'sealed\'] and m[\'fresh_evaluations\']==0 for m in originals)\n    tok,_=R.tokenizer(tokpath,30); tok.save(str(output/\'tokenizer.json\'))\n    jobs=[(str(root),r,str(output),str(tokpath),math.ceil(sample_rows/sum(len(m[\'shards\']) for m in originals)))\n          for root,m in zip(roots,originals) for r in m[\'shards\']]\n    converted=[]\n    with ProcessPoolExecutor(max_workers=workers,mp_context=mp.get_context(\'spawn\')) as pool:\n        for f in as_completed([pool.submit(convert_shard,j) for j in jobs]): converted.append(f.result())\n    converted.sort(key=lambda r:r[\'shard\'])\n    manifest={\'format\':\'nlp-hw02-compact-cache-v1\',\'representation\':30,\'mode_tokens\':R.MODES,\n              \'representation_code_sha256\':sha(Path(R.__file__)), \'original_manifest_sha256\':hashes,\n              \'original_tokenizer_sha256\':sha(tokpath),\'tokenizer\':record(output/\'tokenizer.json\'),\n              \'rows\':sum(r[\'rows\'] for r in converted),\'shards\':converted,\'sealed\':True,\n              \'fresh_evaluations\':0,\'selection\':\'exact sealed finite rows26a0-41 +26c42-59; no new filtering or deduplication\',\n              \'smoke\':smoke}\n    write(output/\'compact-manifest.json\',manifest)\n    cache=CompactCache(output); order=make_order(cache); np.save(output/\'order.npy\',order,allow_pickle=False)\n    order_info={\'seed\':ORDER_SEED,\'rows\':len(cache),\'batch\':128,\'chunk\':4096,\'bijection\':True,\n                \'file\':record(output/\'order.npy\'),\'cache_manifest_sha256\':cache.manifest_sha}\n    write(output/\'order.json\',order_info)\n    samples=[s for r in converted for s in json.loads((output/r[\'folder\']/\'roundtrip-sample.json\').read_text())]\n    # Numeric execution has a low virtual-address-space limit. Its spawn entrypoint\n    # must not import NumPy/OpenBLAS or Torch before that limit is installed.\n    env=dict(os.environ,OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\')\n    subprocess.run([sys.executable,str(Path(__file__).with_name(\'l1_roundtrip.py\')),\'--root\',str(output),\n                    \'--workers\',str(workers)],env=env,check=True,timeout=1200)\n    roundtrip=json.loads((output/\'roundtrip-report.json\').read_text());score=roundtrip[\'score\']\n    assert score[\'denominator\']==score[\'matches\']==len(samples), \'Finite answer round-trip failed\'\n    if not smoke: assert len(samples)>=50000\n    result={\'status\':\'complete\',\'rows\':len(cache),\'shards\':[r[\'shard\'] for r in converted],\n            \'manifest_sha256\':cache.manifest_sha,\'order\':order_info,\'roundtrip_rows\':len(samples),\n            \'answer_mismatches\':0,\'roundtrip\':score,\'seconds\':time.monotonic()-started,\'GPU\':False,\n            \'roundtrip_executor\':roundtrip[\'executor_entrypoint\'],\n            \'representation_code_sha256\':manifest[\'representation_code_sha256\'],\'tokenizer_sha256\':manifest[\'tokenizer\'][\'sha256\']}\n    write(output/\'cache-summary.json\',result)\n    for r in converted:(output/r[\'folder\']/\'roundtrip-sample.json\').unlink()\n    print(\'COMPACT CACHE COMPLETE\',json.dumps({k:result[k] for k in (\'rows\',\'manifest_sha256\',\'roundtrip_rows\',\'answer_mismatches\',\'seconds\')}),flush=True)\n    return result\n\ndef main():\n    p=argparse.ArgumentParser(); p.add_argument(\'--config\',required=True); a=p.parse_args(); cfg=json.loads(Path(a.config).read_text())\n    inp=Path(cfg[\'input_root\']); scratch=Path(cfg[\'temp_root\'])\n    base=materialize_dataset(resolve_dataset(inp,\'nlp-hw02-26-token-cache\'),scratch/\'base\',\'cache-manifest.json\')\n    ext=materialize_dataset(resolve_dataset(inp,\'nlp-hw02-26c-cpu-cache\'),scratch/\'ext\',\'cache-manifest.json\')\n    from kaggle_runtime import prepare_starter\n    starter=prepare_starter(resolve_dataset(inp,\'nlp-hw02-26b-inputs\'),scratch/\'starter\',\'training-inputs\',read_only=True)\n    build(base,ext,starter/\'tokenizer.json\',Path(cfg[\'working_root\'])/\'compact-cache\')\n\nif __name__==\'__main__\':main()\n', 'l1_eval.py': '"""Independent FP16 4D greedy, restored exec, token CE and fresh full-test timer."""\nimport gc, json, os, signal, subprocess, sys, tempfile, time\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom tokenizers import Tokenizer\nimport exp26_core as C, exp26_exec as N, exp26_eval as E\nimport exp27b_fast as Fast\nimport s3_representation as R\nfrom exp26_fast_exec import run_chunks\nfrom exp26_order import collate\nfrom kaggle_io import sha, write\n\ndef load(path,device):\n    payload=torch.load(path,map_location=\'cpu\',weights_only=True)\n    assert payload[\'format\']==\'nlp-hw02-model-only-v1\' and payload[\'optimizer_included\'] is False\n    model=C.SimpleTransformer(**payload[\'model_spec\'],dropout=0.).eval()\n    if device.type==\'cuda\':model.half()\n    model.to(device);model.load_state_dict(payload[\'model\']);del payload\n    return model\n\n@torch.no_grad()\ndef generate(model,rows,tok,table,device,deadline,smoke=False,batch=256):\n    encoded=[[C.BOS]+C.encode(tok,r[\'question\'])+[C.EOS] for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));examples=[None]*len(rows)\n    lo=0; cap=8 if smoke else 512;attempts=[];C.sync(device);started=time.perf_counter()\n    while lo<len(order):\n        C.check_deadline(deadline); indices=order[lo:lo+batch]\n        src=ctx=ids=lps=None\n        try:\n            src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=C.PAD).to(device)\n            ctx=Fast.Context(model,src,cap);ids,lps,_=Fast.decode(ctx,deadline=deadline)\n            for i,seq in zip(indices,ids[:,0].tolist()):\n                active=C.active_tokens(seq);text=C.decode_bytes(table,active).decode(\'utf8\',errors=\'replace\')\n                try:code=R.restore_compact(text);error=None\n                except (ValueError,IndexError):code=\'\';error=\'invalid representation\'\n                examples[i]={\'id\':i,\'question\':rows[i][\'question\'],\'generated_token_ids\':active,\n                             \'represented_code\':text,\'generated_code\':code,\'restore_error\':error,\'eos_emitted\':C.EOS in active}\n            lo+=len(indices)\n        except torch.cuda.OutOfMemoryError:\n            attempts.append({\'lo\':lo,\'batch\':batch,\'status\':\'OOM\'});batch//=2\n            if batch<8:raise\n        finally:\n            del src,ctx,ids,lps\n            if attempts and device.type==\'cuda\':gc.collect();torch.cuda.empty_cache()\n    C.sync(device)\n    return examples,{\'seconds\':time.perf_counter()-started,\'batch\':batch,\'OOM_attempts\':attempts,\n                     \'SDPA_dimensions\':4,\'precision\':\'FP16\' if device.type==\'cuda\' else \'FP32 tiny smoke\',\n                     \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None}\n\ndef execute(examples,smoke=False,deadline=None):\n    # The entire executor, including pool joins/retries, has a parent wall limit.\n    C.check_deadline(deadline)\n    with tempfile.TemporaryDirectory(prefix=\'l1-exec-\') as folder:\n        source=Path(folder)/\'codes.json\';target=Path(folder)/\'result.json\'\n        write(source,[r[\'generated_code\'] for r in examples])\n        env=dict(os.environ,OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\')\n        child=subprocess.Popen([sys.executable,str(Path(__file__).with_name(\'l1_execute.py\')),\n            str(source),str(target),\'1\' if smoke else \'2\'],env=env,start_new_session=True,\n            stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)\n        until=min(time.monotonic()+900.,deadline) if deadline else time.monotonic()+900.\n        try:\n            while child.poll() is None:\n                if time.monotonic()>=until:raise C.BudgetExpired(\'Execution wall budget exhausted\')\n                time.sleep(.05)\n            if child.returncode:raise RuntimeError(\'Bounded execution child failed\')\n            return json.loads(target.read_text())\n        finally:\n            try:os.killpg(child.pid,signal.SIGKILL)\n            except ProcessLookupError:pass\n            try:child.wait(timeout=.2)\n            except subprocess.TimeoutExpired:pass\n\n@torch.no_grad()\ndef ce(model,rows,tok,device,deadline,smoke=False):\n    cached=[([1]+C.encode(tok,r[\'question\'])+[2],C.encode(tok,R.compact(r[\'code\']))) for r in rows]\n    order=sorted(range(len(cached)),key=lambda i:len(cached[i][0])+len(cached[i][1]))\n    total=0.;tokens=0;started=time.perf_counter()\n    lo=0;batch=2 if smoke else 16;retries=[]\n    while lo<len(order):\n        C.check_deadline(deadline);src=tgt=labels=logits=None\n        try:\n            indices=order[lo:lo+batch];src,tgt,labels,counts=collate(cached,indices,device)\n            logits=model(src,tgt).float()\n            loss=float(F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=0,reduction=\'sum\'))\n            if not torch.isfinite(torch.tensor(loss)):raise FloatingPointError(\'Nonfinite evaluation CE\')\n            total+=loss;tokens+=counts[\'target_tokens\'];lo+=len(indices)\n        except torch.cuda.OutOfMemoryError:\n            if batch==1:raise\n            retries.append(batch);batch=max(1,batch//2)\n        finally:\n            del src,tgt,labels,logits\n            if retries and device.type==\'cuda\':gc.collect();torch.cuda.empty_cache()\n    return {\'CE\':total/tokens,\'tokens\':tokens,\'seconds\':time.perf_counter()-started,\n            \'forward\':\'FP16 evaluated weights on GPU; FP32 softmax/sum\',\'representation\':30,\'comparable_to_raw_CE\':False,\n            \'batch\':batch,\'OOM_retries\':retries}\n\ndef evaluate(point,out,inputs,sets,refs,device,deadline,smoke=False,test_rows=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);inputs=Path(inputs)\n    tok,table=R.tokenizer(inputs/\'tokenizer.json\',30);original=Tokenizer.from_file(str(inputs/\'tokenizer.json\'))\n    result={\'weights_sha256\':sha(point),\'splits\':{},\'restoration\':\'exact S3 compact30/mode-token code\',\n            \'decoder\':\'exp27b_fast physical 4D greedy FP16\',\'fresh_evaluated\':0}\n    model=load(point,device)\n    try:\n        for split,rows in sets.items():\n            examples,timing=generate(model,rows,tok,table,device,deadline,smoke,batch=2 if smoke else 256)\n            write(out/(split+\'-generations.json\'),examples)\n            execution=execute(examples,smoke,deadline);write(out/(split+\'-execution.json\'),execution)\n            score=N.compare_exec(refs[split],execution)\n            if not smoke and test_rows is None:assert score[\'denominator\']==(4096 if split==\'heldout4k\' else 991)\n            result[\'splits\'][split]={\'exec\':score,\'CE\':ce(model,rows,tok,device,deadline,smoke),\n                                    \'length_accuracy\':E.length_accuracy(rows,original,score),\'generation\':timing,\n                                    \'restore_errors\':sum(r[\'restore_error\'] is not None for r in examples)}\n            write(out/\'partial-metrics.json\',result)\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    # Entire test timer is fresh: file read, weights/tokenizer load, generation/restore,\n    # bounded execution and final answers close. No previous generations reused.\n    C.sync(device);started=time.perf_counter()\n    test=json.loads((inputs/\'test.json\').read_text())\n    if test_rows is not None:test=test[:test_rows]\n    model=load(point,device);tok,table=R.tokenizer(inputs/\'tokenizer.json\',30)\n    try:\n        examples,timing=generate(model,test,tok,table,device,deadline,smoke,batch=2 if smoke else 256)\n        execution=execute(examples,smoke,deadline)\n        answers=out/\'test-answers.jsonl\'\n        with answers.open(\'w\') as f:\n            for row in examples:f.write(json.dumps({\'id\':row[\'id\'],\'question\':row[\'question\'],\'code\':row[\'generated_code\']},ensure_ascii=False)+\'\\n\')\n        C.sync(device);elapsed=time.perf_counter()-started\n        write(out/\'test-generations.json\',examples);write(out/\'test-execution.json\',execution)\n        result[\'test\']={\'rows\':len(test),\'full_pipeline_seconds\':elapsed,\'generation\':timing,\n                        \'execution\':execution[\'summary\'],\'answers_sha256\':sha(answers),\'cache_reuse\':False,\n                        \'timer\':\'test read + model/tokenizer load + FP16 4D greedy + restore + exec + answers close\',\n                        \'accuracy\':None}\n        write(out/\'metrics.json\',result);return result\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n', 'l1_runtime.py': '"""Long constant-LR main trajectory, disposable cooldowns, exact complete resume."""\nimport copy, gc, json, math, random, time, uuid\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nimport exp26_core as C\nimport exp26_train as T\nfrom exp26_order import collate\nfrom kaggle_io import sha, write\n\nSPECS = {\n    \'C\':dict(d_model=512,heads=8,encoder_layers=6,decoder_layers=6,swiglu_dim=1368,vocab=8195),\n    \'D\':dict(d_model=640,heads=10,encoder_layers=8,decoder_layers=8,swiglu_dim=1712,vocab=8195)}\nPEAK_LR={\'C\':0.0015,\'D\':0.0012}\nMICRO_BATCHES=(128,96,64,32,16,8,4,2,1)\nclass UnsafeAdamOOM(RuntimeError):pass\n\ndef seed_all(seed=42):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef new_model(spec, device):\n    seed_all(); model=C.SimpleTransformer(**spec,dropout=0.).to(device)\n    with torch.no_grad(): model.tokens.weight.div_(math.sqrt(model.width))\n    return model\n\ndef optimizer(model, lr, smoke):\n    return torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=.01,**({} if smoke else {\'fused\':True}))\n\ndef take(order,cursor,batch):\n    return np.asarray(order[(np.arange(batch,dtype=np.int64)+cursor)%len(order)])\n\ndef lr_at(update, peak, warmup):\n    return peak*min(1.,update/max(1,warmup))\n\ndef update(model,opt,scaler,cache,indices,device,micro,lr):\n    """An exact effective batch; 96+32 and 64+64 have token-weighted CE."""\n    C.sync(device); started=time.perf_counter(); opt.zero_grad(set_to_none=True)\n    tokens=sum(len(cache[int(i)][1])+1 for i in indices); total=0.\n    for lo in range(0,len(indices),micro):\n        src,tgt,labels,counts=collate(cache,indices[lo:lo+micro],device)\n        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=scaler.is_enabled()):\n            logits=model(src,tgt)\n            loss=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=C.PAD,reduction=\'sum\')/tokens\n        if not bool(torch.isfinite(loss).detach()):raise FloatingPointError(\'Nonfinite loss\')\n        scaler.scale(loss).backward(); total+=float(loss.detach()); del src,tgt,labels,logits,loss\n    for group in opt.param_groups:group[\'lr\']=lr\n    scaler.unscale_(opt); norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n    skipped=bool(any(float(t)>0 for t in scaler._found_inf_per_device(opt).values())) if scaler.is_enabled() else False\n    if not skipped and not bool(torch.isfinite(norm)):raise FloatingPointError(\'Nonfinite successful gradient\')\n    try:\n        scaler.step(opt)\n    except torch.cuda.OutOfMemoryError as error:\n        # Adam may have modified only some parameters. Never retry this state.\n        error.optimizer_may_be_modified=True\n        raise\n    scaler.update(); C.sync(device)\n    return {\'seconds\':time.perf_counter()-started,\'CE\':total,\'AMP_skip\':int(skipped),\'lr\':lr,\n            \'pairs\':len(indices),\'tokens\':tokens,\'gradient_norm\':float(norm) if torch.isfinite(norm) else None}\n\ndef resilient_update(model,opt,scaler,cache,indices,device,micro,lr):\n    rng=T.cpu(T.rng_state());scale=copy.deepcopy(scaler.state_dict());attempts=[]\n    while True:\n        try:\n            result=update(model,opt,scaler,cache,indices,device,micro,lr)\n            result[\'OOM_retries\']=attempts\n            return result,micro\n        except torch.cuda.OutOfMemoryError as error:\n            unsafe=getattr(error,\'optimizer_may_be_modified\',False)\n        # The exception frame (and its CUDA tensors) must be gone before retry.\n        model.zero_grad(set_to_none=True);gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n        if unsafe:raise UnsafeAdamOOM(\'OOM inside Adam: restore last committed full checkpoint\')\n        smaller=[m for m in MICRO_BATCHES if m<micro]\n        if not smaller:raise RuntimeError(\'OOM even at micro-batch1; last checkpoint preserved\')\n        attempts.append({\'micro_batch\':micro,\'status\':\'late OOM before optimizer step\'})\n        micro=smaller[0];scaler.load_state_dict(scale)\n        if scaler.is_enabled():scaler._per_optimizer_states.clear()\n        T.restore_rng(rng,device)\n        print(\'OOM RETRY micro\',micro,flush=True)\n\ndef probe(model, cache, device, peak, smoke=False, batch=128):\n    """Full forward/backward/Adam allocation on longest rows, then exact reset."""\n    initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; rng=T.cpu(T.rng_state()); attempts=[]; choice=None\n    lengths=cache.source_lengths.astype(np.int32)+cache.target_lengths.astype(np.int32)\n    count=min(batch,len(cache)); selected=np.argpartition(lengths,-count)[-count:]\n    # Also exercise the individually longest source and target.\n    selected[0]=int(cache.source_lengths.argmax())\n    selected[min(1,len(selected)-1)]=int(cache.target_lengths.argmax())\n    indices=np.resize(selected,batch)\n    for micro in ([batch] if smoke else [m for m in MICRO_BATCHES if m<=batch]):\n        opt=None; scaler=None\n        try:\n            model.load_state_dict(initial); T.restore_rng(rng,device)\n            if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n            opt=optimizer(model,peak,smoke); scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1024)\n            result=update(model,opt,scaler,cache,indices,device,micro,peak/1000)\n            memory=torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None\n            enough=smoke or memory<.93*torch.cuda.get_device_properties(device).total_memory\n            attempts.append({\'micro_batch\':micro,\'status\':\'passed\' if enough else \'insufficient_headroom\',\n                             \'peak_reserved_bytes\':memory,\'probe_seconds\':result[\'seconds\']})\n            if enough:choice=micro;break\n        except torch.cuda.OutOfMemoryError:attempts.append({\'micro_batch\':micro,\'status\':\'OOM\'})\n        finally:\n            model.zero_grad(set_to_none=True); del opt,scaler;gc.collect()\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n    model.load_state_dict(initial); T.restore_rng(rng,device);del initial\n    if choice is None:raise RuntimeError(\'No safe micro-batch down to1\')\n    return choice,attempts\n\ndef state_payload(model,opt,scaler,state,spec,identity):\n    return {\'format\':\'nlp-hw02-l1-full-v1\',\'model\':T.cpu(model.state_dict()),\'optimizer\':T.cpu(opt.state_dict()),\n            \'scaler\':scaler.state_dict(),\'scaler_enabled\':scaler.is_enabled(),\'rng\':T.cpu(T.rng_state()),\n            \'state\':copy.deepcopy(state),\'model_spec\':spec,\'identity\':identity,\n            \'scheduler\':{\'kind\':\'warmup-constant; final cooldown only at end of all sessions\',\n                         \'warmup_updates\':identity[\'recipe\'][\'warmup_updates\'],\'last_lr\':state[\'last_lr\']}}\n\ndef save_full(root,model,opt,scaler,state,spec,identity):\n    root=Path(root); folder=root/\'checkpoints\';folder.mkdir(parents=True,exist_ok=True)\n    path=folder/(\'main-\'+uuid.uuid4().hex[:12]+\'.pt\'); temporary=path.with_suffix(\'.pt.tmp\')\n    C.sync(next(model.parameters()).device)\n    torch.save(state_payload(model,opt,scaler,state,spec,identity),temporary); temporary.replace(path)\n    receipt={\'file\':path.relative_to(root).as_posix(),\'sha256\':sha(path),\'bytes\':path.stat().st_size,\n             \'updates\':state[\'updates\'],\'cursor\':state[\'cursor\'],\'main_seconds\':state[\'main_seconds\'],\n             \'model_spec\':spec,\'identity\':identity,\'purpose\':\'constant-LR main resume; never evaluated cooldown weights\'}\n    write(root/\'latest.json\',receipt)\n    # Keep a predecessor during concurrent live snapshots; never accumulate all periodic Adam states.\n    older=sorted(folder.glob(\'main-*.pt\'),key=lambda p:p.stat().st_mtime_ns,reverse=True)\n    for p in older[2:]:p.unlink(missing_ok=True)\n    return receipt\n\ndef read_full(path,identity):\n    p=torch.load(path,map_location=\'cpu\',weights_only=True)\n    assert p[\'format\']==\'nlp-hw02-l1-full-v1\' and p[\'identity\']==identity,\'Resume identity mismatch\'\n    assert p[\'scheduler\'][\'last_lr\']==p[\'state\'][\'last_lr\']\n    assert p[\'scheduler\'][\'warmup_updates\']==identity[\'recipe\'][\'warmup_updates\']\n    assert p[\'state\'][\'last_lr\']==lr_at(p[\'state\'][\'updates\'],identity[\'recipe\'][\'peak_lr\'],identity[\'recipe\'][\'warmup_updates\'])\n    if p[\'state\'][\'updates\']:\n        assert all(g[\'lr\']==p[\'state\'][\'last_lr\'] for g in p[\'optimizer\'][\'param_groups\'])\n    assert all(int(s[\'step\'])==p[\'state\'][\'updates\']-p[\'state\'][\'AMP_skips\'] for s in p[\'optimizer\'][\'state\'].values())\n    assert p[\'state\'][\'cursor\']==p[\'state\'][\'updates\']*identity[\'recipe\'][\'effective_batch\']\n    return p\n\ndef restore_full(payload,model,opt,scaler,device):\n    assert payload[\'scaler_enabled\']==scaler.is_enabled(),\'Resume AMP/scaler mode mismatch\'\n    model.load_state_dict(payload[\'model\']);opt.load_state_dict(payload[\'optimizer\']);scaler.load_state_dict(payload[\'scaler\'])\n    T.restore_rng(payload[\'rng\'],device);model.train(); return copy.deepcopy(payload[\'state\'])\n\ndef save_point(root,tag,model,spec,identity,training):\n    path=Path(root)/\'weights\'/f\'{tag}-fp16.pt\';path.parent.mkdir(parents=True,exist_ok=True)\n    tensors={k:(v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()) for k,v in model.state_dict().items()}\n    torch.save({\'format\':\'nlp-hw02-model-only-v1\',\'model\':tensors,\'model_spec\':spec,\'identity\':identity,\n                \'dtype\':\'FP16\',\'optimizer_included\':False,\'point\':tag},path)\n    receipt={\'file\':path.relative_to(root).as_posix(),\'sha256\':sha(path),\'bytes\':path.stat().st_size,\n             \'dtype\':\'FP16\',\'optimizer_included\':False}\n    write(Path(root)/\'points\'/tag/\'checkpoint.json\',receipt);write(Path(root)/\'points\'/tag/\'training.json\',training)\n    return path\n\ndef identity_for(name,spec,cache,order_sha,smoke=False,batch=128,warmup=1000):\n    # Semantic recipe hashes permit L2/L3 packaging changes without weakening data/state checks.\n    from exp26_common import json_hash\n    recipe={\'architecture\':spec,\'representation\':30,\'normalized_token_init\':True,\'seed\':42,\'dropout\':0.,\n            \'AMP\':\'FP16+GradScaler\',\'fused_AdamW\':not smoke,\'weight_decay\':.01,\'clip\':1.,\n            \'effective_batch\':batch,\'peak_lr\':PEAK_LR[name],\'warmup_updates\':warmup,\'main_schedule\':\'constant\',\n            \'data_policy\':\'fixed bucketed permutation, circular continuation without dropping tail\',\n            \'implementation_sha256\':{n:sha(Path(__file__).with_name(n)) for n in\n                (\'l1_runtime.py\',\'exp26_core.py\',\'exp26_train.py\',\'exp26_order.py\',\'s3_representation.py\')},\n            \'torch\':str(torch.__version__),\'CUDA\':torch.version.cuda}\n    return {\'model\':name,\'recipe\':recipe,\'recipe_sha256\':json_hash(recipe),\n            \'cache_manifest_sha256\':cache.manifest_sha,\'order_sha256\':order_sha,\'smoke\':smoke}\n\ndef run(name,cache,order,order_sha,root,scratch,device,deadline,evaluate,smoke=False,spec=None,\n        checkpoint_seconds=1800.,point_seconds=9000.,cooldown_cap=1500.,max_updates=None,resume=None,\n        batch=128,warmup=1000,point_updates=None,evaluation_seconds=900.,max_points=None):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True);scratch=Path(scratch);scratch.mkdir(parents=True,exist_ok=True)\n    spec=spec or SPECS[name]; identity=identity_for(name,spec,cache,order_sha,smoke,batch,warmup)\n    model=new_model(spec,device); params=sum(p.numel() for p in model.parameters())\n    print(\'MODEL\',name,\'parameters\',params,\'spec\',spec,flush=True)\n    opt=optimizer(model,PEAK_LR[name],smoke);scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1024)\n    state={\'updates\':0,\'cursor\':0,\'main_seconds\':0.,\'branch_seconds\':0.,\'branch_attempted_updates\':0,\n           \'AMP_skips\':0,\'last_lr\':0.,\'next_point_seconds\':point_seconds,\'points\':[], \'micro_batch\':batch}\n    if resume:\n        receipt=json.loads((Path(resume)/\'latest.json\').read_text());file=Path(resume)/receipt[\'file\']\n        assert sha(file)==receipt[\'sha256\'] and file.stat().st_size==receipt[\'bytes\']\n        p=read_full(file,identity);state=restore_full(p,model,opt,scaler,device);del p\n        if \'deferred_point_seconds\' in state:\n            state[\'missed_point_seconds\']=state.pop(\'deferred_point_seconds\')\n            state[\'next_point_seconds\']=(math.floor(state[\'main_seconds\']/point_seconds)+1)*point_seconds\n        print(\'RESUME\',name,state[\'updates\'],\'cursor\',state[\'cursor\'],\'main seconds\',state[\'main_seconds\'],flush=True)\n    else:\n        del opt,scaler\n        micro,attempts=probe(model,cache,device,PEAK_LR[name],smoke,batch)\n        state[\'micro_batch\']=micro\n        write(root/\'memory-probe.json\',{\'effective_batch\':batch,\'micro_batch\':micro,\'attempts\':attempts,\'data_cursor_consumed\':0})\n        opt=optimizer(model,PEAK_LR[name],smoke);scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1024)\n    write(root/\'identity.json\',identity)\n    write(root/\'summary.json\',{\'model\':name,\'status\':\'training\',\'parameter_count\':params,\'state\':state,\'identity\':identity})\n    save_full(root,model,opt,scaler,state,spec,identity);last_save=time.monotonic();start_updates=state[\'updates\']\n    logs=[];unsafe_ooms=0\n    try:\n        while time.monotonic()<deadline-(.2 if smoke else 120.):\n            if (root.parent/\'stop-request.json\').exists():break\n            if max_updates is not None and state[\'updates\']-start_updates>=max_updates:break\n            due=(state[\'updates\']>=point_updates and not state.get(\'point_attempts\',0)) if point_updates else state[\'main_seconds\']>=state[\'next_point_seconds\']\n            if max_points is not None and state.get(\'point_attempts\',0)>=max_points:due=False\n            if due:\n                remaining=deadline-time.monotonic()\n                if smoke or remaining>=cooldown_cap+evaluation_seconds+180:\n                    tag=f\'t{int(round(state["next_point_seconds"]*(1000 if smoke else 1))):06d}\'\n                    anchor=state_payload(model,opt,scaler,state,spec,identity)\n                    anchor_path=scratch/\'anchor.pt\';torch.save(anchor,anchor_path);del anchor\n                    save_full(root,model,opt,scaler,state,spec,identity);last_save=time.monotonic()\n                    mean_seconds=(sum(r[\'seconds\'] for r in logs[-100:])/len(logs[-100:]) if logs else\n                                  state[\'main_seconds\']/max(1,state[\'updates\']))\n                    length=max(1,min(int(.1*state[\'updates\']),int(.90*cooldown_cap/max(.001,mean_seconds))))\n                    branch_start=time.monotonic(); branch_seconds=0.; done=0; branch_skips=0\n                    for j in range(1,length+1):\n                        # Emergency reserve also covers an unexpectedly slow update.\n                        if (root.parent/\'stop-request.json\').exists() or time.monotonic()>=min(branch_start+cooldown_cap-mean_seconds*2,deadline-(.1 if smoke else evaluation_seconds+150)):break\n                        lr=state[\'last_lr\']*(1-j/length)\n                        try:\n                            log,micro=resilient_update(model,opt,scaler,cache,take(order,state[\'cursor\']+done*batch,batch),device,state[\'micro_batch\'],lr)\n                        except UnsafeAdamOOM:\n                            print(\'BRANCH Adam OOM: discard branch and restore anchor\',flush=True)\n                            break\n                        state[\'micro_batch\']=micro\n                        branch_seconds+=log[\'seconds\'];done+=1;branch_skips+=log[\'AMP_skip\']\n                    training={\'main_seconds\':state[\'main_seconds\'],\'target_main_seconds\':state[\'next_point_seconds\'],\n                              \'anchor_updates\':state[\'updates\'],\'anchor_cursor\':state[\'cursor\'],\n                              \'cooldown_updates\':done,\'planned_cooldown_updates\':length,\'cooldown_seconds\':branch_seconds,\n                              \'branch_AMP_skips\':branch_skips,\'all_training_seconds\':state[\'main_seconds\']+state[\'branch_seconds\']+branch_seconds,\n                              \'main_AMP_skips\':state[\'AMP_skips\'],\'parameter_count\':params,\n                              \'common_time_basis\':\'pure main update time; excludes setup, probes, I/O, evaluation and side cooldowns\'}\n                    metrics=None\n                    eval_started=time.monotonic();branch_micro=state[\'micro_batch\']\n                    if done==length:\n                        point=save_point(root,tag,model,spec,identity,training)\n                        # Evaluation uses an independent model: no half/float conversion of training parameters.\n                        del model,opt,scaler;model=opt=scaler=None;gc.collect()\n                        if device.type==\'cuda\':torch.cuda.empty_cache()\n                        try:\n                            metrics=evaluate(point,root/\'points\'/tag/\'evaluation\',min(time.monotonic()+evaluation_seconds,deadline-(.1 if smoke else 150)))\n                        except Exception as error:\n                            write(root/\'points\'/tag/\'incomplete.json\',{\'reason\':\'evaluation failed; main anchor restored\',\n                                  \'error_type\':type(error).__name__,**training})\n                            print(\'EVAL INCOMPLETE\',name,type(error).__name__,flush=True)\n                        if metrics is not None:\n                            training[\'evaluation_seconds\']=time.monotonic()-eval_started\n                            write(root/\'points\'/tag/\'complete.json\',{\'training\':training,\'metrics\':metrics,\'weights_sha256\':sha(point)})\n                        model=C.SimpleTransformer(**spec,dropout=0.).to(device)\n                        opt=optimizer(model,PEAK_LR[name],smoke);scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1024)\n                    else:write(root/\'points\'/tag/\'incomplete.json\',{\'reason\':\'cooldown interrupted; no evaluated point\',**training})\n                    anchor=read_full(anchor_path,identity);state=restore_full(anchor,model,opt,scaler,device);del anchor\n                    state[\'micro_batch\']=min(state[\'micro_batch\'],branch_micro)\n                    anchor_path.unlink(missing_ok=True)\n                    state[\'branch_seconds\']+=branch_seconds;state[\'branch_attempted_updates\']+=done\n                    state[\'point_attempts\']=state.get(\'point_attempts\',0)+1\n                    if metrics is not None:state[\'points\'].append({\'tag\':tag,**training,\'metrics\':metrics})\n                    state[\'next_point_seconds\']+=point_seconds\n                    save_full(root,model,opt,scaler,state,spec,identity);last_save=time.monotonic()\n                    write(root/\'summary.json\',{\'model\':name,\'status\':\'training\',\'parameter_count\':params,\'state\':state,\'identity\':identity})\n                    continue\n                # Preserve the pending time point for L2; spend remaining session on stable main.\n                state[\'deferred_point_seconds\']=state[\'next_point_seconds\'];state[\'next_point_seconds\']=float(\'1e100\')\n            try:\n                log,micro=resilient_update(model,opt,scaler,cache,take(order,state[\'cursor\'],batch),device,state[\'micro_batch\'],\n                                          lr_at(state[\'updates\']+1,PEAK_LR[name],warmup))\n            except UnsafeAdamOOM:\n                unsafe_ooms+=1;smaller=[m for m in MICRO_BATCHES if m<state[\'micro_batch\']]\n                if unsafe_ooms>2 or not smaller:raise\n                old_updates=state[\'updates\'];del model,opt,scaler;gc.collect()\n                if device.type==\'cuda\':torch.cuda.empty_cache()\n                rec=json.loads((root/\'latest.json\').read_text());assert sha(root/rec[\'file\'])==rec[\'sha256\']\n                payload=read_full(root/rec[\'file\'],identity)\n                model=C.SimpleTransformer(**spec,dropout=0.).to(device);opt=optimizer(model,PEAK_LR[name],smoke)\n                scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\',init_scale=1024)\n                state=restore_full(payload,model,opt,scaler,device);del payload\n                state[\'micro_batch\']=smaller[0];logs=[]\n                write(root/\'adam-oom-recovery.json\',{\'rolled_back_from_updates\':old_updates,\n                      \'restored_updates\':state[\'updates\'],\'micro_batch\':state[\'micro_batch\'],\'attempt\':unsafe_ooms})\n                continue\n            state[\'micro_batch\']=micro\n            state[\'updates\']+=1;state[\'cursor\']+=batch;state[\'main_seconds\']+=log[\'seconds\']\n            state[\'AMP_skips\']+=log[\'AMP_skip\'];state[\'last_lr\']=log[\'lr\'];logs.append(log)\n            if len(logs)>100:logs=logs[-100:]\n            if state[\'updates\']%(1 if smoke else 100)==0:\n                print(\'TRAIN\',name,\'updates\',state[\'updates\'],\'main_s\',round(state[\'main_seconds\'],1),\n                      \'CE\',round(log[\'CE\'],4),\'micro\',state[\'micro_batch\'],\'LR\',state[\'last_lr\'],flush=True)\n                write(root/\'progress.json\',dict(state,last_chunk=log,epoch=state[\'cursor\']//len(order)))\n            if time.monotonic()-last_save>=checkpoint_seconds:\n                save_full(root,model,opt,scaler,state,spec,identity);last_save=time.monotonic()\n        if \'deferred_point_seconds\' in state:\n            # Resume evaluates the first future common grid point, never labels a later anchor as the missed time.\n            deferred=state.pop(\'deferred_point_seconds\');state[\'missed_point_seconds\']=deferred\n            state[\'next_point_seconds\']=(math.floor(state[\'main_seconds\']/point_seconds)+1)*point_seconds\n        save_full(root,model,opt,scaler,state,spec,identity)\n        summary={\'model\':name,\'status\':\'session_complete; main LR remains constant\',\'parameter_count\':params,\'state\':state,\'identity\':identity}\n        write(root/\'summary.json\',summary);return summary\n    finally:\n        del model,opt,scaler;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n', 'l1_session.py': '"""Bounded two-process supervisor and registered common-time C/D decision."""\nimport json, math, os, subprocess, sys, time\nfrom pathlib import Path\nfrom session_watchdog import Watchdog as ShortWatchdog, atomic_json\nfrom session_runner import stop_workers, tail\nfrom kaggle_io import write, sha, disk_guard\n\nclass Watchdog(ShortWatchdog):\n    def __init__(self,config):\n        super().__init__(config)\n        self.hard_deadline=min(config[\'hard_deadline_monotonic\'],config[\'session_start_monotonic\']+43200)\n\ndef decision(working):\n    working=Path(working);models={}\n    for name in (\'C\',\'D\'):\n        points={}\n        # Full resume carries completed historical metrics even when old generation/weight\n        # files stay in the attached previous kernel output or an older live version.\n        histories=[]\n        for filename in (\'summary.json\',\'progress.json\'):\n            path=working/f\'model-{name}\'/filename\n            if path.exists():\n                value=json.loads(path.read_text());state=value.get(\'state\',value)\n                histories.append(state)\n        if histories:\n            state=max(histories,key=lambda s:s.get(\'updates\',0))\n            for item in state.get(\'points\',[]):\n                target=item[\'target_main_seconds\']\n                if abs(item[\'main_seconds\']-target)<=max(5.,target*.005):\n                    points[item[\'tag\']]={\'training\':item,\'metrics\':item[\'metrics\']}\n        for p in (working/f\'model-{name}\'/\'points\').glob(\'*/complete.json\'):\n            point=json.loads(p.read_text());train=point[\'training\'];target=train[\'target_main_seconds\']\n            if abs(train[\'main_seconds\']-target)<=max(5.,target*.005):points[p.parent.name]=point\n        models[name]=points\n    common=sorted(set(models[\'C\'])&set(models[\'D\']),key=lambda tag:models[\'C\'][tag][\'training\'][\'target_main_seconds\'])\n    if not common:return {\'status\':\'pending\',\'reason\':\'No completed common pure-main-time point\'}\n    def accuracy(name,tag):return models[name][tag][\'metrics\'][\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\']\n    last=common[-1];c=accuracy(\'C\',last);d=accuracy(\'D\',last);slopes={};previous=None\n    if len(common)>=2:\n        previous=common[-2]\n        for name in (\'C\',\'D\'):\n            a=models[name][previous][\'training\'][\'target_main_seconds\'];b=models[name][last][\'training\'][\'target_main_seconds\']\n            slopes[name]=100*(accuracy(name,last)-accuracy(name,previous))/math.log2(b/a)\n    both=(d<c and (c-d)*100<2 and slopes and slopes[\'D\']>slopes[\'C\'])\n    return {\'status\':\'measured\',\'common_points\':common,\'last_common_point\':last,\'previous_common_point\':previous,\n            \'heldout4k_accuracy\':{\'C\':c,\'D\':d},\'D_minus_C_pp\':100*(d-c),\'slope_pp_per_doubling\':slopes,\n            \'continue_L2\':[\'C\',\'D\'] if both else [\'D\' if d>c else \'C\'],\n            \'rule\':\'best last common exec4k; both if D lags <2pp and has steeper latest log2-time slope; exact ties C\',\n            \'time_basis\':\'pure main training time; cooldown compute recorded separately\'}\n\ndef launch(config,worker,watchdog,live=None,smoke=False,heartbeat=30.):\n    working=Path(config[\'working_root\']);working.mkdir(parents=True,exist_ok=True)\n    path=working/\'session-config.json\';write(path,config);handles=[];processes=[];reason=\'workers completed\';stop_until=None\n    try:\n        for gpu in (0,1):\n            env=dict(os.environ,CUDA_VISIBLE_DEVICES=\'\' if smoke else str(gpu),OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',\n                     OPENBLAS_NUM_THREADS=\'1\',TOKENIZERS_PARALLELISM=\'false\')\n            for secret in (\'KAGGLE_USERNAME\',\'KAGGLE_KEY\',\'KAGGLE_API_TOKEN\'):env.pop(secret,None)\n            h=(working/f\'gpu{gpu}-process.log\').open(\'w\');handles.append(h)\n            p=subprocess.Popen([sys.executable,str(worker),\'--gpu\',str(gpu),\'--config\',str(path)],env=env,\n                               stdout=h,stderr=subprocess.STDOUT,start_new_session=True)\n            processes.append(p);watchdog.register(p)\n        while any(p.poll() is None for p in processes):\n            if live is not None and (live.uninsured or time.monotonic()-live.last_success>=7200.):\n                reason=\'live insurance stale >=2h\';stop_until=stop_until or min(time.monotonic()+120.,config[\'deadline_monotonic\'])\n            if time.monotonic()>=config[\'deadline_monotonic\']:reason=\'session deadline\';break\n            # If one model failed, stop the other rather than leave an idle GPU billed for hours.\n            if any(p.poll() not in (None,0) for p in processes):\n                reason=\'worker failed\';stop_until=stop_until or min(time.monotonic()+120.,config[\'deadline_monotonic\'])\n            if stop_until is not None:\n                if not (working/\'stop-request.json\').exists():write(working/\'stop-request.json\',{\'reason\':reason})\n                if time.monotonic()>=stop_until:break\n            for gpu,p in enumerate(processes):\n                print(\'L1 GPU\',gpu,\'status\',p.poll(),\' | \'.join(tail(working/f\'gpu{gpu}-process.log\'))[:700],flush=True)\n            write(working/\'session-live.json\',{\'elapsed_seconds\':time.monotonic()-config[\'session_start_monotonic\'],\n                  \'exit_codes\':[p.poll() for p in processes],\'live_insurance_stale\':bool(live and live.uninsured)})\n            disk_guard(working,limit=10*1024**3)\n            time.sleep(min(heartbeat,max(.01,config[\'deadline_monotonic\']-time.monotonic())))\n    finally:\n        survivors=stop_workers(processes,watchdog)\n        for h in handles:h.close()\n        comparison=decision(working)\n        report={\'status\':\'complete\' if len(processes)==2 and all(p.returncode==0 for p in processes) else \'partial\',\n                \'reason\':reason,\'exit_codes\':[p.poll() for p in processes],\'unreaped_worker_pids\':survivors,\n                \'wall_seconds\':time.monotonic()-config[\'session_start_monotonic\'],\'comparison\':comparison,\n                \'output_policy\':\'individual files; no ZIP; full stable states + all evaluated FP16 weights\',\n                \'hard_shutdown_seconds\':config[\'hard_shutdown_seconds\'],\'server_ceiling_seconds\':43200}\n        write(working/\'session-summary.json\',report)\n        if live is not None:report[\'final_live_uploaded\']=live.final_sync(\'normal-exit\',timeout=max(.1,min(600.,watchdog.hard_deadline-time.monotonic()-60.)))\n        write(working/\'session-summary.json\',report)\n        files=[p for p in working.rglob(\'*\') if p.is_file() and p.relative_to(working).parts[0]!=\'src\'\n               and p.name!=\'l1-output-manifest.json\' and \'.tmp\' not in p.name]\n        write(working/\'l1-output-manifest.json\',{\'files\':[{\'file\':p.relative_to(working).as_posix(),\n              \'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in files]})\n        if not survivors:watchdog.finish()\n    return report\n', 'l2_compare.py': '#!/usr/bin/env python3\n"""CPU paired greedy report and fixed C-main K4 + C-long2x K4 voting."""\nimport argparse, json, math, sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parent;sys.path.insert(0,str(ROOT/\'src\'))\nfrom kaggle_io import write\nfrom l2_sampler import fixed_split\n\ndef paired(a,b,ids):\n    gains=sum(not a[i][\'correct\'] and b[i][\'correct\'] for i in ids)\n    losses=sum(a[i][\'correct\'] and not b[i][\'correct\'] for i in ids); n=len(ids)\n    delta=(gains-losses)/n if n else 0.; variance=(gains+losses)/n-delta**2 if n else 0.\n    half=1.96*math.sqrt(max(0.,variance)/max(1,n))\n    return {\'n\':n,\'gains\':gains,\'losses\':losses,\'delta_pp\':100*delta,\'paired_95CI_pp\':[100*(delta-half),100*(delta+half)]}\n\ndef compare(working):\n    working=Path(working);result={\'points\':{}}\n    for a in sorted((working/\'model-C-main/points\').glob(\'*/complete.json\')):\n        b=working/\'model-C-long2x/points\'/a.parent.name/\'complete.json\'\n        if not b.exists():continue\n        aa=json.loads(a.read_text());bb=json.loads(b.read_text());ta=aa[\'training\'];tb=bb[\'training\']\n        if abs(ta[\'l2_main_seconds\']-tb[\'l2_main_seconds\'])>5:continue\n        if ta[\'cooldown_target_seconds\']!=tb[\'cooldown_target_seconds\']:continue\n        pa=a.parent/\'evaluation\';pb=b.parent/\'evaluation\'\n        x=json.loads((pa/\'heldout4k-per-example.json\').read_text());y=json.loads((pb/\'heldout4k-per-example.json\').read_text())\n        assert [(r[\'id\'],r[\'question_sha256\']) for r in x]==[(r[\'id\'],r[\'question_sha256\']) for r in y]\n        split=fixed_split()\n        subsets={**split,\'heldout4k\':list(range(len(x))),\'ge90\':[r[\'id\'] for r in x if r[\'original_bpe_length\']>=90],\n                 \'lt60\':[r[\'id\'] for r in x if r[\'original_bpe_length\']<60]}\n        subsets[\'tune-ge90\']=[i for i in split[\'tune2048\'] if i<len(x) and x[i][\'original_bpe_length\']>=90]\n        subsets[\'tune-lt60\']=[i for i in split[\'tune2048\'] if i<len(x) and x[i][\'original_bpe_length\']<60]\n        result[\'points\'][a.parent.name]={k:paired(x,y,[i for i in v if i<len(x)]) for k,v in subsets.items()}\n        olda=pa/\'old991-per-example.json\';oldb=pb/\'old991-per-example.json\'\n        if olda.exists() and oldb.exists():\n            ox=json.loads(olda.read_text());oy=json.loads(oldb.read_text())\n            assert [(r[\'id\'],r[\'question_sha256\']) for r in ox]==[(r[\'id\'],r[\'question_sha256\']) for r in oy]\n            result[\'points\'][a.parent.name][\'old991\']=paired(ox,oy,list(range(len(ox))))\n        # Choosing recipes is a human tune-only decision; eval stored for descriptive paired audit.\n    return result\n\ndef mixed(a,b):\n    import exp27_decode as D\n    import l2_eval as E\n    def load(root):\n        rows=[r for p in sorted(Path(root).glob(\'batch-*.json\')) for r in json.loads(p.read_text())]\n        assert len({r[\'id\'] for r in rows})==len(rows)\n        return {r[\'id\']:r for r in rows}\n    x,y=load(a),load(b);ids=sorted(set(x)&set(y));output=[]\n    for i in ids:\n        assert x[i][\'question_sha256\']==y[i][\'question_sha256\'] and x[i][\'reference\']==y[i][\'reference\']\n        pool=x[i][\'samples\'][\'0.7\'][:4]+y[i][\'samples\'][\'0.7\'][:4]\n        row=dict(x[i],samples={\'0.7\':[c for c in pool if c[\'mean_logprob\'] is not None]})\n        chosen=D.choose(row,{\'mode\':\'vote\',\'temperature\':.7,\'k\':8,\'greedy_vote\':False})\n        output.append({\'id\':i,\'execution\':chosen[\'execution\'],\'correct\':E.matches(row[\'reference\'],chosen[\'execution\'])})\n    return {\'rows\':len(ids),\'complete_tune2048\':len(ids)==2048,\'mixed_matches\':sum(r[\'correct\'] for r in output),\n            \'C_main_K8_matches\':sum(x[i][\'correct\'] for i in ids),\'C_long_K8_matches\':sum(y[i][\'correct\'] for i in ids),\n            \'per_example\':output,\'GPU\':False,\'production_timer_established\':False}\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--working\');p.add_argument(\'--main-k8\');p.add_argument(\'--long-k8\');p.add_argument(\'--out\',required=True);a=p.parse_args()\n    write(a.out,mixed(a.main_k8,a.long_k8) if a.main_k8 else compare(a.working))\n', 'l2_eval.py': '"""Greedy paired journals and bounded K8, preserving every executed sample."""\nimport gc, json, math, time\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom tokenizers import Tokenizer\nimport exp26_core as C, exp26_exec as N, exp27_decode as D, exp27b_fast as Fast\nimport l1_eval as E, s3_representation as R\nfrom kaggle_io import sha, write\nfrom l2_sampler import fixed_split\n\ndef scores(records, ids):\n    rows = [records[i] for i in ids]; n = len(rows)\n    return {\'matches\': sum(r[\'correct\'] for r in rows), \'denominator\': n,\n            \'accuracy\': sum(r[\'correct\'] for r in rows)/n if n else None,\n            \'length_bins\': {label: {\'matches\':sum(r[\'correct\'] for r in rows if lo <= r[\'original_bpe_length\'] < hi),\n                                   \'denominator\':sum(lo <= r[\'original_bpe_length\'] < hi for r in rows)}\n                            for label,lo,hi in [(\'<60\',0,60),(\'60-90\',60,90),(\'90-120\',90,120),(\'>=120\',120,10**9)]}}\n\ndef matches(reference, execution):\n    return reference[\'status\'] == execution[\'status\'] == \'numeric\' and math.isclose(\n        reference[\'value\'], execution[\'value\'], rel_tol=1e-6, abs_tol=1e-6)\n\ndef greedy_journals(out, inputs, sets, refs):\n    original = Tokenizer.from_file(str(Path(inputs)/\'tokenizer.json\')); records = {}\n    for split, rows in sets.items():\n        if not (out/(split+\'-execution.json\')).exists(): continue\n        generated = json.loads((out/(split+\'-generations.json\')).read_text())\n        executed = json.loads((out/(split+\'-execution.json\')).read_text())[\'rows\']\n        score = N.compare_exec(refs[split], {\'rows\':executed}); correct = set(score[\'match_indices\'])\n        records[split] = [dict(id=i, question_sha256=__import__(\'hashlib\').sha256(rows[i][\'question\'].encode()).hexdigest(),\n                               correct=i in correct, execution=executed[i], reference=refs[split][\'rows\'][i],\n                               original_bpe_length=len(original.encode(rows[i][\'code\'],add_special_tokens=False).ids),\n                               generated_code=generated[i][\'generated_code\']) for i in range(len(rows))]\n        write(out/(split+\'-per-example.json\'), records[split])\n    return records\n\n@torch.no_grad()\ndef vote_k8(point, out, inputs, rows, greedy, device, deadline, seed=270042, batch=32, cap=512):\n    out = Path(out); out.mkdir(parents=True,exist_ok=True); ids = fixed_split()[\'tune2048\']\n    ids = [i for i in ids if i < len(rows)]\n    tok, table = R.tokenizer(Path(inputs)/\'tokenizer.json\',30); model = E.load(point,device)\n    completed = []; started = time.monotonic(); lo = 0; attempts = []\n    try:\n        torch.manual_seed(seed)\n        if device.type == \'cuda\': torch.cuda.manual_seed_all(seed)\n        while lo < len(ids):\n            C.check_deadline(deadline); selected = ids[lo:lo+batch]; src=ctx=seq=lps=None\n            # Restore the RNG if a CUDA OOM discards a partially sampled batch.\n            import exp26_train as T\n            rng = T.cpu(T.rng_state())\n            try:\n                encoded = [[C.BOS]+C.encode(tok,rows[i][\'question\'])+[C.EOS] for i in selected]\n                src = nn.utils.rnn.pad_sequence([torch.tensor(s) for s in encoded],batch_first=True,padding_value=C.PAD).to(device)\n                ctx = Fast.Context(model,src,cap); seq,lps,_ = Fast.decode(ctx,k=8,temperature=.7,deadline=deadline)\n                candidates = []\n                for i,group,means in zip(selected,seq.tolist(),lps.tolist()):\n                    samples = []\n                    for j,(tokens,lp) in enumerate(zip(group,means)):\n                        active = C.active_tokens(tokens); represented = C.decode_bytes(table,active).decode(\'utf8\',errors=\'replace\')\n                        try: code = R.restore_compact(represented); error = None\n                        except (ValueError,IndexError): code = \'\'; error = \'invalid representation\'\n                        samples.append({\'sample\':j,\'code\':code,\'represented_code\':represented,\'restore_error\':error,\n                                        \'mean_logprob\':float(lp) if math.isfinite(lp) else None,\'length\':len(active)})\n                    candidates.append({\'id\':i,\'question_sha256\':greedy[i][\'question_sha256\'],\n                                       \'greedy\':{\'code\':greedy[i][\'generated_code\'],\'execution\':greedy[i][\'execution\'],\'mean_logprob\':0.},\n                                       \'reference\':greedy[i][\'reference\'],\'samples\':{\'0.7\':samples}})\n                flat = [c for r in candidates for c in r[\'samples\'][\'0.7\']]\n                executions = E.execute([{\'generated_code\':c[\'code\']} for c in flat],device.type==\'cpu\',deadline)[\'rows\']\n                for candidate,execution in zip(flat,executions):\n                    candidate[\'execution\'] = execution\n                    candidate[\'sampling_status\'] = \'nonfinite_logprob\' if candidate[\'mean_logprob\'] is None else \'complete\'\n                for r in candidates:\n                    valid = dict(r,samples={\'0.7\':[c for c in r[\'samples\'][\'0.7\'] if c[\'mean_logprob\'] is not None]})\n                    winner = D.choose(valid,{\'mode\':\'vote\',\'temperature\':.7,\'k\':8,\'greedy_vote\':False})\n                    r[\'correct\'] = matches(r[\'reference\'],winner[\'execution\']); r[\'vote_execution\'] = winner[\'execution\']\n                write(out/f\'batch-{lo:05d}.json\',candidates); completed.extend(candidates); lo += len(selected)\n            except torch.cuda.OutOfMemoryError:\n                T.restore_rng(rng,device); attempts.append({\'offset\':lo,\'batch\':batch}); batch //= 2\n                if batch < 1: raise\n            finally:\n                del src,ctx,seq,lps; gc.collect()\n                if device.type == \'cuda\': torch.cuda.empty_cache()\n    except C.BudgetExpired:\n        pass\n    finally:\n        del model; gc.collect()\n        if device.type == \'cuda\': torch.cuda.empty_cache()\n        summary = {\'complete\':len(completed)==len(ids),\'rows\':len(completed),\'target_rows\':len(ids),\'K\':8,\'T\':.7,\n                   \'seed\':seed,\'matches\':sum(r[\'correct\'] for r in completed),\'seconds\':time.monotonic()-started,\n                   \'weights_sha256\':sha(point),\'OOM_attempts\':attempts,\n                   \'logprob\':\'untempered logsoftmax; includes EOS; PAD/BOS masked, as existing sampler\',\n                   \'vote\':\'exp27_decode.choose; numeric connected components isclose1e-6, ties count/greedy/mean LP/order\',\n                   \'offline_mixed\':\'first4 from each worker; C-main greedy tie anchor; same K8, no gold in selection\'}\n        write(out/\'summary.json\',summary)\n    return summary\n\ndef evaluate(point,out,inputs,sets,refs,device,deadline,final=False,smoke=False,vote_cap_seconds=900.,test_rows=None):\n    out = Path(out); out.mkdir(parents=True,exist_ok=True)\n    # At most 900s for greedy including CE and complete test; independently bounded vote.\n    try:\n        metrics = E.evaluate(point,out,inputs,sets,refs,device,min(deadline,time.monotonic()+900.),smoke,test_rows=test_rows)\n    finally:\n        # CE/test timeout must not discard already executed greedy correctness.\n        records = greedy_journals(out,inputs,sets,refs)\n    splits = fixed_split()\n    metrics[\'paired_subsets\'] = {name:scores(records[\'heldout4k\'],[i for i in ids if i < len(records[\'heldout4k\'])])\n                               for name,ids in splits.items()}\n    if final:\n        metrics[\'K8_tune2048\'] = vote_k8(point,out/\'k8-tune2048\',inputs,sets[\'heldout4k\'],records[\'heldout4k\'],device,\n                                        min(deadline,time.monotonic()+vote_cap_seconds),cap=8 if smoke else 512,\n                                        batch=2 if smoke else 32)\n    write(out/\'metrics.json\',metrics); return metrics\n', 'l2_gpu_smoke.py': '"""Reviewer-only CUDA faults around the exact production update/run/evaluation functions."""\nimport copy, gc, hashlib, json, time\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport torch\nimport l1_runtime as L, l2_runtime as R, l2_eval as E, exp26_core as C, exp26_train as T\nfrom l2_resume import initial_state\nfrom kaggle_io import write\n\ndef digest(value):\n    h=hashlib.sha256()\n    def visit(v):\n        if torch.is_tensor(v):\n            a=v.detach().cpu().contiguous().numpy();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes())\n        elif isinstance(v,dict):\n            for k in sorted(v,key=str):h.update(str(k).encode());visit(v[k])\n        elif isinstance(v,(tuple,list)):\n            for x in v:visit(x)\n        else:h.update(repr(v).encode())\n    visit(value);return h.hexdigest()\n\ndef numerical(model,opt,scaler):\n    return digest({\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'scaler\':scaler.state_dict(),\'rng\':T.rng_state()})\n\ndef exercise(gpu,name,anchor,identity,cache,order,inputs,sets,refs,device,cfg):\n    assert device.type==\'cuda\' and not torch.is_inference_mode_enabled() and torch.is_grad_enabled()\n    root=Path(cfg[\'working_root\'])/(\'model-\'+name);root.mkdir(parents=True,exist_ok=True)\n    report={\'passed\':False,\'GPU\':True,\'device\':torch.cuda.get_device_name(0),\'checks\':{},\'stage\':\'start\'}\n    def stage(label,**fields):\n        report[\'stage\']=label;report.update(fields);write(root/\'gpu-smoke.json\',report)\n        print(\'CUDA SMOKE\',name,label,flush=True)\n    def check(label,**fields):\n        report[\'checks\'][label]=fields or {\'passed\':True};stage(label)\n    spec=anchor[\'model_spec\'];lr=anchor[\'state\'][\'last_lr\'];micro=anchor[\'state\'][\'micro_batch\']\n    stage(\'longest-row memory probe\')\n    model=C.SimpleTransformer(**spec,dropout=0.).to(device);model.load_state_dict(anchor[\'model\'])\n    choice,attempts=L.probe(model,cache,device,lr)\n    assert choice>=micro, \'Inherited micro-batch failed longest-row CUDA probe\'\n    report[\'probe\']={\'choice\':choice,\'inherited_micro_batch\':micro,\'attempts\':attempts}\n    opt=L.optimizer(model,lr,False);scaler=torch.amp.GradScaler(\'cuda\')\n    state=L.restore_full(anchor,model,opt,scaler,device)\n    cursor=0 if gpu==1 else state[\'cursor\'];ids=L.take(order,cursor,128)\n    stage(\'L1 versus L2 finite CUDA update\')\n    reference=L.update(model,opt,scaler,cache,ids,device,micro,lr);expected=numerical(model,opt,scaler)\n    L.restore_full(anchor,model,opt,scaler,device)\n    actual,_=R.safe_update(model,opt,scaler,cache,ids,device,micro,lr)\n    assert expected==numerical(model,opt,scaler) and reference[\'AMP_skip\']==actual[\'AMP_skip\']\n    check(\'bit_exact_L1_CUDA_update\',AMP_skip=actual[\'AMP_skip\'],micro_batch=micro)\n    L.restore_full(anchor,model,opt,scaler,device)\n    before=digest((model.state_dict(),opt.state_dict()));old_scale=scaler.get_scale()\n    hook=next(model.parameters()).register_hook(lambda grad:torch.full_like(grad,float(\'inf\')))\n    try:overflow,_=R.safe_update(model,opt,scaler,cache,ids,device,micro,lr)\n    finally:hook.remove()\n    assert overflow[\'AMP_skip\']==1 and overflow[\'nonfinite_skip\']==0\n    assert scaler.get_scale()==old_scale/2 and before==digest((model.state_dict(),opt.state_dict()))\n    check(\'GradScaler_overflow_skip\')\n    L.restore_full(anchor,model,opt,scaler,device)\n    before=numerical(model,opt,scaler);ce=L.F.cross_entropy;calls=[0]\n    def bad_loss(*args,**kwargs):\n        calls[0]+=1;return ce(*args,**kwargs)*(float(\'nan\') if calls[0]==2 else 1.)\n    with patch.object(L.F,\'cross_entropy\',bad_loss):rejected,_=R.safe_update(model,opt,scaler,cache,ids,device,micro,lr)\n    assert rejected[\'nonfinite_skip\']==1 and before==numerical(model,opt,scaler)\n    assert all(p.grad is None for p in model.parameters())\n    check(\'partial_backward_nonfinite_loss_skip\')\n    # Serialize/restore and compare the next real CUDA update with uninterrupted execution.\n    state=initial_state(anchor,name);R.save_full(root,model,opt,scaler,state,spec,identity)\n    R.safe_update(model,opt,scaler,cache,ids,device,micro,lr);expected=numerical(model,opt,scaler)\n    del model,opt,scaler;gc.collect();torch.cuda.empty_cache()\n    model,opt,scaler,state=R.reload_latest(root,identity,device)\n    R.safe_update(model,opt,scaler,cache,ids,device,micro,lr)\n    assert expected==numerical(model,opt,scaler)\n    check(\'serialized_resume_bit_exact_CUDA\')\n    del model,opt,scaler;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n    stage(\'production train cooldown eval restore train\')\n    calls=[0];original=R.safe_update\n    def inject(*args,**kwargs):\n        calls[0]+=1\n        if gpu==0 and calls[0]<=3:\n            with patch.object(L.F,\'cross_entropy\',lambda *a,**k:ce(*a,**k)*float(\'nan\')):\n                return original(*args,**kwargs)\n        return original(*args,**kwargs)\n    evaluate=lambda point,out,deadline,final:E.evaluate(point,out,inputs,sets,refs,device,deadline,final,\n                                                         vote_cap_seconds=cfg[\'vote_cap_seconds\'],test_rows=256)\n    with patch.object(R,\'safe_update\',inject):\n        result=R.run(name,anchor,identity,cache,order,root,device,cfg[\'deadline_monotonic\'],evaluate,**cfg[\'phase_recipe\'])\n    assert result[\'status\']==\'complete\', result[\'status\']\n    state=result[\'state\'];assert len(state[\'l2_points\'])==2 and all(p[\'complete\'] for p in state[\'l2_points\'])\n    assert state[\'updates\']-anchor[\'state\'][\'updates\']>=300\n    assert state[\'stability\'][\'escalations\']==(1 if gpu==0 else 0)\n    assert state[\'last_lr\']==lr*(.5 if gpu==0 else 1.)\n    check(\'nonfinite_escalation_full_rollback_LR\' if gpu==0 else \'constant_LR_weighted_fork\',\n          updates=state[\'updates\'],cursor=state[\'cursor\'],LR=state[\'last_lr\'],stability=state[\'stability\'])\n    for item in state[\'l2_points\']:\n        out=root/\'points\'/item[\'tag\']/\'evaluation\';rows=json.loads((out/\'heldout4k-per-example.json\').read_text())\n        assert len(rows)==64 and all(isinstance(r[\'correct\'],bool) for r in rows)\n        assert item[\'metrics\'][\'test\'][\'rows\']==256\n    final=root/\'points\'/state[\'l2_points\'][-1][\'tag\']/\'evaluation/k8-tune2048\'\n    vote=json.loads((final/\'summary.json\').read_text());assert vote[\'complete\'] and vote[\'rows\']>0\n    journals=[r for p in sorted(final.glob(\'batch-*.json\')) for r in json.loads(p.read_text())]\n    assert len(journals)==vote[\'rows\'] and all(len(r[\'samples\'][\'0.7\'])==8 for r in journals)\n    assert all(\'execution\' in c and \'mean_logprob\' in c and c[\'sampling_status\']==\'complete\'\n               for r in journals for c in r[\'samples\'][\'0.7\'])\n    check(\'K8_executed_samples_and_greedy256\',rows=vote[\'rows\'],samples=8*vote[\'rows\'],K8_seconds=vote[\'seconds\'],\n          test_seconds=[p[\'metrics\'][\'test\'][\'full_pipeline_seconds\'] for p in state[\'l2_points\']])\n    report[\'peak_reserved_bytes\']=torch.cuda.max_memory_reserved(device)\n    report[\'peak_allocated_bytes\']=torch.cuda.max_memory_allocated(device)\n    report[\'micro_batch\']=state[\'micro_batch\'];report[\'main_added_updates\']=state[\'updates\']-anchor[\'state\'][\'updates\']\n    report[\'main_seconds\']=state[\'l2_main_seconds\']\n    receipt=json.loads((root/\'latest.json\').read_text());saved=R.read_full(root/receipt[\'file\'],identity)\n    report[\'passed\']=True;stage(\'all primary CUDA phases passed\')\n    if gpu==0:\n        # Intentional nonzero exit only after all CUDA diagnostics/stable checkpoints exist.\n        write(root.parent/\'smoke-worker-crash.json\',{\'expected_exit\':1,\'GPU\':0,\'primary_checks_passed\':True})\n        raise RuntimeError(\'Reviewer injected worker failure\')\n    stage(\'wait for sibling crash, then resume healthy worker\')\n    marker=root.parent/\'smoke-worker-crash.json\'\n    while not marker.exists():\n        C.check_deadline(cfg[\'deadline_monotonic\']-150);time.sleep(.1)\n    old=copy.deepcopy(saved[\'state\'])\n    tail=R.run(name,saved,identity,cache,order,root,device,cfg[\'deadline_monotonic\'],evaluate,\n               main_targets=(1e6,),cooldown_seconds=(1.,),eval_seconds=(1.,),max_updates=64)\n    state=tail[\'state\'];assert state[\'updates\']==old[\'updates\']+64 and state[\'cursor\']==old[\'cursor\']+64*128\n    assert state[\'last_lr\']==old[\'last_lr\'] and state[\'l2_points\']==old[\'l2_points\']\n    check(\'healthy_worker_resumed_after_sibling_crash\',continued_updates=64,cursor=state[\'cursor\'])\n    report[\'passed\']=True;stage(\'complete\')\n    tail[\'status\']=\'complete\';tail[\'reviewer_smoke_tail_updates\']=64;write(root/\'summary.json\',tail)\n', 'l2_live.py': '"""Private live insurance: latest full states + latest evaluated points, flat <=2.5GB."""\nimport argparse, gzip, json, os, shutil, sys, time\nfrom pathlib import Path\nfrom kaggle_io import sha, write, safe_name\nfrom s2_live_sync import LiveSync, audit_tree, credential_patterns, privacy_probe\n\nSTAGE_LIMIT=2500000000\nSECRET_NAMES=(\'KAGGLE_USERNAME\',\'KAGGLE_KEY\')\n\ndef gate(torch_module, secret_getter=None,require_secrets=True):\n    """Secrets are fetched only into env; a missing secret always aborts before install/data."""\n    if not require_secrets:\n        if torch_module.cuda.device_count()!=2 or not all(\'T4\' in torch_module.cuda.get_device_name(i) for i in (0,1)):\n            return {\'passed\':False,\'reason\':\'GPU smoke requires exactly two T4 GPUs\'}\n        return {\'passed\':True,\'reason\':\'GPU smoke T4x2 gate passed; Secrets/live not required\'}\n    missing=[]\n    try:\n        if secret_getter is None:\n            from kaggle_secrets import UserSecretsClient\n            secret_getter=UserSecretsClient().get_secret\n        for name in SECRET_NAMES:\n            try:\n                value=secret_getter(name)\n                if not value:missing.append(name)\n                else:os.environ[name]=value\n            except Exception:missing.append(name)\n    except Exception:missing=list(SECRET_NAMES)\n    if missing:return {\'passed\':False,\'reason\':\'Kaggle Secrets unavailable: \'+\', \'.join(missing),\n                       \'action\':\'Attach both Secrets and use Edit -> Save Version -> Save & Run All\'}\n    if torch_module.cuda.device_count()!=2:return {\'passed\':False,\'reason\':\'Requires exactly two T4 GPUs\'}\n    if not all(\'T4\' in torch_module.cuda.get_device_name(i) for i in (0,1)):\n        return {\'passed\':False,\'reason\':\'Both GPU devices must be T4\'}\n    return {\'passed\':True,\'reason\':\'Kaggle Secrets and T4x2 gate passed\'}\n\ndef stage_snapshot(working,stage,ref,reason,limit=STAGE_LIMIT):\n    working,stage=Path(working),Path(stage);stage.mkdir(parents=True,exist_ok=True)\n    # Includes sources, binaries and nonselected files, as in S2v2. Secrets never go to logs/config.\n    cache_path=stage.parent/\'audit-cache.json\'\n    cache=json.loads(cache_path.read_text()) if cache_path.exists() else {}\n    cache=audit_tree(working,ref,cache);write(cache_path,cache)\n    for p in stage.iterdir():\n        if p.is_dir():shutil.rmtree(p)\n        else:p.unlink()\n    patterns=credential_patterns(ref);used=0;files=[];selected=[]\n    for root in sorted(working.glob(\'model-*\')):\n        if not root.is_dir():continue\n        selected.extend(root.glob(\'*.json\'))\n        latest=root/\'latest.json\'\n        if latest.exists():\n            rec=json.loads(latest.read_text());weight=root/safe_name(rec[\'file\'])\n            if not weight.resolve().is_relative_to(root.resolve()):raise ValueError(\'Unsafe state path\')\n            selected.append(weight)\n        # Older point weights and generations remain in earlier immutable Dataset versions.\n        complete=sorted((root/\'points\').glob(\'*/checkpoint.json\'))\n        if complete:\n            point=complete[-1].parent;selected.extend(p for p in point.rglob(\'*\') if p.is_file())\n            rec=json.loads((point/\'checkpoint.json\').read_text());selected.append(root/safe_name(rec[\'file\']))\n    selected.extend(p for p in working.glob(\'*.json\') if p.name!=\'session-config.json\')\n    # L3 reuses the exact weighted permutation, rather than rebuilding it under a new NumPy.\n    selected.extend(p for p in (working/\'weighted-order\').glob(\'*\') if p.is_file())\n    for p in sorted(set(selected)):\n        if p.is_symlink() or \'.tmp\' in p.name:raise ValueError(\'Unsafe staging source\')\n        relative=p.relative_to(working).as_posix();flat=relative.replace(\'/\',\'__\');target=stage/flat\n        if p.suffix in (\'.json\',\'.jsonl\'):\n            data=p.read_bytes()\n            if any(v in data for v in patterns):raise ValueError(\'Secret detected\')\n            if len(data)>128*1024:data=gzip.compress(data,compresslevel=1,mtime=0);flat+=\'.gz\';target=stage/flat\n            size=len(data)\n            if used+size>limit-1024**2:raise ValueError(\'Live staging size limit\')\n            target.write_bytes(data)\n        else:\n            size=p.stat().st_size\n            if used+size>limit-1024**2:raise ValueError(\'Live staging size limit\')\n            shutil.copyfile(p,target)\n        used+=size;files.append({\'file\':flat,\'original_path\':relative,\'bytes\':size,\'sha256\':sha(target),\n                                 \'gzip\':flat.endswith(\'.gz\')})\n    # A concurrent atomic checkpoint replacement must not produce a false resume receipt.\n    by_original={r[\'original_path\']:r for r in files}\n    for root in working.glob(\'model-*\'):\n        entry=by_original.get(root.name+\'/latest.json\')\n        if not entry:continue\n        data=(stage/entry[\'file\']).read_bytes();rec=json.loads(gzip.decompress(data) if entry[\'gzip\'] else data)\n        item=by_original.get(root.name+\'/\'+rec[\'file\'])\n        assert item and item[\'sha256\']==rec[\'sha256\'] and item[\'bytes\']==rec[\'bytes\']\n    write(stage/\'recover-manifest.json\',{\'format\':\'nlp-hw02-l2-live-v1\',\'reason\':reason,\'unix_time\':time.time(),\n          \'files\':files,\'payload_bytes\':used,\'staging_limit_bytes\':limit,\'credentials_in_files\':False,\n          \'point_policy\':\'latest cooled point (including partial evaluation) per model; each preceding point retained in its uploaded Dataset version\'})\n    write(stage/\'dataset-metadata.json\',{\'id\':ref,\'title\':\'NLP HW02 L2 live\',\'licenses\':[{\'name\':\'other\'}],\n          \'description\':\'Private training insurance: full main model/AdamW/scaler/RNG/cursor and evaluated FP16 points.\'})\n    audit_tree(stage,ref)\n    assert sum(p.stat().st_size for p in stage.iterdir() if p.is_file())<=limit\n\ndef restore_snapshot(source,destination):\n    source,destination=Path(source),Path(destination)\n    manifest=json.loads((source/\'recover-manifest.json\').read_text())\n    assert manifest[\'format\']==\'nlp-hw02-l2-live-v1\'\n    for rec in manifest[\'files\']:\n        p=source/safe_name(rec[\'file\']);assert sha(p)==rec[\'sha256\'] and p.stat().st_size==rec[\'bytes\']\n        target=destination/safe_name(rec[\'original_path\']);target.parent.mkdir(parents=True,exist_ok=True)\n        if rec[\'gzip\']:target.write_bytes(gzip.decompress(p.read_bytes()))\n        else:shutil.copyfile(p,target)\n    return destination\n\nclass L2LiveSync(LiveSync):\n    def __init__(self,working,scratch,ref,interval=5400.,poll_seconds=10.,upload_timeout=600.):\n        super().__init__(working,scratch,ref,interval,poll_seconds,upload_timeout)\n        # Helpers authenticate with the explicitly attached API key only, never platform JWTs.\n        for name in (\'KAGGLE_USER_SECRETS_TOKEN\',\'KAGGLE_IAP_TOKEN\'):self.env.pop(name,None)\n        self.upload_timeout=min(upload_timeout,600.);self.last_success=time.monotonic();self.failures=0\n        self.uninsured=False;self.last_attempt=0.\n    def log(self,event,**fields):\n        record={\'unix_time\':time.time(),\'event\':event,**fields}\n        with (self.working/\'l2-live-sync.log\').open(\'a\') as f:f.write(json.dumps(record)+\'\\n\')\n        write(self.working/\'l2-live-sync-status.json\',record)\n    def signature(self):\n        return sorted((str(p.relative_to(self.working)),p.stat().st_mtime_ns)\n                      for p in self.working.glob(\'model-*/points/*/complete.json\'))\n    def sync(self,reason,timeout=900.,final=False,wait_ready=False):\n        until=time.monotonic()+min(timeout,900.);helper=str(Path(__file__).resolve())\n        def call(args,cap):\n            remaining=until-time.monotonic()\n            if remaining<=0:raise TimeoutError(\'Live deadline\')\n            self.bounded_call(args,min(remaining,cap),final=final)\n        if not self.lock.acquire(timeout=min(1.,timeout)):return False\n        try:\n            for p in (self.scratch/\'cli-temp\').iterdir():\n                if p.is_dir():shutil.rmtree(p)\n                else:p.unlink()\n            call([sys.executable,helper,\'stage\',\'--working\',str(self.working),\'--stage\',str(self.stage),\n                  \'--ref\',self.ref,\'--reason\',reason],240.)\n            privacy=self.scratch/\'privacy.json\';privacy.unlink(missing_ok=True)\n            call([sys.executable,helper,\'privacy\',\'--ref\',self.ref,\'--receipt\',str(privacy)],60.)\n            info=json.loads(privacy.read_text())\n            if info.get(\'exists\') and not info.get(\'private\'):raise ValueError(\'Public Dataset refused\')\n            command=[\'kaggle\',\'datasets\',\'version\' if info[\'exists\'] else \'create\',\'-p\',str(self.stage),\n                     \'--dir-mode\',\'skip\',\'--keep-tabular\',\'--quiet\']\n            if info[\'exists\']:command+=[\'-m\',\'L2 recovery \'+reason]\n            call(command,self.upload_timeout)\n            if wait_ready:\n                # Re-probe owner-private server identity, not merely a successful local CLI exit.\n                ready=False\n                while time.monotonic()<until:\n                    privacy.unlink(missing_ok=True)\n                    call([sys.executable,helper,\'privacy\',\'--ref\',self.ref,\'--receipt\',str(privacy)],30.)\n                    current=json.loads(privacy.read_text())\n                    if current.get(\'exists\'):\n                        if not current.get(\'private\'):raise ValueError(\'Server privacy check failed\')\n                        ready=True;break\n                    time.sleep(min(5.,max(0.,until-time.monotonic())))\n                if not ready:raise TimeoutError(\'Dataset not visible after startup upload\')\n            # Successful upload is insured only after Dataset processing reports ready.\n            status=self.scratch/\'ready.json\';ready=False\n            while time.monotonic()<until:\n                status.unlink(missing_ok=True)\n                call([sys.executable,helper,\'ready\',\'--ref\',self.ref,\'--receipt\',str(status)],30.)\n                state=json.loads(status.read_text())[\'status\']\n                if state==\'ready\':ready=True;break\n                if state==\'error\':raise RuntimeError(\'Dataset processing failed\')\n                time.sleep(min(10.,max(0.,until-time.monotonic())))\n            if not ready:raise TimeoutError(\'Dataset processing deadline\')\n            self.last_success=time.monotonic();self.failures=0\n            self.log(\'uploaded\',reason=reason,operation=\'version\' if info[\'exists\'] else \'create\',\n                     staging_bytes=sum(p.stat().st_size for p in self.stage.iterdir() if p.is_file()))\n            return True\n        except Exception as e:\n            self.failures+=1\n            self.log(\'sync_error\',reason=reason,error_type=type(e).__name__,returncode=getattr(e,\'returncode\',None))\n            return False\n        finally:self.lock.release()\n    def run(self):\n        while not self.stop_event.is_set():\n            try:\n                now=time.monotonic();signature=self.signature()\n                new=signature!=self.last_signature\n                due=now-self.last_success>=self.interval\n                eligible=now-self.last_attempt>=(300. if self.failures else 30.)\n                if (new or due or self.failures) and eligible:\n                    self.last_attempt=now\n                    if self.sync(\'new-point\' if new else \'full-state-heartbeat\'):self.last_signature=signature\n                    self.last_attempt=time.monotonic()\n                if time.monotonic()-self.last_success>=7200.:\n                    self.uninsured=True;self.log(\'insurance_expired\',age_seconds=time.monotonic()-self.last_success)\n                    return\n            except Exception as e:self.log(\'thread_error\',error_type=type(e).__name__)\n            self.stop_event.wait(self.poll_seconds)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=[\'stage\',\'privacy\',\'ready\',\'restore\'])\n    for name in (\'working\',\'stage\',\'ref\',\'receipt\',\'reason\',\'destination\'):p.add_argument(\'--\'+name)\n    a=p.parse_args()\n    try:\n        if a.mode==\'stage\':stage_snapshot(a.working,a.stage,a.ref,a.reason)\n        elif a.mode==\'privacy\':privacy_probe(a.ref,a.receipt)\n        elif a.mode==\'ready\':\n            from kaggle.api.kaggle_api_extended import KaggleApi\n            api=KaggleApi();api.authenticate();write(a.receipt,{\'status\':api.dataset_status(a.ref)})\n        else:restore_snapshot(a.stage,a.destination)\n    except Exception:sys.exit(1)\n', 'l2_resume.py': '"""Fail-fast pinned L1 C preflight. No D blobs, no newest-state guessing."""\nimport copy, json\nfrom pathlib import Path\nimport torch\nimport l1_runtime as L\nfrom kaggle_io import sha, safe_name\nfrom exp26_common import json_hash\n\ndef verify_anchor(source, expected, check_environment=True):\n    source = Path(source); path = source/safe_name(expected[\'file\'])\n    assert path.stat().st_size == expected[\'bytes\'], \'L1 C bytes mismatch\'\n    assert sha(path) == expected[\'sha256\'], \'L1 C SHA mismatch\'\n    p = L.read_full(path, expected[\'identity\'])\n    s = p[\'state\']\n    for key in (\'updates\', \'cursor\', \'main_seconds\'):\n        assert s[key] == expected[key], \'L1 C \'+key+\' mismatch\'\n    assert p[\'model_spec\'] == L.SPECS[\'C\'] and p[\'identity\'][\'model\'] == \'C\'\n    assert s[\'last_lr\'] == L.PEAK_LR[\'C\'] and s[\'updates\'] > 1000\n    assert p[\'scaler_enabled\'] and p[\'rng\'][\'torch_cuda\'], \'Missing real AMP/CUDA RNG state\'\n    recipe = p[\'identity\'][\'recipe\']\n    for name, digest in recipe[\'implementation_sha256\'].items():\n        assert sha(Path(L.__file__).with_name(name)) == digest, \'L1 implementation changed: \'+name\n    if check_environment:\n        assert recipe[\'torch\'] == str(torch.__version__) and recipe[\'CUDA\'] == torch.version.cuda, \'Torch/CUDA resume mismatch\'\n    assert all(bool(torch.isfinite(v).all()) for v in p[\'model\'].values() if v.is_floating_point())\n    assert all(bool(torch.isfinite(v).all()) for a in p[\'optimizer\'][\'state\'].values()\n               for v in a.values() if torch.is_tensor(v) and v.is_floating_point())\n    return path, p\n\ndef branch_identity(parent, order_meta):\n    identity = copy.deepcopy(parent); identity[\'model\'] = \'C-long2x\'\n    identity[\'parent_recipe_sha256\'] = parent[\'recipe_sha256\']\n    identity[\'order_sha256\'] = order_meta[\'file\'][\'sha256\']\n    identity[\'recipe\'][\'data_policy\'] = \'fixed bucketed weighted order; long rows twice, others once; circular\'\n    identity[\'recipe\'][\'sampler\'] = copy.deepcopy(order_meta)\n    identity[\'recipe_sha256\'] = json_hash(identity[\'recipe\'])\n    return identity\n\ndef initial_state(payload, name):\n    state = copy.deepcopy(payload[\'state\'])\n    state[\'l2_origin\'] = {\'updates\': state[\'updates\'], \'cursor\': state[\'cursor\'], \'main_seconds\': state[\'main_seconds\']}\n    if name == \'C-long2x\': state[\'cursor\'] = 0\n    state[\'cursor_origin\'] = state[\'cursor\']-state[\'updates\']*128\n    state[\'l2_main_seconds\'] = 0.; state[\'nonfinite_skips\'] = 0\n    state[\'l2_points\'] = []; state[\'stability\'] = {\'total\': 0, \'consecutive\': 0, \'window\': [], \'escalations\': 0, \'history\': []}\n    return state\n\ndef mounted_expected(source, expected_user):\n    """Read the mounted live receipt before any cache scan; validate the requested C, never D."""\n    import gzip\n    source = Path(source); manifest = json.loads((source/\'recover-manifest.json\').read_text())\n    assert manifest[\'format\'] == \'nlp-hw02-l1-live-v1\'\n    blob = next(r for r in manifest[\'files\'] if r[\'file\'] == expected_user[\'file\'])\n    latest = next(r for r in manifest[\'files\'] if r[\'original_path\'] == \'model-C/latest.json\')\n    raw = (source/latest[\'file\']).read_bytes()\n    assert sha(source/latest[\'file\']) == latest[\'sha256\'] and len(raw) == latest[\'bytes\']\n    receipt = json.loads(gzip.decompress(raw) if latest[\'gzip\'] else raw)\n    assert receipt[\'file\'].replace(\'/\',\'__\') == expected_user[\'file\'].removeprefix(\'model-C__\')\n    assert receipt[\'sha256\'] == blob[\'sha256\'] and receipt[\'bytes\'] == blob[\'bytes\']\n    assert receipt[\'updates\'] == expected_user[\'updates\'] and receipt[\'cursor\'] == expected_user[\'cursor\']\n    assert abs(receipt[\'main_seconds\']-expected_user[\'main_seconds\']) <= 5., \'Rounded user main_s mismatch\'\n    return dict(receipt, file=blob[\'file\'])\n\ndef cpu_restore(payload, model, opt):\n    """CPU audit only: preserve CUDA RNG/scaler in payload; never claim CUDA numerical identity."""\n    import exp26_train as T\n    model.load_state_dict(payload[\'model\']); opt.load_state_dict(payload[\'optimizer\'])\n    # CUDA fused flag cannot run on CPU, but moments/steps/LR remain unchanged.\n    for g in opt.param_groups: g[\'fused\'] = False\n    T.restore_rng(payload[\'rng\'], torch.device(\'cpu\')); model.train()\n    return copy.deepcopy(payload[\'state\'])\n', 'l2_runtime.py': '"""L1 finite-update math, independent nonfinite recovery and disposable timed cooldowns."""\nimport copy, gc, json, time, uuid\nfrom pathlib import Path\nimport torch\nimport l1_runtime as L\nimport exp26_core as C\nimport exp26_train as T\nfrom kaggle_io import sha, write\nfrom l2_resume import initial_state\n\nPOLICY = {\'consecutive\': 3, \'window_size\': 100, \'window_failures\': 10, \'max_escalations\': 3,\n          \'amp_consecutive\': 32}\nclass UnsafeState(RuntimeError): pass\n\ndef safe_update(model, opt, scaler, cache, indices, device, micro, lr):\n    """Never call Adam after rejected loss/gradient. Normal AMP skips stay normal."""\n    started = time.perf_counter(); scale = copy.deepcopy(scaler.state_dict())\n    try:\n        log, micro = L.resilient_update(model, opt, scaler, cache, indices, device, micro, lr)\n        tensors = list(model.parameters()) + [v for s in opt.state.values() for v in s.values()\n                                              if torch.is_tensor(v) and v.is_floating_point()]\n        norms = torch.stack(torch._foreach_norm(tensors))\n        if not bool(torch.isfinite(norms).all()): raise UnsafeState(\'Nonfinite model/Adam after update\')\n        log[\'nonfinite_skip\'] = 0\n        log[\'seconds\'] = time.perf_counter()-started\n        return log, micro\n    except FloatingPointError:\n        opt.zero_grad(set_to_none=True)\n        scaler.load_state_dict(scale)\n        if scaler.is_enabled(): scaler._per_optimizer_states.clear()\n        for group in opt.param_groups: group[\'lr\'] = lr\n        C.sync(device)\n        return {\'seconds\': time.perf_counter()-started, \'CE\': None, \'AMP_skip\': 0, \'nonfinite_skip\': 1,\n                \'lr\': lr, \'pairs\': len(indices), \'tokens\': 0, \'gradient_norm\': None, \'OOM_retries\': []}, micro\n\ndef observe(stability, rejected, policy=POLICY, amp_skip=0):\n    stability[\'total\'] += int(rejected)\n    stability[\'consecutive\'] = stability[\'consecutive\']+1 if rejected else 0\n    stability[\'window\'] = (stability[\'window\']+[int(rejected)])[-policy[\'window_size\']:]\n    stability[\'amp_total\'] = stability.get(\'amp_total\', 0)+int(amp_skip)\n    stability[\'amp_consecutive\'] = stability.get(\'amp_consecutive\', 0)+1 if amp_skip else 0\n    return (stability[\'consecutive\'] >= policy[\'consecutive\'] or sum(stability[\'window\']) >= policy[\'window_failures\']\n            or stability[\'amp_consecutive\'] >= policy[\'amp_consecutive\'])\n\ndef finite_payload(payload):\n    for value in list(payload[\'model\'].values()) + [v for s in payload[\'optimizer\'][\'state\'].values() for v in s.values()]:\n        if torch.is_tensor(value) and value.is_floating_point() and not bool(torch.isfinite(value).all()): return False\n    return True\n\ndef payload(model, opt, scaler, state, spec, identity):\n    p = L.state_payload(model, opt, scaler, state, spec, identity)\n    p[\'format\'] = \'nlp-hw02-l2-full-v1\'; p[\'stability_policy\'] = POLICY\n    p[\'runtime_sha256\'] = {n: sha(Path(__file__).with_name(n)) for n in (\'l2_runtime.py\', \'l2_resume.py\', \'l2_sampler.py\')}\n    p[\'scheduler\'][\'kind\'] = \'constant inherited LR; recorded emergency halvings; no repeated warmup\'\n    return p\n\ndef read_full(path, identity, check_environment=True):\n    p = torch.load(path, map_location=\'cpu\', weights_only=True)\n    assert p[\'format\'] == \'nlp-hw02-l2-full-v1\' and p[\'identity\'] == identity\n    assert p[\'stability_policy\'] == POLICY\n    for n, digest in p[\'runtime_sha256\'].items(): assert sha(Path(__file__).with_name(n)) == digest\n    for n, digest in identity[\'recipe\'][\'implementation_sha256\'].items(): assert sha(Path(__file__).with_name(n)) == digest\n    if check_environment:\n        assert identity[\'recipe\'][\'torch\'] == str(torch.__version__) and identity[\'recipe\'][\'CUDA\'] == torch.version.cuda\n    s = p[\'state\']; assert s[\'cursor\'] == s[\'cursor_origin\']+s[\'updates\']*128\n    assert p[\'scheduler\'][\'last_lr\'] == s[\'last_lr\']\n    assert all(g[\'lr\'] == s[\'last_lr\'] for g in p[\'optimizer\'][\'param_groups\'])\n    steps = s[\'updates\']-s[\'AMP_skips\']-s[\'nonfinite_skips\']\n    assert all(int(v[\'step\']) == steps for v in p[\'optimizer\'][\'state\'].values())\n    assert finite_payload(p), \'Nonfinite stable payload\'\n    return p\n\ndef save_full(root, model, opt, scaler, state, spec, identity):\n    root = Path(root); folder = root/\'checkpoints\'; folder.mkdir(parents=True, exist_ok=True)\n    C.sync(next(model.parameters()).device); p = payload(model, opt, scaler, state, spec, identity)\n    if not finite_payload(p): raise FloatingPointError(\'Refuse nonfinite stable state\')\n    path = folder/(\'main-\'+uuid.uuid4().hex[:12]+\'.pt\'); tmp = path.with_suffix(\'.pt.tmp\')\n    torch.save(p, tmp); tmp.replace(path); del p\n    rec = {\'file\': path.relative_to(root).as_posix(), \'sha256\': sha(path), \'bytes\': path.stat().st_size,\n           \'updates\': state[\'updates\'], \'cursor\': state[\'cursor\'], \'main_seconds\': state[\'main_seconds\'],\n           \'l2_main_seconds\': state[\'l2_main_seconds\'], \'identity\': identity, \'model_spec\': spec,\n           \'purpose\': \'stable full L2 resume; uncooled; model/AdamW/scaler/all RNG/history\'}\n    write(root/\'latest.json\', rec)\n    for old in sorted(folder.glob(\'main-*.pt\'), key=lambda p:p.stat().st_mtime_ns, reverse=True)[2:]: old.unlink(missing_ok=True)\n    return rec\n\ndef reload_latest(root, identity, device, smoke=False):\n    rec = json.loads((Path(root)/\'latest.json\').read_text()); path = Path(root)/rec[\'file\']\n    assert sha(path) == rec[\'sha256\'] and path.stat().st_size == rec[\'bytes\']\n    p = read_full(path, identity)\n    m = C.SimpleTransformer(**p[\'model_spec\'], dropout=0.).to(device)\n    o = L.optimizer(m, p[\'state\'][\'last_lr\'], smoke)\n    s = torch.amp.GradScaler(device.type, enabled=device.type==\'cuda\', init_scale=1024)\n    state = L.restore_full(p, m, o, s, device); del p\n    return m, o, s, state\n\ndef release(device, *objects):\n    # Callers must drop their references before empty_cache.\n    gc.collect()\n    if device.type == \'cuda\': torch.cuda.empty_cache()\n\ndef run(name, anchor, identity, cache, order, root, device, deadline, evaluate,\n        main_targets=(10800.,21600.,28800.), cooldown_seconds=(900.,900.,3000.),\n        eval_seconds=(900.,900.,1800.), checkpoint_seconds=1800., smoke=False, max_updates=None):\n    root = Path(root); root.mkdir(parents=True, exist_ok=True); spec = anchor[\'model_spec\']\n    model = C.SimpleTransformer(**spec, dropout=0.).to(device)\n    opt = L.optimizer(model, anchor[\'state\'][\'last_lr\'], smoke)\n    scaler = torch.amp.GradScaler(device.type, enabled=device.type==\'cuda\', init_scale=1024)\n    state = L.restore_full(anchor, model, opt, scaler, device)\n    if anchor[\'format\'] == \'nlp-hw02-l1-full-v1\': state = initial_state(anchor, name)\n    # L3 can pass read_full() directly: order/cursor/Adam/scaler/RNG/history survive.\n    del anchor\n    write(root/\'identity.json\', identity); write(root/\'stability-policy.json\', POLICY)\n    save_full(root, model, opt, scaler, state, spec, identity)\n    last_save = time.monotonic(); started_updates = state[\'updates\']; status = \'complete\'; logs = []; pending = None\n    def stopped(): return (root.parent/\'stop-request.json\').exists() or time.monotonic() >= deadline-120*(not smoke)\n    def summary(status):\n        result = {\'model\': name, \'status\': status, \'state\': state, \'identity\': identity}\n        write(root/\'summary.json\', result); return result\n    summary(\'training\')\n    try:\n        for mark, cool_s, eval_s in zip(main_targets, cooldown_seconds, eval_seconds):\n            if f\'l2-{int(mark):05d}\' in {p[\'tag\'] for p in state[\'l2_points\']}: continue\n            while state[\'l2_main_seconds\'] < mark and not stopped():\n                if max_updates is not None and state[\'updates\']-started_updates >= max_updates: break\n                unsafe = pending; pending = None; log = None\n                if unsafe is None:\n                    attempt_started = time.perf_counter()\n                    try:\n                        log, micro = safe_update(model,opt,scaler,cache,L.take(order,state[\'cursor\'],128),device,state[\'micro_batch\'],state[\'last_lr\'])\n                    except L.UnsafeAdamOOM: unsafe = \'Adam OOM\'\n                    except UnsafeState: unsafe = \'nonfinite post-Adam model\'\n                    if unsafe is not None:\n                        C.sync(device)\n                        spent = time.perf_counter()-attempt_started\n                        state[\'main_seconds\'] += spent; state[\'l2_main_seconds\'] += spent\n                if log is not None:\n                    state[\'micro_batch\'] = micro; state[\'updates\'] += 1; state[\'cursor\'] += 128\n                    state[\'main_seconds\'] += log[\'seconds\']; state[\'l2_main_seconds\'] += log[\'seconds\']\n                    state[\'AMP_skips\'] += log[\'AMP_skip\']; state[\'nonfinite_skips\'] += log[\'nonfinite_skip\']\n                    logs = (logs+[log])[-100:]\n                    escalate = observe(state[\'stability\'], log[\'nonfinite_skip\'], amp_skip=log[\'AMP_skip\'])\n                else: escalate = True\n                if escalate:\n                    history = copy.deepcopy(state[\'stability\']); elapsed = state[\'l2_main_seconds\']\n                    history[\'escalations\'] += 1\n                    event = {\'reason\': unsafe or \'nonfinite threshold\', \'from_updates\': state[\'updates\'],\n                             \'l2_main_seconds\': elapsed, \'escalation\': history[\'escalations\'], \'old_lr\': state[\'last_lr\']}\n                    old_micro = state[\'micro_batch\']; old_lr = state[\'last_lr\']\n                    del model,opt,scaler; release(device)\n                    model,opt,scaler,state = reload_latest(root,identity,device,smoke)\n                    # Work already spent remains in the common main-time axis after rollback.\n                    state[\'l2_main_seconds\'] = elapsed; state[\'main_seconds\'] = state[\'l2_origin\'][\'main_seconds\']+elapsed\n                    history[\'consecutive\'] = 0; history[\'window\'] = []; history[\'amp_consecutive\'] = 0\n                    event.update(restored_updates=state[\'updates\'], restored_cursor=state[\'cursor\'])\n                    if history[\'escalations\'] >= POLICY[\'max_escalations\']:\n                        event[\'action\'] = \'graceful worker stop; keep last good LR/state\'; status = \'stability_stop\'\n                    else:\n                        state[\'last_lr\'] = old_lr*.5\n                        for group in opt.param_groups: group[\'lr\'] = state[\'last_lr\']\n                        event.update(action=\'rollback and halve LR\', new_lr=state[\'last_lr\'])\n                        if unsafe == \'Adam OOM\':\n                            choices = [m for m in L.MICRO_BATCHES if m < old_micro]\n                            if not choices: status = \'stability_stop\'\n                            else: state[\'micro_batch\'] = choices[0]\n                    history[\'history\'].append(event); state[\'stability\'] = history\n                    write(root/\'stability-history.json\', history)\n                    save_full(root,model,opt,scaler,state,spec,identity); last_save = time.monotonic()\n                    if status == \'stability_stop\': break\n                    continue\n                if state[\'updates\'] % (1 if smoke else 100) == 0:\n                    write(root/\'progress.json\', state)\n                    print(\'L2 TRAIN\',name,\'updates\',state[\'updates\'],\'main_s\',round(state[\'l2_main_seconds\'],1),\n                          \'CE\',log[\'CE\'],\'LR\',state[\'last_lr\'],\'NF\',state[\'stability\'][\'total\'],flush=True)\n                if time.monotonic()-last_save >= checkpoint_seconds:\n                    try: save_full(root,model,opt,scaler,state,spec,identity)\n                    except FloatingPointError:\n                        # A corrupted Adam/model must never replace the last known-good receipt.\n                        pending = \'nonfinite stable candidate\'; continue\n                    last_save = time.monotonic()\n            if status == \'stability_stop\' or stopped() or state[\'l2_main_seconds\'] < mark:\n                status = status if status == \'stability_stop\' else \'partial; target main time not reached\'; break\n            # Both models evaluate the same L2 main-time target and the same timed cooldown.\n            rec = save_full(root,model,opt,scaler,state,spec,identity); last_save = time.monotonic()\n            tag = f\'l2-{int(mark):05d}\'; out = root/\'points\'/tag; out.mkdir(parents=True,exist_ok=True)\n            train = {\'target_l2_main_seconds\': mark, \'l2_main_seconds\': state[\'l2_main_seconds\'],\n                     \'main_seconds\': state[\'main_seconds\'], \'anchor_updates\': state[\'updates\'],\n                     \'anchor_cursor\': state[\'cursor\'], \'anchor_sha256\': rec[\'sha256\'],\n                     \'cooldown_target_seconds\': cool_s, \'cooldown_updates\': 0, \'cooldown_AMP_skips\': 0,\n                     \'cooldown_nonfinite_skips\': 0, \'final\': mark == main_targets[-1]}\n            write(out/\'training.json\',train)\n            branch_start = time.monotonic(); branch_s = 0.; branch_failed = False; metrics = None\n            branch_stability = copy.deepcopy(state[\'stability\']); branch_failure_reason = None\n            try:\n                if deadline-time.monotonic() < cool_s+eval_s+120*(not smoke):\n                    raise C.BudgetExpired(\'Insufficient reserved cooldown/evaluation time\')\n                while branch_s < cool_s and not stopped():\n                    lr = state[\'last_lr\']*max(0.,1.-branch_s/cool_s)\n                    log,micro = safe_update(model,opt,scaler,cache,L.take(order,state[\'cursor\']+train[\'cooldown_updates\']*128,128),\n                                           device,state[\'micro_batch\'],lr)\n                    state[\'micro_batch\'] = micro; branch_s += log[\'seconds\']; train[\'cooldown_updates\'] += 1\n                    train[\'cooldown_AMP_skips\'] += log[\'AMP_skip\']; train[\'cooldown_nonfinite_skips\'] += log[\'nonfinite_skip\']\n                    if observe(branch_stability,log[\'nonfinite_skip\'],amp_skip=log[\'AMP_skip\']):\n                        branch_failed = True; branch_failure_reason = \'nonfinite cooldown threshold\'; break\n                train[\'cooldown_seconds\'] = branch_s; train[\'cooldown_wall_seconds\'] = time.monotonic()-branch_start\n                if branch_failed or branch_s < cool_s: raise RuntimeError(\'Incomplete cooldown\')\n                if not all(bool(torch.isfinite(v).all()) for v in model.parameters()): raise FloatingPointError(\'Nonfinite cooled model\')\n                point = L.save_point(root,tag,model,spec,identity,train)\n                del model,opt,scaler; model=opt=scaler=None; release(device)\n                begin = time.monotonic()\n                metrics = evaluate(point,out/\'evaluation\',min(deadline-60*(not smoke),time.monotonic()+eval_s),train[\'final\'])\n                train[\'evaluation_seconds\'] = time.monotonic()-begin\n                write(out/\'complete.json\',{\'training\':train,\'metrics\':metrics,\'weights_sha256\':sha(point)})\n            except Exception as error:\n                if isinstance(error,(UnsafeState,L.UnsafeAdamOOM)):\n                    branch_failed = True; branch_failure_reason = type(error).__name__+\' in cooldown\'\n                write(out/\'incomplete.json\',{\'reason\':type(error).__name__,\'training\':train})\n                print(\'L2 POINT INCOMPLETE\',name,tag,type(error).__name__,flush=True)\n            finally:\n                if model is not None: del model,opt,scaler\n                release(device); model,opt,scaler,state = reload_latest(root,identity,device,smoke)\n                state[\'branch_seconds\'] += branch_s; state[\'branch_attempted_updates\'] += train[\'cooldown_updates\']\n                state[\'stability\'] = branch_stability\n                if branch_failed:\n                    h = state[\'stability\']; h[\'escalations\'] += 1\n                    event = {\'reason\':branch_failure_reason,\'point\':tag,\'restored_updates\':state[\'updates\'],\n                             \'restored_cursor\':state[\'cursor\'],\'old_lr\':state[\'last_lr\'],\'escalation\':h[\'escalations\']}\n                    h[\'consecutive\'] = 0; h[\'window\'] = []; h[\'amp_consecutive\'] = 0\n                    if h[\'escalations\'] >= POLICY[\'max_escalations\']:\n                        event[\'action\'] = \'graceful worker stop; cooldown discarded\'; status = \'stability_stop\'\n                    else:\n                        state[\'last_lr\'] *= .5\n                        for group in opt.param_groups: group[\'lr\'] = state[\'last_lr\']\n                        event.update(action=\'cooldown rollback and halve worker LR\',new_lr=state[\'last_lr\'])\n                    h[\'history\'].append(event); write(root/\'stability-history.json\',h)\n                state[\'l2_points\'].append({\'tag\':tag,\'training\':train,\'complete\':metrics is not None,\'metrics\':metrics})\n                save_full(root,model,opt,scaler,state,spec,identity); last_save = time.monotonic(); summary(\'training\')\n            if status == \'stability_stop\': break\n        # Even a stability stop publishes the restored last-good state, with cumulative history.\n        save_full(root,model,opt,scaler,state,spec,identity)\n        return summary(status)\n    finally:\n        del model,opt,scaler; release(device)\n', 'l2_sampler.py': '"""Deterministic 1/2 row multiplicity; reference lengths never enter training loss."""\nimport hashlib, json\nfrom pathlib import Path\nimport numpy as np\nfrom kaggle_io import sha, write\nfrom exp26_common import json_hash\n\nSEED = 2026101302\n\ndef fixed_split(n=4096):\n    ordered = sorted(range(n), key=lambda i: hashlib.sha256(\n        f\'candidate-selection-v1:heldout4k:{i}\'.encode()).digest())\n    return {\'tune2048\': sorted(ordered[:n//2]), \'eval2048\': sorted(ordered[n//2:])}\n\ndef threshold_for(lengths, fraction):\n    """Nearest attainable >=threshold fraction; ties choose the higher threshold."""\n    counts = np.bincount(np.asarray(lengths, dtype=np.int64))\n    tails = np.cumsum(counts[::-1])[::-1]\n    options = [(abs(int(tails[t])/len(lengths)-fraction), -t, t)\n               for t in np.flatnonzero(counts)]\n    options.append((fraction, -len(counts), len(counts)))\n    return min(options)[2]\n\ndef original_lengths(cache, root):\n    """Accept raw offsets only with exact source/provenance/row SHA alignment."""\n    root = Path(root) if root else None\n    if root is None: return None\n    values = []\n    for rec in cache.manifest[\'shards\']:\n        folder = root/rec[\'folder\']; compact = cache.root/rec[\'folder\']\n        expected = {r[\'file\']: r for r in rec[\'original_files\']}\n        for name in (\'source-offsets.npy\', \'provenance-rows.npy\', \'question-sha256.npy\', \'target-offsets.npy\'):\n            p = folder/name; r = expected[name]\n            assert p.stat().st_size == r[\'bytes\'] and sha(p) == r[\'sha256\'], \'Unverified original lengths\'\n            if name != \'target-offsets.npy\': assert sha(p) == sha(compact/name), \'Row identity mismatch\'\n        offsets = np.load(folder/\'target-offsets.npy\', mmap_mode=\'r\', allow_pickle=False)\n        assert len(offsets)-1 == rec[\'rows\']\n        values.append(np.diff(offsets))\n    return np.concatenate(values)\n\ndef build_order(cache, heldout_lengths, destination, original_root=None, seed=SEED, batch=128, chunk=4096):\n    assert 0 < len(cache) <= np.iinfo(np.uint32).max and batch > 0 and chunk > 0\n    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)\n    raw = original_lengths(cache, original_root)\n    lengths = raw if raw is not None else cache.target_lengths\n    fraction = float(np.mean(np.asarray(heldout_lengths) >= 90))\n    threshold = threshold_for(lengths, fraction); long = np.asarray(lengths) >= threshold\n    slots = np.concatenate((np.arange(len(cache), dtype=np.uint32), np.flatnonzero(long).astype(np.uint32)))\n    rng = np.random.default_rng(seed); rng.shuffle(slots)\n    # Identical L1 bucketing: sort inside 4096, then permute complete batches.\n    for lo in range(0, len(slots), chunk):\n        part = slots[lo:lo+chunk]; ns = cache.source_lengths[part].astype(np.int32)\n        nt = cache.target_lengths[part].astype(np.int32)\n        part = part[np.lexsort((nt, ns, ns+nt+1))]; full = len(part)//batch\n        if full: part[:full*batch] = part[:full*batch].reshape(full, batch)[rng.permutation(full)].reshape(-1)\n        slots[lo:lo+len(part)] = part\n    assert np.array_equal(np.bincount(slots, minlength=len(cache)), 1+long.astype(np.int64))\n    np.save(destination/\'order.npy\', slots, allow_pickle=False)\n    meta = {\'seed\': seed, \'rows\': len(cache), \'slots\': len(slots), \'batch\': batch, \'chunk\': chunk,\n            \'length_source\': \'original BPE offsets; exact row identity\' if raw is not None else \'verified compact target-offsets\',\n            \'original_lengths_available\': raw is not None, \'threshold\': int(threshold),\n            \'heldout_original_ge90\': int(sum(np.asarray(heldout_lengths) >= 90)),\n            \'heldout_rows\': len(heldout_lengths), \'target_long_fraction\': fraction,\n            \'train_long_rows\': int(long.sum()), \'train_long_fraction\': float(long.mean()),\n            \'fraction_error\': float(long.mean())-fraction, \'weighted_long_fraction\': float(2*long.sum()/len(slots)),\n            \'multiplicity\': {\'short\': 1, \'long\': 2}, \'cache_manifest_sha256\': cache.manifest_sha,\n            \'file\': {\'file\': \'order.npy\', \'sha256\': sha(destination/\'order.npy\'), \'bytes\': (destination/\'order.npy\').stat().st_size},\n            \'selection\': \'nearest attainable threshold; higher threshold wins ties; no loss reweighting\',\n            \'lengths_sha256\': hashlib.sha256(np.asarray(lengths, dtype=\'<i8\').tobytes()).hexdigest()}\n    write(destination/\'order.json\', meta)\n    write(destination/\'split.json\', {k: {\'ids\': v, \'sha256\': json_hash(v)} for k, v in fixed_split().items()})\n    return slots, meta\n\ndef load_order(root, cache, expected_sha):\n    """Check metadata and every index before the first CUDA update, including weighted tails."""\n    root = Path(root); meta = json.loads((root/\'order.json\').read_text())\n    path = root/\'order.npy\'\n    assert sha(path) == meta[\'file\'][\'sha256\'] == expected_sha, \'Order SHA mismatch\'\n    assert path.stat().st_size == meta[\'file\'][\'bytes\'] and meta[\'rows\'] == len(cache)\n    order = np.load(path, mmap_mode=\'r\', allow_pickle=False)\n    assert order.ndim == 1 and order.dtype == np.uint32 and len(order) > 0\n    assert int(order.min()) >= 0 and int(order.max()) < len(cache), \'Order index out of bounds\'\n    if \'slots\' in meta: assert len(order) == meta[\'slots\']\n    return order, meta\n', 'l2_session.py': '"""Independent workers: a failed sibling never requests a healthy worker stop."""\nimport json, os, shutil, subprocess, sys, time\nfrom pathlib import Path\nfrom l1_session import Watchdog\nfrom session_runner import stop_workers, tail\nfrom kaggle_io import write, sha\n\ndef disk_guard(root,limit):\n    """Rotation and atomic JSON publication may remove a path during this scan."""\n    used=0\n    for p in Path(root).rglob(\'*\'):\n        try:\n            if p.is_file():used+=p.stat().st_size\n        except FileNotFoundError:continue\n    if used>limit:raise RuntimeError(\'Working output exceeds reserve\')\n    if shutil.disk_usage(root).free<512*1024**2:raise RuntimeError(\'Less than 512 MiB free\')\n    return used\n\ndef launch(config,worker,watchdog,live=None,smoke=False,heartbeat=30.):\n    working=Path(config[\'working_root\']);working.mkdir(parents=True,exist_ok=True)\n    path=working/\'session-config.json\';write(path,config);handles=[];processes=[];reason=\'workers completed\';stop_until=None;failed=set()\n    try:\n        for gpu in (0,1):\n            env=dict(os.environ,CUDA_VISIBLE_DEVICES=\'\' if smoke else str(gpu),OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',\n                     OPENBLAS_NUM_THREADS=\'1\',TOKENIZERS_PARALLELISM=\'false\')\n            for secret in (\'KAGGLE_USERNAME\',\'KAGGLE_KEY\',\'KAGGLE_API_TOKEN\',\n                           \'KAGGLE_USER_SECRETS_TOKEN\',\'KAGGLE_IAP_TOKEN\'):env.pop(secret,None)\n            h=(working/f\'gpu{gpu}-process.log\').open(\'w\');handles.append(h)\n            p=subprocess.Popen([sys.executable,str(worker),\'--gpu\',str(gpu),\'--config\',str(path)],env=env,\n                               stdout=h,stderr=subprocess.STDOUT,start_new_session=True)\n            processes.append(p);watchdog.register(p)\n        while any(p.poll() is None for p in processes):\n            if live is not None and (live.uninsured or time.monotonic()-live.last_success>=7200.):\n                reason=\'live insurance stale >=2h\';stop_until=stop_until or min(time.monotonic()+120.,config[\'deadline_monotonic\'])\n            if time.monotonic()>=config[\'deadline_monotonic\']:reason=\'session deadline\';break\n            for gpu,p in enumerate(processes):\n                if p.poll() not in (None,0) and gpu not in failed:\n                    failed.add(gpu)\n                    write(working/f\'gpu{gpu}-failure.json\',{\'exit_code\':p.poll(),\'healthy_worker_stopped\':False,\n                          \'available_state\':(working/(\'model-C-main\' if gpu==0 else \'model-C-long2x\')/\'latest.json\').exists()})\n                    print(\'L2 worker failed\',gpu,\'; other worker continues independently\',flush=True)\n            if stop_until is not None:\n                if not (working/\'stop-request.json\').exists():write(working/\'stop-request.json\',{\'reason\':reason})\n                if time.monotonic()>=stop_until:break\n            for gpu,p in enumerate(processes):\n                print(\'L2 GPU\',gpu,\'status\',p.poll(),\' | \'.join(tail(working/f\'gpu{gpu}-process.log\'))[:700],flush=True)\n            write(working/\'session-live.json\',{\'elapsed_seconds\':time.monotonic()-config[\'session_start_monotonic\'],\n                  \'exit_codes\':[p.poll() for p in processes],\'live_insurance_stale\':bool(live and live.uninsured)})\n            disk_guard(working,limit=10*1024**3)\n            time.sleep(min(heartbeat,max(.01,config[\'deadline_monotonic\']-time.monotonic())))\n    except Exception as error:\n        reason=\'supervisor error: \'+type(error).__name__\n        write(working/\'supervisor-failure.json\',{\'error_type\':type(error).__name__})\n    finally:\n        survivors=stop_workers(processes,watchdog)\n        for h in handles:h.close()\n        if not survivors:\n            for p in working.glob(\'model-*/checkpoints/*.pt.tmp\'):p.unlink(missing_ok=True)\n        from l2_compare import compare\n        try:comparison=compare(working)\n        except Exception as error:comparison={\'status\':\'incomplete\',\'error_type\':type(error).__name__}\n        for gpu,p in enumerate(processes):\n            if p.returncode not in (None,0):\n                failed.add(gpu)\n                if not (working/f\'gpu{gpu}-failure.json\').exists():\n                    write(working/f\'gpu{gpu}-failure.json\',{\'exit_code\':p.returncode,\'healthy_worker_stopped\':False})\n        worker_status={}\n        for name in (\'C-main\',\'C-long2x\'):\n            p=working/f\'model-{name}\'/\'summary.json\'\n            try:worker_status[name]=json.loads(p.read_text()).get(\'status\') if p.exists() else \'missing\'\n            except (OSError,ValueError):worker_status[name]=\'unreadable\'\n        report={\'status\':\'complete\' if len(processes)==2 and all(p.returncode==0 for p in processes)\n                 and all(s==\'complete\' for s in worker_status.values()) else \'partial\',\'worker_status\':worker_status,\n                \'reason\':reason,\'failed_workers\':sorted(failed),\'exit_codes\':[p.poll() for p in processes],\'unreaped_worker_pids\':survivors,\n                \'wall_seconds\':time.monotonic()-config[\'session_start_monotonic\'],\'comparison\':comparison,\n                \'output_policy\':\'individual files; no ZIP; full stable states + all evaluated FP16 weights\',\n                \'hard_shutdown_seconds\':config[\'hard_shutdown_seconds\'],\'server_ceiling_seconds\':43200}\n        write(working/\'session-summary.json\',report)\n        if live is not None:\n            try:report[\'final_live_uploaded\']=live.final_sync(\'normal-exit\',timeout=max(.1,min(1200.,watchdog.hard_deadline-time.monotonic()-120.)))\n            except Exception as error:\n                report.update(final_live_uploaded=False,final_live_error_type=type(error).__name__)\n        write(working/\'session-summary.json\',report)\n        files=[p for p in working.rglob(\'*\') if p.is_file() and p.relative_to(working).parts[0]!=\'src\'\n               and p.name!=\'l2-output-manifest.json\' and \'.tmp\' not in p.name]\n        write(working/\'l2-output-manifest.json\',{\'files\':[{\'file\':p.relative_to(working).as_posix(),\n              \'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in files]})\n        if not survivors:watchdog.finish()\n    return report\n', 'l3_resume.py': '"""Pinned L2 continuation: keep both identities, weighted order, cursor and full history."""\nimport gzip, json, shutil\nfrom pathlib import Path\nimport torch\nimport l1_runtime as L, l2_runtime as R\nfrom kaggle_io import sha, safe_name, write, resolve_dataset\n\ndef resolve_cache_mount(input_root, source, kind):\n    """Foreign sources use owner-qualified mounts; own datasets may still be flat."""\n    root=Path(input_root);owner,slug=source.split(\'/\')[:2]\n    for candidate in (root/kind/owner/slug,root/slug):\n        if candidate.is_dir():return candidate\n    return resolve_dataset(root,owner+\'/\'+slug)\n\ndef live_bytes(source, entry):\n    """Accept original gzip or Kaggle\'s expanded mount, verifying the producer\'s bytes."""\n    path=Path(source)/safe_name(entry[\'file\'])\n    if path.exists():\n        raw=path.read_bytes()\n        assert len(raw)==entry[\'bytes\'] and sha(path)==entry[\'sha256\'], \'Live entry mismatch\'\n        return gzip.decompress(raw) if entry[\'gzip\'] else raw\n    assert entry[\'gzip\'] and path.suffix==\'.gz\', \'Missing live entry\'\n    raw=path.with_suffix(\'\').read_bytes()\n    packed=gzip.compress(raw,compresslevel=1,mtime=0)\n    import hashlib\n    assert len(packed)==entry[\'bytes\'] and hashlib.sha256(packed).hexdigest()==entry[\'sha256\']\n    return raw\n\ndef mounted_expected(source, name, pin):\n    source=Path(source);manifest=json.loads((source/\'recover-manifest.json\').read_text())\n    assert manifest[\'format\']==\'nlp-hw02-l2-live-v1\'\n    latest=next(r for r in manifest[\'files\'] if r[\'original_path\']==f\'model-{name}/latest.json\')\n    receipt=json.loads(live_bytes(source,latest));original=receipt[\'file\']\n    blob=next(r for r in manifest[\'files\'] if r[\'original_path\']==f\'model-{name}/{original}\')\n    assert blob[\'file\']==pin[\'file\'] and blob[\'sha256\']==receipt[\'sha256\']==pin[\'sha256\']\n    assert blob[\'bytes\']==receipt[\'bytes\']==pin[\'bytes\']\n    assert dict(receipt,file=blob[\'file\'],original_file=original)==pin, \'Pinned L2 latest changed\'\n    return pin\n\ndef verify_anchor(source, expected, check_environment=True, expected_points=3):\n    path=Path(source)/safe_name(expected[\'file\'])\n    assert path.stat().st_size==expected[\'bytes\'] and sha(path)==expected[\'sha256\'], \'L2 anchor SHA/bytes mismatch\'\n    p=R.read_full(path,expected[\'identity\'],check_environment=check_environment)\n    for key in (\'updates\',\'cursor\',\'main_seconds\',\'l2_main_seconds\'):\n        assert p[\'state\'][key]==expected[key], \'L2 anchor \'+key+\' mismatch\'\n    assert p[\'model_spec\']==expected[\'model_spec\']==L.SPECS[\'C\']\n    assert p[\'state\'][\'last_lr\']==.0015 and p[\'scaler_enabled\'] and p[\'rng\'][\'torch_cuda\']\n    assert p[\'state\'][\'stability\'][\'escalations\']==0, \'Unexpected recovered L2 recipe\'\n    history=p[\'state\'][\'l2_points\']\n    assert len(history)==expected_points and all(r[\'complete\'] for r in history)\n    if expected_points==5:\n        assert [r[\'tag\'] for r in history]==[\'l2-10800\',\'l2-21600\',\'l2-28800\',\'l2-45540\',\'l2-62280\']\n        assert history[-1].get(\'metrics\',{}).get(\'K8_tune2048\',{}).get(\'complete\') is True, \'Incomplete L3 final K8\'\n    return path,p\n\ndef restore_order(source, destination, expected):\n    source,destination=Path(source),Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    manifest=json.loads((source/\'recover-manifest.json\').read_text())\n    for filename in (\'order.npy\',\'order.json\',\'split.json\'):\n        entry=next(r for r in manifest[\'files\'] if r[\'original_path\']==\'weighted-order/\'+filename)\n        target=destination/filename\n        if filename==\'order.npy\':\n            path=source/safe_name(entry[\'file\'])\n            assert path.stat().st_size==entry[\'bytes\'] and sha(path)==entry[\'sha256\']==expected[\'file\'][\'sha256\']\n            shutil.copyfile(path,target)\n        else:target.write_bytes(live_bytes(source,entry))\n    assert json.loads((destination/\'order.json\').read_text())==expected, \'Weighted sampler metadata changed\'\n    return destination\n\ndef phase_recipe(config, anchor):\n    # R.run uses cumulative L2 main time. Offset each model from its exact saved value.\n    origin=anchor[\'state\'][\'l2_main_seconds\'];recipe=config[\'phase_recipe\']\n    return dict(recipe,main_targets=[origin+seconds for seconds in recipe[\'main_targets\']])\n\ndef completion_status(result, targets, final_k8_rows=None):\n    """The inherited runtime preserves state on eval failure, but can still say complete."""\n    if result[\'status\'] != \'complete\':\n        return result\n    points={p[\'tag\']:p for p in result[\'state\'][\'l2_points\']}\n    for target in targets:\n        point=points.get(f\'l2-{int(target):05d}\')\n        metrics=point.get(\'metrics\') if point else None\n        if not point or not point[\'complete\'] or metrics is None or not metrics.get(\'K8_tune2048\',{\'complete\':True}).get(\'complete\',False):\n            result[\'status\']=\'partial; L3 cooldown/evaluation incomplete\'\n            break\n    if result[\'status\']==\'complete\' and final_k8_rows is not None:\n        final=points[f\'l2-{int(targets[-1]):05d}\'][\'metrics\']\n        vote=final.get(\'K8_tune2048\',{})\n        if (vote.get(\'complete\') is not True or vote.get(\'rows\')!=final_k8_rows\n                or vote.get(\'target_rows\')!=final_k8_rows):\n            result[\'status\']=\'partial; L4 final K8 missing or incomplete\'\n    return result\n\ndef run_continuation(name, anchor, identity, cache, order, root, device, deadline, evaluate, config):\n    recipe=phase_recipe(config,anchor)\n    result=R.run(name,anchor,identity,cache,order,root,device,deadline,evaluate,**recipe)\n    completion_status(result,recipe[\'main_targets\'],\n                      final_k8_rows=2048 if config.get(\'session_kind\')==\'l4\' and not config.get(\'gpu_smoke\') else None)\n    write(Path(root)/\'summary.json\',result)\n    return result\n\ndef exercise_smoke(gpu,name,anchor,identity,cache,order,inputs,sets,refs,device,config):\n    """L2\'s existing CUDA checks, rebased without modifying its frozen runtime modules."""\n    import copy\n    from unittest.mock import patch\n    import l2_gpu_smoke as S\n    original_run=R.run;origin=anchor[\'state\'][\'l2_main_seconds\'];history=copy.deepcopy(anchor[\'state\'][\'l2_points\'])\n    def continued_state(payload, unused_name):\n        # The L2 smoke\'s serialization probe must save the actual inherited full state.\n        return copy.deepcopy(payload[\'state\'])\n    def run(*args,**kwargs):\n        primary=len(kwargs[\'main_targets\'])==2\n        kwargs[\'main_targets\']=[origin+t for t in kwargs[\'main_targets\']]\n        result=original_run(*args,**kwargs)\n        assert result[\'state\'][\'l2_points\'][:len(history)]==history\n        # Its legacy assertions count only the two newly evaluated smoke marks.\n        if primary:\n            result=copy.deepcopy(result)\n            result[\'state\'][\'l2_points\']=result[\'state\'][\'l2_points\'][len(history):]\n        return result\n    # Legacy smoke uses GPU1 cursor0. Feed an equivalent rotated view for its finite-math\n    # comparison only; production run/reload always receives the original order/cursor.\n    old_take=L.take\n    def take(candidate,cursor,count):\n        return old_take(order,anchor[\'state\'][\'cursor\'],count) if gpu==1 and cursor==0 else old_take(candidate,cursor,count)\n    # R.run uses actual nonzero cursors; the patched take only changes the legacy probe.\n    with patch.object(S,\'initial_state\',continued_state),patch.object(R,\'run\',run),patch.object(L,\'take\',take):\n        S.exercise(gpu,name,anchor,identity,cache,order,inputs,sets,refs,device,config)\n', 's2_live_sync.py': '"""Best-effort private Dataset snapshots. Supervisor stays Torch-free and bounded."""\nimport argparse,gzip,json,os,shutil,signal,subprocess,sys,threading,time\nfrom pathlib import Path\nfrom session_watchdog import atomic_json\n\nSECRET_NAMES=(\'KAGGLE_USERNAME\',\'KAGGLE_KEY\')\nSTAGE_LIMIT=450_000_000  # FP16 weights + compressed journals; reserve another copy for CLI ZIP.\n\ndef load_secrets():\n    # Values live only in process memory/environment. Never return, log or serialize them.\n    try:\n        from kaggle_secrets import UserSecretsClient\n        client=UserSecretsClient()\n        for name in SECRET_NAMES:os.environ[name]=client.get_secret(name)\n        return all(os.environ.get(name) for name in SECRET_NAMES)\n    except Exception:\n        print(\'LIVE: Kaggle Secrets unavailable; uploads disabled\',flush=True)\n        return False\n\ndef credential_patterns(ref):\n    values=[os.environ.get(name,\'\').encode() for name in SECRET_NAMES]\n    # The fixed public owner in Dataset IDs is not authentication data. All other\n    # username values and every key byte are forbidden (including in binary files).\n    owner=ref.split(\'/\')[0].encode()\n    return [v for i,v in enumerate(values) if v and not (i==0 and v==owner)]\n\ndef contains_secret(path,patterns):\n    overlap=max([len(v) for v in patterns]+[1])-1;previous=b\'\'\n    with Path(path).open(\'rb\') as handle:\n        while True:\n            part=handle.read(1024*1024)\n            if not part:return False\n            block=previous+part\n            if any(value in block for value in patterns):return True\n            previous=block[-overlap:] if overlap else b\'\'\n\ndef audit_tree(root,ref,cache=None):\n    patterns=credential_patterns(ref);cache={} if cache is None else cache\n    for path in Path(root).rglob(\'*\'):\n        if not path.is_file():continue\n        if path.is_symlink():raise RuntimeError(\'Symlink in snapshot source\')\n        # Active atomic temporaries are never uploaded; their published versions are audited.\n        if \'.tmp\' in path.name:continue\n        stat=path.stat();stamp=[stat.st_size,stat.st_mtime_ns]\n        key=str(path.relative_to(root))\n        if cache.get(key)!=stamp:\n            if contains_secret(path,patterns):raise RuntimeError(\'Credential bytes detected; upload refused\')\n            cache[key]=stamp\n    return cache\n\ndef stage_snapshot(working,stage,ref,reason):\n    """Runs in a killable CPU child; no Torch or heavy IO on watchdog thread."""\n    working,stage=Path(working),Path(stage);stage.mkdir(parents=True,exist_ok=True)\n    audit_path=stage.parent/\'live-audit-cache.json\'\n    cache=json.loads(audit_path.read_text()) if audit_path.exists() else {}\n    cache=audit_tree(working,ref,cache);atomic_json(audit_path,cache)\n    payload=stage/\'snapshot\';payload.mkdir(exist_ok=True)\n    old_path=stage.parent/\'live-weight-cache.json\'\n    weights=json.loads(old_path.read_text()) if old_path.exists() else {}\n    keep=set();manifest=[];used=0\n    patterns=credential_patterns(ref)\n    def reserve(size):\n        nonlocal used\n        if used+size>STAGE_LIMIT:raise RuntimeError(\'Live staging size limit\')\n        used+=size\n    def copy_file(path):\n        relative=path.relative_to(working);target=payload/relative\n        if path.is_symlink():raise RuntimeError(\'Symlink in snapshot source\')\n        data=path.read_bytes()\n        if any(v in data for v in patterns):raise RuntimeError(\'Credential bytes detected; upload refused\')\n        if path.suffix in (\'.json\',\'.jsonl\') and len(data)>256*1024:\n            data=gzip.compress(data,compresslevel=1,mtime=0);target=target.with_name(target.name+\'.gz\')\n        reserve(len(data));target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)\n        keep.add(target);manifest.append({\'file\':target.relative_to(stage).as_posix(),\'bytes\':len(data)})\n    selected=set()\n    for path in working.glob(\'*.json\'):\n        if path.name!=\'session-config.json\':selected.add(path)\n    for root in working.glob(\'gpu*-exp*\'):\n        selected.update(root.glob(\'*summary.json\'));selected.update(root.glob(\'*gate.json\'))\n        selected.update(root.glob(\'progress.json\'));selected.update(root.glob(\'training-finished.json\'))\n        for point in (root/\'points\').glob(\'*\'):\n            metric=point/\'evaluation/metrics.json\'\n            if not metric.exists():continue\n            selected.update(point.glob(\'training.json\'))\n            selected.update((point/\'evaluation\').rglob(\'*.json\'))\n            receipt_path=point/\'checkpoint.json\'\n            if not receipt_path.exists():continue\n            receipt=json.loads(receipt_path.read_text());source=root/receipt[\'file\']\n            if not source.resolve().is_relative_to(root.resolve()) or source.is_symlink():raise RuntimeError(\'Invalid point weight path\')\n            destination=payload/root.name/\'weights\'/f\'{point.name}-fp16.pt\'\n            stat=source.stat();stamp=[stat.st_size,stat.st_mtime_ns];key=root.name+\'/\'+point.name\n            if weights.get(key)!=stamp or not destination.exists():\n                import torch\n                torch.set_num_threads(1)\n                original=torch.load(source,map_location=\'cpu\',weights_only=True)\n                model={k:(v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()) for k,v in original[\'model\'].items()}\n                estimate=sum(v.numel()*v.element_size() for v in model.values())+1024*1024\n                if used+estimate>STAGE_LIMIT:raise RuntimeError(\'Live staging weight size limit\')\n                destination.parent.mkdir(parents=True,exist_ok=True)\n                torch.save({\'format\':\'nlp-hw02-model-only-v1\',\'model\':model,\'model_spec\':original[\'model_spec\'],\n                    \'experiment\':original[\'experiment\'],\'point\':point.name,\'dtype\':\'FP16\',\'optimizer_included\':False,\n                    \'source_fp32_sha256\':receipt[\'sha256\'],\'purpose\':\'live recovery; quantized evaluated weights\'},destination)\n                weights[key]=stamp\n                del original,model\n            reserve(destination.stat().st_size);keep.add(destination)\n            manifest.append({\'file\':destination.relative_to(stage).as_posix(),\'bytes\':destination.stat().st_size,\'dtype\':\'FP16\',\'optimizer_included\':False})\n            target=payload/receipt_path.relative_to(working);target.parent.mkdir(parents=True,exist_ok=True)\n            import hashlib\n            digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n            atomic_json(target,{\'file\':destination.relative_to(payload/root.name).as_posix(),\'sha256\':digest,\n                \'bytes\':destination.stat().st_size,\'dtype\':\'FP16\',\'optimizer_included\':False,\'source_fp32\':receipt})\n            reserve(target.stat().st_size);keep.add(target)\n        # Heldout 27b candidate journals and complete test generations/answers.\n        for folder in (\'heldout4k\',\'test\'):\n            for path in (root/folder).rglob(\'*\'):\n                if path.is_file() and path.suffix in (\'.json\',\'.jsonl\'):selected.add(path)\n    for path in (working/\'a-test\').rglob(\'*\'):\n        if path.is_file() and path.suffix in (\'.json\',\'.jsonl\'):selected.add(path)\n    for path in sorted(selected):copy_file(path)\n    for path in working.glob(\'gpu*-process.log\'):\n        with path.open(\'rb\') as handle:\n            handle.seek(0,2);handle.seek(max(0,handle.tell()-32*1024));data=handle.read()\n        for value in patterns:data=data.replace(value,b\'[REDACTED]\')\n        target=payload/path.name;target.write_bytes(data);reserve(len(data));keep.add(target)\n    for path in payload.rglob(\'*\'):\n        if path.is_file() and path not in keep:path.unlink()\n    atomic_json(stage/\'dataset-metadata.json\',{\'id\':ref,\'title\':\'NLP HW02 S2 live\',\'licenses\':[{\'name\':\'other\'}],\n        \'description\':\'Private incremental S2 recovery: evaluated FP16 model weights, metrics and generation journals.\'})\n    atomic_json(payload/\'live-manifest.json\',{\'reason\':reason,\'unix_time\':time.time(),\'files\':manifest,\n        \'payload_bytes\':used,\'staging_limit_bytes\':STAGE_LIMIT,\'credentials_in_files\':False,\n        \'weights\':\'FP16 model only; optimizer excluded; FP32 evaluation remains in notebook output\'})\n    atomic_json(old_path,weights)\n    audit_tree(stage,ref)\n    assert sum(p.stat().st_size for p in stage.rglob(\'*\') if p.is_file())<STAGE_LIMIT+64*1024\n\ndef privacy_probe(ref,path):\n    from kaggle.api.kaggle_api_extended import KaggleApi\n    api=KaggleApi();api.authenticate();page=1\n    while True:\n        rows=api.dataset_list(mine=True,page=page) or []\n        found=next((d for d in rows if d.ref==ref),None)\n        if found:\n            atomic_json(path,{\'exists\':True,\'private\':bool(found.is_private)});return\n        if not rows:atomic_json(path,{\'exists\':False});return\n        page+=1\n\nclass LiveSync:\n    def __init__(self,working,scratch,ref,interval=900.,poll_seconds=5.,upload_timeout=540.):\n        self.working=Path(working);self.scratch=Path(scratch)/\'live-sync\';self.scratch.mkdir(parents=True,exist_ok=True)\n        self.stage=self.scratch/\'stage\';self.ref=ref;self.interval=interval;self.poll_seconds=poll_seconds\n        self.upload_timeout=min(upload_timeout,540.);self.lock=threading.Lock();self.active_lock=threading.Lock()\n        self.active=None;self.stop_event=threading.Event();self.thread=None;self.last_attempt=0.;self.last_signature=None\n        self.env=dict(os.environ,CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\')\n        # No credential file is needed or written by CLI; only inherited env auth.\n        config_dir=self.scratch/\'empty-config\';config_dir.mkdir(exist_ok=True)\n        cli_tmp=self.scratch/\'cli-temp\';cli_tmp.mkdir(exist_ok=True)\n        self.env.update(KAGGLE_CONFIG_DIR=str(config_dir),TMPDIR=str(cli_tmp))\n        self.env.pop(\'KAGGLE_API_TOKEN\',None)\n\n    def log(self,event,**fields):\n        # No subprocess output, exception text, argv, URLs or env in this log.\n        record={\'unix_time\':time.time(),\'event\':event,**fields}\n        with (self.working/\'s2-live-sync.log\').open(\'a\') as handle:handle.write(json.dumps(record)+\'\\n\')\n        atomic_json(self.working/\'s2-live-sync-status.json\',record)\n\n    def kill_active(self):\n        with self.active_lock:process=self.active\n        if process is not None:\n            try:os.killpg(process.pid,signal.SIGKILL)\n            except OSError:pass\n\n    def bounded_call(self,args,timeout,final=False):\n        until=time.monotonic()+min(timeout,600.)\n        process=subprocess.Popen(args,env=self.env,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,start_new_session=True)\n        with self.active_lock:self.active=process\n        try:\n            while process.poll() is None:\n                if time.monotonic()>=until or (self.stop_event.is_set() and not final):\n                    try:os.killpg(process.pid,signal.SIGKILL)\n                    except OSError:pass\n                    # No communicate(), and no implicit unbounded wait in subprocess.run().\n                    try:process.wait(timeout=.1)\n                    except subprocess.TimeoutExpired:pass\n                    raise subprocess.TimeoutExpired(\'live child\',timeout)\n                time.sleep(min(.05,max(.001,until-time.monotonic())))\n            if process.returncode:raise subprocess.CalledProcessError(process.returncode,\'live child\')\n        finally:\n            with self.active_lock:\n                if self.active is process:self.active=None\n\n    def signature(self):\n        paths=list(self.working.glob(\'gpu*-exp*/points/*/evaluation/metrics.json\'))\n        paths+=list(self.working.glob(\'gpu*-exp*/test/**/timing.json\'))\n        paths+=list(self.working.glob(\'gpu*-exp*/*summary.json\'))\n        return sorted((str(p.relative_to(self.working)),p.stat().st_size,p.stat().st_mtime_ns) for p in paths)\n\n    def sync(self,reason,timeout=590.,final=False):\n        until=time.monotonic()+min(timeout,600.);helper=str(Path(__file__).resolve())\n        def call(args,cap):\n            remaining=until-time.monotonic()\n            if remaining<=0:raise subprocess.TimeoutExpired(\'live snapshot\',timeout)\n            return self.bounded_call(args,min(cap,remaining),final=final)\n        if not self.lock.acquire(timeout=min(1.,timeout)):return False\n        try:\n            # Remove abandoned CLI archives after a timed-out upload, before another copy.\n            for path in (self.scratch/\'cli-temp\').iterdir():\n                if path.is_dir():shutil.rmtree(path)\n                else:path.unlink()\n            call([sys.executable,helper,\'stage\',\'--working\',str(self.working),\'--stage\',str(self.stage),\'--ref\',self.ref,\'--reason\',reason],120.)\n            privacy=self.scratch/\'privacy.json\';privacy.unlink(missing_ok=True)\n            call([sys.executable,helper,\'privacy\',\'--ref\',self.ref,\'--receipt\',str(privacy)],30.)\n            info=json.loads(privacy.read_text())\n            if info.get(\'exists\') and not info.get(\'private\'):raise RuntimeError(\'Public target refused\')\n            command=[\'kaggle\',\'datasets\',\'version\' if info[\'exists\'] else \'create\',\'-p\',str(self.stage),\'--dir-mode\',\'zip\',\'--keep-tabular\',\'--quiet\']\n            if info[\'exists\']:command+=[\'-m\',\'S2 incremental recovery \'+reason]\n            call(command,self.upload_timeout)\n            self.log(\'uploaded\',operation=\'version\' if info[\'exists\'] else \'create\',reason=reason,\n                staging_bytes=sum(p.stat().st_size for p in self.stage.rglob(\'*\') if p.is_file()))\n            return True\n        except Exception as error:\n            self.log(\'sync_error\',error_type=type(error).__name__,returncode=getattr(error,\'returncode\',None),reason=reason)\n            return False\n        finally:self.lock.release()\n\n    def run(self):\n        while not self.stop_event.is_set():\n            try:\n                signature=self.signature();now=time.monotonic()\n                if signature!=self.last_signature or now-self.last_attempt>=self.interval:\n                    before=signature;self.last_attempt=now\n                    if self.sync(\'new-point\' if signature else \'heartbeat\'):\n                        # Changes published during upload trigger the next snapshot too.\n                        self.last_signature=before\n                    else:self.last_signature=before  # Failed upload retries at heartbeat/new metric.\n            except Exception as error:self.log(\'thread_error\',error_type=type(error).__name__)\n            self.stop_event.wait(self.poll_seconds)\n\n    def start(self):\n        self.thread=threading.Thread(target=self.run,name=\'s2-live-sync\',daemon=True);self.thread.start();return self\n\n    def final_sync(self,reason=\'final\',timeout=30.):\n        self.stop_event.set();self.kill_active();until=time.monotonic()+timeout\n        if self.thread is not None:self.thread.join(timeout=min(.5,timeout))\n        # IO/upload on a daemon; even a wedged filesystem or child cannot hold watchdog exit.\n        result=[]\n        thread=threading.Thread(target=lambda:result.append(self.sync(reason,max(.1,until-time.monotonic()-.2),final=True)),daemon=True)\n        thread.start();thread.join(timeout=max(0.,until-time.monotonic()))\n        self.kill_active()\n        return bool(result and result[0])\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'mode\',choices=[\'stage\',\'privacy\'])\n    for name in (\'working\',\'stage\',\'ref\',\'receipt\',\'reason\'):parser.add_argument(\'--\'+name)\n    args=parser.parse_args()\n    try:\n        if args.mode==\'stage\':stage_snapshot(args.working,args.stage,args.ref,args.reason)\n        else:privacy_probe(args.ref,args.receipt)\n    except Exception:sys.exit(1)  # No credential-bearing exception text/traceback to disk.\n', 's2_report.py': '"""A/M/B common-time comparisons and the pre-registered S2 final decision."""\nfrom pathlib import Path\nimport exp26_eval as E\nfrom exp26_common import read_json,write_json\n\ndef decision(common,timings,gates_passed=True):\n    if not gates_passed:return {\'status\':\'pending;27b gates not passed\'}\n    if not common:return {\'status\':\'pending;no aligned A/M/B common point\'}\n    point=max(common,key=lambda p:p[\'target_seconds\']);tag=point[\'tag\']\n    rank=sorted(point[\'models\'],key=lambda m:point[\'models\'][m][\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\'],reverse=True)\n    acc=lambda name:point[\'models\'][name][\'splits\'][\'heldout4k\'][\'exec\'][\'accuracy\']\n    delta=100*(acc(rank[0])-acc(rank[1]));result={\'point\':tag,\'A_point\':point[\'A_point\'],\'leader\':rank[0],\'leader_gap_pp\':delta}\n    if delta>=2.:\n        candidates=[rank[0]];rule=\'leader gap>=2pp and fulltest<=120s\'\n    else:candidates=[name for name in rank if 100*(acc(rank[0])-acc(name))<2.];rule=\'gap<2pp: minimum full inference time among near leaders\'\n    missing=[name for name in candidates if name not in timings.get(tag,{})]\n    if missing:return dict(result,status=\'pending;missing corresponding checkpoint test timings\',missing_models=missing)\n    eligible=[name for name in candidates if timings[tag][name][\'full_pipeline_seconds\']<=120.]\n    if not eligible:return dict(result,status=\'pending;no eligible candidate under120s\',rule=rule)\n    chosen=min(eligible,key=lambda name:timings[tag][name][\'full_pipeline_seconds\'])\n    return dict(result,status=\'measured\',final_model=chosen,rule=rule,full_test_seconds=timings[tag][chosen][\'full_pipeline_seconds\'])\n\ndef collect(working):\n    working=Path(working);a_path=working/\'a-baselines.json\';a=read_json(a_path) if a_path.exists() else {}\n    reports={name:read_json(working/folder/file) if (working/folder/file).exists() else {}\n        for name,folder,file in [(\'M\',\'gpu0-exp29\',\'29-summary.json\'),(\'B\',\'gpu1-exp28\',\'28-summary.json\')]}\n    for name,folder in [(\'M\',\'gpu0-exp29\'),(\'B\',\'gpu1-exp28\')]:\n        # A kill between atomic metrics and the experiment summary still leaves a useful point.\n        for path in (working/folder/\'points\').glob(\'*/training.json\'):\n            metrics=path.parent/\'evaluation/metrics.json\'\n            if metrics.exists():\n                point=read_json(path);point[\'metrics\']=read_json(metrics)\n                if all(s in point[\'metrics\'].get(\'splits\',{}) for s in (\'heldout4k\',\'old1024\')):\n                    point[\'evaluation_status\']=\'complete\'\n                    reports[name].setdefault(\'points\',{})[path.parent.name]=point\n    common=[];comparisons=[];timings={}\n    for tag in (\'t01\',\'t02\',\'t03\',\'t04\'):\n        models={};target=None;ap=None\n        for name,report in reports.items():\n            p=report.get(\'points\',{}).get(tag)\n            if p and p.get(\'time_aligned\'):\n                target=p[\'target_train_seconds\'];ap=p[\'A_point\'];models[name]=p[\'metrics\']\n        if ap in a:models[\'A\']=a[ap]\n        if len(models)<2:continue\n        pairs={}\n        names=sorted(models)\n        for i,name in enumerate(names):\n            for other in names[:i]:pairs[f\'{name}_vs_{other}\']={s:E.paired_mcnemar(models[name][\'splits\'][s][\'exec\'],models[other][\'splits\'][s][\'exec\']) for s in (\'heldout4k\',\'old1024\')}\n        row={\'tag\':tag,\'A_point\':ap,\'target_seconds\':target,\'models\':models,\'McNemar\':pairs};comparisons.append(row)\n        if len(models)==3:common.append(row)\n    for name,report in reports.items():\n        for tag,tests in report.get(\'test_by_point\',{}).items():\n            if tests.get(\'selected\'):timings.setdefault(tag,{})[name]=tests[\'selected\']\n    for p in (working/\'a-test\').glob(\'*/selected/timing.json\'):\n        timings.setdefault(p.parent.parent.name,{})[\'A\']=read_json(p)\n    fast_path=working/\'gpu0-exp27b/27b-summary.json\';fast=read_json(fast_path) if fast_path.exists() else {}\n    passed=fast.get(\'status\')==\'complete\' and fast.get(\'greedy_gate\',{}).get(\'token_identical\',False) and fast.get(\'heldout4k\',{}).get(\'quality_gate\',False)\n    result={\'comparisons\':comparisons,\'common_points\':[r[\'tag\'] for r in common],\'test_timings\':timings,\n        \'decision\':decision(common,timings,passed),\'27b_status\':fast.get(\'status\',\'missing\'),\n        \'models\':{name:{\'status\':r.get(\'status\',\'missing\'),\'missing_points\':r.get(\'missing_points\',[])} for name,r in reports.items()}}\n    write_json(working/\'s2-comparison.json\',result);return result\n', 's3_representation.py': '"""Reversible representations; no new semantic targets or changed row selection."""\nimport ast, io, math, re, tokenize\nfrom fractions import Fraction\nfrom tokenizers import Tokenizer, AddedToken\n\nMODES = {\'header_tail\': \'<|header_tail|>\', \'header\': \'<|header|>\', \'raw\': \'<|raw|>\'}\nNUMBERS = [f\'<|N{i}|>\' for i in range(1, 21)]\nNUMBER = re.compile(r\'(?<![A-Za-z_])[-+]?(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?(?:[eE][-+]?\\d+)?(?:/\\d+)?\')\n\ndef no_comments(code):\n    lines = code.splitlines(keepends=True)\n    try:\n        spans = [t for t in tokenize.generate_tokens(io.StringIO(code).readline) if t.type == tokenize.COMMENT]\n    except (tokenize.TokenError, IndentationError, SyntaxError):\n        return code\n    for t in reversed(spans):\n        i = t.start[0] - 1\n        lines[i] = lines[i][:t.start[1]].rstrip(\' \\t\') + lines[i][t.end[1]:]\n    return \'\'.join(lines)\n\ndef no_blank(code):\n    protected = set()\n    try:\n        for t in tokenize.generate_tokens(io.StringIO(code).readline):\n            if t.type == tokenize.STRING:\n                protected.update(range(t.start[0], t.end[0] + 1))\n        candidate = \'\'.join(line for i, line in enumerate(code.splitlines(keepends=True), 1)\n                            if line.strip() or i in protected)\n        if ast.dump(ast.parse(code)) == ast.dump(ast.parse(candidate)):\n            return candidate\n    except (tokenize.TokenError, SyntaxError, ValueError, RecursionError):\n        pass\n    return code\n\ndef restore_compact(text):\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError(\'Missing representation mode\')\n    fragment = text[len(MODES[mode]):]\n    if mode == \'raw\':\n        return fragment\n    if mode == \'header\':\n        return \'def solve() -> int:\\n\' + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith(\'    \'):\n        raise ValueError(\'Missing final result expression\')\n    lines[-1] = \'    result = \' + lines[-1][4:]\n    body = \'\'.join(lines)\n    return \'def solve() -> int:\\n\' + body + (\'\' if body.endswith(\'\\n\') else \'\\n\') + \'    return result\\n\'\n\ndef compact(code):\n    normalized = no_blank(no_comments(code))\n    raw = MODES[\'raw\'] + normalized\n    lines = normalized.splitlines(keepends=True)\n    if not lines or lines[0] != \'def solve() -> int:\\n\':\n        return raw\n    try:\n        tree = ast.parse(normalized)\n        f = tree.body[0]\n        if len(tree.body) != 1 or not isinstance(f, ast.FunctionDef):\n            return raw\n        tail = (len(f.body) >= 2 and isinstance(f.body[-1], ast.Return)\n                and isinstance(f.body[-1].value, ast.Name) and f.body[-1].value.id == \'result\'\n                and isinstance(f.body[-2], ast.Assign) and len(f.body[-2].targets) == 1\n                and isinstance(f.body[-2].targets[0], ast.Name) and f.body[-2].targets[0].id == \'result\'\n                and lines[-1].strip() == \'return result\' and lines[-2].startswith(\'    result = \'))\n        if tail:\n            lines[-2] = \'    \' + lines[-2][len(\'    result = \'):]\n            text = MODES[\'header_tail\'] + \'\'.join(lines[1:-1])\n        else:\n            text = MODES[\'header\'] + \'\'.join(lines[1:])\n        # Protect multiline assignments/tails and any unusual source layout.\n        return text if ast.dump(ast.parse(restore_compact(text))) == ast.dump(tree) else raw\n    except (SyntaxError, ValueError, IndexError, RecursionError):\n        return raw\n\ndef question_mapping(question):\n    mapping, occurrences = {}, []\n    for match in NUMBER.finditer(question):\n        try:\n            text = match.group().replace(\',\', \'\')\n            value = Fraction(text) if \'/\' not in text else Fraction(text.split(\'/\')[0]) / Fraction(text.split(\'/\')[1])\n            if not math.isfinite(float(value)):\n                continue\n        except (ValueError, ZeroDivisionError, OverflowError):\n            continue\n        if value not in mapping and len(mapping) < 20:\n            # Integers stay integers (range/index/slices); decimal/fraction values stay floats.\n            number = int(value) if value.denominator == 1 else float(value)\n            mapping[value] = (NUMBERS[len(mapping)], number)\n        if value in mapping:\n            occurrences.append((match.start(), match.end(), mapping[value][0]))\n    marked = question\n    for start, end, token in reversed(occurrences):\n        marked = marked[:start] + token + \' \' + marked[start:]\n    return marked, mapping\n\ndef placeholder_code(code, mapping):\n    try:\n        tree = ast.parse(code)\n    except (SyntaxError, ValueError, RecursionError):\n        return code\n    lines = code.encode(\'utf-8\').splitlines(keepends=True)\n    offsets = [0]\n    for line in lines:\n        offsets.append(offsets[-1] + len(line))\n    changes = []\n    def literal(node):\n        if isinstance(node, ast.Constant) and type(node.value) in (int, float):\n            return node.value\n        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):\n            v = literal(node.operand)\n            if v is not None:\n                return -v if isinstance(node.op, ast.USub) else v\n        return None\n    def visit(node):\n        v = literal(node)\n        if v is not None and (type(v) is int or math.isfinite(v)):\n            key = Fraction(str(v))\n            if key not in mapping and type(v) is float:\n                # Fraction question values such as2/3 map to the same Python float literal.\n                equivalent = [q for q,(token,number) in mapping.items() if float(number) == v]\n                if equivalent:key = equivalent[0]\n            if key in mapping:\n                token, number = mapping[key]\n                # Preserve literal type; merely replacing 5.0 by integer 5 can change execution.\n                replacement = token if type(v) is type(number) else (\'float(\' + token + \')\' if type(v) is float else \'int(\' + token + \')\')\n                changes.append((offsets[node.lineno - 1] + node.col_offset,\n                                offsets[node.end_lineno - 1] + node.end_col_offset, replacement.encode()))\n                return\n        for child in ast.iter_child_nodes(node):\n            visit(child)\n    visit(tree)\n    raw = code.encode(\'utf-8\')\n    for start, end, replacement in sorted(changes, reverse=True):\n        raw = raw[:start] + replacement + raw[end:]\n    return raw.decode(\'utf-8\')\n\ndef restore_numbers(text, question):\n    _, mapping = question_mapping(question)\n    reverse = {token: \'(\' + repr(number) + \')\' for token, number in mapping.values()}\n    def replace(match):\n        if match.group() not in reverse:\n            raise ValueError(\'Unknown generated number placeholder\')\n        return reverse[match.group()]\n    return re.sub(r\'<\\|N\\d+\\|>\', replace, text)\n\ndef transform(question, code, variant):\n    if variant == 30:\n        return question, compact(code)\n    marked, mapping = question_mapping(question)\n    return marked, placeholder_code(code, mapping)\n\ndef restore(text, question, variant):\n    return restore_compact(text) if variant == 30 else restore_numbers(text, question)\n\ndef tokenizer(path, variant):\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else NUMBERS\n    assert tok.add_special_tokens([AddedToken(t, normalized=False, special=True) for t in added]) == len(added)\n    # Original byte alphabet is unchanged; added specials decode literally, never get skipped.\n    visible = list(range(33,127)) + list(range(161,173)) + list(range(174,256))\n    bchars = {b:chr(b) for b in visible}\n    for i,b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256+i)\n    charbytes = {c:b for b,c in bchars.items()}\n    tokenbytes = [b\'\',b\'\',b\'\'] + [bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)]\n    tokenbytes += [t.encode() for t in added]\n    assert all(tok.encode(t, add_special_tokens=False).ids == [8189+i] for i,t in enumerate(added))\n    return tok, tokenbytes\n\ndef init_encoder(path, variant):\n    global _TOKENIZER, _VARIANT\n    _TOKENIZER, _ = tokenizer(path, variant)\n    _VARIANT = variant\n\ndef encode_block(rows):\n    import numpy as np\n    from array import array\n    texts = [transform(q,c,_VARIANT) for q,c in rows]\n    sources = _TOKENIZER.encode_batch([q for q,c in texts],add_special_tokens=False)\n    targets = _TOKENIZER.encode_batch([c for q,c in texts],add_special_tokens=False)\n    s,t,so,to = array(\'h\'),array(\'h\'),array(\'q\',[0]),array(\'q\',[0])\n    for source,target in zip(sources,targets):\n        s.extend([1]+[i+3 for i in source.ids]+[2]);so.append(len(s))\n        t.extend(i+3 for i in target.ids);to.append(len(t))\n    return [np.frombuffer(a,dtype=d) for a,d in zip((s,t,so,to),(\'int16\',\'int16\',\'int64\',\'int64\'))]\n', 'session_runner.py': '"""Two independent processes with one visible GPU each; wall deadline starts at first cell."""\nimport json,os,shutil,signal,subprocess,sys,time,zipfile\nfrom session_watchdog import Watchdog,atomic_json,clean_checkpoints\nfrom pathlib import Path\nfrom kaggle_io import write,sha,disk_guard\n\ndef archive(working,kind=\'s1\'):\n    working=Path(working);target=working/f\'{kind}-results.zip\';tmp=working/f\'{kind}-results.zip.tmp\'\n    if kind in (\'s2\',\'s3\'):\n        clean_checkpoints(working)\n        for path in working.rglob(\'*\'):\n            if path.is_file() and (path.name.endswith(\'.tmp\') or \'.tmp-\' in path.name):path.unlink(missing_ok=True)\n    # Always leave a per-experiment summary, including interruptions before the first metric.\n    experiments=([(\'gpu0-exp30\',\'30-summary.json\',30),(\'gpu1-exp31\',\'31-summary.json\',31),(\'gpu0-exp32\',\'32-summary.json\',32)] if kind==\'s3\' else\n        [(\'gpu0-exp27b\',\'27b-summary.json\',\'27b\'),(\'gpu0-exp29\',\'29-summary.json\',29),(\'gpu1-exp28\',\'28-summary.json\',28)]\n        if kind==\'s2\' else [(\'gpu0-a26\',\'26b-final-summary.json\',\'26b\'),(\'gpu0-exp27\',\'27-summary.json\',27),(\'gpu1-exp28\',\'28-summary.json\',28)])\n    for directory,filename,experiment in experiments:\n        root=working/directory;p=root/filename\n        if not p.exists():\n            progress=root/\'progress.json\';partial=json.loads(progress.read_text()) if progress.exists() else {}\n            partial.update(experiment=experiment,status=\'partial; see process/failure log and latest complete checkpoint\')\n            write(p,partial)\n    files=[p for p in sorted(working.rglob(\'*\')) if p.is_file() and\n           p.relative_to(working).parts[0] not in (\'src\',\'inputs\',\'unpacked-cache\',\'unpacked-resume\') and\n           p.name not in (f\'{kind}-results.zip\',f\'{kind}-results.zip.tmp\',f\'{kind}-archive-receipt.json\')]\n    omitted=[]\n    if kind in (\'s2\',\'s3\'):\n        # Individual Kaggle outputs only: weights/AdamW are never duplicated in a ZIP.\n        manifest={\'output_policy\':\'individual files; no ZIP\',\'files\':[\n            {\'file\':p.relative_to(working).as_posix(),\'bytes\':p.stat().st_size} for p in files\n            if p.name not in (\'session-summary.json\',kind+\'-summary.json\',kind+\'-output-manifest.json\')],\n            \'mutable_summaries_outside_manifest\':[\'session-summary.json\',kind+\'-summary.json\']}\n        manifest[\'output_bytes\']=disk_guard(working,limit=4*1024**3)\n        atomic_json(working/(kind+\'-output-manifest.json\'),manifest)\n        return working\n    # A ZIP temporarily duplicates retained artifacts; reserve before writing, including archive overhead.\n    total=sum(p.stat().st_size for p in working.rglob(\'*\') if p.is_file())\n    if total+sum(p.stat().st_size for p in files)+64*1024**2>19*1024**3:\n        raise RuntimeError(\'ZIP would exceed19GiB working safety cap\')\n    manifest=[{\'file\':p.relative_to(working).as_posix(),\'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in files]\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:\n        for p in files:z.write(p,p.relative_to(working))\n        z.writestr(\'zip-manifest.json\',json.dumps({\'files\':manifest,\'retained_on_kaggle_outside_zip\':[p.relative_to(working).as_posix() for p in omitted]},indent=2))\n    tmp.replace(target);write(working/f\'{kind}-archive-receipt.json\',{\'file\':target.name,\'bytes\':target.stat().st_size,\'sha256\':sha(target)})\n    disk_guard(working);return target\n\ndef tail(path):\n    with Path(path).open(\'rb\') as handle:\n        handle.seek(0,2);handle.seek(max(0,handle.tell()-4096))\n        return handle.read().decode(errors=\'replace\').splitlines()[-3:]\n\ndef group_alive(group):\n    try:os.killpg(group,0);return True\n    except ProcessLookupError:return False\n\ndef stop_workers(processes,watchdog):\n    # One shared grace period, never a per-worker extension; signal groups even if leaders exited.\n    watchdog.terminate()\n    until=min(watchdog.term_at+watchdog.grace,watchdog.hard_deadline)\n    while time.monotonic()<until and any(group_alive(p.pid) for p in processes):\n        for p in processes:p.poll()\n        time.sleep(min(.1,max(0.,until-time.monotonic())))\n    watchdog.kill()\n    until=min(time.monotonic()+2.,watchdog.hard_deadline)\n    while time.monotonic()<until and any(p.poll() is None for p in processes):time.sleep(.05)\n    for p in processes:\n        try:p.wait(timeout=.01)\n        except subprocess.TimeoutExpired:pass  # D-state must not hold up report/output.\n    return [p.pid for p in processes if p.poll() is None]\n\ndef launch(config,worker,smoke=False,heartbeat_seconds=30,watchdog=None,live_sync=None):\n    working=Path(config[\'working_root\']);working.mkdir(parents=True,exist_ok=True)\n    watchdog=watchdog or Watchdog(config).start()\n    config_path=working/\'session-config.json\';atomic_json(config_path,config)\n    handles=[];processes=[];kind=config.get(\'session_kind\',\'s1\');report={};survivors=[]\n    try:\n        for gpu in (0,1):\n            if time.monotonic()>=config[\'deadline_monotonic\']:break\n            env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu) if not smoke else \'\',OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',TOKENIZERS_PARALLELISM=\'false\')\n            for name in (\'KAGGLE_USERNAME\',\'KAGGLE_KEY\',\'KAGGLE_API_TOKEN\'):env.pop(name,None)\n            h=(working/f\'gpu{gpu}-process.log\').open(\'w\');handles.append(h)\n            cmd=[sys.executable,str(worker),\'--gpu\',str(gpu),\'--config\',str(config_path)]\n            p=subprocess.Popen(cmd,env=env,stdout=h,stderr=subprocess.STDOUT,start_new_session=True)\n            processes.append(p);watchdog.register(p)\n        while any(p.poll() is None for p in processes):\n            remaining=config[\'deadline_monotonic\']-time.monotonic()\n            for gpu,p in enumerate(processes):\n                print(\'GPU\',gpu,\'status\',p.poll(),\'deadline in\',round(remaining,1),\'s; last:\',\n                      \' | \'.join(tail(working/f\'gpu{gpu}-process.log\'))[:1800],flush=True)\n            atomic_json(working/\'session-live.json\',{\'elapsed_seconds\':time.monotonic()-config[\'session_start_monotonic\'],\n                \'remaining_seconds\':remaining,\'processes\':[p.poll() for p in processes]})\n            disk_guard(working,limit=(4 if kind in (\'s2\',\'s3\') else 18)*1024**3)\n            if remaining<=0:break\n            time.sleep(min(heartbeat_seconds,max(.01,remaining)))\n    except BaseException as error:\n        report[\'supervisor_error\']={\'type\':type(error).__name__,\'message\':str(error)[:1000]}\n    finally:\n        survivors=stop_workers(processes,watchdog)\n        for h in handles:h.close()\n        report.update(status=\'complete\' if len(processes)==2 and all(p.returncode==0 for p in processes) and not report else \'partial\',\n            exit_codes=[p.returncode for p in processes],unreaped_worker_pids=survivors,\n            process_groups=[p.pid for p in processes],wall_seconds=time.monotonic()-config[\'session_start_monotonic\'],\n            budget_seconds=config[\'budget_seconds\'],hard_deadline_monotonic=watchdog.hard_deadline,\n            termination_grace_seconds=watchdog.grace,term_to_kill_seconds=watchdog.kill_at-watchdog.term_at,\n            GPU_launch=not smoke,processes=len(processes))\n        # Persist a useful session result before any potentially slow report/archive operation.\n        atomic_json(working/\'session-summary.json\',report);atomic_json(working/f\'{kind}-summary.json\',report)\n        try:\n            if kind==\'s2\':\n                from s2_report import collect\n                report[\'comparison\']=collect(working)\n                if any(r[\'status\']!=\'complete\' for r in report[\'comparison\'][\'models\'].values()) or report[\'comparison\'][\'decision\'][\'status\']!=\'measured\':report[\'status\']=\'partial\'\n            elif kind==\'s3\':\n                report[\'experiments\']={}\n                for directory,experiment in [(\'gpu0-exp30\',30),(\'gpu1-exp31\',31),(\'gpu0-exp32\',32)]:\n                    path=working/directory/(str(experiment)+\'-summary.json\')\n                    partial=json.loads(path.read_text()) if path.exists() else {}\n                    report[\'experiments\'][str(experiment)]={\'status\':partial.get(\'status\',\'missing\'),\n                        \'points\':list(partial.get(\'points\',{})), \'missing_points\':partial.get(\'missing_points\'),\n                        \'scored_candidates\':sum(s.get(\'candidates\',0) for s in partial.get(\'splits\',{}).values())}\n                if any(e[\'status\']!=\'complete\' for e in report[\'experiments\'].values()):report[\'status\']=\'partial\'\n            # Unpack/anchor scratch is outside working and never becomes a Kaggle output.\n            if config.get(\'temp_root\') and not survivors and live_sync is None:shutil.rmtree(config[\'temp_root\'],ignore_errors=True)\n            archive(working,kind)\n            report[\'output_bytes\']=disk_guard(working,limit=(4 if kind in (\'s2\',\'s3\') else 18)*1024**3)\n            report[\'output_policy\']=\'individual files; no ZIP\' if kind in (\'s2\',\'s3\') else \'ZIP\'\n        except BaseException as error:\n            report.update(status=\'partial\',report_error={\'type\':type(error).__name__,\'message\':str(error)[:1000]})\n        report[\'wall_seconds\']=time.monotonic()-config[\'session_start_monotonic\']\n        atomic_json(working/\'session-summary.json\',report);atomic_json(working/f\'{kind}-summary.json\',report)\n        if live_sync is not None:\n            try:report[\'live_final_uploaded\']=live_sync.final_sync(\'normal-exit\',timeout=30.)\n            except Exception:report[\'live_final_uploaded\']=False\n            atomic_json(working/\'session-summary.json\',report);atomic_json(working/f\'{kind}-summary.json\',report)\n            # Live staging is outside output; leave it alone if an upload thread is unreaped.\n        # Leave the hard fuse armed if SIGKILL did not reap a worker.\n        if not survivors:watchdog.finish()\n    return report\n', 'session_watchdog.py': '"""Torch-free notebook watchdog, armed before installs, input preparation and workers."""\nimport json,os,signal,subprocess,threading,time,uuid\nfrom pathlib import Path\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp-\'+uuid.uuid4().hex[:8])\n    with tmp.open(\'w\') as handle:\n        json.dump(value,handle,ensure_ascii=False,indent=2,allow_nan=False)\n        handle.write(\'\\n\');handle.flush();os.fsync(handle.fileno())\n    tmp.replace(path)\n\ndef clean_checkpoints(working):\n    """Small, Torch-free cleanup also works if a kill interrupted receipt publication."""\n    for root in Path(working).glob(\'gpu*-exp*\'):\n        keep=set();blobs=set();valid=True\n        for selector in (\'latest.json\',\'final-checkpoint.json\'):\n            path=root/selector\n            if not path.exists():continue\n            try:\n                receipt=json.loads(path.read_text());keep.add(receipt[\'file\'])\n                if receipt.get(\'tensor_blob\'):blobs.add(receipt[\'tensor_blob\'][\'relative_path\'])\n            except (OSError,ValueError,KeyError):valid=False\n        for directory in (\'checkpoints\',\'weights\'):\n            for path in (root/directory).glob(\'*.pt.tmp\'):path.unlink(missing_ok=True)\n        if valid:\n            for path in (root/\'checkpoints\').glob(\'*.pt\'):\n                if path.relative_to(root).as_posix() not in keep:\n                    path.unlink(missing_ok=True);path.with_suffix(\'.pt.json\').unlink(missing_ok=True)\n            for path in (root/\'checkpoints/blobs\').glob(\'*.pt\'):\n                if \'blobs/\'+path.name not in blobs:path.unlink(missing_ok=True)\n\nclass Watchdog:\n    def __init__(self,config):\n        self.config=config;self.working=Path(config[\'working_root\'])\n        self.deadline=config[\'deadline_monotonic\']\n        self.hard_deadline=min(config.get(\'hard_deadline_monotonic\',self.deadline+600),\n                               config[\'session_start_monotonic\']+3*3600+40*60)\n        self.grace=config.get(\'terminate_grace_seconds\',20.)\n        self.groups=set();self.lock=threading.Lock();self.done=threading.Event()\n        self.term_at=None;self.kill_at=None;self.thread=None\n        self.own_group=os.getpgrp()\n        self.signal_errors=[]\n        self.live_sync=None\n\n    def register(self,process):\n        # Popen(start_new_session=True): PID is the process group ID, including exec pools.\n        with self.lock:self.groups.add(process.pid)\n        if self.term_at is not None:self.signal_groups(signal.SIGTERM)\n        if self.kill_at is not None:self.signal_groups(signal.SIGKILL)\n\n    def signal_groups(self,sig,extra=()):\n        with self.lock:groups=self.groups.copy()\n        groups.update(extra)\n        for group in groups:\n            if group==self.own_group:continue\n            try:os.killpg(group,sig)\n            except ProcessLookupError:pass\n            except OSError as error:self.signal_errors.append({\'group\':group,\'signal\':sig,\'error\':str(error)})\n\n    def terminate(self):\n        with self.lock:\n            if self.term_at is None:self.term_at=time.monotonic()\n        self.signal_groups(signal.SIGTERM)\n\n    def kill(self):\n        with self.lock:self.kill_at=time.monotonic()\n        self.signal_groups(signal.SIGKILL)\n\n    def descendants(self):\n        # Also catch setup subprocesses or grandchildren that created a new session.\n        rows=[]\n        if Path(\'/proc\').is_dir():\n            for path in Path(\'/proc\').glob(\'[0-9]*/stat\'):\n                try:\n                    fields=path.read_text().rsplit(\')\',1)[1].split()\n                    rows.append((int(path.parent.name),int(fields[1]),int(fields[2])))\n                except (OSError,ValueError,IndexError):pass\n        try:\n            if not rows:\n                result=subprocess.run([\'ps\',\'-eo\',\'pid=,ppid=,pgid=\'],capture_output=True,text=True,\n                                      timeout=1.,start_new_session=True,check=True)\n                rows=[tuple(map(int,line.split())) for line in result.stdout.splitlines()]\n        except (OSError,ValueError,subprocess.SubprocessError):pass\n        parents={os.getpid()};children=set()\n        while True:\n            found={pid for pid,ppid,_ in rows if ppid in parents}-children\n            if not found:break\n            children.update(found);parents.update(found)\n        return sorted(children),{group for pid,_,group in rows if pid in children}\n\n    def force_exit(self):\n        # Heavy snapshot/FP16 work is isolated in a child; final_sync has a hard wall timeout.\n        report={\'status\':\'partial; notebook watchdog forced exit\',\'watchdog_forced_exit\':True,\n                \'wall_seconds\':time.monotonic()-self.config[\'session_start_monotonic\'],\n                \'budget_seconds\':self.config[\'budget_seconds\'],\'hard_deadline_monotonic\':self.hard_deadline,\n                \'process_groups\':sorted(self.groups),\'output_policy\':\'individual files; no ZIP\',\n                \'signal_errors\':self.signal_errors,\n                \'partial_results\':\'per-experiment summaries and points/*/evaluation/metrics.json\'}\n        try:\n            self.signal_groups(signal.SIGKILL)\n            children,groups=self.descendants()\n            self.signal_groups(signal.SIGKILL,groups)\n            for pid in children:\n                try:os.kill(pid,signal.SIGKILL)\n                except OSError:pass\n            try:clean_checkpoints(self.working)\n            except OSError as error:report[\'cleanup_error\']=str(error)\n            atomic_json(self.working/\'session-summary.json\',report)\n            atomic_json(self.working/(self.config.get(\'session_kind\',\'s2\')+\'-summary.json\'),report)\n            if self.live_sync is not None:\n                try:self.live_sync.final_sync(\'watchdog\',timeout=min(20.,max(.1,self.hard_deadline-time.monotonic()-1.)))\n                except Exception:pass  # A failed upload must never prevent emergency exit.\n            print(\'WATCHDOG: saved session-summary; os._exit(0)\',flush=True)\n        finally:os._exit(0)\n\n    def run(self):\n        while not self.done.is_set():\n            now=time.monotonic()\n            # Reserve 25s inside the existing hard ceiling for the final best-effort upload.\n            if now>=self.hard_deadline-(25. if self.live_sync is not None else 0.):self.force_exit()\n            if now>=self.deadline and self.term_at is None:self.terminate()\n            if self.term_at is not None and now>=self.term_at+self.grace and self.kill_at is None:self.kill()\n            self.done.wait(.1)\n\n    def start(self):\n        if self.thread is not None:return self\n        atomic_json(self.working/\'session-summary.json\',{\'status\':\'partial; watchdog armed\',\n            \'budget_seconds\':self.config[\'budget_seconds\'],\'deadline_monotonic\':self.deadline,\n            \'hard_deadline_monotonic\':self.hard_deadline})\n        self.thread=threading.Thread(target=self.run,name=\'notebook-watchdog\',daemon=True)\n        self.thread.start();return self\n\n    def finish(self):\n        self.done.set()\n        if self.thread is not None:self.thread.join(timeout=1.)\n'}
for name,source in SOURCES.items():(MODULE_DIR/name).write_text(source)
sys.path.insert(0,str(MODULE_DIR));os.environ['TOKENIZERS_PARALLELISM']='false'
from d1_session import Watchdog
from kaggle_io import write
WATCHDOG=Watchdog(CONFIG).start()
import hashlib
write(WORKING/'d1-protocol.json',{'config':{k:v for k,v in CONFIG.items() if not k.endswith('_monotonic')},
      'modules':{n:hashlib.sha256(s.encode()).hexdigest() for n,s in SOURCES.items()},'training':False})
import torch
from l2_live import gate
GATE=gate(torch,require_secrets=CONFIG['require_secrets']);write(WORKING/'startup-gate.json',GATE)
if not GATE['passed']:WATCHDOG.finish();raise RuntimeError(GATE['reason'])
if CONFIG['require_secrets'] and os.environ.get('KAGGLE_USERNAME')!=CONFIG['owner']:
    WATCHDOG.finish();raise RuntimeError('D1 requires friend1 Secrets')


In [ ]:
import importlib.metadata,subprocess,sys
required={'tokenizers':'0.23.2','pyarrow':'23.0.1','sympy':'1.14.0','kaggle':'2.2.4'}
missing=[]
for package,version in required.items():
    try:installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:installed=None
    if installed!=version:missing.append(package+'=='+version)
if missing:
    installer=subprocess.Popen([sys.executable,'-m','pip','install','--quiet',*missing],start_new_session=True)
    if 'WATCHDOG' in globals():WATCHDOG.register(installer)
    try:
        if installer.wait(timeout=180)!=0:raise RuntimeError('Dependency install failed')
    except subprocess.TimeoutExpired:
        import os,signal
        os.killpg(installer.pid,signal.SIGKILL)
        raise
print({p:importlib.metadata.version(p) for p in required})


In [ ]:
from d1_session import setup,launch
from d1_live import D1LiveSync
setup(CONFIG)
LIVE=None
if CONFIG['enable_live_sync']:
    LIVE=D1LiveSync(WORKING,CONFIG['temp_root'],CONFIG['live_dataset'],interval=1800.)
    assert LIVE.sync('startup-probe',timeout=300.,wait_ready=True),'Private D1 live probe failed'
    WATCHDOG.live_sync=LIVE;LIVE.start()
assert time.monotonic()-SESSION_START<CONFIG['setup_cap_seconds'],'D1 setup including live probe exceeded budget'
report=launch(CONFIG,MODULE_DIR/'d1_worker.py',WATCHDOG,LIVE)
print(json.dumps(report,indent=2))
# Partial is a valid saved result. The review gate inspects completeness explicitly.
assert not report['unreaped_worker_pids']
